# ChildSafe: expanded object classifier
Adapted from the original EfficientNet/MobileNet training pipeline. No results are claimed before running.

1. Upload `childsafe_supplement.zip` as a private Kaggle Dataset, and add ChildSUn as another input.
2. The starter contains 169 reviewed crops. Recheck the sheets and edit `manifest.csv` as needed.
   New downloads start with `reviewed=false`; set true only after checking category and privacy.
   You can review locally before uploading, or export/edit/re-upload the manifest on Kaggle.
3. Enable GPU and Internet (ImageNet weights/dependencies). Set the two explicit paths below.
4. Run all. Grouped train/validation/calibration/test partitions are checked before training.
   Canonical ChildSUn test stays fixed. New source photos never cross partitions.

`unsafe` means a project policy for object-to-mouth contact, not a claim that a pen is always dangerous.
Fruit/spoon being classified safe does not establish safe eating or absence of choking.
Do not use children, sharp objects, or real insertion into anyone's mouth for scenario collection.


In [ ]:
%pip install -q timm imagehash scikit-learn pandas scipy matplotlib pyyaml tqdm
import os, sys, subprocess
from pathlib import Path
REPO=Path('/kaggle/working/childsafe')
REPO.mkdir(exist_ok=True); os.chdir(REPO)
sys.path.insert(0,str(REPO/'src')); os.environ['PYTHONPATH']=str(REPO/'src')
for folder in ['configs','scripts','src/childsafe/data','src/childsafe/classifier','data/splits','outputs']:
    (REPO/folder).mkdir(parents=True,exist_ok=True)
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'Enable a Kaggle GPU before the full training run'


In [ ]:
%%writefile configs/default.yaml
# Central config. Every threshold here is a starting value, NOT a tuned result.
# Tune D3 thresholds by grid search on the dev split of the scenario videos (scripts/grid_search.py),
# and report on the held-out test split only.

paths:
  raw_data: data/raw/childsun          # extracted ChildSUn root
  splits: data/splits/splits.csv
  hashes: data/splits/hashes.npz
  outputs: outputs
  docs: docs                           # generated reports (dataset, split, results)
  face_model: models/face_landmarker.task
  hand_model: models/hand_landmarker.task

# Category name (folder name, or filename prefix for the flat ChildSUn layout; normalized:
# lowercase, no spaces/_/-) -> binary label. Grouping checked against the Kaggle dataset
# description (2026-10-01): unsafe = knives, sockets, scissors, electric heaters, forks;
# safe = toys, baby items, fruits (spoon and ball are not named there; kept safe).
label_map:
  knife: unsafe
  socket: unsafe
  sockets: unsafe
  scissors: unsafe
  scissor: unsafe
  seissor: unsafe     # typo used in ChildSUn filenames
  electricheater: unsafe
  heater: unsafe
  fork: unsafe
  toy: safe
  toys: safe
  ball: safe
  babyitems: safe
  babyitem: safe
  fruit: safe
  fruits: safe
  spoon: safe
  # objects outside the 10 ChildSUn classes (Open Images pen, phone, mug, hand...). "safe" here only means
  # "not an unsafe class" for P(unsafe); it is not a safety claim (a pen cap is a choking hazard).
  other: safe
  # if the dataset is already organised as safe/ and unsafe/ folders:
  safe: safe
  unsafe: unsafe

# Merge spelling variants into one class name.
# socket/sockets are separate filename sets in ChildSUn (86 + 124 source photos); merged here
# because the paper lists a single "socket" category.
category_aliases:
  sockets: socket
  seissor: scissors
  scissor: scissors
  fruits: fruit
  toys: toy
  babyitem: babyitems
  electricheater: heater

dedup:
  hash_size: 8              # 64-bit pHash / dHash
  phash_threshold: 8        # chosen by inspecting montages (docs/split_report.md): true dups have low dHash
  dhash_threshold: 5
  dihedral: true            # also compare flipped / 90-degree-rotated variants (catches flip/rotate augmentation)

split:
  test_frac: 0.15
  val_frac: 0.15
  seed: 42

train:
  img_size: 224
  batch_size: 32
  epochs: 25
  lr: 3.0e-4
  lr_overrides:             # VGG19 diverged at 3e-4 (best epoch 0 in the first runs)
    vgg19_bn: 5.0e-5
  weight_decay: 1.0e-4
  patience: 6
  num_workers: 4
  amp: true
  robust_aug: false         # A5: random erasing, motion blur, gamma, ISO noise
  models: [efficientnet_b0, mobilenetv3_large_100, vgg19_bn]

calibration:
  n_bins: 15
  reject_threshold: 0.70    # calibrated max prob below this -> "uncertain"

landmarks:
  normalizer: face_width    # face_width | inter_eye | hand_size (B1 compares all three)
  min_face_conf: 0.5
  min_hand_conf: 0.5
  num_hands: 2

smoothing:
  method: one_euro          # none | ema | one_euro
  ema_alpha: 0.4
  one_euro_min_cutoff: 1.0
  one_euro_beta: 0.05
  lost_hold_s: 0.8          # B2: hold last state this long when the mouth landmark disappears
  lost_max_s: 3.0           # legacy metadata only; runtime never extends held evidence beyond lost_hold_s

temporal:                   # D1
  window_s: 0.3333333333333333
  required_evidence_s: 0.1
  max_gap_s: 0.25
  d_enter: 0.35             # normalized distance to enter "near" (hysteresis: enter < exit)
  d_exit: 0.50
  n_of_m: [3, 10]           # legacy metadata only; runtime uses the duration window above
  merge_s: 2.0              # merge events closer than this
  cooldown_s: 5.0           # limit repeat notifications; never lower displayed danger

cascade:                    # D2
  enabled: true
  d_far: 1.2                # run Module A only when normalized distance < d_far
  classify_every_n: 3       # and at most every N frames (cached in between)
  classify_interval_s: 0.1
  crop_scale: 2.2           # hand crop = hand bbox * scale

rules:                      # D3
  p_unsafe_danger: 0.60     # calibrated P(unsafe) needed for danger
  dwell_danger_s: 0.5       # near for at least this long before danger
  dwell_safe_caution_s: 3.0 # safe object near mouth this long -> caution (capped, never danger)
  ttc_caution_s: 0.8        # unsafe object approaching with time-to-contact below this -> caution

runtime:
  max_people: 4
  stale_frame_s: 0.25
  camera_index: 0
  width: 1280
  height: 720
  log_dir: outputs/events   # events only; video is never written

detector:
  weights: models/yolo11n-seg.pt
  min_conf: 0.5
  complete_object_coverage: false # COCO excludes coins/pens; never infer their location

quality:
  calibrated: false
  max_hand_object_px: 80
  min_face_px: 40
  min_hand_px: 10
  min_mouth_px: 4
  max_yaw_deg: 60
  max_pitch_deg: 45
  max_landmark_jump: 0.35
  min_sharpness: 20
  min_brightness: 40
  max_brightness: 220
  min_mask_px: 16
  min_mask_conf: 0.6


In [ ]:
%%writefile scripts/prepare_expanded_data.py
"""Merge reviewed supplemental crops, preserve canonical test, audit duplicates before training."""
from __future__ import annotations
import argparse
import hashlib
import json
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from childsafe.data.dedup import compute_hashes, find_duplicate_pairs

def merge_manifest(base, raw_root, supplement, seed=42, audit=True):
    base = base.copy()
    base['path'] = [str((Path(raw_root)/p).resolve()) for p in base.path]
    base['group'] = 'childsun:' + base.group.astype(str)
    base['domain'] = 'childsun'
    base['reviewed'] = True
    # Reserve half the existing validation groups for temperature fitting.
    rng = np.random.default_rng(seed)
    for category, part in base[base.split_group == 'val'].groupby('category'):
        groups = sorted(part.group.unique()); rng.shuffle(groups)
        if len(groups) < 2:
            raise ValueError(f'Need two validation groups for {category}')
        base.loc[base.group.isin(groups[:max(1,len(groups)//2)]), 'split_group'] = 'cal'
    source = Path(supplement)
    extra = pd.read_csv(source).fillna('')
    approved = extra.reviewed.astype(str).str.lower().isin(['true','1','yes'])
    print(f'Review: {approved.sum()} approved, {(~approved).sum()} pending/excluded', flush=True)
    extra = extra[approved].copy()
    if extra.empty:
        raise ValueError('Review manifest.csv first: set reviewed=true only for verified object labels and privacy')
    if not set(extra.label).issubset({'safe','unsafe'}):
        raise ValueError('Unknown samples need a separately designed rejection evaluation; never map them to safe')
    extra['path'] = [str((source.parent/p).resolve()) for p in extra.path]
    extra['split_group'] = ''
    df = pd.concat([base, extra], ignore_index=True)
    parent = list(range(len(df)))
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]; i = parent[i]
        return i
    def union(i,j):
        parent[find(j)] = find(i)
    for _, rows in df.groupby('group'):
        ids=list(rows.index)
        for i in ids[1:]: union(ids[0],i)
    # Decode every image; fail instead of silently hashing corrupt files as duplicates.
    pixels = {}
    for i, path in enumerate(df.path):
        with Image.open(path) as image:
            image = image.convert('RGB')
            digest=hashlib.sha256(str(image.size).encode()+image.tobytes()).hexdigest()
        if digest in pixels:
            j=pixels[digest]
            if df.at[i,'category'] != df.at[j,'category']:
                raise ValueError('Identical image has conflicting category labels; review annotations')
            union(i,j)
        else: pixels[digest]=i
    duplicate_count = 0
    if audit:
        ph,dh=compute_hashes(df.path.tolist())
        pairs=find_duplicate_pairs(ph,dh,8,5)
        duplicate_count=len(pairs)
        for i,j,*_ in pairs:
            i,j=int(i),int(j)
            if df.at[i,'category'] != df.at[j,'category']:
                raise ValueError(f'Near-duplicate crops have conflicting categories: {df.at[i,"path"]}, {df.at[j,"path"]}')
            union(i,j)
    df['group'] = [f'merged:{find(i)}' for i in range(len(df))]
    for _, rows in df.groupby('group'):
        roles = set(rows.split_group)-{''}
        if len(roles)>1:
            raise ValueError(f'Duplicate links canonical partitions: {rows.path.tolist()[:4]}; review before training')
        if roles: df.loc[rows.index,'split_group']=next(iter(roles))
    for category in sorted(df.loc[df.split_group=='','category'].unique()):
        part=df[(df.split_group=='') & (df.category==category)]
        groups=sorted(part.group.unique()); rng.shuffle(groups)
        if category not in set(base.category) and len(groups)<10:
            raise ValueError(f'{category}: need >=10 independent source groups; got {len(groups)}')
        # With sparse supplements of existing classes, keep all in train (base holdouts remain).
        n=len(groups)
        n_hold=max(1,int(n*.1)) if n>=10 else 0
        for role, subset in [('test',groups[:n_hold]),('cal',groups[n_hold:2*n_hold]),
                             ('val',groups[2*n_hold:3*n_hold]),('train',groups[3*n_hold:])]:
            df.loc[df.group.isin(subset), 'split_group']=role
    if df.groupby('group').split_group.nunique().max()!=1:
        raise ValueError('Group leakage')
    return df, {'images':len(df), 'near_duplicate_pairs':duplicate_count, 'seed':seed,
                'counts':df.groupby(['category','domain','split_group']).size().to_dict()}

def main():
    ap=argparse.ArgumentParser()
    ap.add_argument('--base', default='data/splits/splits.csv')
    ap.add_argument('--raw-root', required=True)
    ap.add_argument('--supplement', required=True)
    ap.add_argument('--out', default='data/splits/expanded.csv')
    args=ap.parse_args()
    df,report=merge_manifest(pd.read_csv(args.base),args.raw_root,args.supplement)
    out=Path(args.out); out.parent.mkdir(parents=True,exist_ok=True); df.to_csv(out,index=False)
    report['counts']={' / '.join(k):int(v) for k,v in report['counts'].items()}
    out.with_suffix('.report.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
    print(df.groupby(['category','split_group']).size().unstack(fill_value=0).to_string())

if __name__=='__main__': main()


In [ ]:
%%writefile data/splits/splits.csv
path,category,label,source_split,source_id,group,split_group,split_random
baby_items_10_0_1720.jpg,babyitems,safe,,babyitems_10,0,train,train
baby_items_10_0_316.jpg,babyitems,safe,,babyitems_10,0,train,val
baby_items_10_0_6049.jpg,babyitems,safe,,babyitems_10,0,train,train
baby_items_10_0_8479.jpg,babyitems,safe,,babyitems_10,0,train,train
baby_items_10_0_9975.jpg,babyitems,safe,,babyitems_10,0,train,val
baby_items_11_0_3451.jpg,babyitems,safe,,babyitems_11,1,train,train
baby_items_11_0_5551.jpg,babyitems,safe,,babyitems_11,1,train,train
baby_items_11_0_6229.jpg,babyitems,safe,,babyitems_11,1,train,train
baby_items_11_0_7464.jpg,babyitems,safe,,babyitems_11,1,train,train
baby_items_11_0_8242.jpg,babyitems,safe,,babyitems_11,1,train,val
baby_items_12_0_1282.jpg,babyitems,safe,,babyitems_12,2,train,train
baby_items_12_0_1456.jpg,babyitems,safe,,babyitems_12,2,train,train
baby_items_12_0_3461.jpg,babyitems,safe,,babyitems_12,2,train,train
baby_items_12_0_480.jpg,babyitems,safe,,babyitems_12,2,train,test
baby_items_12_0_8628.jpg,babyitems,safe,,babyitems_12,2,train,train
baby_items_13_0_4602.jpg,babyitems,safe,,babyitems_13,3,train,train
baby_items_13_0_4964.jpg,babyitems,safe,,babyitems_13,3,train,train
baby_items_13_0_6249.jpg,babyitems,safe,,babyitems_13,3,train,train
baby_items_13_0_7031.jpg,babyitems,safe,,babyitems_13,3,train,train
baby_items_13_0_7327.jpg,babyitems,safe,,babyitems_13,3,train,train
baby_items_14_0_1991.jpg,babyitems,safe,,babyitems_14,4,train,test
baby_items_14_0_3126.jpg,babyitems,safe,,babyitems_14,4,train,test
baby_items_14_0_7356.jpg,babyitems,safe,,babyitems_14,4,train,train
baby_items_14_0_8457.jpg,babyitems,safe,,babyitems_14,4,train,train
baby_items_14_0_8630.jpg,babyitems,safe,,babyitems_14,4,train,train
baby_items_17_0_2477.jpg,babyitems,safe,,babyitems_17,5,train,train
baby_items_17_0_3502.jpg,babyitems,safe,,babyitems_17,5,train,train
baby_items_17_0_3657.jpg,babyitems,safe,,babyitems_17,5,train,train
baby_items_17_0_5567.jpg,babyitems,safe,,babyitems_17,5,train,train
baby_items_17_0_6068.jpg,babyitems,safe,,babyitems_17,5,train,train
baby_items_18_0_3651.jpg,babyitems,safe,,babyitems_18,6,train,train
baby_items_18_0_5839.jpg,babyitems,safe,,babyitems_18,6,train,train
baby_items_18_0_6794.jpg,babyitems,safe,,babyitems_18,6,train,train
baby_items_18_0_8228.jpg,babyitems,safe,,babyitems_18,6,train,val
baby_items_18_0_9619.jpg,babyitems,safe,,babyitems_18,6,train,test
baby_items_19_0_1301.jpg,babyitems,safe,,babyitems_19,7,train,train
baby_items_19_0_5228.jpg,babyitems,safe,,babyitems_19,7,train,train
baby_items_19_0_5400.jpg,babyitems,safe,,babyitems_19,7,train,val
baby_items_19_0_7725.jpg,babyitems,safe,,babyitems_19,7,train,val
baby_items_19_0_9355.jpg,babyitems,safe,,babyitems_19,7,train,train
baby_items_1_0_274.jpg,babyitems,safe,,babyitems_1,8,train,train
baby_items_1_0_3408.jpg,babyitems,safe,,babyitems_1,8,train,train
baby_items_1_0_6097.jpg,babyitems,safe,,babyitems_1,8,train,val
baby_items_1_0_8430.jpg,babyitems,safe,,babyitems_1,8,train,train
baby_items_1_0_8634.jpg,babyitems,safe,,babyitems_1,8,train,val
baby_items_20_0_1209.jpg,babyitems,safe,,babyitems_20,9,train,train
baby_items_20_0_2053.jpg,babyitems,safe,,babyitems_20,9,train,train
baby_items_20_0_5692.jpg,babyitems,safe,,babyitems_20,9,train,train
baby_items_20_0_5787.jpg,babyitems,safe,,babyitems_20,9,train,test
baby_items_20_0_8158.jpg,babyitems,safe,,babyitems_20,9,train,train
baby_items_22_0_2577.jpg,babyitems,safe,,babyitems_22,10,train,test
baby_items_22_0_2607.jpg,babyitems,safe,,babyitems_22,10,train,val
baby_items_22_0_8612.jpg,babyitems,safe,,babyitems_22,10,train,test
baby_items_22_0_9580.jpg,babyitems,safe,,babyitems_22,10,train,train
baby_items_22_0_9594.jpg,babyitems,safe,,babyitems_22,10,train,test
baby_items_24_0_1185.jpg,babyitems,safe,,babyitems_24,11,train,train
baby_items_24_0_2262.jpg,babyitems,safe,,babyitems_24,11,train,train
baby_items_24_0_4373.jpg,babyitems,safe,,babyitems_24,11,train,train
baby_items_24_0_7259.jpg,babyitems,safe,,babyitems_24,11,train,test
baby_items_24_0_7503.jpg,babyitems,safe,,babyitems_24,11,train,train
baby_items_25_0_21.jpg,babyitems,safe,,babyitems_25,12,train,train
baby_items_25_0_4426.jpg,babyitems,safe,,babyitems_25,12,train,train
baby_items_25_0_4471.jpg,babyitems,safe,,babyitems_25,12,train,train
baby_items_25_0_6300.jpg,babyitems,safe,,babyitems_25,12,train,train
baby_items_25_0_677.jpg,babyitems,safe,,babyitems_25,12,train,val
baby_items_26_0_2939.jpg,babyitems,safe,,babyitems_26,13,train,train
baby_items_26_0_3309.jpg,babyitems,safe,,babyitems_26,13,train,train
baby_items_26_0_7660.jpg,babyitems,safe,,babyitems_26,13,train,val
baby_items_26_0_9789.jpg,babyitems,safe,,babyitems_26,13,train,train
baby_items_26_0_9830.jpg,babyitems,safe,,babyitems_26,13,train,val
baby_items_27_0_5271.jpg,babyitems,safe,,babyitems_27,14,train,train
baby_items_27_0_5870.jpg,babyitems,safe,,babyitems_27,14,train,test
baby_items_27_0_71.jpg,babyitems,safe,,babyitems_27,14,train,train
baby_items_27_0_7879.jpg,babyitems,safe,,babyitems_27,14,train,train
baby_items_27_0_9895.jpg,babyitems,safe,,babyitems_27,14,train,train
baby_items_29_0_264.jpg,babyitems,safe,,babyitems_29,15,train,train
baby_items_29_0_2799.jpg,babyitems,safe,,babyitems_29,15,train,val
baby_items_29_0_3692.jpg,babyitems,safe,,babyitems_29,15,train,train
baby_items_29_0_388.jpg,babyitems,safe,,babyitems_29,15,train,train
baby_items_29_0_7305.jpg,babyitems,safe,,babyitems_29,15,train,train
baby_items_2_0_2361.jpg,babyitems,safe,,babyitems_2,16,test,train
baby_items_2_0_6787.jpg,babyitems,safe,,babyitems_2,16,test,train
baby_items_2_0_8416.jpg,babyitems,safe,,babyitems_2,16,test,train
baby_items_2_0_8515.jpg,babyitems,safe,,babyitems_2,16,test,train
baby_items_2_0_9156.jpg,babyitems,safe,,babyitems_2,16,test,train
baby_items_30_0_129.jpg,babyitems,safe,,babyitems_30,17,train,train
baby_items_30_0_2623.jpg,babyitems,safe,,babyitems_30,17,train,val
baby_items_30_0_3682.jpg,babyitems,safe,,babyitems_30,17,train,test
baby_items_30_0_6364.jpg,babyitems,safe,,babyitems_30,17,train,test
baby_items_30_0_9359.jpg,babyitems,safe,,babyitems_30,17,train,train
baby_items_31_0_1300.jpg,babyitems,safe,,babyitems_31,18,val,train
baby_items_31_0_3223.jpg,babyitems,safe,,babyitems_31,18,val,test
baby_items_31_0_5241.jpg,babyitems,safe,,babyitems_31,18,val,train
baby_items_31_0_8505.jpg,babyitems,safe,,babyitems_31,18,val,val
baby_items_31_0_9419.jpg,babyitems,safe,,babyitems_31,18,val,train
baby_items_32_0_2885.jpg,babyitems,safe,,babyitems_32,19,val,train
baby_items_32_0_3221.jpg,babyitems,safe,,babyitems_32,19,val,train
baby_items_32_0_3410.jpg,babyitems,safe,,babyitems_32,19,val,train
baby_items_32_0_3631.jpg,babyitems,safe,,babyitems_32,19,val,train
baby_items_32_0_4011.jpg,babyitems,safe,,babyitems_32,19,val,test
baby_items_33_0_1913.jpg,babyitems,safe,,babyitems_33,20,train,val
baby_items_33_0_2402.jpg,babyitems,safe,,babyitems_33,20,train,train
baby_items_33_0_3560.jpg,babyitems,safe,,babyitems_33,20,train,test
baby_items_33_0_6184.jpg,babyitems,safe,,babyitems_33,20,train,train
baby_items_33_0_7734.jpg,babyitems,safe,,babyitems_33,20,train,train
baby_items_34_0_3468.jpg,babyitems,safe,,babyitems_34,21,train,val
baby_items_34_0_3591.jpg,babyitems,safe,,babyitems_34,21,train,test
baby_items_34_0_5650.jpg,babyitems,safe,,babyitems_34,21,train,train
baby_items_34_0_6662.jpg,babyitems,safe,,babyitems_34,21,train,train
baby_items_34_0_9576.jpg,babyitems,safe,,babyitems_34,21,train,val
baby_items_35_0_4505.jpg,babyitems,safe,,babyitems_35,22,train,train
baby_items_35_0_545.jpg,babyitems,safe,,babyitems_35,22,train,val
baby_items_35_0_6555.jpg,babyitems,safe,,babyitems_35,22,train,train
baby_items_35_0_6976.jpg,babyitems,safe,,babyitems_35,22,train,train
baby_items_35_0_7003.jpg,babyitems,safe,,babyitems_35,22,train,train
baby_items_36_0_2852.jpg,babyitems,safe,,babyitems_36,23,train,val
baby_items_36_0_4725.jpg,babyitems,safe,,babyitems_36,23,train,train
baby_items_36_0_719.jpg,babyitems,safe,,babyitems_36,23,train,val
baby_items_36_0_7475.jpg,babyitems,safe,,babyitems_36,23,train,train
baby_items_36_0_8446.jpg,babyitems,safe,,babyitems_36,23,train,train
baby_items_37_0_1098.jpg,babyitems,safe,,babyitems_37,24,train,train
baby_items_37_0_2958.jpg,babyitems,safe,,babyitems_37,24,train,test
baby_items_37_0_4103.jpg,babyitems,safe,,babyitems_37,24,train,val
baby_items_37_0_4828.jpg,babyitems,safe,,babyitems_37,24,train,train
baby_items_37_0_7239.jpg,babyitems,safe,,babyitems_37,24,train,train
baby_items_38_0_1318.jpg,babyitems,safe,,babyitems_38,25,train,val
baby_items_38_0_1883.jpg,babyitems,safe,,babyitems_38,25,train,train
baby_items_38_0_6270.jpg,babyitems,safe,,babyitems_38,25,train,train
baby_items_38_0_8496.jpg,babyitems,safe,,babyitems_38,25,train,train
baby_items_38_0_9596.jpg,babyitems,safe,,babyitems_38,25,train,train
baby_items_39_0_305.jpg,babyitems,safe,,babyitems_39,26,val,test
baby_items_39_0_4113.jpg,babyitems,safe,,babyitems_39,26,val,train
baby_items_39_0_715.jpg,babyitems,safe,,babyitems_39,26,val,train
baby_items_39_0_8133.jpg,babyitems,safe,,babyitems_39,26,val,train
baby_items_39_0_8158.jpg,babyitems,safe,,babyitems_39,26,val,test
baby_items_3_0_4701.jpg,babyitems,safe,,babyitems_3,27,train,train
baby_items_3_0_5414.jpg,babyitems,safe,,babyitems_3,27,train,train
baby_items_3_0_6547.jpg,babyitems,safe,,babyitems_3,27,train,train
baby_items_3_0_7501.jpg,babyitems,safe,,babyitems_3,27,train,train
baby_items_3_0_9920.jpg,babyitems,safe,,babyitems_3,27,train,train
baby_items_41_0_3801.jpg,babyitems,safe,,babyitems_41,28,train,train
baby_items_41_0_4498.jpg,babyitems,safe,,babyitems_41,28,train,train
baby_items_41_0_5607.jpg,babyitems,safe,,babyitems_41,28,train,train
baby_items_41_0_7367.jpg,babyitems,safe,,babyitems_41,28,train,train
baby_items_41_0_9058.jpg,babyitems,safe,,babyitems_41,28,train,train
baby_items_42_0_1133.jpg,babyitems,safe,,babyitems_42,29,train,train
baby_items_42_0_434.jpg,babyitems,safe,,babyitems_42,29,train,train
baby_items_42_0_5493.jpg,babyitems,safe,,babyitems_42,29,train,train
baby_items_42_0_6521.jpg,babyitems,safe,,babyitems_42,29,train,train
baby_items_42_0_7524.jpg,babyitems,safe,,babyitems_42,29,train,test
baby_items_43_0_2473.jpg,babyitems,safe,,babyitems_43,30,train,train
baby_items_43_0_5421.jpg,babyitems,safe,,babyitems_43,30,train,test
baby_items_43_0_7334.jpg,babyitems,safe,,babyitems_43,30,train,train
baby_items_43_0_7388.jpg,babyitems,safe,,babyitems_43,30,train,train
baby_items_43_0_9178.jpg,babyitems,safe,,babyitems_43,30,train,train
baby_items_44_0_2777.jpg,babyitems,safe,,babyitems_44,31,train,train
baby_items_44_0_4000.jpg,babyitems,safe,,babyitems_44,31,train,train
baby_items_44_0_8143.jpg,babyitems,safe,,babyitems_44,31,train,val
baby_items_44_0_896.jpg,babyitems,safe,,babyitems_44,31,train,val
baby_items_44_0_9814.jpg,babyitems,safe,,babyitems_44,31,train,val
baby_items_45_0_1482.jpg,babyitems,safe,,babyitems_45,32,val,test
baby_items_45_0_2098.jpg,babyitems,safe,,babyitems_45,32,val,val
baby_items_45_0_445.jpg,babyitems,safe,,babyitems_45,32,val,train
baby_items_45_0_6776.jpg,babyitems,safe,,babyitems_45,32,val,train
baby_items_45_0_8949.jpg,babyitems,safe,,babyitems_45,32,val,train
baby_items_46_0_4267.jpg,babyitems,safe,,babyitems_46,33,train,train
baby_items_46_0_8264.jpg,babyitems,safe,,babyitems_46,33,train,test
baby_items_46_0_834.jpg,babyitems,safe,,babyitems_46,33,train,train
baby_items_46_0_8398.jpg,babyitems,safe,,babyitems_46,33,train,train
baby_items_46_0_9910.jpg,babyitems,safe,,babyitems_46,33,train,test
baby_items_47_0_386.jpg,babyitems,safe,,babyitems_47,34,val,train
baby_items_47_0_5322.jpg,babyitems,safe,,babyitems_47,34,val,train
baby_items_47_0_6145.jpg,babyitems,safe,,babyitems_47,34,val,val
baby_items_47_0_7338.jpg,babyitems,safe,,babyitems_47,34,val,train
baby_items_47_0_8262.jpg,babyitems,safe,,babyitems_47,34,val,test
baby_items_4_0_315.jpg,babyitems,safe,,babyitems_4,35,train,train
baby_items_4_0_4328.jpg,babyitems,safe,,babyitems_4,35,train,val
baby_items_4_0_7574.jpg,babyitems,safe,,babyitems_4,35,train,train
baby_items_4_0_9832.jpg,babyitems,safe,,babyitems_4,35,train,train
baby_items_4_0_9927.jpg,babyitems,safe,,babyitems_4,35,train,train
baby_items_51_0_2258.jpg,babyitems,safe,,babyitems_51,36,train,val
baby_items_51_0_5372.jpg,babyitems,safe,,babyitems_51,36,train,train
baby_items_51_0_7122.jpg,babyitems,safe,,babyitems_51,36,train,train
baby_items_51_0_8241.jpg,babyitems,safe,,babyitems_51,36,train,train
baby_items_51_0_8694.jpg,babyitems,safe,,babyitems_51,36,train,train
baby_items_52_0_177.jpg,babyitems,safe,,babyitems_52,37,test,val
baby_items_52_0_5239.jpg,babyitems,safe,,babyitems_52,37,test,train
baby_items_52_0_5935.jpg,babyitems,safe,,babyitems_52,37,test,test
baby_items_52_0_7505.jpg,babyitems,safe,,babyitems_52,37,test,test
baby_items_52_0_9235.jpg,babyitems,safe,,babyitems_52,37,test,test
baby_items_53_0_18.jpg,babyitems,safe,,babyitems_53,38,train,train
baby_items_53_0_4382.jpg,babyitems,safe,,babyitems_53,38,train,test
baby_items_53_0_8020.jpg,babyitems,safe,,babyitems_53,38,train,train
baby_items_53_0_8148.jpg,babyitems,safe,,babyitems_53,38,train,train
baby_items_53_0_8518.jpg,babyitems,safe,,babyitems_53,38,train,train
baby_items_54_0_1069.jpg,babyitems,safe,,babyitems_54,39,test,train
baby_items_54_0_2679.jpg,babyitems,safe,,babyitems_54,39,test,train
baby_items_54_0_3697.jpg,babyitems,safe,,babyitems_54,39,test,train
baby_items_54_0_3812.jpg,babyitems,safe,,babyitems_54,39,test,train
baby_items_54_0_818.jpg,babyitems,safe,,babyitems_54,39,test,train
baby_items_55_0_3993.jpg,babyitems,safe,,babyitems_55,40,val,train
baby_items_55_0_4835.jpg,babyitems,safe,,babyitems_55,40,val,train
baby_items_55_0_5651.jpg,babyitems,safe,,babyitems_55,40,val,train
baby_items_55_0_6788.jpg,babyitems,safe,,babyitems_55,40,val,train
baby_items_55_0_9094.jpg,babyitems,safe,,babyitems_55,40,val,train
baby_items_57_0_2818.jpg,babyitems,safe,,babyitems_57,41,train,train
baby_items_57_0_5366.jpg,babyitems,safe,,babyitems_57,41,train,val
baby_items_57_0_6220.jpg,babyitems,safe,,babyitems_57,41,train,train
baby_items_57_0_8531.jpg,babyitems,safe,,babyitems_57,41,train,train
baby_items_57_0_9489.jpg,babyitems,safe,,babyitems_57,41,train,train
baby_items_58_0_4143.jpg,babyitems,safe,,babyitems_58,42,train,train
baby_items_58_0_5386.jpg,babyitems,safe,,babyitems_58,42,train,train
baby_items_58_0_5595.jpg,babyitems,safe,,babyitems_58,42,train,train
baby_items_58_0_7606.jpg,babyitems,safe,,babyitems_58,42,train,train
baby_items_58_0_8570.jpg,babyitems,safe,,babyitems_58,42,train,test
baby_items_59_0_481.jpg,babyitems,safe,,babyitems_59,43,train,train
baby_items_59_0_7029.jpg,babyitems,safe,,babyitems_59,43,train,train
baby_items_59_0_7204.jpg,babyitems,safe,,babyitems_59,43,train,train
baby_items_59_0_8814.jpg,babyitems,safe,,babyitems_59,43,train,train
baby_items_59_0_894.jpg,babyitems,safe,,babyitems_59,43,train,train
baby_items_5_0_3221.jpg,babyitems,safe,,babyitems_5,44,test,train
baby_items_5_0_3365.jpg,babyitems,safe,,babyitems_5,44,test,train
baby_items_5_0_3914.jpg,babyitems,safe,,babyitems_5,44,test,train
baby_items_5_0_6404.jpg,babyitems,safe,,babyitems_5,44,test,train
baby_items_5_0_7998.jpg,babyitems,safe,,babyitems_5,44,test,train
baby_items_60_0_2337.jpg,babyitems,safe,,babyitems_60,45,train,val
baby_items_60_0_250.jpg,babyitems,safe,,babyitems_60,45,train,train
baby_items_60_0_4811.jpg,babyitems,safe,,babyitems_60,45,train,train
baby_items_60_0_6866.jpg,babyitems,safe,,babyitems_60,45,train,train
baby_items_60_0_7108.jpg,babyitems,safe,,babyitems_60,45,train,train
baby_items_62_0_186.jpg,babyitems,safe,,babyitems_62,46,train,train
baby_items_62_0_2946.jpg,babyitems,safe,,babyitems_62,46,train,train
baby_items_62_0_3059.jpg,babyitems,safe,,babyitems_62,46,train,train
baby_items_62_0_6109.jpg,babyitems,safe,,babyitems_62,46,train,val
baby_items_62_0_7166.jpg,babyitems,safe,,babyitems_62,46,train,train
baby_items_63_0_2434.jpg,babyitems,safe,,babyitems_63,47,train,test
baby_items_63_0_2774.jpg,babyitems,safe,,babyitems_63,47,train,train
baby_items_63_0_5506.jpg,babyitems,safe,,babyitems_63,47,train,val
baby_items_63_0_754.jpg,babyitems,safe,,babyitems_63,47,train,train
baby_items_63_0_9672.jpg,babyitems,safe,,babyitems_63,47,train,val
baby_items_64_0_1786.jpg,babyitems,safe,,babyitems_64,48,train,val
baby_items_64_0_2701.jpg,babyitems,safe,,babyitems_64,48,train,test
baby_items_64_0_2969.jpg,babyitems,safe,,babyitems_64,48,train,train
baby_items_64_0_3217.jpg,babyitems,safe,,babyitems_64,48,train,test
baby_items_64_0_4957.jpg,babyitems,safe,,babyitems_64,48,train,train
baby_items_65_0_1530.jpg,babyitems,safe,,babyitems_65,49,test,val
baby_items_65_0_2633.jpg,babyitems,safe,,babyitems_65,49,test,val
baby_items_65_0_3459.jpg,babyitems,safe,,babyitems_65,49,test,train
baby_items_65_0_5101.jpg,babyitems,safe,,babyitems_65,49,test,val
baby_items_65_0_8601.jpg,babyitems,safe,,babyitems_65,49,test,val
baby_items_66_0_4534.jpg,babyitems,safe,,babyitems_66,50,test,train
baby_items_66_0_532.jpg,babyitems,safe,,babyitems_66,50,test,train
baby_items_66_0_5357.jpg,babyitems,safe,,babyitems_66,50,test,test
baby_items_66_0_8274.jpg,babyitems,safe,,babyitems_66,50,test,train
baby_items_66_0_8303.jpg,babyitems,safe,,babyitems_66,50,test,train
baby_items_67_0_3063.jpg,babyitems,safe,,babyitems_67,51,train,train
baby_items_67_0_5862.jpg,babyitems,safe,,babyitems_67,51,train,train
baby_items_67_0_6007.jpg,babyitems,safe,,babyitems_67,51,train,test
baby_items_67_0_7012.jpg,babyitems,safe,,babyitems_67,51,train,train
baby_items_67_0_7636.jpg,babyitems,safe,,babyitems_67,51,train,train
baby_items_68_0_1266.jpg,babyitems,safe,,babyitems_68,52,val,train
baby_items_68_0_5540.jpg,babyitems,safe,,babyitems_68,52,val,train
baby_items_68_0_6558.jpg,babyitems,safe,,babyitems_68,52,val,train
baby_items_68_0_7083.jpg,babyitems,safe,,babyitems_68,52,val,test
baby_items_68_0_8197.jpg,babyitems,safe,,babyitems_68,52,val,test
baby_items_69_0_225.jpg,babyitems,safe,,babyitems_69,53,train,train
baby_items_69_0_3663.jpg,babyitems,safe,,babyitems_69,53,train,train
baby_items_69_0_6266.jpg,babyitems,safe,,babyitems_69,53,train,train
baby_items_69_0_7608.jpg,babyitems,safe,,babyitems_69,53,train,train
baby_items_69_0_7642.jpg,babyitems,safe,,babyitems_69,53,train,train
baby_items_6_0_213.jpg,babyitems,safe,,babyitems_6,54,train,test
baby_items_6_0_2565.jpg,babyitems,safe,,babyitems_6,54,train,val
baby_items_6_0_5306.jpg,babyitems,safe,,babyitems_6,54,train,train
baby_items_6_0_5591.jpg,babyitems,safe,,babyitems_6,54,train,test
baby_items_6_0_8380.jpg,babyitems,safe,,babyitems_6,54,train,val
baby_items_70_0_6207.jpg,babyitems,safe,,babyitems_70,55,train,train
baby_items_70_0_7295.jpg,babyitems,safe,,babyitems_70,55,train,test
baby_items_70_0_9287.jpg,babyitems,safe,,babyitems_70,55,train,val
baby_items_70_0_9746.jpg,babyitems,safe,,babyitems_70,55,train,train
baby_items_70_0_9756.jpg,babyitems,safe,,babyitems_70,55,train,val
baby_items_71_0_1481.jpg,babyitems,safe,,babyitems_71,56,train,train
baby_items_71_0_3036.jpg,babyitems,safe,,babyitems_71,56,train,val
baby_items_71_0_7032.jpg,babyitems,safe,,babyitems_71,56,train,train
baby_items_71_0_7303.jpg,babyitems,safe,,babyitems_71,56,train,val
baby_items_71_0_7907.jpg,babyitems,safe,,babyitems_71,56,train,train
baby_items_72_0_1904.jpg,babyitems,safe,,babyitems_72,57,test,train
baby_items_72_0_3207.jpg,babyitems,safe,,babyitems_72,57,test,train
baby_items_72_0_613.jpg,babyitems,safe,,babyitems_72,57,test,train
baby_items_72_0_6235.jpg,babyitems,safe,,babyitems_72,57,test,train
baby_items_72_0_6673.jpg,babyitems,safe,,babyitems_72,57,test,train
baby_items_73_0_2549.jpg,babyitems,safe,,babyitems_73,58,test,train
baby_items_73_0_3446.jpg,babyitems,safe,,babyitems_73,58,test,train
baby_items_73_0_4864.jpg,babyitems,safe,,babyitems_73,58,test,train
baby_items_73_0_8580.jpg,babyitems,safe,,babyitems_73,58,test,test
baby_items_73_0_9243.jpg,babyitems,safe,,babyitems_73,58,test,val
baby_items_74_0_159.jpg,babyitems,safe,,babyitems_74,59,test,val
baby_items_74_0_2845.jpg,babyitems,safe,,babyitems_74,59,test,train
baby_items_74_0_5923.jpg,babyitems,safe,,babyitems_74,59,test,train
baby_items_74_0_8373.jpg,babyitems,safe,,babyitems_74,59,test,train
baby_items_74_0_9061.jpg,babyitems,safe,,babyitems_74,59,test,train
baby_items_75_0_1010.jpg,babyitems,safe,,babyitems_75,60,train,train
baby_items_75_0_2144.jpg,babyitems,safe,,babyitems_75,60,train,train
baby_items_75_0_7730.jpg,babyitems,safe,,babyitems_75,60,train,train
baby_items_75_0_8293.jpg,babyitems,safe,,babyitems_75,60,train,train
baby_items_75_0_919.jpg,babyitems,safe,,babyitems_75,60,train,train
baby_items_76_0_1433.jpg,babyitems,safe,,babyitems_76,61,train,train
baby_items_76_0_4512.jpg,babyitems,safe,,babyitems_76,61,train,test
baby_items_76_0_6740.jpg,babyitems,safe,,babyitems_76,61,train,train
baby_items_76_0_8417.jpg,babyitems,safe,,babyitems_76,61,train,test
baby_items_76_0_9996.jpg,babyitems,safe,,babyitems_76,61,train,train
baby_items_77_0_1881.jpg,babyitems,safe,,babyitems_77,62,val,train
baby_items_77_0_3360.jpg,babyitems,safe,,babyitems_77,62,val,train
baby_items_77_0_4118.jpg,babyitems,safe,,babyitems_77,62,val,train
baby_items_77_0_6609.jpg,babyitems,safe,,babyitems_77,62,val,test
baby_items_77_0_9739.jpg,babyitems,safe,,babyitems_77,62,val,train
baby_items_79_0_1515.jpg,babyitems,safe,,babyitems_79,63,test,test
baby_items_79_0_3652.jpg,babyitems,safe,,babyitems_79,63,test,train
baby_items_79_0_4460.jpg,babyitems,safe,,babyitems_79,63,test,train
baby_items_79_0_6183.jpg,babyitems,safe,,babyitems_79,63,test,train
baby_items_79_0_9905.jpg,babyitems,safe,,babyitems_79,63,test,test
baby_items_80_0_2335.jpg,babyitems,safe,,babyitems_80,64,train,train
baby_items_80_0_3972.jpg,babyitems,safe,,babyitems_80,64,train,train
baby_items_80_0_8200.jpg,babyitems,safe,,babyitems_80,64,train,val
baby_items_80_0_9333.jpg,babyitems,safe,,babyitems_80,64,train,train
baby_items_80_0_9593.jpg,babyitems,safe,,babyitems_80,64,train,train
baby_items_81_0_4607.jpg,babyitems,safe,,babyitems_81,65,val,train
baby_items_81_0_5590.jpg,babyitems,safe,,babyitems_81,65,val,train
baby_items_81_0_6003.jpg,babyitems,safe,,babyitems_81,65,val,train
baby_items_81_0_6333.jpg,babyitems,safe,,babyitems_81,65,val,test
baby_items_81_0_7369.jpg,babyitems,safe,,babyitems_81,65,val,train
baby_items_82_0_113.jpg,babyitems,safe,,babyitems_82,66,train,test
baby_items_82_0_2359.jpg,babyitems,safe,,babyitems_82,66,train,val
baby_items_82_0_3690.jpg,babyitems,safe,,babyitems_82,66,train,train
baby_items_82_0_6889.jpg,babyitems,safe,,babyitems_82,66,train,train
baby_items_82_0_7937.jpg,babyitems,safe,,babyitems_82,66,train,train
baby_items_84_0_2154.jpg,babyitems,safe,,babyitems_84,67,train,train
baby_items_84_0_3035.jpg,babyitems,safe,,babyitems_84,67,train,train
baby_items_84_0_5176.jpg,babyitems,safe,,babyitems_84,67,train,train
baby_items_84_0_5629.jpg,babyitems,safe,,babyitems_84,67,train,train
baby_items_84_0_8811.jpg,babyitems,safe,,babyitems_84,67,train,val
baby_items_85_0_2286.jpg,babyitems,safe,,babyitems_85,68,val,train
baby_items_85_0_2798.jpg,babyitems,safe,,babyitems_85,68,val,train
baby_items_85_0_3599.jpg,babyitems,safe,,babyitems_85,68,val,val
baby_items_85_0_5084.jpg,babyitems,safe,,babyitems_85,68,val,test
baby_items_85_0_5113.jpg,babyitems,safe,,babyitems_85,68,val,test
baby_items_86_0_6989.jpg,babyitems,safe,,babyitems_86,69,train,train
baby_items_86_0_8342.jpg,babyitems,safe,,babyitems_86,69,train,train
baby_items_86_0_8386.jpg,babyitems,safe,,babyitems_86,69,train,train
baby_items_86_0_8773.jpg,babyitems,safe,,babyitems_86,69,train,test
baby_items_86_0_9548.jpg,babyitems,safe,,babyitems_86,69,train,val
baby_items_87_0_115.jpg,babyitems,safe,,babyitems_87,70,test,train
baby_items_87_0_119.jpg,babyitems,safe,,babyitems_87,70,test,train
baby_items_87_0_3026.jpg,babyitems,safe,,babyitems_87,70,test,train
baby_items_87_0_7806.jpg,babyitems,safe,,babyitems_87,70,test,train
baby_items_87_0_8888.jpg,babyitems,safe,,babyitems_87,70,test,val
baby_items_88_0_2163.jpg,babyitems,safe,,babyitems_88,71,train,train
baby_items_88_0_411.jpg,babyitems,safe,,babyitems_88,71,train,train
baby_items_88_0_5536.jpg,babyitems,safe,,babyitems_88,71,train,train
baby_items_88_0_8622.jpg,babyitems,safe,,babyitems_88,71,train,train
baby_items_88_0_9819.jpg,babyitems,safe,,babyitems_88,71,train,test
baby_items_89_0_1806.jpg,babyitems,safe,,babyitems_89,72,train,train
baby_items_89_0_3250.jpg,babyitems,safe,,babyitems_89,72,train,test
baby_items_89_0_5502.jpg,babyitems,safe,,babyitems_89,72,train,train
baby_items_89_0_7790.jpg,babyitems,safe,,babyitems_89,72,train,train
baby_items_89_0_9801.jpg,babyitems,safe,,babyitems_89,72,train,train
baby_items_90_0_1514.jpg,babyitems,safe,,babyitems_90,73,train,train
baby_items_90_0_2830.jpg,babyitems,safe,,babyitems_90,73,train,train
baby_items_90_0_3980.jpg,babyitems,safe,,babyitems_90,73,train,train
baby_items_90_0_4122.jpg,babyitems,safe,,babyitems_90,73,train,test
baby_items_90_0_8721.jpg,babyitems,safe,,babyitems_90,73,train,train
baby_items_9_0_1015.jpg,babyitems,safe,,babyitems_9,74,val,val
baby_items_9_0_555.jpg,babyitems,safe,,babyitems_9,74,val,train
baby_items_9_0_5561.jpg,babyitems,safe,,babyitems_9,74,val,test
baby_items_9_0_7910.jpg,babyitems,safe,,babyitems_9,74,val,train
baby_items_9_0_8427.jpg,babyitems,safe,,babyitems_9,74,val,test
ball_1109_0_2662.jpg,ball,safe,,ball_1109,75,val,train
ball_1109_0_2751.jpg,ball,safe,,ball_1109,75,val,val
ball_1109_0_4178.jpg,ball,safe,,ball_1109,75,val,val
ball_1109_0_5009.jpg,ball,safe,,ball_1109,75,val,train
ball_1109_0_9387.jpg,ball,safe,,ball_1109,75,val,test
ball_1110_0_2175.jpg,ball,safe,,ball_1110,76,train,val
ball_1110_0_2376.jpg,ball,safe,,ball_1110,76,train,train
ball_1110_0_4358.jpg,ball,safe,,ball_1110,76,train,val
ball_1110_0_4886.jpg,ball,safe,,ball_1110,76,train,val
ball_1110_0_974.jpg,ball,safe,,ball_1110,76,train,train
ball_1111_0_1373.jpg,ball,safe,,ball_1111,77,train,train
ball_1111_0_4802.jpg,ball,safe,,ball_1111,77,train,val
ball_1111_0_5861.jpg,ball,safe,,ball_1111,77,train,train
ball_1111_0_6003.jpg,ball,safe,,ball_1111,77,train,train
ball_1111_0_8597.jpg,ball,safe,,ball_1111,77,train,val
ball_1112_0_5267.jpg,ball,safe,,ball_1112,78,train,train
ball_1112_0_6819.jpg,ball,safe,,ball_1112,78,train,train
ball_1112_0_6929.jpg,ball,safe,,ball_1112,78,train,train
ball_1112_0_8879.jpg,ball,safe,,ball_1112,78,train,train
ball_1112_0_9628.jpg,ball,safe,,ball_1112,78,train,test
ball_1113_0_1837.jpg,ball,safe,,ball_1113,79,train,train
ball_1113_0_1986.jpg,ball,safe,,ball_1113,79,train,train
ball_1113_0_5156.jpg,ball,safe,,ball_1113,79,train,train
ball_1113_0_5158.jpg,ball,safe,,ball_1113,79,train,val
ball_1113_0_7794.jpg,ball,safe,,ball_1113,79,train,train
ball_1114_0_2576.jpg,ball,safe,,ball_1114,80,train,val
ball_1114_0_4748.jpg,ball,safe,,ball_1114,80,train,train
ball_1114_0_7535.jpg,ball,safe,,ball_1114,80,train,train
ball_1114_0_7621.jpg,ball,safe,,ball_1114,80,train,train
ball_1114_0_9945.jpg,ball,safe,,ball_1114,80,train,train
ball_129_0_1236.jpg,ball,safe,,ball_129,81,test,train
ball_129_0_3297.jpg,ball,safe,,ball_129,81,test,val
ball_129_0_578.jpg,ball,safe,,ball_129,81,test,val
ball_129_0_6172.jpg,ball,safe,,ball_129,81,test,val
ball_129_0_8646.jpg,ball,safe,,ball_129,81,test,train
ball_130_0_2003.jpg,ball,safe,,ball_130,82,train,test
ball_130_0_3337.jpg,ball,safe,,ball_130,82,train,train
ball_130_0_5706.jpg,ball,safe,,ball_130,82,train,train
ball_130_0_6227.jpg,ball,safe,,ball_130,82,train,val
ball_130_0_9615.jpg,ball,safe,,ball_130,82,train,train
ball_131_0_1360.jpg,ball,safe,,ball_131,83,train,train
ball_131_0_2473.jpg,ball,safe,,ball_131,83,train,train
ball_131_0_600.jpg,ball,safe,,ball_131,83,train,train
ball_131_0_7021.jpg,ball,safe,,ball_131,83,train,val
ball_131_0_9730.jpg,ball,safe,,ball_131,83,train,train
ball_132_0_1086.jpg,ball,safe,,ball_132,84,val,train
ball_132_0_2840.jpg,ball,safe,,ball_132,84,val,test
ball_132_0_4909.jpg,ball,safe,,ball_132,84,val,train
ball_132_0_6969.jpg,ball,safe,,ball_132,84,val,val
ball_132_0_8074.jpg,ball,safe,,ball_132,84,val,train
ball_133_0_3549.jpg,ball,safe,,ball_133,85,test,test
ball_133_0_6.jpg,ball,safe,,ball_133,85,test,train
ball_133_0_6908.jpg,ball,safe,,ball_133,85,test,test
ball_133_0_8148.jpg,ball,safe,,ball_133,85,test,test
ball_133_0_961.jpg,ball,safe,,ball_133,85,test,train
ball_134_0_2483.jpg,ball,safe,,ball_134,86,train,train
ball_134_0_3798.jpg,ball,safe,,ball_134,86,train,train
ball_134_0_4906.jpg,ball,safe,,ball_134,86,train,test
ball_134_0_5270.jpg,ball,safe,,ball_134,86,train,val
ball_134_0_9284.jpg,ball,safe,,ball_134,86,train,test
ball_135_0_1969.jpg,ball,safe,,ball_135,87,train,train
ball_135_0_2762.jpg,ball,safe,,ball_135,87,train,train
ball_135_0_2841.jpg,ball,safe,,ball_135,87,train,test
ball_135_0_9524.jpg,ball,safe,,ball_135,87,train,train
ball_135_0_9868.jpg,ball,safe,,ball_135,87,train,train
ball_136_0_2861.jpg,ball,safe,,ball_136,88,train,train
ball_136_0_3126.jpg,ball,safe,,ball_136,88,train,train
ball_136_0_4477.jpg,ball,safe,,ball_136,88,train,train
ball_136_0_4706.jpg,ball,safe,,ball_136,88,train,train
ball_136_0_6682.jpg,ball,safe,,ball_136,88,train,train
ball_137_0_3932.jpg,ball,safe,,ball_137,89,val,val
ball_137_0_445.jpg,ball,safe,,ball_137,89,val,train
ball_137_0_5250.jpg,ball,safe,,ball_137,89,val,val
ball_137_0_5577.jpg,ball,safe,,ball_137,89,val,val
ball_137_0_5895.jpg,ball,safe,,ball_137,89,val,train
ball_138_0_3011.jpg,ball,safe,,ball_138,90,val,val
ball_138_0_3023.jpg,ball,safe,,ball_138,90,val,train
ball_138_0_6086.jpg,ball,safe,,ball_138,90,val,val
ball_138_0_8368.jpg,ball,safe,,ball_138,90,val,train
ball_138_0_9900.jpg,ball,safe,,ball_138,90,val,train
ball_139_0_1272.jpg,ball,safe,,ball_139,91,train,train
ball_139_0_3921.jpg,ball,safe,,ball_139,91,train,train
ball_139_0_541.jpg,ball,safe,,ball_139,91,train,train
ball_139_0_6964.jpg,ball,safe,,ball_139,91,train,train
ball_139_0_7758.jpg,ball,safe,,ball_139,91,train,train
ball_140_0_2881.jpg,ball,safe,,ball_140,92,train,train
ball_140_0_5021.jpg,ball,safe,,ball_140,92,train,train
ball_140_0_5931.jpg,ball,safe,,ball_140,92,train,val
ball_140_0_7336.jpg,ball,safe,,ball_140,92,train,train
ball_140_0_8188.jpg,ball,safe,,ball_140,92,train,val
ball_141_0_2783.jpg,ball,safe,,ball_141,93,val,train
ball_141_0_5622.jpg,ball,safe,,ball_141,93,val,test
ball_141_0_5792.jpg,ball,safe,,ball_141,93,val,train
ball_141_0_8539.jpg,ball,safe,,ball_141,93,val,val
ball_141_0_9580.jpg,ball,safe,,ball_141,93,val,train
ball_142_0_1080.jpg,ball,safe,,ball_142,94,train,train
ball_142_0_1265.jpg,ball,safe,,ball_142,94,train,train
ball_142_0_5071.jpg,ball,safe,,ball_142,94,train,train
ball_142_0_7690.jpg,ball,safe,,ball_142,94,train,train
ball_142_0_8678.jpg,ball,safe,,ball_142,94,train,train
ball_143_0_2208.jpg,ball,safe,,ball_143,95,test,test
ball_143_0_2288.jpg,ball,safe,,ball_143,95,test,train
ball_143_0_290.jpg,ball,safe,,ball_143,95,test,test
ball_143_0_41.jpg,ball,safe,,ball_143,95,test,train
ball_143_0_6189.jpg,ball,safe,,ball_143,95,test,train
ball_144_0_3077.jpg,ball,safe,,ball_144,96,train,test
ball_144_0_4611.jpg,ball,safe,,ball_144,96,train,train
ball_144_0_4767.jpg,ball,safe,,ball_144,96,train,test
ball_144_0_7642.jpg,ball,safe,,ball_144,96,train,train
ball_144_0_9104.jpg,ball,safe,,ball_144,96,train,val
ball_145_0_1587.jpg,ball,safe,,ball_145,97,train,train
ball_145_0_4092.jpg,ball,safe,,ball_145,97,train,train
ball_145_0_5168.jpg,ball,safe,,ball_145,97,train,val
ball_145_0_794.jpg,ball,safe,,ball_145,97,train,train
ball_145_0_8949.jpg,ball,safe,,ball_145,97,train,train
ball_146_0_1322.jpg,ball,safe,,ball_146,98,train,train
ball_146_0_1861.jpg,ball,safe,,ball_146,98,train,test
ball_146_0_7053.jpg,ball,safe,,ball_146,98,train,train
ball_146_0_9403.jpg,ball,safe,,ball_146,98,train,train
ball_146_0_9625.jpg,ball,safe,,ball_146,98,train,test
ball_147_0_2563.jpg,ball,safe,,ball_147,99,train,train
ball_147_0_2987.jpg,ball,safe,,ball_147,99,train,train
ball_147_0_3888.jpg,ball,safe,,ball_147,99,train,val
ball_147_0_9525.jpg,ball,safe,,ball_147,99,train,test
ball_147_0_9646.jpg,ball,safe,,ball_147,99,train,test
ball_148_0_5746.jpg,ball,safe,,ball_148,100,train,train
ball_148_0_7507.jpg,ball,safe,,ball_148,100,train,train
ball_148_0_7835.jpg,ball,safe,,ball_148,100,train,train
ball_148_0_871.jpg,ball,safe,,ball_148,100,train,train
ball_148_0_981.jpg,ball,safe,,ball_148,100,train,train
ball_149_0_2157.jpg,ball,safe,,ball_149,101,train,train
ball_149_0_2759.jpg,ball,safe,,ball_149,101,train,train
ball_149_0_2995.jpg,ball,safe,,ball_149,101,train,train
ball_149_0_4146.jpg,ball,safe,,ball_149,101,train,train
ball_149_0_5187.jpg,ball,safe,,ball_149,101,train,train
ball_150_0_2215.jpg,ball,safe,,ball_150,102,train,train
ball_150_0_7396.jpg,ball,safe,,ball_150,102,train,train
ball_150_0_7718.jpg,ball,safe,,ball_150,102,train,val
ball_150_0_8083.jpg,ball,safe,,ball_150,102,train,train
ball_150_0_9212.jpg,ball,safe,,ball_150,102,train,train
ball_151_0_3873.jpg,ball,safe,,ball_151,103,train,val
ball_151_0_4415.jpg,ball,safe,,ball_151,103,train,test
ball_151_0_5514.jpg,ball,safe,,ball_151,103,train,test
ball_151_0_8795.jpg,ball,safe,,ball_151,103,train,test
ball_151_0_9648.jpg,ball,safe,,ball_151,103,train,test
ball_152_0_3882.jpg,ball,safe,,ball_152,104,test,train
ball_152_0_4038.jpg,ball,safe,,ball_152,104,test,test
ball_152_0_632.jpg,ball,safe,,ball_152,104,test,val
ball_152_0_7507.jpg,ball,safe,,ball_152,104,test,train
ball_152_0_833.jpg,ball,safe,,ball_152,104,test,test
ball_153_0_224.jpg,ball,safe,,ball_153,105,val,train
ball_153_0_5007.jpg,ball,safe,,ball_153,105,val,train
ball_153_0_639.jpg,ball,safe,,ball_153,105,val,test
ball_153_0_796.jpg,ball,safe,,ball_153,105,val,train
ball_153_0_9679.jpg,ball,safe,,ball_153,105,val,train
ball_154_0_3745.jpg,ball,safe,,ball_154,106,train,train
ball_154_0_494.jpg,ball,safe,,ball_154,106,train,val
ball_154_0_7718.jpg,ball,safe,,ball_154,106,train,test
ball_154_0_857.jpg,ball,safe,,ball_154,106,train,train
ball_154_0_8697.jpg,ball,safe,,ball_154,106,train,train
ball_155_0_1808.jpg,ball,safe,,ball_155,107,test,train
ball_155_0_4076.jpg,ball,safe,,ball_155,107,test,val
ball_155_0_6919.jpg,ball,safe,,ball_155,107,test,val
ball_155_0_8482.jpg,ball,safe,,ball_155,107,test,train
ball_155_0_9965.jpg,ball,safe,,ball_155,107,test,train
ball_156_0_1840.jpg,ball,safe,,ball_156,108,test,train
ball_156_0_6148.jpg,ball,safe,,ball_156,108,test,train
ball_156_0_6250.jpg,ball,safe,,ball_156,108,test,train
ball_156_0_807.jpg,ball,safe,,ball_156,108,test,train
ball_156_0_8865.jpg,ball,safe,,ball_156,108,test,test
ball_157_0_4137.jpg,ball,safe,,ball_157,109,test,train
ball_157_0_5646.jpg,ball,safe,,ball_157,109,test,train
ball_157_0_5850.jpg,ball,safe,,ball_157,109,test,test
ball_157_0_7913.jpg,ball,safe,,ball_157,109,test,train
ball_157_0_8893.jpg,ball,safe,,ball_157,109,test,train
ball_158_0_5486.jpg,ball,safe,,ball_158,110,train,val
ball_158_0_7624.jpg,ball,safe,,ball_158,110,train,train
ball_158_0_7840.jpg,ball,safe,,ball_158,110,train,train
ball_158_0_8843.jpg,ball,safe,,ball_158,110,train,train
ball_158_0_9582.jpg,ball,safe,,ball_158,110,train,test
ball_159_0_1161.jpg,ball,safe,,ball_159,111,train,train
ball_159_0_3561.jpg,ball,safe,,ball_159,111,train,train
ball_159_0_3640.jpg,ball,safe,,ball_159,111,train,train
ball_159_0_4782.jpg,ball,safe,,ball_159,111,train,train
ball_159_0_750.jpg,ball,safe,,ball_159,111,train,train
ball_160_0_1382.jpg,ball,safe,,ball_160,112,val,test
ball_160_0_1998.jpg,ball,safe,,ball_160,112,val,train
ball_160_0_34.jpg,ball,safe,,ball_160,112,val,train
ball_160_0_3839.jpg,ball,safe,,ball_160,112,val,train
ball_160_0_8521.jpg,ball,safe,,ball_160,112,val,test
ball_161_0_3020.jpg,ball,safe,,ball_161,113,train,train
ball_161_0_4468.jpg,ball,safe,,ball_161,113,train,train
ball_161_0_5374.jpg,ball,safe,,ball_161,113,train,train
ball_161_0_7943.jpg,ball,safe,,ball_161,113,train,train
ball_161_0_9616.jpg,ball,safe,,ball_161,113,train,train
ball_162_0_4080.jpg,ball,safe,,ball_162,114,val,test
ball_162_0_462.jpg,ball,safe,,ball_162,114,val,train
ball_162_0_5675.jpg,ball,safe,,ball_162,114,val,train
ball_162_0_8481.jpg,ball,safe,,ball_162,114,val,train
ball_162_0_9595.jpg,ball,safe,,ball_162,114,val,train
ball_163_0_3337.jpg,ball,safe,,ball_163,115,train,val
ball_163_0_4527.jpg,ball,safe,,ball_163,115,train,train
ball_163_0_4712.jpg,ball,safe,,ball_163,115,train,train
ball_163_0_9194.jpg,ball,safe,,ball_163,115,train,train
ball_163_0_946.jpg,ball,safe,,ball_163,115,train,train
ball_164_0_6829.jpg,ball,safe,,ball_164,116,train,train
ball_164_0_6865.jpg,ball,safe,,ball_164,116,train,train
ball_164_0_9257.jpg,ball,safe,,ball_164,116,train,train
ball_164_0_9321.jpg,ball,safe,,ball_164,116,train,val
ball_164_0_9775.jpg,ball,safe,,ball_164,116,train,train
ball_165_0_10.jpg,ball,safe,,ball_165,117,test,train
ball_165_0_6235.jpg,ball,safe,,ball_165,117,test,train
ball_165_0_6701.jpg,ball,safe,,ball_165,117,test,train
ball_165_0_7879.jpg,ball,safe,,ball_165,117,test,train
ball_165_0_9887.jpg,ball,safe,,ball_165,117,test,test
ball_166_0_1209.jpg,ball,safe,,ball_166,118,test,train
ball_166_0_1919.jpg,ball,safe,,ball_166,118,test,train
ball_166_0_1982.jpg,ball,safe,,ball_166,118,test,train
ball_166_0_8654.jpg,ball,safe,,ball_166,118,test,train
ball_166_0_9290.jpg,ball,safe,,ball_166,118,test,train
ball_167_0_1587.jpg,ball,safe,,ball_167,119,train,train
ball_167_0_3466.jpg,ball,safe,,ball_167,119,train,train
ball_167_0_5312.jpg,ball,safe,,ball_167,119,train,train
ball_167_0_7728.jpg,ball,safe,,ball_167,119,train,train
ball_167_0_906.jpg,ball,safe,,ball_167,119,train,train
ball_168_0_1717.jpg,ball,safe,,ball_168,120,test,train
ball_168_0_3808.jpg,ball,safe,,ball_168,120,test,train
ball_168_0_5274.jpg,ball,safe,,ball_168,120,test,train
ball_168_0_5763.jpg,ball,safe,,ball_168,120,test,test
ball_168_0_670.jpg,ball,safe,,ball_168,120,test,val
ball_169_0_334.jpg,ball,safe,,ball_169,121,val,train
ball_169_0_3510.jpg,ball,safe,,ball_169,121,val,train
ball_169_0_5190.jpg,ball,safe,,ball_169,121,val,val
ball_169_0_7360.jpg,ball,safe,,ball_169,121,val,train
ball_169_0_8273.jpg,ball,safe,,ball_169,121,val,train
ball_170_0_2523.jpg,ball,safe,,ball_170,122,train,train
ball_170_0_4072.jpg,ball,safe,,ball_170,122,train,train
ball_170_0_5440.jpg,ball,safe,,ball_170,122,train,train
ball_170_0_8074.jpg,ball,safe,,ball_170,122,train,train
ball_170_0_9864.jpg,ball,safe,,ball_170,122,train,train
ball_171_0_1226.jpg,ball,safe,,ball_171,123,train,train
ball_171_0_4175.jpg,ball,safe,,ball_171,123,train,train
ball_171_0_555.jpg,ball,safe,,ball_171,123,train,train
ball_171_0_5641.jpg,ball,safe,,ball_171,123,train,train
ball_171_0_6635.jpg,ball,safe,,ball_171,123,train,val
ball_172_0_2102.jpg,ball,safe,,ball_172,124,train,train
ball_172_0_6513.jpg,ball,safe,,ball_172,124,train,train
ball_172_0_7776.jpg,ball,safe,,ball_172,124,train,train
ball_172_0_8369.jpg,ball,safe,,ball_172,124,train,train
ball_172_0_9869.jpg,ball,safe,,ball_172,124,train,train
ball_173_0_2781.jpg,ball,safe,,ball_173,125,train,train
ball_173_0_3092.jpg,ball,safe,,ball_173,125,train,train
ball_173_0_4132.jpg,ball,safe,,ball_173,125,train,train
ball_173_0_8408.jpg,ball,safe,,ball_173,125,train,train
ball_173_0_8525.jpg,ball,safe,,ball_173,125,train,val
ball_174_0_291.jpg,ball,safe,,ball_174,126,train,train
ball_174_0_3217.jpg,ball,safe,,ball_174,126,train,train
ball_174_0_329.jpg,ball,safe,,ball_174,126,train,train
ball_174_0_4009.jpg,ball,safe,,ball_174,126,train,train
ball_174_0_9880.jpg,ball,safe,,ball_174,126,train,train
ball_175_0_1162.jpg,ball,safe,,ball_175,127,train,train
ball_175_0_4305.jpg,ball,safe,,ball_175,127,train,test
ball_175_0_4518.jpg,ball,safe,,ball_175,127,train,val
ball_175_0_512.jpg,ball,safe,,ball_175,127,train,test
ball_175_0_8650.jpg,ball,safe,,ball_175,127,train,train
ball_176_0_2119.jpg,ball,safe,,ball_176,128,train,train
ball_176_0_3204.jpg,ball,safe,,ball_176,128,train,train
ball_176_0_841.jpg,ball,safe,,ball_176,128,train,train
ball_176_0_8879.jpg,ball,safe,,ball_176,128,train,test
ball_176_0_9008.jpg,ball,safe,,ball_176,128,train,train
ball_177_0_2813.jpg,ball,safe,,ball_177,129,train,test
ball_177_0_498.jpg,ball,safe,,ball_177,129,train,train
ball_177_0_6674.jpg,ball,safe,,ball_177,129,train,val
ball_177_0_8279.jpg,ball,safe,,ball_177,129,train,train
ball_177_0_9605.jpg,ball,safe,,ball_177,129,train,test
ball_178_0_1336.jpg,ball,safe,,ball_178,130,train,train
ball_178_0_2424.jpg,ball,safe,,ball_178,130,train,train
ball_178_0_423.jpg,ball,safe,,ball_178,130,train,train
ball_178_0_7482.jpg,ball,safe,,ball_178,130,train,test
ball_178_0_8066.jpg,ball,safe,,ball_178,130,train,train
ball_179_0_1442.jpg,ball,safe,,ball_179,131,train,train
ball_179_0_280.jpg,ball,safe,,ball_179,131,train,train
ball_179_0_4258.jpg,ball,safe,,ball_179,131,train,val
ball_179_0_607.jpg,ball,safe,,ball_179,131,train,val
ball_179_0_9642.jpg,ball,safe,,ball_179,131,train,train
ball_180_0_3121.jpg,ball,safe,,ball_180,132,train,train
ball_180_0_5679.jpg,ball,safe,,ball_180,132,train,train
ball_180_0_5703.jpg,ball,safe,,ball_180,132,train,test
ball_180_0_9498.jpg,ball,safe,,ball_180,132,train,train
ball_180_0_9675.jpg,ball,safe,,ball_180,132,train,train
ball_181_0_132.jpg,ball,safe,,ball_181,133,train,train
ball_181_0_166.jpg,ball,safe,,ball_181,133,train,train
ball_181_0_1797.jpg,ball,safe,,ball_181,133,train,train
ball_181_0_3035.jpg,ball,safe,,ball_181,133,train,val
ball_181_0_3735.jpg,ball,safe,,ball_181,133,train,train
ball_182_0_1227.jpg,ball,safe,,ball_182,134,train,train
ball_182_0_4213.jpg,ball,safe,,ball_182,134,train,train
ball_182_0_6085.jpg,ball,safe,,ball_182,134,train,train
ball_182_0_7837.jpg,ball,safe,,ball_182,134,train,train
ball_182_0_8038.jpg,ball,safe,,ball_182,134,train,train
ball_183_0_3687.jpg,ball,safe,,ball_183,135,train,train
ball_183_0_5354.jpg,ball,safe,,ball_183,135,train,train
ball_183_0_5827.jpg,ball,safe,,ball_183,135,train,val
ball_183_0_7701.jpg,ball,safe,,ball_183,135,train,train
ball_183_0_8446.jpg,ball,safe,,ball_183,135,train,test
ball_184_0_1043.jpg,ball,safe,,ball_184,136,train,train
ball_184_0_184.jpg,ball,safe,,ball_184,136,train,train
ball_184_0_7047.jpg,ball,safe,,ball_184,136,train,test
ball_184_0_7865.jpg,ball,safe,,ball_184,136,train,train
ball_184_0_8993.jpg,ball,safe,,ball_184,136,train,val
ball_185_0_2821.jpg,ball,safe,,ball_185,137,test,test
ball_185_0_362.jpg,ball,safe,,ball_185,137,test,train
ball_185_0_4003.jpg,ball,safe,,ball_185,137,test,train
ball_185_0_693.jpg,ball,safe,,ball_185,137,test,test
ball_185_0_7178.jpg,ball,safe,,ball_185,137,test,val
ball_186_0_4440.jpg,ball,safe,,ball_186,138,train,train
ball_186_0_5755.jpg,ball,safe,,ball_186,138,train,train
ball_186_0_6149.jpg,ball,safe,,ball_186,138,train,test
ball_186_0_7268.jpg,ball,safe,,ball_186,138,train,test
ball_186_0_9101.jpg,ball,safe,,ball_186,138,train,train
ball_187_0_2981.jpg,ball,safe,,ball_187,139,train,test
ball_187_0_2988.jpg,ball,safe,,ball_187,139,train,test
ball_187_0_4339.jpg,ball,safe,,ball_187,139,train,train
ball_187_0_4518.jpg,ball,safe,,ball_187,139,train,train
ball_187_0_8832.jpg,ball,safe,,ball_187,139,train,train
ball_188_0_487.jpg,ball,safe,,ball_188,140,train,test
ball_188_0_6752.jpg,ball,safe,,ball_188,140,train,val
ball_188_0_7111.jpg,ball,safe,,ball_188,140,train,train
ball_188_0_822.jpg,ball,safe,,ball_188,140,train,val
ball_188_0_975.jpg,ball,safe,,ball_188,140,train,train
ball_189_0_1462.jpg,ball,safe,,ball_189,141,val,val
ball_189_0_3003.jpg,ball,safe,,ball_189,141,val,train
ball_189_0_4204.jpg,ball,safe,,ball_189,141,val,train
ball_189_0_5192.jpg,ball,safe,,ball_189,141,val,val
ball_189_0_7003.jpg,ball,safe,,ball_189,141,val,train
ball_190_0_217.jpg,ball,safe,,ball_190,142,train,train
ball_190_0_2374.jpg,ball,safe,,ball_190,142,train,train
ball_190_0_2744.jpg,ball,safe,,ball_190,142,train,train
ball_190_0_3399.jpg,ball,safe,,ball_190,142,train,train
ball_190_0_4835.jpg,ball,safe,,ball_190,142,train,train
ball_191_0_11.jpg,ball,safe,,ball_191,143,train,train
ball_191_0_2194.jpg,ball,safe,,ball_191,143,train,test
ball_191_0_3384.jpg,ball,safe,,ball_191,143,train,train
ball_191_0_4654.jpg,ball,safe,,ball_191,143,train,train
ball_191_0_5570.jpg,ball,safe,,ball_191,143,train,train
ball_192_0_1332.jpg,ball,safe,,ball_192,144,val,train
ball_192_0_4286.jpg,ball,safe,,ball_192,144,val,train
ball_192_0_4526.jpg,ball,safe,,ball_192,144,val,val
ball_192_0_671.jpg,ball,safe,,ball_192,144,val,train
ball_192_0_8950.jpg,ball,safe,,ball_192,144,val,train
ball_193_0_3189.jpg,ball,safe,,ball_193,145,train,train
ball_193_0_3680.jpg,ball,safe,,ball_193,145,train,train
ball_193_0_6981.jpg,ball,safe,,ball_193,145,train,train
ball_193_0_7000.jpg,ball,safe,,ball_193,145,train,test
ball_193_0_8135.jpg,ball,safe,,ball_193,145,train,train
ball_194_0_178.jpg,ball,safe,,ball_194,146,train,train
ball_194_0_1880.jpg,ball,safe,,ball_194,146,train,train
ball_194_0_1955.jpg,ball,safe,,ball_194,146,train,train
ball_194_0_4567.jpg,ball,safe,,ball_194,146,train,train
ball_194_0_6448.jpg,ball,safe,,ball_194,146,train,train
ball_195_0_4110.jpg,ball,safe,,ball_195,147,train,train
ball_195_0_4531.jpg,ball,safe,,ball_195,147,train,train
ball_195_0_5275.jpg,ball,safe,,ball_195,147,train,test
ball_195_0_6122.jpg,ball,safe,,ball_195,147,train,train
ball_195_0_8972.jpg,ball,safe,,ball_195,147,train,train
ball_196_0_1969.jpg,ball,safe,,ball_196,148,train,train
ball_196_0_389.jpg,ball,safe,,ball_196,148,train,val
ball_196_0_7184.jpg,ball,safe,,ball_196,148,train,train
ball_196_0_8348.jpg,ball,safe,,ball_196,148,train,train
ball_196_0_957.jpg,ball,safe,,ball_196,148,train,val
ball_197_0_288.jpg,ball,safe,,ball_197,149,val,test
ball_197_0_685.jpg,ball,safe,,ball_197,149,val,test
ball_197_0_7017.jpg,ball,safe,,ball_197,149,val,train
ball_197_0_8732.jpg,ball,safe,,ball_197,149,val,train
ball_197_0_8956.jpg,ball,safe,,ball_197,149,val,val
ball_198_0_1878.jpg,ball,safe,,ball_198,150,train,train
ball_198_0_4908.jpg,ball,safe,,ball_198,150,train,test
ball_198_0_7453.jpg,ball,safe,,ball_198,150,train,val
ball_198_0_7854.jpg,ball,safe,,ball_198,150,train,train
ball_198_0_9278.jpg,ball,safe,,ball_198,150,train,train
ball_199_0_2315.jpg,ball,safe,,ball_199,151,train,test
ball_199_0_5660.jpg,ball,safe,,ball_199,151,train,train
ball_199_0_7308.jpg,ball,safe,,ball_199,151,train,val
ball_199_0_7655.jpg,ball,safe,,ball_199,151,train,train
ball_199_0_8528.jpg,ball,safe,,ball_199,151,train,train
fork_100_0_1876.jpg,fork,unsafe,,fork_100,152,train,train
fork_100_0_5423.jpg,fork,unsafe,,fork_100,152,train,train
fork_100_0_5693.jpg,fork,unsafe,,fork_100,152,train,test
fork_100_0_8127.jpg,fork,unsafe,,fork_100,152,train,val
fork_100_0_8925.jpg,fork,unsafe,,fork_100,152,train,test
fork_101_0_1437.jpg,fork,unsafe,,fork_101,153,train,val
fork_101_0_3857.jpg,fork,unsafe,,fork_101,153,train,test
fork_101_0_4542.jpg,fork,unsafe,,fork_101,153,train,test
fork_101_0_5624.jpg,fork,unsafe,,fork_101,153,train,train
fork_101_0_829.jpg,fork,unsafe,,fork_101,153,train,val
fork_102_0_4700.jpg,fork,unsafe,,fork_102,154,train,train
fork_102_0_4764.jpg,fork,unsafe,,fork_102,154,train,train
fork_102_0_6936.jpg,fork,unsafe,,fork_102,154,train,train
fork_102_0_7391.jpg,fork,unsafe,,fork_102,154,train,train
fork_102_0_8978.jpg,fork,unsafe,,fork_102,154,train,train
fork_103_0_1321.jpg,fork,unsafe,,fork_103,155,train,train
fork_103_0_4794.jpg,fork,unsafe,,fork_103,155,train,train
fork_103_0_7073.jpg,fork,unsafe,,fork_103,155,train,val
fork_103_0_8251.jpg,fork,unsafe,,fork_103,155,train,train
fork_103_0_9980.jpg,fork,unsafe,,fork_103,155,train,train
fork_104_0_2904.jpg,fork,unsafe,,fork_104,156,train,train
fork_104_0_5590.jpg,fork,unsafe,,fork_104,156,train,train
fork_104_0_5869.jpg,fork,unsafe,,fork_104,156,train,train
fork_104_0_6089.jpg,fork,unsafe,,fork_104,156,train,val
fork_104_0_9089.jpg,fork,unsafe,,fork_104,156,train,val
fork_105_0_3133.jpg,fork,unsafe,,fork_105,157,train,test
fork_105_0_3757.jpg,fork,unsafe,,fork_105,157,train,val
fork_105_0_404.jpg,fork,unsafe,,fork_105,157,train,test
fork_105_0_6516.jpg,fork,unsafe,,fork_105,157,train,train
fork_105_0_9587.jpg,fork,unsafe,,fork_105,157,train,train
fork_106_0_2516.jpg,fork,unsafe,,fork_106,158,train,train
fork_106_0_2721.jpg,fork,unsafe,,fork_106,158,train,train
fork_106_0_4496.jpg,fork,unsafe,,fork_106,158,train,val
fork_106_0_7011.jpg,fork,unsafe,,fork_106,158,train,train
fork_106_0_7663.jpg,fork,unsafe,,fork_106,158,train,train
fork_107_0_460.jpg,fork,unsafe,,fork_107,159,val,train
fork_107_0_8732.jpg,fork,unsafe,,fork_107,159,val,train
fork_107_0_8814.jpg,fork,unsafe,,fork_107,159,val,train
fork_107_0_9538.jpg,fork,unsafe,,fork_107,159,val,train
fork_107_0_957.jpg,fork,unsafe,,fork_107,159,val,train
fork_108_0_407.jpg,fork,unsafe,,fork_108,160,val,train
fork_108_0_5306.jpg,fork,unsafe,,fork_108,160,val,train
fork_108_0_5761.jpg,fork,unsafe,,fork_108,160,val,test
fork_108_0_6136.jpg,fork,unsafe,,fork_108,160,val,val
fork_108_0_9285.jpg,fork,unsafe,,fork_108,160,val,train
fork_109_0_1237.jpg,fork,unsafe,,fork_109,161,test,test
fork_109_0_1521.jpg,fork,unsafe,,fork_109,161,test,test
fork_109_0_3467.jpg,fork,unsafe,,fork_109,161,test,test
fork_109_0_8318.jpg,fork,unsafe,,fork_109,161,test,val
fork_109_0_9707.jpg,fork,unsafe,,fork_109,161,test,train
fork_10_0_2555.jpg,fork,unsafe,,fork_10,162,train,test
fork_10_0_3153.jpg,fork,unsafe,,fork_10,162,train,train
fork_10_0_3401.jpg,fork,unsafe,,fork_10,162,train,test
fork_10_0_5107.jpg,fork,unsafe,,fork_10,162,train,test
fork_10_0_7413.jpg,fork,unsafe,,fork_10,162,train,val
fork_110_0_7227.jpg,fork,unsafe,,fork_110,163,train,test
fork_110_0_7554.jpg,fork,unsafe,,fork_110,163,train,train
fork_110_0_775.jpg,fork,unsafe,,fork_110,163,train,train
fork_110_0_8010.jpg,fork,unsafe,,fork_110,163,train,val
fork_110_0_9157.jpg,fork,unsafe,,fork_110,163,train,train
fork_111_0_1343.jpg,fork,unsafe,,fork_111,164,test,test
fork_111_0_3068.jpg,fork,unsafe,,fork_111,164,test,train
fork_111_0_3699.jpg,fork,unsafe,,fork_111,164,test,train
fork_111_0_7408.jpg,fork,unsafe,,fork_111,164,test,train
fork_111_0_9829.jpg,fork,unsafe,,fork_111,164,test,train
fork_112_0_2183.jpg,fork,unsafe,,fork_112,165,test,val
fork_112_0_2887.jpg,fork,unsafe,,fork_112,165,test,train
fork_112_0_2962.jpg,fork,unsafe,,fork_112,165,test,train
fork_112_0_7351.jpg,fork,unsafe,,fork_112,165,test,test
fork_112_0_8905.jpg,fork,unsafe,,fork_112,165,test,train
fork_113_0_4336.jpg,fork,unsafe,,fork_113,166,train,train
fork_113_0_6014.jpg,fork,unsafe,,fork_113,166,train,train
fork_113_0_6937.jpg,fork,unsafe,,fork_113,166,train,test
fork_113_0_8568.jpg,fork,unsafe,,fork_113,166,train,train
fork_113_0_9588.jpg,fork,unsafe,,fork_113,166,train,test
fork_114_0_4627.jpg,fork,unsafe,,fork_114,167,train,val
fork_114_0_4755.jpg,fork,unsafe,,fork_114,167,train,val
fork_114_0_5168.jpg,fork,unsafe,,fork_114,167,train,test
fork_114_0_6611.jpg,fork,unsafe,,fork_114,167,train,train
fork_114_0_8616.jpg,fork,unsafe,,fork_114,167,train,train
fork_115_0_3197.jpg,fork,unsafe,,fork_115,168,test,test
fork_115_0_406.jpg,fork,unsafe,,fork_115,168,test,train
fork_115_0_4448.jpg,fork,unsafe,,fork_115,168,test,train
fork_115_0_6666.jpg,fork,unsafe,,fork_115,168,test,val
fork_115_0_8121.jpg,fork,unsafe,,fork_115,168,test,train
fork_116_0_2181.jpg,fork,unsafe,,fork_116,169,train,train
fork_116_0_254.jpg,fork,unsafe,,fork_116,169,train,train
fork_116_0_2898.jpg,fork,unsafe,,fork_116,169,train,train
fork_116_0_559.jpg,fork,unsafe,,fork_116,169,train,train
fork_116_0_6992.jpg,fork,unsafe,,fork_116,169,train,train
fork_117_0_103.jpg,fork,unsafe,,fork_117,170,train,val
fork_117_0_3494.jpg,fork,unsafe,,fork_117,170,train,train
fork_117_0_5353.jpg,fork,unsafe,,fork_117,170,train,test
fork_117_0_5899.jpg,fork,unsafe,,fork_117,170,train,train
fork_117_0_7859.jpg,fork,unsafe,,fork_117,170,train,test
fork_118_0_1277.jpg,fork,unsafe,,fork_118,171,train,train
fork_118_0_2524.jpg,fork,unsafe,,fork_118,171,train,val
fork_118_0_3329.jpg,fork,unsafe,,fork_118,171,train,test
fork_118_0_5407.jpg,fork,unsafe,,fork_118,171,train,train
fork_118_0_9216.jpg,fork,unsafe,,fork_118,171,train,val
fork_119_0_2064.jpg,fork,unsafe,,fork_119,153,train,train
fork_119_0_376.jpg,fork,unsafe,,fork_119,153,train,train
fork_119_0_4836.jpg,fork,unsafe,,fork_119,153,train,test
fork_119_0_8061.jpg,fork,unsafe,,fork_119,153,train,train
fork_119_0_9913.jpg,fork,unsafe,,fork_119,153,train,train
fork_11_0_2182.jpg,fork,unsafe,,fork_11,172,train,test
fork_11_0_4553.jpg,fork,unsafe,,fork_11,172,train,test
fork_11_0_5097.jpg,fork,unsafe,,fork_11,172,train,train
fork_11_0_9565.jpg,fork,unsafe,,fork_11,172,train,train
fork_11_0_9727.jpg,fork,unsafe,,fork_11,172,train,train
fork_120_0_2457.jpg,fork,unsafe,,fork_120,173,train,test
fork_120_0_2774.jpg,fork,unsafe,,fork_120,173,train,train
fork_120_0_6402.jpg,fork,unsafe,,fork_120,173,train,train
fork_120_0_6467.jpg,fork,unsafe,,fork_120,173,train,val
fork_120_0_6968.jpg,fork,unsafe,,fork_120,173,train,train
fork_121_0_5977.jpg,fork,unsafe,,fork_121,174,test,train
fork_121_0_6570.jpg,fork,unsafe,,fork_121,174,test,val
fork_121_0_6640.jpg,fork,unsafe,,fork_121,174,test,val
fork_121_0_6993.jpg,fork,unsafe,,fork_121,174,test,train
fork_121_0_7854.jpg,fork,unsafe,,fork_121,174,test,train
fork_122_0_3168.jpg,fork,unsafe,,fork_122,175,train,test
fork_122_0_3492.jpg,fork,unsafe,,fork_122,175,train,train
fork_122_0_3790.jpg,fork,unsafe,,fork_122,175,train,train
fork_122_0_7495.jpg,fork,unsafe,,fork_122,175,train,train
fork_122_0_8187.jpg,fork,unsafe,,fork_122,175,train,train
fork_123_0_400.jpg,fork,unsafe,,fork_123,176,train,test
fork_123_0_4931.jpg,fork,unsafe,,fork_123,176,train,train
fork_123_0_6810.jpg,fork,unsafe,,fork_123,176,train,train
fork_123_0_7134.jpg,fork,unsafe,,fork_123,176,train,val
fork_123_0_886.jpg,fork,unsafe,,fork_123,176,train,train
fork_124_0_1683.jpg,fork,unsafe,,fork_124,177,test,val
fork_124_0_4971.jpg,fork,unsafe,,fork_124,177,test,train
fork_124_0_5272.jpg,fork,unsafe,,fork_124,177,test,test
fork_124_0_6342.jpg,fork,unsafe,,fork_124,177,test,train
fork_124_0_72.jpg,fork,unsafe,,fork_124,177,test,train
fork_125_0_1482.jpg,fork,unsafe,,fork_125,178,test,train
fork_125_0_3963.jpg,fork,unsafe,,fork_125,178,test,train
fork_125_0_4762.jpg,fork,unsafe,,fork_125,178,test,test
fork_125_0_6387.jpg,fork,unsafe,,fork_125,178,test,train
fork_125_0_7169.jpg,fork,unsafe,,fork_125,178,test,train
fork_126_0_3353.jpg,fork,unsafe,,fork_126,179,train,train
fork_126_0_3454.jpg,fork,unsafe,,fork_126,179,train,train
fork_126_0_4057.jpg,fork,unsafe,,fork_126,179,train,train
fork_126_0_4359.jpg,fork,unsafe,,fork_126,179,train,val
fork_126_0_5833.jpg,fork,unsafe,,fork_126,179,train,train
fork_127_0_3027.jpg,fork,unsafe,,fork_127,180,train,train
fork_127_0_3605.jpg,fork,unsafe,,fork_127,180,train,val
fork_127_0_5800.jpg,fork,unsafe,,fork_127,180,train,train
fork_127_0_7809.jpg,fork,unsafe,,fork_127,180,train,train
fork_127_0_9882.jpg,fork,unsafe,,fork_127,180,train,val
fork_128_0_1100.jpg,fork,unsafe,,fork_128,181,val,train
fork_128_0_3243.jpg,fork,unsafe,,fork_128,181,val,train
fork_128_0_3649.jpg,fork,unsafe,,fork_128,181,val,train
fork_128_0_3955.jpg,fork,unsafe,,fork_128,181,val,train
fork_128_0_9286.jpg,fork,unsafe,,fork_128,181,val,train
fork_129_0_1334.jpg,fork,unsafe,,fork_129,182,test,train
fork_129_0_1931.jpg,fork,unsafe,,fork_129,182,test,train
fork_129_0_7122.jpg,fork,unsafe,,fork_129,182,test,train
fork_129_0_7755.jpg,fork,unsafe,,fork_129,182,test,train
fork_129_0_793.jpg,fork,unsafe,,fork_129,182,test,train
fork_12_0_4010.jpg,fork,unsafe,,fork_12,183,test,train
fork_12_0_5460.jpg,fork,unsafe,,fork_12,183,test,train
fork_12_0_564.jpg,fork,unsafe,,fork_12,183,test,train
fork_12_0_710.jpg,fork,unsafe,,fork_12,183,test,train
fork_12_0_814.jpg,fork,unsafe,,fork_12,183,test,val
fork_130_0_1716.jpg,fork,unsafe,,fork_130,184,train,test
fork_130_0_6346.jpg,fork,unsafe,,fork_130,184,train,train
fork_130_0_643.jpg,fork,unsafe,,fork_130,184,train,train
fork_130_0_6840.jpg,fork,unsafe,,fork_130,184,train,train
fork_130_0_8341.jpg,fork,unsafe,,fork_130,184,train,train
fork_131_0_1834.jpg,fork,unsafe,,fork_131,185,train,train
fork_131_0_48.jpg,fork,unsafe,,fork_131,185,train,train
fork_131_0_4800.jpg,fork,unsafe,,fork_131,185,train,train
fork_131_0_6905.jpg,fork,unsafe,,fork_131,185,train,train
fork_131_0_699.jpg,fork,unsafe,,fork_131,185,train,train
fork_132_0_1035.jpg,fork,unsafe,,fork_132,186,train,val
fork_132_0_2341.jpg,fork,unsafe,,fork_132,186,train,train
fork_132_0_2640.jpg,fork,unsafe,,fork_132,186,train,train
fork_132_0_3471.jpg,fork,unsafe,,fork_132,186,train,train
fork_132_0_3878.jpg,fork,unsafe,,fork_132,186,train,train
fork_133_0_383.jpg,fork,unsafe,,fork_133,187,train,train
fork_133_0_4199.jpg,fork,unsafe,,fork_133,187,train,train
fork_133_0_4643.jpg,fork,unsafe,,fork_133,187,train,test
fork_133_0_611.jpg,fork,unsafe,,fork_133,187,train,test
fork_133_0_8672.jpg,fork,unsafe,,fork_133,187,train,train
fork_134_0_230.jpg,fork,unsafe,,fork_134,188,test,val
fork_134_0_2939.jpg,fork,unsafe,,fork_134,188,test,val
fork_134_0_6043.jpg,fork,unsafe,,fork_134,188,test,train
fork_134_0_7518.jpg,fork,unsafe,,fork_134,188,test,val
fork_134_0_8904.jpg,fork,unsafe,,fork_134,188,test,train
fork_135_0_3178.jpg,fork,unsafe,,fork_135,189,train,val
fork_135_0_4474.jpg,fork,unsafe,,fork_135,189,train,train
fork_135_0_5075.jpg,fork,unsafe,,fork_135,189,train,train
fork_135_0_7320.jpg,fork,unsafe,,fork_135,189,train,train
fork_135_0_9079.jpg,fork,unsafe,,fork_135,189,train,test
fork_13_0_1537.jpg,fork,unsafe,,fork_13,190,test,train
fork_13_0_3255.jpg,fork,unsafe,,fork_13,190,test,train
fork_13_0_5560.jpg,fork,unsafe,,fork_13,190,test,train
fork_13_0_5986.jpg,fork,unsafe,,fork_13,190,test,train
fork_13_0_7874.jpg,fork,unsafe,,fork_13,190,test,train
fork_14_0_348.jpg,fork,unsafe,,fork_14,191,test,train
fork_14_0_3913.jpg,fork,unsafe,,fork_14,191,test,train
fork_14_0_7569.jpg,fork,unsafe,,fork_14,191,test,train
fork_14_0_8211.jpg,fork,unsafe,,fork_14,191,test,train
fork_14_0_8587.jpg,fork,unsafe,,fork_14,191,test,test
fork_15_0_308.jpg,fork,unsafe,,fork_15,192,train,train
fork_15_0_4085.jpg,fork,unsafe,,fork_15,192,train,test
fork_15_0_7133.jpg,fork,unsafe,,fork_15,192,train,val
fork_15_0_729.jpg,fork,unsafe,,fork_15,192,train,train
fork_15_0_9957.jpg,fork,unsafe,,fork_15,192,train,train
fork_16_0_5382.jpg,fork,unsafe,,fork_16,193,train,train
fork_16_0_5863.jpg,fork,unsafe,,fork_16,193,train,train
fork_16_0_7671.jpg,fork,unsafe,,fork_16,193,train,train
fork_16_0_8008.jpg,fork,unsafe,,fork_16,193,train,test
fork_16_0_9232.jpg,fork,unsafe,,fork_16,193,train,train
fork_17_0_3914.jpg,fork,unsafe,,fork_17,194,val,test
fork_17_0_4156.jpg,fork,unsafe,,fork_17,194,val,train
fork_17_0_5204.jpg,fork,unsafe,,fork_17,194,val,train
fork_17_0_5214.jpg,fork,unsafe,,fork_17,194,val,train
fork_17_0_536.jpg,fork,unsafe,,fork_17,194,val,train
fork_18_0_2823.jpg,fork,unsafe,,fork_18,195,train,train
fork_18_0_4898.jpg,fork,unsafe,,fork_18,195,train,train
fork_18_0_6239.jpg,fork,unsafe,,fork_18,195,train,train
fork_18_0_7025.jpg,fork,unsafe,,fork_18,195,train,val
fork_18_0_8001.jpg,fork,unsafe,,fork_18,195,train,val
fork_19_0_4479.jpg,fork,unsafe,,fork_19,196,train,train
fork_19_0_7723.jpg,fork,unsafe,,fork_19,196,train,train
fork_19_0_8228.jpg,fork,unsafe,,fork_19,196,train,train
fork_19_0_8600.jpg,fork,unsafe,,fork_19,196,train,test
fork_19_0_8621.jpg,fork,unsafe,,fork_19,196,train,train
fork_1_0_2257.jpg,fork,unsafe,,fork_1,197,train,train
fork_1_0_6582.jpg,fork,unsafe,,fork_1,197,train,train
fork_1_0_7087.jpg,fork,unsafe,,fork_1,197,train,test
fork_1_0_8381.jpg,fork,unsafe,,fork_1,197,train,train
fork_1_0_8641.jpg,fork,unsafe,,fork_1,197,train,train
fork_20_0_29.jpg,fork,unsafe,,fork_20,198,train,test
fork_20_0_4268.jpg,fork,unsafe,,fork_20,198,train,train
fork_20_0_6091.jpg,fork,unsafe,,fork_20,198,train,train
fork_20_0_6449.jpg,fork,unsafe,,fork_20,198,train,test
fork_20_0_9721.jpg,fork,unsafe,,fork_20,198,train,train
fork_21_0_1310.jpg,fork,unsafe,,fork_21,199,train,train
fork_21_0_2181.jpg,fork,unsafe,,fork_21,199,train,train
fork_21_0_2633.jpg,fork,unsafe,,fork_21,199,train,train
fork_21_0_3718.jpg,fork,unsafe,,fork_21,199,train,train
fork_21_0_5020.jpg,fork,unsafe,,fork_21,199,train,val
fork_22_0_118.jpg,fork,unsafe,,fork_22,200,test,train
fork_22_0_2493.jpg,fork,unsafe,,fork_22,200,test,val
fork_22_0_3480.jpg,fork,unsafe,,fork_22,200,test,train
fork_22_0_6408.jpg,fork,unsafe,,fork_22,200,test,test
fork_22_0_9863.jpg,fork,unsafe,,fork_22,200,test,val
fork_23_0_1912.jpg,fork,unsafe,,fork_23,201,train,train
fork_23_0_1998.jpg,fork,unsafe,,fork_23,201,train,val
fork_23_0_5774.jpg,fork,unsafe,,fork_23,201,train,test
fork_23_0_7387.jpg,fork,unsafe,,fork_23,201,train,train
fork_23_0_9720.jpg,fork,unsafe,,fork_23,201,train,train
fork_24_0_3151.jpg,fork,unsafe,,fork_24,202,train,train
fork_24_0_6117.jpg,fork,unsafe,,fork_24,202,train,test
fork_24_0_7971.jpg,fork,unsafe,,fork_24,202,train,train
fork_24_0_9240.jpg,fork,unsafe,,fork_24,202,train,train
fork_24_0_9624.jpg,fork,unsafe,,fork_24,202,train,val
fork_25_0_6458.jpg,fork,unsafe,,fork_25,203,train,val
fork_25_0_7485.jpg,fork,unsafe,,fork_25,203,train,train
fork_25_0_7595.jpg,fork,unsafe,,fork_25,203,train,train
fork_25_0_7711.jpg,fork,unsafe,,fork_25,203,train,train
fork_25_0_8592.jpg,fork,unsafe,,fork_25,203,train,train
fork_26_0_3577.jpg,fork,unsafe,,fork_26,204,test,train
fork_26_0_4089.jpg,fork,unsafe,,fork_26,204,test,train
fork_26_0_5870.jpg,fork,unsafe,,fork_26,204,test,test
fork_26_0_6039.jpg,fork,unsafe,,fork_26,204,test,train
fork_26_0_8595.jpg,fork,unsafe,,fork_26,204,test,train
fork_27_0_1524.jpg,fork,unsafe,,fork_27,205,val,train
fork_27_0_2351.jpg,fork,unsafe,,fork_27,205,val,train
fork_27_0_2397.jpg,fork,unsafe,,fork_27,205,val,train
fork_27_0_7104.jpg,fork,unsafe,,fork_27,205,val,test
fork_27_0_9535.jpg,fork,unsafe,,fork_27,205,val,train
fork_28_0_1343.jpg,fork,unsafe,,fork_28,206,train,train
fork_28_0_387.jpg,fork,unsafe,,fork_28,206,train,train
fork_28_0_4648.jpg,fork,unsafe,,fork_28,206,train,train
fork_28_0_5111.jpg,fork,unsafe,,fork_28,206,train,train
fork_28_0_8654.jpg,fork,unsafe,,fork_28,206,train,train
fork_29_0_1252.jpg,fork,unsafe,,fork_29,207,train,test
fork_29_0_8349.jpg,fork,unsafe,,fork_29,207,train,train
fork_29_0_9046.jpg,fork,unsafe,,fork_29,207,train,val
fork_29_0_9553.jpg,fork,unsafe,,fork_29,207,train,train
fork_29_0_98.jpg,fork,unsafe,,fork_29,207,train,train
fork_2_0_3655.jpg,fork,unsafe,,fork_2,208,train,val
fork_2_0_5898.jpg,fork,unsafe,,fork_2,208,train,val
fork_2_0_7194.jpg,fork,unsafe,,fork_2,208,train,train
fork_2_0_961.jpg,fork,unsafe,,fork_2,208,train,train
fork_2_0_9823.jpg,fork,unsafe,,fork_2,208,train,train
fork_30_0_3586.jpg,fork,unsafe,,fork_30,209,val,train
fork_30_0_4140.jpg,fork,unsafe,,fork_30,209,val,train
fork_30_0_5484.jpg,fork,unsafe,,fork_30,209,val,train
fork_30_0_8482.jpg,fork,unsafe,,fork_30,209,val,train
fork_30_0_9556.jpg,fork,unsafe,,fork_30,209,val,train
fork_31_0_5179.jpg,fork,unsafe,,fork_31,210,test,val
fork_31_0_5253.jpg,fork,unsafe,,fork_31,210,test,val
fork_31_0_5496.jpg,fork,unsafe,,fork_31,210,test,train
fork_31_0_8185.jpg,fork,unsafe,,fork_31,210,test,train
fork_31_0_8897.jpg,fork,unsafe,,fork_31,210,test,val
fork_32_0_192.jpg,fork,unsafe,,fork_32,211,val,train
fork_32_0_3662.jpg,fork,unsafe,,fork_32,211,val,val
fork_32_0_5538.jpg,fork,unsafe,,fork_32,211,val,test
fork_32_0_712.jpg,fork,unsafe,,fork_32,211,val,train
fork_32_0_9930.jpg,fork,unsafe,,fork_32,211,val,train
fork_33_0_3461.jpg,fork,unsafe,,fork_33,212,val,test
fork_33_0_3911.jpg,fork,unsafe,,fork_33,212,val,train
fork_33_0_6026.jpg,fork,unsafe,,fork_33,212,val,test
fork_33_0_7761.jpg,fork,unsafe,,fork_33,212,val,val
fork_33_0_8012.jpg,fork,unsafe,,fork_33,212,val,train
fork_34_0_1926.jpg,fork,unsafe,,fork_34,213,train,val
fork_34_0_196.jpg,fork,unsafe,,fork_34,213,train,test
fork_34_0_2923.jpg,fork,unsafe,,fork_34,213,train,train
fork_34_0_3767.jpg,fork,unsafe,,fork_34,213,train,train
fork_34_0_5573.jpg,fork,unsafe,,fork_34,213,train,train
fork_35_0_4625.jpg,fork,unsafe,,fork_35,214,train,train
fork_35_0_569.jpg,fork,unsafe,,fork_35,214,train,train
fork_35_0_6605.jpg,fork,unsafe,,fork_35,214,train,train
fork_35_0_7299.jpg,fork,unsafe,,fork_35,214,train,val
fork_35_0_8346.jpg,fork,unsafe,,fork_35,214,train,train
fork_36_0_3838.jpg,fork,unsafe,,fork_36,215,val,train
fork_36_0_6226.jpg,fork,unsafe,,fork_36,215,val,train
fork_36_0_6462.jpg,fork,unsafe,,fork_36,215,val,train
fork_36_0_6698.jpg,fork,unsafe,,fork_36,215,val,train
fork_36_0_8811.jpg,fork,unsafe,,fork_36,215,val,train
fork_37_0_1981.jpg,fork,unsafe,,fork_37,153,train,train
fork_37_0_2940.jpg,fork,unsafe,,fork_37,153,train,train
fork_37_0_3413.jpg,fork,unsafe,,fork_37,153,train,train
fork_37_0_874.jpg,fork,unsafe,,fork_37,153,train,train
fork_37_0_9785.jpg,fork,unsafe,,fork_37,153,train,test
fork_38_0_6839.jpg,fork,unsafe,,fork_38,216,train,train
fork_38_0_7052.jpg,fork,unsafe,,fork_38,216,train,train
fork_38_0_7122.jpg,fork,unsafe,,fork_38,216,train,train
fork_38_0_7632.jpg,fork,unsafe,,fork_38,216,train,train
fork_38_0_9133.jpg,fork,unsafe,,fork_38,216,train,test
fork_39_0_3000.jpg,fork,unsafe,,fork_39,217,train,train
fork_39_0_3452.jpg,fork,unsafe,,fork_39,217,train,test
fork_39_0_5623.jpg,fork,unsafe,,fork_39,217,train,val
fork_39_0_604.jpg,fork,unsafe,,fork_39,217,train,train
fork_39_0_9181.jpg,fork,unsafe,,fork_39,217,train,train
fork_3_0_4331.jpg,fork,unsafe,,fork_3,218,train,train
fork_3_0_6387.jpg,fork,unsafe,,fork_3,218,train,train
fork_3_0_719.jpg,fork,unsafe,,fork_3,218,train,train
fork_3_0_8342.jpg,fork,unsafe,,fork_3,218,train,train
fork_3_0_8749.jpg,fork,unsafe,,fork_3,218,train,train
fork_40_0_7829.jpg,fork,unsafe,,fork_40,219,train,test
fork_40_0_8557.jpg,fork,unsafe,,fork_40,219,train,train
fork_40_0_94.jpg,fork,unsafe,,fork_40,219,train,train
fork_40_0_9489.jpg,fork,unsafe,,fork_40,219,train,val
fork_40_0_9811.jpg,fork,unsafe,,fork_40,219,train,test
fork_41_0_3080.jpg,fork,unsafe,,fork_41,220,train,val
fork_41_0_4628.jpg,fork,unsafe,,fork_41,220,train,train
fork_41_0_5679.jpg,fork,unsafe,,fork_41,220,train,train
fork_41_0_8775.jpg,fork,unsafe,,fork_41,220,train,val
fork_41_0_9413.jpg,fork,unsafe,,fork_41,220,train,train
fork_42_0_676.jpg,fork,unsafe,,fork_42,221,train,val
fork_42_0_7864.jpg,fork,unsafe,,fork_42,221,train,train
fork_42_0_8193.jpg,fork,unsafe,,fork_42,221,train,val
fork_42_0_8207.jpg,fork,unsafe,,fork_42,221,train,train
fork_42_0_9276.jpg,fork,unsafe,,fork_42,221,train,test
fork_43_0_1851.jpg,fork,unsafe,,fork_43,222,train,train
fork_43_0_2504.jpg,fork,unsafe,,fork_43,222,train,train
fork_43_0_3468.jpg,fork,unsafe,,fork_43,222,train,test
fork_43_0_5226.jpg,fork,unsafe,,fork_43,222,train,train
fork_43_0_5947.jpg,fork,unsafe,,fork_43,222,train,test
fork_44_0_1173.jpg,fork,unsafe,,fork_44,223,train,train
fork_44_0_2077.jpg,fork,unsafe,,fork_44,223,train,train
fork_44_0_478.jpg,fork,unsafe,,fork_44,223,train,train
fork_44_0_6620.jpg,fork,unsafe,,fork_44,223,train,train
fork_44_0_9597.jpg,fork,unsafe,,fork_44,223,train,train
fork_45_0_1943.jpg,fork,unsafe,,fork_45,224,train,train
fork_45_0_2258.jpg,fork,unsafe,,fork_45,224,train,train
fork_45_0_4355.jpg,fork,unsafe,,fork_45,224,train,train
fork_45_0_6838.jpg,fork,unsafe,,fork_45,224,train,test
fork_45_0_891.jpg,fork,unsafe,,fork_45,224,train,train
fork_46_0_1293.jpg,fork,unsafe,,fork_46,225,val,train
fork_46_0_2639.jpg,fork,unsafe,,fork_46,225,val,val
fork_46_0_3772.jpg,fork,unsafe,,fork_46,225,val,val
fork_46_0_6118.jpg,fork,unsafe,,fork_46,225,val,train
fork_46_0_7894.jpg,fork,unsafe,,fork_46,225,val,train
fork_47_0_1516.jpg,fork,unsafe,,fork_47,226,train,train
fork_47_0_1679.jpg,fork,unsafe,,fork_47,226,train,train
fork_47_0_2724.jpg,fork,unsafe,,fork_47,226,train,train
fork_47_0_3295.jpg,fork,unsafe,,fork_47,226,train,train
fork_47_0_9136.jpg,fork,unsafe,,fork_47,226,train,train
fork_48_0_2561.jpg,fork,unsafe,,fork_48,227,val,test
fork_48_0_3009.jpg,fork,unsafe,,fork_48,227,val,train
fork_48_0_5036.jpg,fork,unsafe,,fork_48,227,val,train
fork_48_0_5334.jpg,fork,unsafe,,fork_48,227,val,test
fork_48_0_8159.jpg,fork,unsafe,,fork_48,227,val,train
fork_49_0_4285.jpg,fork,unsafe,,fork_49,228,test,train
fork_49_0_4883.jpg,fork,unsafe,,fork_49,228,test,val
fork_49_0_6553.jpg,fork,unsafe,,fork_49,228,test,train
fork_49_0_69.jpg,fork,unsafe,,fork_49,228,test,train
fork_49_0_9913.jpg,fork,unsafe,,fork_49,228,test,train
fork_4_0_37.jpg,fork,unsafe,,fork_4,229,train,val
fork_4_0_3768.jpg,fork,unsafe,,fork_4,229,train,train
fork_4_0_5508.jpg,fork,unsafe,,fork_4,229,train,train
fork_4_0_7782.jpg,fork,unsafe,,fork_4,229,train,train
fork_4_0_8476.jpg,fork,unsafe,,fork_4,229,train,train
fork_50_0_3234.jpg,fork,unsafe,,fork_50,230,train,test
fork_50_0_419.jpg,fork,unsafe,,fork_50,230,train,train
fork_50_0_8487.jpg,fork,unsafe,,fork_50,230,train,test
fork_50_0_9454.jpg,fork,unsafe,,fork_50,230,train,train
fork_50_0_9785.jpg,fork,unsafe,,fork_50,230,train,test
fork_51_0_4240.jpg,fork,unsafe,,fork_51,231,train,train
fork_51_0_4677.jpg,fork,unsafe,,fork_51,231,train,train
fork_51_0_5012.jpg,fork,unsafe,,fork_51,231,train,train
fork_51_0_8598.jpg,fork,unsafe,,fork_51,231,train,train
fork_51_0_9917.jpg,fork,unsafe,,fork_51,231,train,val
fork_52_0_1231.jpg,fork,unsafe,,fork_52,232,train,train
fork_52_0_440.jpg,fork,unsafe,,fork_52,232,train,train
fork_52_0_7146.jpg,fork,unsafe,,fork_52,232,train,train
fork_52_0_7175.jpg,fork,unsafe,,fork_52,232,train,train
fork_52_0_9178.jpg,fork,unsafe,,fork_52,232,train,train
fork_53_0_3711.jpg,fork,unsafe,,fork_53,233,train,train
fork_53_0_4456.jpg,fork,unsafe,,fork_53,233,train,val
fork_53_0_553.jpg,fork,unsafe,,fork_53,233,train,train
fork_53_0_7414.jpg,fork,unsafe,,fork_53,233,train,train
fork_53_0_7689.jpg,fork,unsafe,,fork_53,233,train,train
fork_54_0_2650.jpg,fork,unsafe,,fork_54,234,train,val
fork_54_0_2751.jpg,fork,unsafe,,fork_54,234,train,test
fork_54_0_2966.jpg,fork,unsafe,,fork_54,234,train,train
fork_54_0_3752.jpg,fork,unsafe,,fork_54,234,train,val
fork_54_0_4006.jpg,fork,unsafe,,fork_54,234,train,train
fork_55_0_2245.jpg,fork,unsafe,,fork_55,235,val,train
fork_55_0_4665.jpg,fork,unsafe,,fork_55,235,val,train
fork_55_0_5387.jpg,fork,unsafe,,fork_55,235,val,train
fork_55_0_674.jpg,fork,unsafe,,fork_55,235,val,val
fork_55_0_8581.jpg,fork,unsafe,,fork_55,235,val,train
fork_56_0_3201.jpg,fork,unsafe,,fork_56,236,train,train
fork_56_0_3555.jpg,fork,unsafe,,fork_56,236,train,train
fork_56_0_4111.jpg,fork,unsafe,,fork_56,236,train,val
fork_56_0_717.jpg,fork,unsafe,,fork_56,236,train,val
fork_56_0_7869.jpg,fork,unsafe,,fork_56,236,train,test
fork_57_0_6062.jpg,fork,unsafe,,fork_57,237,train,val
fork_57_0_6477.jpg,fork,unsafe,,fork_57,237,train,val
fork_57_0_7398.jpg,fork,unsafe,,fork_57,237,train,train
fork_57_0_7827.jpg,fork,unsafe,,fork_57,237,train,train
fork_57_0_7852.jpg,fork,unsafe,,fork_57,237,train,train
fork_58_0_2098.jpg,fork,unsafe,,fork_58,238,train,train
fork_58_0_4781.jpg,fork,unsafe,,fork_58,238,train,train
fork_58_0_657.jpg,fork,unsafe,,fork_58,238,train,train
fork_58_0_8014.jpg,fork,unsafe,,fork_58,238,train,train
fork_58_0_9527.jpg,fork,unsafe,,fork_58,238,train,train
fork_59_0_3088.jpg,fork,unsafe,,fork_59,239,train,train
fork_59_0_670.jpg,fork,unsafe,,fork_59,239,train,test
fork_59_0_6923.jpg,fork,unsafe,,fork_59,239,train,train
fork_59_0_7425.jpg,fork,unsafe,,fork_59,239,train,train
fork_59_0_8096.jpg,fork,unsafe,,fork_59,239,train,test
fork_5_0_4700.jpg,fork,unsafe,,fork_5,240,train,train
fork_5_0_6241.jpg,fork,unsafe,,fork_5,240,train,train
fork_5_0_649.jpg,fork,unsafe,,fork_5,240,train,train
fork_5_0_7688.jpg,fork,unsafe,,fork_5,240,train,train
fork_5_0_9781.jpg,fork,unsafe,,fork_5,240,train,train
fork_60_0_1999.jpg,fork,unsafe,,fork_60,241,train,train
fork_60_0_2684.jpg,fork,unsafe,,fork_60,241,train,train
fork_60_0_3757.jpg,fork,unsafe,,fork_60,241,train,train
fork_60_0_4970.jpg,fork,unsafe,,fork_60,241,train,train
fork_60_0_5298.jpg,fork,unsafe,,fork_60,241,train,train
fork_61_0_2738.jpg,fork,unsafe,,fork_61,242,val,train
fork_61_0_2759.jpg,fork,unsafe,,fork_61,242,val,train
fork_61_0_2833.jpg,fork,unsafe,,fork_61,242,val,train
fork_61_0_8170.jpg,fork,unsafe,,fork_61,242,val,train
fork_61_0_8677.jpg,fork,unsafe,,fork_61,242,val,train
fork_62_0_288.jpg,fork,unsafe,,fork_62,243,train,test
fork_62_0_6091.jpg,fork,unsafe,,fork_62,243,train,val
fork_62_0_625.jpg,fork,unsafe,,fork_62,243,train,train
fork_62_0_6882.jpg,fork,unsafe,,fork_62,243,train,train
fork_62_0_819.jpg,fork,unsafe,,fork_62,243,train,train
fork_63_0_1760.jpg,fork,unsafe,,fork_63,244,train,train
fork_63_0_4298.jpg,fork,unsafe,,fork_63,244,train,train
fork_63_0_4752.jpg,fork,unsafe,,fork_63,244,train,train
fork_63_0_633.jpg,fork,unsafe,,fork_63,244,train,test
fork_63_0_7673.jpg,fork,unsafe,,fork_63,244,train,train
fork_64_0_3383.jpg,fork,unsafe,,fork_64,245,train,val
fork_64_0_6279.jpg,fork,unsafe,,fork_64,245,train,train
fork_64_0_6418.jpg,fork,unsafe,,fork_64,245,train,train
fork_64_0_8903.jpg,fork,unsafe,,fork_64,245,train,train
fork_64_0_92.jpg,fork,unsafe,,fork_64,245,train,train
fork_65_0_3166.jpg,fork,unsafe,,fork_65,241,train,test
fork_65_0_5385.jpg,fork,unsafe,,fork_65,241,train,train
fork_65_0_6761.jpg,fork,unsafe,,fork_65,241,train,train
fork_65_0_8330.jpg,fork,unsafe,,fork_65,241,train,train
fork_65_0_9807.jpg,fork,unsafe,,fork_65,241,train,train
fork_66_0_1687.jpg,fork,unsafe,,fork_66,246,val,train
fork_66_0_315.jpg,fork,unsafe,,fork_66,246,val,val
fork_66_0_3906.jpg,fork,unsafe,,fork_66,246,val,train
fork_66_0_4953.jpg,fork,unsafe,,fork_66,246,val,train
fork_66_0_918.jpg,fork,unsafe,,fork_66,246,val,train
fork_67_0_1562.jpg,fork,unsafe,,fork_67,247,test,train
fork_67_0_342.jpg,fork,unsafe,,fork_67,247,test,train
fork_67_0_3842.jpg,fork,unsafe,,fork_67,247,test,train
fork_67_0_4847.jpg,fork,unsafe,,fork_67,247,test,train
fork_67_0_9506.jpg,fork,unsafe,,fork_67,247,test,val
fork_68_0_1172.jpg,fork,unsafe,,fork_68,248,train,train
fork_68_0_2249.jpg,fork,unsafe,,fork_68,248,train,train
fork_68_0_5802.jpg,fork,unsafe,,fork_68,248,train,train
fork_68_0_7813.jpg,fork,unsafe,,fork_68,248,train,train
fork_68_0_8674.jpg,fork,unsafe,,fork_68,248,train,train
fork_69_0_158.jpg,fork,unsafe,,fork_69,249,train,train
fork_69_0_2803.jpg,fork,unsafe,,fork_69,249,train,train
fork_69_0_4173.jpg,fork,unsafe,,fork_69,249,train,train
fork_69_0_8165.jpg,fork,unsafe,,fork_69,249,train,train
fork_69_0_8735.jpg,fork,unsafe,,fork_69,249,train,train
fork_6_0_5117.jpg,fork,unsafe,,fork_6,250,train,test
fork_6_0_600.jpg,fork,unsafe,,fork_6,250,train,train
fork_6_0_7206.jpg,fork,unsafe,,fork_6,250,train,test
fork_6_0_7793.jpg,fork,unsafe,,fork_6,250,train,train
fork_6_0_9734.jpg,fork,unsafe,,fork_6,250,train,train
fork_70_0_1228.jpg,fork,unsafe,,fork_70,251,train,train
fork_70_0_3648.jpg,fork,unsafe,,fork_70,251,train,train
fork_70_0_4641.jpg,fork,unsafe,,fork_70,251,train,test
fork_70_0_5027.jpg,fork,unsafe,,fork_70,251,train,val
fork_70_0_8652.jpg,fork,unsafe,,fork_70,251,train,train
fork_72_0_3174.jpg,fork,unsafe,,fork_72,252,train,val
fork_72_0_5386.jpg,fork,unsafe,,fork_72,252,train,train
fork_72_0_5565.jpg,fork,unsafe,,fork_72,252,train,val
fork_72_0_6276.jpg,fork,unsafe,,fork_72,252,train,train
fork_72_0_7426.jpg,fork,unsafe,,fork_72,252,train,test
fork_73_0_2415.jpg,fork,unsafe,,fork_73,253,train,val
fork_73_0_3378.jpg,fork,unsafe,,fork_73,253,train,train
fork_73_0_3814.jpg,fork,unsafe,,fork_73,253,train,train
fork_73_0_6273.jpg,fork,unsafe,,fork_73,253,train,test
fork_73_0_8435.jpg,fork,unsafe,,fork_73,253,train,train
fork_74_0_2879.jpg,fork,unsafe,,fork_74,254,train,test
fork_74_0_5843.jpg,fork,unsafe,,fork_74,254,train,val
fork_74_0_6702.jpg,fork,unsafe,,fork_74,254,train,train
fork_74_0_9876.jpg,fork,unsafe,,fork_74,254,train,train
fork_74_0_9912.jpg,fork,unsafe,,fork_74,254,train,train
fork_75_0_2410.jpg,fork,unsafe,,fork_75,255,train,train
fork_75_0_3281.jpg,fork,unsafe,,fork_75,255,train,train
fork_75_0_6281.jpg,fork,unsafe,,fork_75,255,train,train
fork_75_0_7318.jpg,fork,unsafe,,fork_75,255,train,test
fork_75_0_943.jpg,fork,unsafe,,fork_75,255,train,train
fork_76_0_3894.jpg,fork,unsafe,,fork_76,256,test,train
fork_76_0_7198.jpg,fork,unsafe,,fork_76,256,test,train
fork_76_0_7777.jpg,fork,unsafe,,fork_76,256,test,train
fork_76_0_8314.jpg,fork,unsafe,,fork_76,256,test,train
fork_76_0_9322.jpg,fork,unsafe,,fork_76,256,test,train
fork_77_0_2691.jpg,fork,unsafe,,fork_77,257,train,train
fork_77_0_5767.jpg,fork,unsafe,,fork_77,257,train,test
fork_77_0_6245.jpg,fork,unsafe,,fork_77,257,train,train
fork_77_0_715.jpg,fork,unsafe,,fork_77,257,train,train
fork_77_0_773.jpg,fork,unsafe,,fork_77,257,train,val
fork_78_0_302.jpg,fork,unsafe,,fork_78,258,val,test
fork_78_0_3804.jpg,fork,unsafe,,fork_78,258,val,val
fork_78_0_4698.jpg,fork,unsafe,,fork_78,258,val,train
fork_78_0_7436.jpg,fork,unsafe,,fork_78,258,val,train
fork_78_0_9268.jpg,fork,unsafe,,fork_78,258,val,train
fork_79_0_3760.jpg,fork,unsafe,,fork_79,259,train,train
fork_79_0_539.jpg,fork,unsafe,,fork_79,259,train,train
fork_79_0_6712.jpg,fork,unsafe,,fork_79,259,train,train
fork_79_0_6795.jpg,fork,unsafe,,fork_79,259,train,train
fork_79_0_93.jpg,fork,unsafe,,fork_79,259,train,train
fork_7_0_102.jpg,fork,unsafe,,fork_7,260,train,train
fork_7_0_2628.jpg,fork,unsafe,,fork_7,260,train,train
fork_7_0_3427.jpg,fork,unsafe,,fork_7,260,train,val
fork_7_0_3647.jpg,fork,unsafe,,fork_7,260,train,train
fork_7_0_48.jpg,fork,unsafe,,fork_7,260,train,train
fork_80_0_2793.jpg,fork,unsafe,,fork_80,261,test,train
fork_80_0_3608.jpg,fork,unsafe,,fork_80,261,test,train
fork_80_0_4256.jpg,fork,unsafe,,fork_80,261,test,test
fork_80_0_6372.jpg,fork,unsafe,,fork_80,261,test,train
fork_80_0_7332.jpg,fork,unsafe,,fork_80,261,test,val
fork_81_0_185.jpg,fork,unsafe,,fork_81,262,train,train
fork_81_0_4734.jpg,fork,unsafe,,fork_81,262,train,val
fork_81_0_5633.jpg,fork,unsafe,,fork_81,262,train,train
fork_81_0_5666.jpg,fork,unsafe,,fork_81,262,train,train
fork_81_0_8365.jpg,fork,unsafe,,fork_81,262,train,train
fork_82_0_1474.jpg,fork,unsafe,,fork_82,263,train,train
fork_82_0_1645.jpg,fork,unsafe,,fork_82,263,train,train
fork_82_0_3346.jpg,fork,unsafe,,fork_82,263,train,test
fork_82_0_6704.jpg,fork,unsafe,,fork_82,263,train,train
fork_82_0_8672.jpg,fork,unsafe,,fork_82,263,train,train
fork_83_0_1689.jpg,fork,unsafe,,fork_83,163,train,train
fork_83_0_3758.jpg,fork,unsafe,,fork_83,163,train,train
fork_83_0_4758.jpg,fork,unsafe,,fork_83,163,train,test
fork_83_0_6864.jpg,fork,unsafe,,fork_83,163,train,train
fork_83_0_8977.jpg,fork,unsafe,,fork_83,163,train,train
fork_84_0_1050.jpg,fork,unsafe,,fork_84,163,train,train
fork_84_0_2877.jpg,fork,unsafe,,fork_84,163,train,train
fork_84_0_3664.jpg,fork,unsafe,,fork_84,163,train,train
fork_84_0_6735.jpg,fork,unsafe,,fork_84,163,train,train
fork_84_0_9462.jpg,fork,unsafe,,fork_84,163,train,test
fork_85_0_115.jpg,fork,unsafe,,fork_85,264,train,train
fork_85_0_164.jpg,fork,unsafe,,fork_85,264,train,val
fork_85_0_3826.jpg,fork,unsafe,,fork_85,264,train,train
fork_85_0_6704.jpg,fork,unsafe,,fork_85,264,train,train
fork_85_0_6727.jpg,fork,unsafe,,fork_85,264,train,train
fork_86_0_139.jpg,fork,unsafe,,fork_86,265,val,train
fork_86_0_2800.jpg,fork,unsafe,,fork_86,265,val,train
fork_86_0_3395.jpg,fork,unsafe,,fork_86,265,val,val
fork_86_0_4834.jpg,fork,unsafe,,fork_86,265,val,train
fork_86_0_5792.jpg,fork,unsafe,,fork_86,265,val,train
fork_87_0_2678.jpg,fork,unsafe,,fork_87,186,train,train
fork_87_0_4833.jpg,fork,unsafe,,fork_87,186,train,test
fork_87_0_6657.jpg,fork,unsafe,,fork_87,186,train,train
fork_87_0_7373.jpg,fork,unsafe,,fork_87,186,train,val
fork_87_0_9885.jpg,fork,unsafe,,fork_87,186,train,train
fork_88_0_3438.jpg,fork,unsafe,,fork_88,266,test,train
fork_88_0_3659.jpg,fork,unsafe,,fork_88,266,test,train
fork_88_0_7419.jpg,fork,unsafe,,fork_88,266,test,val
fork_88_0_8118.jpg,fork,unsafe,,fork_88,266,test,test
fork_88_0_8251.jpg,fork,unsafe,,fork_88,266,test,train
fork_89_0_5906.jpg,fork,unsafe,,fork_89,267,train,val
fork_89_0_621.jpg,fork,unsafe,,fork_89,267,train,train
fork_89_0_6827.jpg,fork,unsafe,,fork_89,267,train,train
fork_89_0_7398.jpg,fork,unsafe,,fork_89,267,train,train
fork_89_0_7919.jpg,fork,unsafe,,fork_89,267,train,val
fork_8_0_1535.jpg,fork,unsafe,,fork_8,268,train,train
fork_8_0_4425.jpg,fork,unsafe,,fork_8,268,train,train
fork_8_0_6192.jpg,fork,unsafe,,fork_8,268,train,test
fork_8_0_8283.jpg,fork,unsafe,,fork_8,268,train,train
fork_8_0_8918.jpg,fork,unsafe,,fork_8,268,train,train
fork_90_0_2307.jpg,fork,unsafe,,fork_90,269,train,train
fork_90_0_4633.jpg,fork,unsafe,,fork_90,269,train,val
fork_90_0_5078.jpg,fork,unsafe,,fork_90,269,train,train
fork_90_0_6491.jpg,fork,unsafe,,fork_90,269,train,train
fork_90_0_7731.jpg,fork,unsafe,,fork_90,269,train,train
fork_91_0_1626.jpg,fork,unsafe,,fork_91,270,val,test
fork_91_0_2550.jpg,fork,unsafe,,fork_91,270,val,val
fork_91_0_408.jpg,fork,unsafe,,fork_91,270,val,train
fork_91_0_4881.jpg,fork,unsafe,,fork_91,270,val,test
fork_91_0_9632.jpg,fork,unsafe,,fork_91,270,val,train
fork_92_0_1595.jpg,fork,unsafe,,fork_92,271,val,train
fork_92_0_4481.jpg,fork,unsafe,,fork_92,271,val,val
fork_92_0_5722.jpg,fork,unsafe,,fork_92,271,val,train
fork_92_0_7119.jpg,fork,unsafe,,fork_92,271,val,train
fork_92_0_8178.jpg,fork,unsafe,,fork_92,271,val,val
fork_93_0_2515.jpg,fork,unsafe,,fork_93,272,train,train
fork_93_0_4688.jpg,fork,unsafe,,fork_93,272,train,train
fork_93_0_5885.jpg,fork,unsafe,,fork_93,272,train,train
fork_93_0_6295.jpg,fork,unsafe,,fork_93,272,train,test
fork_93_0_7567.jpg,fork,unsafe,,fork_93,272,train,train
fork_94_0_2793.jpg,fork,unsafe,,fork_94,248,train,test
fork_94_0_3302.jpg,fork,unsafe,,fork_94,248,train,train
fork_94_0_4626.jpg,fork,unsafe,,fork_94,248,train,val
fork_94_0_5262.jpg,fork,unsafe,,fork_94,248,train,val
fork_94_0_7468.jpg,fork,unsafe,,fork_94,248,train,train
fork_95_0_1402.jpg,fork,unsafe,,fork_95,273,train,train
fork_95_0_1843.jpg,fork,unsafe,,fork_95,273,train,train
fork_95_0_4021.jpg,fork,unsafe,,fork_95,273,train,val
fork_95_0_8557.jpg,fork,unsafe,,fork_95,273,train,train
fork_95_0_9973.jpg,fork,unsafe,,fork_95,273,train,train
fork_96_0_211.jpg,fork,unsafe,,fork_96,274,val,val
fork_96_0_3091.jpg,fork,unsafe,,fork_96,274,val,train
fork_96_0_6588.jpg,fork,unsafe,,fork_96,274,val,train
fork_96_0_7065.jpg,fork,unsafe,,fork_96,274,val,test
fork_96_0_9140.jpg,fork,unsafe,,fork_96,274,val,train
fork_97_0_3864.jpg,fork,unsafe,,fork_97,275,train,train
fork_97_0_5146.jpg,fork,unsafe,,fork_97,275,train,train
fork_97_0_5448.jpg,fork,unsafe,,fork_97,275,train,test
fork_97_0_637.jpg,fork,unsafe,,fork_97,275,train,train
fork_97_0_803.jpg,fork,unsafe,,fork_97,275,train,train
fork_98_0_2080.jpg,fork,unsafe,,fork_98,276,train,train
fork_98_0_2907.jpg,fork,unsafe,,fork_98,276,train,train
fork_98_0_480.jpg,fork,unsafe,,fork_98,276,train,test
fork_98_0_562.jpg,fork,unsafe,,fork_98,276,train,train
fork_98_0_8456.jpg,fork,unsafe,,fork_98,276,train,train
fork_99_0_2828.jpg,fork,unsafe,,fork_99,163,train,train
fork_99_0_284.jpg,fork,unsafe,,fork_99,163,train,train
fork_99_0_31.jpg,fork,unsafe,,fork_99,163,train,test
fork_99_0_3408.jpg,fork,unsafe,,fork_99,163,train,train
fork_99_0_5390.jpg,fork,unsafe,,fork_99,163,train,test
fork_9_0_3006.jpg,fork,unsafe,,fork_9,277,val,train
fork_9_0_3441.jpg,fork,unsafe,,fork_9,277,val,val
fork_9_0_4865.jpg,fork,unsafe,,fork_9,277,val,train
fork_9_0_8343.jpg,fork,unsafe,,fork_9,277,val,val
fork_9_0_8670.jpg,fork,unsafe,,fork_9,277,val,test
fruits_1.jpg,fruit,safe,,fruits_1.jpg,278,train,train
fruits_10.jpg,fruit,safe,,fruits_10.jpg,279,train,val
fruits_100.jpg,fruit,safe,,fruits_100.jpg,280,val,test
fruits_101.jpg,fruit,safe,,fruits_101.jpg,281,train,test
fruits_102.jpg,fruit,safe,,fruits_102.jpg,282,val,train
fruits_103.jpg,fruit,safe,,fruits_103.jpg,283,train,val
fruits_104.jpg,fruit,safe,,fruits_104.jpg,284,train,train
fruits_105.jpg,fruit,safe,,fruits_105.jpg,285,train,train
fruits_106.jpg,fruit,safe,,fruits_106.jpg,286,val,train
fruits_107.jpg,fruit,safe,,fruits_107.jpg,287,train,val
fruits_108.jpg,fruit,safe,,fruits_108.jpg,288,train,test
fruits_109.jpg,fruit,safe,,fruits_109.jpg,288,train,train
fruits_11.jpg,fruit,safe,,fruits_11.jpg,289,train,train
fruits_110.jpg,fruit,safe,,fruits_110.jpg,290,val,train
fruits_111.jpg,fruit,safe,,fruits_111.jpg,291,train,test
fruits_112.jpg,fruit,safe,,fruits_112.jpg,292,train,test
fruits_113.jpg,fruit,safe,,fruits_113.jpg,293,test,train
fruits_114.jpg,fruit,safe,,fruits_114.jpg,294,test,train
fruits_115.jpg,fruit,safe,,fruits_115.jpg,295,test,train
fruits_116.jpg,fruit,safe,,fruits_116.jpg,296,val,train
fruits_117.jpg,fruit,safe,,fruits_117.jpg,297,train,train
fruits_118.jpg,fruit,safe,,fruits_118.jpg,298,train,train
fruits_119.jpg,fruit,safe,,fruits_119.jpg,299,train,train
fruits_12.jpg,fruit,safe,,fruits_12.jpg,300,train,train
fruits_120.jpg,fruit,safe,,fruits_120.jpg,301,val,train
fruits_121.jpg,fruit,safe,,fruits_121.jpg,302,train,train
fruits_122.jpg,fruit,safe,,fruits_122.jpg,303,val,val
fruits_123.jpg,fruit,safe,,fruits_123.jpg,304,train,train
fruits_124.jpg,fruit,safe,,fruits_124.jpg,305,train,test
fruits_125.jpg,fruit,safe,,fruits_125.jpg,306,train,train
fruits_126.jpg,fruit,safe,,fruits_126.jpg,307,train,train
fruits_127.jpg,fruit,safe,,fruits_127.jpg,308,val,val
fruits_128.jpg,fruit,safe,,fruits_128.jpg,309,train,train
fruits_129.jpg,fruit,safe,,fruits_129.jpg,310,train,test
fruits_13.jpg,fruit,safe,,fruits_13.jpg,311,train,train
fruits_130.jpg,fruit,safe,,fruits_130.jpg,312,train,train
fruits_131.jpg,fruit,safe,,fruits_131.jpg,313,train,train
fruits_132.jpg,fruit,safe,,fruits_132.jpg,313,train,train
fruits_133.jpg,fruit,safe,,fruits_133.jpg,314,test,test
fruits_134.jpg,fruit,safe,,fruits_134.jpg,315,train,train
fruits_135.jpg,fruit,safe,,fruits_135.jpg,316,val,val
fruits_136.jpg,fruit,safe,,fruits_136.jpg,317,train,train
fruits_137.jpg,fruit,safe,,fruits_137.jpg,318,val,train
fruits_138.jpg,fruit,safe,,fruits_138.jpg,319,train,train
fruits_139.jpg,fruit,safe,,fruits_139.jpg,320,train,train
fruits_14.jpg,fruit,safe,,fruits_14.jpg,311,train,train
fruits_140.jpg,fruit,safe,,fruits_140.jpg,321,train,train
fruits_141.jpg,fruit,safe,,fruits_141.jpg,322,train,train
fruits_142.jpg,fruit,safe,,fruits_142.jpg,323,train,train
fruits_143.jpg,fruit,safe,,fruits_143.jpg,324,train,train
fruits_144.jpg,fruit,safe,,fruits_144.jpg,325,test,train
fruits_145.jpg,fruit,safe,,fruits_145.jpg,326,train,train
fruits_146.jpg,fruit,safe,,fruits_146.jpg,327,train,train
fruits_147.jpg,fruit,safe,,fruits_147.jpg,328,train,val
fruits_148.jpg,fruit,safe,,fruits_148.jpg,329,test,train
fruits_149.jpg,fruit,safe,,fruits_149.jpg,330,test,train
fruits_15.jpg,fruit,safe,,fruits_15.jpg,331,train,train
fruits_150.jpg,fruit,safe,,fruits_150.jpg,332,train,train
fruits_151.jpg,fruit,safe,,fruits_151.jpg,333,train,train
fruits_152.jpg,fruit,safe,,fruits_152.jpg,334,train,train
fruits_153.jpg,fruit,safe,,fruits_153.jpg,335,train,test
fruits_154.jpg,fruit,safe,,fruits_154.jpg,336,train,test
fruits_155.jpg,fruit,safe,,fruits_155.jpg,337,train,train
fruits_156.jpg,fruit,safe,,fruits_156.jpg,338,test,val
fruits_157.jpg,fruit,safe,,fruits_157.jpg,339,val,test
fruits_158.jpg,fruit,safe,,fruits_158.jpg,340,train,val
fruits_159.jpg,fruit,safe,,fruits_159.jpg,341,train,train
fruits_16.jpg,fruit,safe,,fruits_16.jpg,342,train,train
fruits_160.jpg,fruit,safe,,fruits_160.jpg,343,train,train
fruits_161.jpg,fruit,safe,,fruits_161.jpg,344,train,train
fruits_162.jpg,fruit,safe,,fruits_162.jpg,345,val,train
fruits_163.jpg,fruit,safe,,fruits_163.jpg,346,train,val
fruits_164.jpg,fruit,safe,,fruits_164.jpg,347,train,test
fruits_165.jpg,fruit,safe,,fruits_165.jpg,348,train,train
fruits_166.jpg,fruit,safe,,fruits_166.jpg,349,train,train
fruits_167.jpg,fruit,safe,,fruits_167.jpg,350,train,train
fruits_168.jpg,fruit,safe,,fruits_168.jpg,351,train,val
fruits_169.jpg,fruit,safe,,fruits_169.jpg,352,train,train
fruits_17.jpg,fruit,safe,,fruits_17.jpg,353,train,train
fruits_170.jpg,fruit,safe,,fruits_170.jpg,354,train,train
fruits_171.jpg,fruit,safe,,fruits_171.jpg,355,val,train
fruits_172.jpg,fruit,safe,,fruits_172.jpg,356,train,train
fruits_173.jpg,fruit,safe,,fruits_173.jpg,357,train,train
fruits_174.jpg,fruit,safe,,fruits_174.jpg,358,test,train
fruits_175.jpg,fruit,safe,,fruits_175.jpg,359,train,train
fruits_176.jpg,fruit,safe,,fruits_176.jpg,360,train,val
fruits_177.jpg,fruit,safe,,fruits_177.jpg,361,train,train
fruits_178.jpg,fruit,safe,,fruits_178.jpg,362,test,train
fruits_179.jpg,fruit,safe,,fruits_179.jpg,363,train,train
fruits_18.jpg,fruit,safe,,fruits_18.jpg,364,val,train
fruits_180.jpg,fruit,safe,,fruits_180.jpg,365,train,val
fruits_181.jpg,fruit,safe,,fruits_181.jpg,366,test,train
fruits_182.jpg,fruit,safe,,fruits_182.jpg,367,train,train
fruits_183.jpg,fruit,safe,,fruits_183.jpg,368,train,train
fruits_184.jpg,fruit,safe,,fruits_184.jpg,369,train,train
fruits_185.jpg,fruit,safe,,fruits_185.jpg,370,train,test
fruits_186.jpg,fruit,safe,,fruits_186.jpg,371,train,train
fruits_187.jpg,fruit,safe,,fruits_187.jpg,372,train,train
fruits_188.jpg,fruit,safe,,fruits_188.jpg,373,val,val
fruits_189.jpg,fruit,safe,,fruits_189.jpg,374,train,val
fruits_19.jpg,fruit,safe,,fruits_19.jpg,375,val,val
fruits_190.jpg,fruit,safe,,fruits_190.jpg,376,test,val
fruits_191.jpg,fruit,safe,,fruits_191.jpg,377,train,train
fruits_192.jpg,fruit,safe,,fruits_192.jpg,378,train,train
fruits_193.jpg,fruit,safe,,fruits_193.jpg,379,train,train
fruits_194.jpg,fruit,safe,,fruits_194.jpg,380,train,train
fruits_195.jpg,fruit,safe,,fruits_195.jpg,381,train,val
fruits_196.jpg,fruit,safe,,fruits_196.jpg,382,val,test
fruits_197.jpg,fruit,safe,,fruits_197.jpg,383,train,val
fruits_198.jpg,fruit,safe,,fruits_198.jpg,384,test,train
fruits_199.jpg,fruit,safe,,fruits_199.jpg,385,train,train
fruits_2.jpg,fruit,safe,,fruits_2.jpg,386,test,train
fruits_20.jpg,fruit,safe,,fruits_20.jpg,387,train,val
fruits_200.jpg,fruit,safe,,fruits_200.jpg,388,train,train
fruits_201.jpg,fruit,safe,,fruits_201.jpg,389,train,train
fruits_202.jpg,fruit,safe,,fruits_202.jpg,390,train,train
fruits_203.jpg,fruit,safe,,fruits_203.jpg,391,val,train
fruits_204.jpg,fruit,safe,,fruits_204.jpg,392,train,train
fruits_205.jpg,fruit,safe,,fruits_205.jpg,393,test,train
fruits_206.jpg,fruit,safe,,fruits_206.jpg,394,train,train
fruits_207.jpg,fruit,safe,,fruits_207.jpg,390,train,train
fruits_208.jpg,fruit,safe,,fruits_208.jpg,395,train,train
fruits_209.jpg,fruit,safe,,fruits_209.jpg,396,train,train
fruits_21.jpg,fruit,safe,,fruits_21.jpg,397,test,train
fruits_210.jpg,fruit,safe,,fruits_210.jpg,398,test,train
fruits_211.jpg,fruit,safe,,fruits_211.jpg,399,val,train
fruits_212.jpg,fruit,safe,,fruits_212.jpg,400,test,val
fruits_213.jpg,fruit,safe,,fruits_213.jpg,401,train,train
fruits_214.jpg,fruit,safe,,fruits_214.jpg,402,test,train
fruits_215.jpg,fruit,safe,,fruits_215.jpg,403,train,train
fruits_216.jpg,fruit,safe,,fruits_216.jpg,404,val,train
fruits_217.jpg,fruit,safe,,fruits_217.jpg,405,train,train
fruits_218.jpg,fruit,safe,,fruits_218.jpg,405,train,train
fruits_219.jpg,fruit,safe,,fruits_219.jpg,406,train,train
fruits_22.jpg,fruit,safe,,fruits_22.jpg,407,train,train
fruits_220.jpg,fruit,safe,,fruits_220.jpg,408,train,train
fruits_221.jpg,fruit,safe,,fruits_221.jpg,409,train,train
fruits_222.jpg,fruit,safe,,fruits_222.jpg,410,val,train
fruits_223.jpg,fruit,safe,,fruits_223.jpg,411,train,train
fruits_224.jpg,fruit,safe,,fruits_224.jpg,412,train,test
fruits_225.jpg,fruit,safe,,fruits_225.jpg,412,train,test
fruits_226.jpg,fruit,safe,,fruits_226.jpg,413,train,test
fruits_227.jpg,fruit,safe,,fruits_227.jpg,414,train,train
fruits_228.jpg,fruit,safe,,fruits_228.jpg,415,train,train
fruits_229.jpg,fruit,safe,,fruits_229.jpg,416,train,train
fruits_23.jpg,fruit,safe,,fruits_23.jpg,417,val,test
fruits_230.jpg,fruit,safe,,fruits_230.jpg,418,train,train
fruits_231.jpg,fruit,safe,,fruits_231.jpg,419,train,train
fruits_232.jpg,fruit,safe,,fruits_232.jpg,420,train,val
fruits_233.jpg,fruit,safe,,fruits_233.jpg,421,train,train
fruits_234.jpg,fruit,safe,,fruits_234.jpg,421,train,train
fruits_235.jpg,fruit,safe,,fruits_235.jpg,422,train,train
fruits_236.jpg,fruit,safe,,fruits_236.jpg,423,train,test
fruits_237.jpg,fruit,safe,,fruits_237.jpg,422,train,train
fruits_238.jpg,fruit,safe,,fruits_238.jpg,423,train,train
fruits_239.jpg,fruit,safe,,fruits_239.jpg,424,val,train
fruits_24.jpg,fruit,safe,,fruits_24.jpg,425,train,train
fruits_240.jpg,fruit,safe,,fruits_240.jpg,426,train,test
fruits_241.jpg,fruit,safe,,fruits_241.jpg,427,train,train
fruits_242.jpg,fruit,safe,,fruits_242.jpg,428,val,train
fruits_243.jpg,fruit,safe,,fruits_243.jpg,429,train,test
fruits_244.jpg,fruit,safe,,fruits_244.jpg,430,test,train
fruits_245.jpg,fruit,safe,,fruits_245.jpg,431,train,train
fruits_246.jpg,fruit,safe,,fruits_246.jpg,432,test,train
fruits_247.jpg,fruit,safe,,fruits_247.jpg,433,train,train
fruits_248.jpg,fruit,safe,,fruits_248.jpg,434,train,train
fruits_249.jpg,fruit,safe,,fruits_249.jpg,435,train,val
fruits_25.jpg,fruit,safe,,fruits_25.jpg,436,train,train
fruits_250.jpg,fruit,safe,,fruits_250.jpg,437,val,train
fruits_251.jpg,fruit,safe,,fruits_251.jpg,438,train,test
fruits_252.jpg,fruit,safe,,fruits_252.jpg,439,train,train
fruits_253.jpg,fruit,safe,,fruits_253.jpg,440,test,val
fruits_254.jpg,fruit,safe,,fruits_254.jpg,441,test,train
fruits_255.jpg,fruit,safe,,fruits_255.jpg,442,train,train
fruits_256.jpg,fruit,safe,,fruits_256.jpg,443,test,train
fruits_257.jpg,fruit,safe,,fruits_257.jpg,444,train,train
fruits_258.jpg,fruit,safe,,fruits_258.jpg,445,train,test
fruits_259.jpg,fruit,safe,,fruits_259.jpg,446,train,train
fruits_26.jpg,fruit,safe,,fruits_26.jpg,447,train,train
fruits_260.jpg,fruit,safe,,fruits_260.jpg,448,train,train
fruits_261.jpg,fruit,safe,,fruits_261.jpg,449,test,val
fruits_262.jpg,fruit,safe,,fruits_262.jpg,450,test,train
fruits_263.jpg,fruit,safe,,fruits_263.jpg,451,val,test
fruits_264.jpg,fruit,safe,,fruits_264.jpg,452,val,train
fruits_265.jpg,fruit,safe,,fruits_265.jpg,453,train,test
fruits_266.jpg,fruit,safe,,fruits_266.jpg,454,val,train
fruits_267.jpg,fruit,safe,,fruits_267.jpg,455,train,test
fruits_268.jpg,fruit,safe,,fruits_268.jpg,456,train,train
fruits_269.jpg,fruit,safe,,fruits_269.jpg,448,train,train
fruits_27.jpg,fruit,safe,,fruits_27.jpg,457,train,train
fruits_270.jpg,fruit,safe,,fruits_270.jpg,458,val,val
fruits_271.jpg,fruit,safe,,fruits_271.jpg,459,test,train
fruits_272.jpg,fruit,safe,,fruits_272.jpg,460,train,test
fruits_273.jpg,fruit,safe,,fruits_273.jpg,461,train,train
fruits_274.jpg,fruit,safe,,fruits_274.jpg,462,val,train
fruits_275.jpg,fruit,safe,,fruits_275.jpg,463,train,train
fruits_276.jpg,fruit,safe,,fruits_276.jpg,464,test,train
fruits_277.jpg,fruit,safe,,fruits_277.jpg,465,val,test
fruits_278.jpg,fruit,safe,,fruits_278.jpg,466,val,test
fruits_279.jpg,fruit,safe,,fruits_279.jpg,467,train,train
fruits_28.jpg,fruit,safe,,fruits_28.jpg,468,train,train
fruits_280.jpg,fruit,safe,,fruits_280.jpg,469,val,train
fruits_281.jpg,fruit,safe,,fruits_281.jpg,470,train,train
fruits_282.jpg,fruit,safe,,fruits_282.jpg,471,train,train
fruits_283.jpg,fruit,safe,,fruits_283.jpg,472,train,train
fruits_284.jpg,fruit,safe,,fruits_284.jpg,473,train,train
fruits_285.jpg,fruit,safe,,fruits_285.jpg,474,train,train
fruits_286.jpg,fruit,safe,,fruits_286.jpg,475,train,train
fruits_287.jpg,fruit,safe,,fruits_287.jpg,476,train,val
fruits_288.jpg,fruit,safe,,fruits_288.jpg,477,train,test
fruits_289.jpg,fruit,safe,,fruits_289.jpg,475,train,train
fruits_29.jpg,fruit,safe,,fruits_29.jpg,478,train,train
fruits_290.jpg,fruit,safe,,fruits_290.jpg,479,train,train
fruits_291.jpg,fruit,safe,,fruits_291.jpg,480,train,train
fruits_292.jpg,fruit,safe,,fruits_292.jpg,481,train,train
fruits_293.jpg,fruit,safe,,fruits_293.jpg,482,test,train
fruits_294.jpg,fruit,safe,,fruits_294.jpg,483,train,train
fruits_295.jpg,fruit,safe,,fruits_295.jpg,484,train,val
fruits_296.jpg,fruit,safe,,fruits_296.jpg,485,val,train
fruits_297.jpg,fruit,safe,,fruits_297.jpg,486,train,train
fruits_298.jpg,fruit,safe,,fruits_298.jpg,487,val,train
fruits_299.jpg,fruit,safe,,fruits_299.jpg,488,test,train
fruits_3.jpg,fruit,safe,,fruits_3.jpg,489,train,val
fruits_30.jpg,fruit,safe,,fruits_30.jpg,490,test,val
fruits_300.jpg,fruit,safe,,fruits_300.jpg,491,test,train
fruits_301.jpg,fruit,safe,,fruits_301.jpg,492,train,test
fruits_302.jpg,fruit,safe,,fruits_302.jpg,493,train,test
fruits_303.jpg,fruit,safe,,fruits_303.jpg,494,train,val
fruits_304.jpg,fruit,safe,,fruits_304.jpg,495,train,train
fruits_305.jpg,fruit,safe,,fruits_305.jpg,496,test,train
fruits_306.jpg,fruit,safe,,fruits_306.jpg,497,val,train
fruits_307.jpg,fruit,safe,,fruits_307.jpg,498,train,train
fruits_308.jpg,fruit,safe,,fruits_308.jpg,499,test,val
fruits_309.jpg,fruit,safe,,fruits_309.jpg,500,test,train
fruits_31.jpg,fruit,safe,,fruits_31.jpg,501,train,train
fruits_310.jpg,fruit,safe,,fruits_310.jpg,502,train,test
fruits_311.jpg,fruit,safe,,fruits_311.jpg,503,train,train
fruits_312.jpg,fruit,safe,,fruits_312.jpg,504,val,train
fruits_313.jpg,fruit,safe,,fruits_313.jpg,505,train,test
fruits_314.jpg,fruit,safe,,fruits_314.jpg,506,test,train
fruits_315.jpg,fruit,safe,,fruits_315.jpg,507,val,train
fruits_316.jpg,fruit,safe,,fruits_316.jpg,508,train,val
fruits_317.jpg,fruit,safe,,fruits_317.jpg,509,train,val
fruits_318.jpg,fruit,safe,,fruits_318.jpg,510,train,train
fruits_319.jpg,fruit,safe,,fruits_319.jpg,511,train,test
fruits_32.jpg,fruit,safe,,fruits_32.jpg,512,test,train
fruits_320.jpg,fruit,safe,,fruits_320.jpg,513,train,test
fruits_321.jpg,fruit,safe,,fruits_321.jpg,514,train,train
fruits_322.jpg,fruit,safe,,fruits_322.jpg,514,train,val
fruits_323.jpg,fruit,safe,,fruits_323.jpg,513,train,val
fruits_324.jpg,fruit,safe,,fruits_324.jpg,515,train,test
fruits_325.jpg,fruit,safe,,fruits_325.jpg,516,val,train
fruits_326.jpg,fruit,safe,,fruits_326.jpg,515,train,train
fruits_327.jpg,fruit,safe,,fruits_327.jpg,517,val,train
fruits_328.jpg,fruit,safe,,fruits_328.jpg,518,train,val
fruits_329.jpg,fruit,safe,,fruits_329.jpg,519,test,train
fruits_33.jpg,fruit,safe,,fruits_33.jpg,520,train,train
fruits_330.jpg,fruit,safe,,fruits_330.jpg,521,train,val
fruits_331.jpg,fruit,safe,,fruits_331.jpg,522,test,train
fruits_332.jpg,fruit,safe,,fruits_332.jpg,523,test,val
fruits_333.jpg,fruit,safe,,fruits_333.jpg,524,train,train
fruits_334.jpg,fruit,safe,,fruits_334.jpg,524,train,train
fruits_335.jpg,fruit,safe,,fruits_335.jpg,525,test,train
fruits_336.jpg,fruit,safe,,fruits_336.jpg,526,train,train
fruits_337.jpg,fruit,safe,,fruits_337.jpg,527,test,train
fruits_338.jpg,fruit,safe,,fruits_338.jpg,528,test,test
fruits_339.jpg,fruit,safe,,fruits_339.jpg,529,val,train
fruits_34.jpg,fruit,safe,,fruits_34.jpg,501,train,train
fruits_340.jpg,fruit,safe,,fruits_340.jpg,530,train,test
fruits_341.jpg,fruit,safe,,fruits_341.jpg,531,train,train
fruits_342.jpg,fruit,safe,,fruits_342.jpg,531,train,train
fruits_343.jpg,fruit,safe,,fruits_343.jpg,532,train,test
fruits_344.jpg,fruit,safe,,fruits_344.jpg,533,train,train
fruits_345.jpg,fruit,safe,,fruits_345.jpg,534,test,test
fruits_346.jpg,fruit,safe,,fruits_346.jpg,535,val,train
fruits_347.jpg,fruit,safe,,fruits_347.jpg,536,train,train
fruits_348.jpg,fruit,safe,,fruits_348.jpg,537,train,train
fruits_349.jpg,fruit,safe,,fruits_349.jpg,538,train,train
fruits_35.jpg,fruit,safe,,fruits_35.jpg,539,train,train
fruits_350.jpg,fruit,safe,,fruits_350.jpg,540,train,train
fruits_351.jpg,fruit,safe,,fruits_351.jpg,541,train,train
fruits_352.jpg,fruit,safe,,fruits_352.jpg,542,train,val
fruits_353.jpg,fruit,safe,,fruits_353.jpg,543,train,train
fruits_354.jpg,fruit,safe,,fruits_354.jpg,544,train,train
fruits_355.jpg,fruit,safe,,fruits_355.jpg,545,train,train
fruits_356.jpg,fruit,safe,,fruits_356.jpg,546,train,train
fruits_357.jpg,fruit,safe,,fruits_357.jpg,547,train,train
fruits_358.jpg,fruit,safe,,fruits_358.jpg,548,train,val
fruits_359.jpg,fruit,safe,,fruits_359.jpg,549,train,train
fruits_36.jpg,fruit,safe,,fruits_36.jpg,539,train,train
fruits_37.jpg,fruit,safe,,fruits_37.jpg,550,train,train
fruits_38.jpg,fruit,safe,,fruits_38.jpg,551,train,train
fruits_39.jpg,fruit,safe,,fruits_39.jpg,552,train,test
fruits_4.jpg,fruit,safe,,fruits_4.jpg,553,train,train
fruits_40.jpg,fruit,safe,,fruits_40.jpg,554,val,test
fruits_41.jpg,fruit,safe,,fruits_41.jpg,555,test,val
fruits_42.jpg,fruit,safe,,fruits_42.jpg,556,train,train
fruits_43.jpg,fruit,safe,,fruits_43.jpg,556,train,train
fruits_44.jpg,fruit,safe,,fruits_44.jpg,557,train,val
fruits_45.jpg,fruit,safe,,fruits_45.jpg,558,train,val
fruits_46.jpg,fruit,safe,,fruits_46.jpg,559,test,train
fruits_47.jpg,fruit,safe,,fruits_47.jpg,560,val,test
fruits_48.jpg,fruit,safe,,fruits_48.jpg,561,train,train
fruits_49.jpg,fruit,safe,,fruits_49.jpg,562,train,val
fruits_5.jpg,fruit,safe,,fruits_5.jpg,563,train,test
fruits_50.jpg,fruit,safe,,fruits_50.jpg,564,train,train
fruits_51.jpg,fruit,safe,,fruits_51.jpg,565,test,train
fruits_52.jpg,fruit,safe,,fruits_52.jpg,566,test,train
fruits_53.jpg,fruit,safe,,fruits_53.jpg,567,train,train
fruits_54.jpg,fruit,safe,,fruits_54.jpg,568,test,val
fruits_55.jpg,fruit,safe,,fruits_55.jpg,564,train,test
fruits_56.jpg,fruit,safe,,fruits_56.jpg,569,val,train
fruits_57.jpg,fruit,safe,,fruits_57.jpg,552,train,train
fruits_58.jpg,fruit,safe,,fruits_58.jpg,570,val,val
fruits_59.jpg,fruit,safe,,fruits_59.jpg,571,test,val
fruits_6.jpg,fruit,safe,,fruits_6.jpg,572,val,train
fruits_60.jpg,fruit,safe,,fruits_60.jpg,573,train,train
fruits_61.jpg,fruit,safe,,fruits_61.jpg,574,test,test
fruits_62.jpg,fruit,safe,,fruits_62.jpg,575,val,train
fruits_63.jpg,fruit,safe,,fruits_63.jpg,576,train,train
fruits_64.jpg,fruit,safe,,fruits_64.jpg,577,train,test
fruits_65.jpg,fruit,safe,,fruits_65.jpg,578,train,test
fruits_66.jpg,fruit,safe,,fruits_66.jpg,579,train,train
fruits_67.jpg,fruit,safe,,fruits_67.jpg,580,val,train
fruits_68.jpg,fruit,safe,,fruits_68.jpg,581,train,train
fruits_69.jpg,fruit,safe,,fruits_69.jpg,579,train,train
fruits_7.jpg,fruit,safe,,fruits_7.jpg,582,train,train
fruits_70.jpg,fruit,safe,,fruits_70.jpg,583,train,val
fruits_71.jpg,fruit,safe,,fruits_71.jpg,584,train,train
fruits_72.jpg,fruit,safe,,fruits_72.jpg,585,train,val
fruits_73.jpg,fruit,safe,,fruits_73.jpg,586,train,val
fruits_74.jpg,fruit,safe,,fruits_74.jpg,587,train,train
fruits_75.jpg,fruit,safe,,fruits_75.jpg,588,train,train
fruits_76.jpg,fruit,safe,,fruits_76.jpg,589,train,test
fruits_77.jpg,fruit,safe,,fruits_77.jpg,590,val,train
fruits_78.jpg,fruit,safe,,fruits_78.jpg,591,val,train
fruits_79.jpg,fruit,safe,,fruits_79.jpg,592,test,train
fruits_8.jpg,fruit,safe,,fruits_8.jpg,593,train,test
fruits_80.jpg,fruit,safe,,fruits_80.jpg,594,test,train
fruits_81.jpg,fruit,safe,,fruits_81.jpg,595,train,train
fruits_82.jpg,fruit,safe,,fruits_82.jpg,596,train,train
fruits_83.jpg,fruit,safe,,fruits_83.jpg,597,val,train
fruits_84.jpg,fruit,safe,,fruits_84.jpg,598,train,val
fruits_85.jpg,fruit,safe,,fruits_85.jpg,598,train,train
fruits_86.jpg,fruit,safe,,fruits_86.jpg,599,train,val
fruits_87.jpg,fruit,safe,,fruits_87.jpg,600,train,train
fruits_88.jpg,fruit,safe,,fruits_88.jpg,601,train,train
fruits_89.jpg,fruit,safe,,fruits_89.jpg,602,train,train
fruits_9.jpg,fruit,safe,,fruits_9.jpg,279,train,test
fruits_90.jpg,fruit,safe,,fruits_90.jpg,511,train,val
fruits_91.jpg,fruit,safe,,fruits_91.jpg,603,train,train
fruits_92.jpg,fruit,safe,,fruits_92.jpg,604,val,train
fruits_93.jpg,fruit,safe,,fruits_93.jpg,605,test,train
fruits_94.jpg,fruit,safe,,fruits_94.jpg,606,val,train
fruits_95.jpg,fruit,safe,,fruits_95.jpg,607,train,test
fruits_96.jpg,fruit,safe,,fruits_96.jpg,608,train,train
fruits_97.jpg,fruit,safe,,fruits_97.jpg,509,train,train
fruits_98.jpg,fruit,safe,,fruits_98.jpg,609,train,test
fruits_99.jpg,fruit,safe,,fruits_99.jpg,610,train,train
heater_10_0_6825.jpg,heater,unsafe,,heater_10,611,test,train
heater_10_0_6841.jpg,heater,unsafe,,heater_10,611,test,train
heater_10_0_9048.jpg,heater,unsafe,,heater_10,611,test,train
heater_10_0_937.jpg,heater,unsafe,,heater_10,611,test,train
heater_10_0_9713.jpg,heater,unsafe,,heater_10,611,test,val
heater_11_0_4191.jpg,heater,unsafe,,heater_11,612,train,test
heater_11_0_5469.jpg,heater,unsafe,,heater_11,612,train,test
heater_11_0_5635.jpg,heater,unsafe,,heater_11,612,train,test
heater_11_0_7637.jpg,heater,unsafe,,heater_11,612,train,test
heater_11_0_8719.jpg,heater,unsafe,,heater_11,612,train,train
heater_12_0_2391.jpg,heater,unsafe,,heater_12,613,train,train
heater_12_0_2696.jpg,heater,unsafe,,heater_12,613,train,train
heater_12_0_3033.jpg,heater,unsafe,,heater_12,613,train,train
heater_12_0_3975.jpg,heater,unsafe,,heater_12,613,train,train
heater_12_0_5260.jpg,heater,unsafe,,heater_12,613,train,train
heater_14_0_1934.jpg,heater,unsafe,,heater_14,614,train,val
heater_14_0_2410.jpg,heater,unsafe,,heater_14,614,train,val
heater_14_0_4140.jpg,heater,unsafe,,heater_14,614,train,train
heater_14_0_5134.jpg,heater,unsafe,,heater_14,614,train,val
heater_14_0_9659.jpg,heater,unsafe,,heater_14,614,train,val
heater_15_0_2000.jpg,heater,unsafe,,heater_15,615,train,train
heater_15_0_2867.jpg,heater,unsafe,,heater_15,615,train,train
heater_15_0_3386.jpg,heater,unsafe,,heater_15,615,train,val
heater_15_0_80.jpg,heater,unsafe,,heater_15,615,train,train
heater_15_0_8580.jpg,heater,unsafe,,heater_15,615,train,train
heater_16_0_2342.jpg,heater,unsafe,,heater_16,616,val,train
heater_16_0_4067.jpg,heater,unsafe,,heater_16,616,val,train
heater_16_0_4218.jpg,heater,unsafe,,heater_16,616,val,train
heater_16_0_9214.jpg,heater,unsafe,,heater_16,616,val,train
heater_16_0_946.jpg,heater,unsafe,,heater_16,616,val,val
heater_17_0_3910.jpg,heater,unsafe,,heater_17,617,val,train
heater_17_0_4905.jpg,heater,unsafe,,heater_17,617,val,train
heater_17_0_5642.jpg,heater,unsafe,,heater_17,617,val,train
heater_17_0_6548.jpg,heater,unsafe,,heater_17,617,val,train
heater_17_0_8237.jpg,heater,unsafe,,heater_17,617,val,train
heater_18_0_4146.jpg,heater,unsafe,,heater_18,618,train,train
heater_18_0_489.jpg,heater,unsafe,,heater_18,618,train,train
heater_18_0_6190.jpg,heater,unsafe,,heater_18,618,train,train
heater_18_0_7436.jpg,heater,unsafe,,heater_18,618,train,train
heater_18_0_9988.jpg,heater,unsafe,,heater_18,618,train,train
heater_19_0_1622.jpg,heater,unsafe,,heater_19,619,train,test
heater_19_0_2616.jpg,heater,unsafe,,heater_19,619,train,train
heater_19_0_4051.jpg,heater,unsafe,,heater_19,619,train,train
heater_19_0_7206.jpg,heater,unsafe,,heater_19,619,train,train
heater_19_0_8286.jpg,heater,unsafe,,heater_19,619,train,val
heater_1_0_1317.jpg,heater,unsafe,,heater_1,620,test,train
heater_1_0_616.jpg,heater,unsafe,,heater_1,620,test,train
heater_1_0_6685.jpg,heater,unsafe,,heater_1,620,test,train
heater_1_0_8635.jpg,heater,unsafe,,heater_1,620,test,test
heater_1_0_9912.jpg,heater,unsafe,,heater_1,620,test,train
heater_21_0_1030.jpg,heater,unsafe,,heater_21,621,test,train
heater_21_0_5438.jpg,heater,unsafe,,heater_21,621,test,train
heater_21_0_871.jpg,heater,unsafe,,heater_21,621,test,val
heater_21_0_8763.jpg,heater,unsafe,,heater_21,621,test,train
heater_21_0_9075.jpg,heater,unsafe,,heater_21,621,test,train
heater_22_0_1415.jpg,heater,unsafe,,heater_22,622,train,train
heater_22_0_4490.jpg,heater,unsafe,,heater_22,622,train,train
heater_22_0_7849.jpg,heater,unsafe,,heater_22,622,train,test
heater_22_0_9685.jpg,heater,unsafe,,heater_22,622,train,train
heater_22_0_9710.jpg,heater,unsafe,,heater_22,622,train,train
heater_23_0_4438.jpg,heater,unsafe,,heater_23,623,train,test
heater_23_0_5252.jpg,heater,unsafe,,heater_23,623,train,test
heater_23_0_6239.jpg,heater,unsafe,,heater_23,623,train,train
heater_23_0_7091.jpg,heater,unsafe,,heater_23,623,train,train
heater_23_0_8747.jpg,heater,unsafe,,heater_23,623,train,train
heater_24_0_2647.jpg,heater,unsafe,,heater_24,624,train,train
heater_24_0_3807.jpg,heater,unsafe,,heater_24,624,train,train
heater_24_0_4666.jpg,heater,unsafe,,heater_24,624,train,train
heater_24_0_7442.jpg,heater,unsafe,,heater_24,624,train,train
heater_24_0_850.jpg,heater,unsafe,,heater_24,624,train,train
heater_25_0_1089.jpg,heater,unsafe,,heater_25,625,train,val
heater_25_0_4645.jpg,heater,unsafe,,heater_25,625,train,test
heater_25_0_5886.jpg,heater,unsafe,,heater_25,625,train,test
heater_25_0_7074.jpg,heater,unsafe,,heater_25,625,train,train
heater_25_0_9777.jpg,heater,unsafe,,heater_25,625,train,train
heater_26_0_4071.jpg,heater,unsafe,,heater_26,626,train,train
heater_26_0_4505.jpg,heater,unsafe,,heater_26,626,train,train
heater_26_0_4822.jpg,heater,unsafe,,heater_26,626,train,train
heater_26_0_5525.jpg,heater,unsafe,,heater_26,626,train,val
heater_26_0_9370.jpg,heater,unsafe,,heater_26,626,train,val
heater_28_0_1016.jpg,heater,unsafe,,heater_28,627,train,train
heater_28_0_1048.jpg,heater,unsafe,,heater_28,627,train,train
heater_28_0_2039.jpg,heater,unsafe,,heater_28,627,train,train
heater_28_0_2762.jpg,heater,unsafe,,heater_28,627,train,test
heater_28_0_3703.jpg,heater,unsafe,,heater_28,627,train,train
heater_29_0_2193.jpg,heater,unsafe,,heater_29,628,test,val
heater_29_0_3723.jpg,heater,unsafe,,heater_29,628,test,val
heater_29_0_403.jpg,heater,unsafe,,heater_29,628,test,train
heater_29_0_8128.jpg,heater,unsafe,,heater_29,628,test,train
heater_29_0_8739.jpg,heater,unsafe,,heater_29,628,test,train
heater_30_0_1799.jpg,heater,unsafe,,heater_30,629,train,train
heater_30_0_2761.jpg,heater,unsafe,,heater_30,629,train,test
heater_30_0_5267.jpg,heater,unsafe,,heater_30,629,train,train
heater_30_0_5569.jpg,heater,unsafe,,heater_30,629,train,train
heater_30_0_9349.jpg,heater,unsafe,,heater_30,629,train,val
heater_33_0_2021.jpg,heater,unsafe,,heater_33,630,train,train
heater_33_0_3125.jpg,heater,unsafe,,heater_33,630,train,train
heater_33_0_4734.jpg,heater,unsafe,,heater_33,630,train,val
heater_33_0_6345.jpg,heater,unsafe,,heater_33,630,train,train
heater_33_0_7436.jpg,heater,unsafe,,heater_33,630,train,train
heater_34_0_1272.jpg,heater,unsafe,,heater_34,631,train,val
heater_34_0_4091.jpg,heater,unsafe,,heater_34,631,train,val
heater_34_0_6590.jpg,heater,unsafe,,heater_34,631,train,test
heater_34_0_7644.jpg,heater,unsafe,,heater_34,631,train,test
heater_34_0_8910.jpg,heater,unsafe,,heater_34,631,train,val
heater_35_0_1511.jpg,heater,unsafe,,heater_35,632,train,test
heater_35_0_4382.jpg,heater,unsafe,,heater_35,632,train,train
heater_35_0_514.jpg,heater,unsafe,,heater_35,632,train,train
heater_35_0_6497.jpg,heater,unsafe,,heater_35,632,train,train
heater_35_0_6955.jpg,heater,unsafe,,heater_35,632,train,train
heater_36_0_1001.jpg,heater,unsafe,,heater_36,633,train,test
heater_36_0_128.jpg,heater,unsafe,,heater_36,633,train,train
heater_36_0_1700.jpg,heater,unsafe,,heater_36,633,train,train
heater_36_0_5365.jpg,heater,unsafe,,heater_36,633,train,test
heater_36_0_8344.jpg,heater,unsafe,,heater_36,633,train,train
heater_37_0_3557.jpg,heater,unsafe,,heater_37,634,train,train
heater_37_0_654.jpg,heater,unsafe,,heater_37,634,train,train
heater_37_0_6712.jpg,heater,unsafe,,heater_37,634,train,train
heater_37_0_756.jpg,heater,unsafe,,heater_37,634,train,test
heater_37_0_7701.jpg,heater,unsafe,,heater_37,634,train,train
heater_38_0_2059.jpg,heater,unsafe,,heater_38,635,train,train
heater_38_0_4728.jpg,heater,unsafe,,heater_38,635,train,val
heater_38_0_4909.jpg,heater,unsafe,,heater_38,635,train,test
heater_38_0_8419.jpg,heater,unsafe,,heater_38,635,train,train
heater_38_0_8950.jpg,heater,unsafe,,heater_38,635,train,train
heater_39_0_1846.jpg,heater,unsafe,,heater_39,636,train,val
heater_39_0_2612.jpg,heater,unsafe,,heater_39,636,train,test
heater_39_0_5241.jpg,heater,unsafe,,heater_39,636,train,train
heater_39_0_5432.jpg,heater,unsafe,,heater_39,636,train,train
heater_39_0_7213.jpg,heater,unsafe,,heater_39,636,train,train
heater_3_0_2610.jpg,heater,unsafe,,heater_3,637,train,train
heater_3_0_4096.jpg,heater,unsafe,,heater_3,637,train,test
heater_3_0_5768.jpg,heater,unsafe,,heater_3,637,train,train
heater_3_0_730.jpg,heater,unsafe,,heater_3,637,train,test
heater_3_0_7691.jpg,heater,unsafe,,heater_3,637,train,train
heater_40_0_1340.jpg,heater,unsafe,,heater_40,638,test,train
heater_40_0_1404.jpg,heater,unsafe,,heater_40,638,test,val
heater_40_0_3405.jpg,heater,unsafe,,heater_40,638,test,train
heater_40_0_6391.jpg,heater,unsafe,,heater_40,638,test,train
heater_40_0_8427.jpg,heater,unsafe,,heater_40,638,test,train
heater_42_0_4718.jpg,heater,unsafe,,heater_42,639,train,val
heater_42_0_5503.jpg,heater,unsafe,,heater_42,639,train,val
heater_42_0_5774.jpg,heater,unsafe,,heater_42,639,train,train
heater_42_0_6125.jpg,heater,unsafe,,heater_42,639,train,train
heater_42_0_9337.jpg,heater,unsafe,,heater_42,639,train,train
heater_43_0_217.jpg,heater,unsafe,,heater_43,640,train,val
heater_43_0_6367.jpg,heater,unsafe,,heater_43,640,train,train
heater_43_0_8261.jpg,heater,unsafe,,heater_43,640,train,train
heater_43_0_9331.jpg,heater,unsafe,,heater_43,640,train,train
heater_43_0_9433.jpg,heater,unsafe,,heater_43,640,train,train
heater_44_0_1287.jpg,heater,unsafe,,heater_44,641,train,train
heater_44_0_2774.jpg,heater,unsafe,,heater_44,641,train,train
heater_44_0_3815.jpg,heater,unsafe,,heater_44,641,train,train
heater_44_0_5601.jpg,heater,unsafe,,heater_44,641,train,train
heater_44_0_5800.jpg,heater,unsafe,,heater_44,641,train,train
heater_49_0_206.jpg,heater,unsafe,,heater_49,642,val,train
heater_49_0_4399.jpg,heater,unsafe,,heater_49,642,val,val
heater_49_0_7352.jpg,heater,unsafe,,heater_49,642,val,train
heater_49_0_8542.jpg,heater,unsafe,,heater_49,642,val,train
heater_49_0_9029.jpg,heater,unsafe,,heater_49,642,val,test
heater_4_0_193.jpg,heater,unsafe,,heater_4,643,train,train
heater_4_0_6589.jpg,heater,unsafe,,heater_4,643,train,train
heater_4_0_7891.jpg,heater,unsafe,,heater_4,643,train,train
heater_4_0_9257.jpg,heater,unsafe,,heater_4,643,train,train
heater_4_0_9539.jpg,heater,unsafe,,heater_4,643,train,train
heater_51_0_2161.jpg,heater,unsafe,,heater_51,644,test,train
heater_51_0_5698.jpg,heater,unsafe,,heater_51,644,test,train
heater_51_0_658.jpg,heater,unsafe,,heater_51,644,test,train
heater_51_0_8590.jpg,heater,unsafe,,heater_51,644,test,train
heater_51_0_9500.jpg,heater,unsafe,,heater_51,644,test,train
heater_52_0_310.jpg,heater,unsafe,,heater_52,645,val,train
heater_52_0_3100.jpg,heater,unsafe,,heater_52,645,val,test
heater_52_0_4228.jpg,heater,unsafe,,heater_52,645,val,train
heater_52_0_4306.jpg,heater,unsafe,,heater_52,645,val,train
heater_52_0_83.jpg,heater,unsafe,,heater_52,645,val,train
heater_53_0_1027.jpg,heater,unsafe,,heater_53,646,train,train
heater_53_0_2043.jpg,heater,unsafe,,heater_53,646,train,train
heater_53_0_2938.jpg,heater,unsafe,,heater_53,646,train,test
heater_53_0_2947.jpg,heater,unsafe,,heater_53,646,train,train
heater_53_0_7667.jpg,heater,unsafe,,heater_53,646,train,train
heater_54_0_1710.jpg,heater,unsafe,,heater_54,647,train,val
heater_54_0_2360.jpg,heater,unsafe,,heater_54,647,train,train
heater_54_0_5946.jpg,heater,unsafe,,heater_54,647,train,val
heater_54_0_6875.jpg,heater,unsafe,,heater_54,647,train,train
heater_54_0_9817.jpg,heater,unsafe,,heater_54,647,train,train
heater_55_0_1872.jpg,heater,unsafe,,heater_55,648,train,train
heater_55_0_7231.jpg,heater,unsafe,,heater_55,648,train,train
heater_55_0_8323.jpg,heater,unsafe,,heater_55,648,train,test
heater_55_0_9816.jpg,heater,unsafe,,heater_55,648,train,test
heater_55_0_9966.jpg,heater,unsafe,,heater_55,648,train,train
heater_6_0_1.jpg,heater,unsafe,,heater_6,649,val,train
heater_6_0_1084.jpg,heater,unsafe,,heater_6,649,val,val
heater_6_0_2701.jpg,heater,unsafe,,heater_6,649,val,train
heater_6_0_3000.jpg,heater,unsafe,,heater_6,649,val,train
heater_6_0_3146.jpg,heater,unsafe,,heater_6,649,val,test
heater_7_0_11.jpg,heater,unsafe,,heater_7,650,train,val
heater_7_0_1271.jpg,heater,unsafe,,heater_7,650,train,train
heater_7_0_4276.jpg,heater,unsafe,,heater_7,650,train,train
heater_7_0_8478.jpg,heater,unsafe,,heater_7,650,train,train
heater_7_0_8860.jpg,heater,unsafe,,heater_7,650,train,train
heater_8_0_1605.jpg,heater,unsafe,,heater_8,651,train,test
heater_8_0_6647.jpg,heater,unsafe,,heater_8,651,train,train
heater_8_0_7249.jpg,heater,unsafe,,heater_8,651,train,test
heater_8_0_7739.jpg,heater,unsafe,,heater_8,651,train,train
heater_8_0_8518.jpg,heater,unsafe,,heater_8,651,train,train
heater_9_0_347.jpg,heater,unsafe,,heater_9,652,val,val
heater_9_0_5210.jpg,heater,unsafe,,heater_9,652,val,train
heater_9_0_5279.jpg,heater,unsafe,,heater_9,652,val,train
heater_9_0_5550.jpg,heater,unsafe,,heater_9,652,val,val
heater_9_0_6740.jpg,heater,unsafe,,heater_9,652,val,train
knife_10_0_2358.jpg,knife,unsafe,,knife_10,653,train,test
knife_10_0_2363.jpg,knife,unsafe,,knife_10,653,train,train
knife_10_0_5444.jpg,knife,unsafe,,knife_10,653,train,train
knife_10_0_5676.jpg,knife,unsafe,,knife_10,653,train,train
knife_10_0_9219.jpg,knife,unsafe,,knife_10,653,train,test
knife_11_0_4515.jpg,knife,unsafe,,knife_11,654,train,val
knife_11_0_5121.jpg,knife,unsafe,,knife_11,654,train,train
knife_11_0_5671.jpg,knife,unsafe,,knife_11,654,train,val
knife_11_0_7990.jpg,knife,unsafe,,knife_11,654,train,train
knife_11_0_9689.jpg,knife,unsafe,,knife_11,654,train,val
knife_12_0_1932.jpg,knife,unsafe,,knife_12,655,train,train
knife_12_0_26.jpg,knife,unsafe,,knife_12,655,train,train
knife_12_0_2752.jpg,knife,unsafe,,knife_12,655,train,train
knife_12_0_3721.jpg,knife,unsafe,,knife_12,655,train,train
knife_12_0_7297.jpg,knife,unsafe,,knife_12,655,train,train
knife_13_0_1642.jpg,knife,unsafe,,knife_13,656,train,val
knife_13_0_4949.jpg,knife,unsafe,,knife_13,656,train,train
knife_13_0_6712.jpg,knife,unsafe,,knife_13,656,train,train
knife_13_0_8089.jpg,knife,unsafe,,knife_13,656,train,test
knife_13_0_9851.jpg,knife,unsafe,,knife_13,656,train,train
knife_14_0_1197.jpg,knife,unsafe,,knife_14,657,train,train
knife_14_0_1406.jpg,knife,unsafe,,knife_14,657,train,val
knife_14_0_3413.jpg,knife,unsafe,,knife_14,657,train,val
knife_14_0_5002.jpg,knife,unsafe,,knife_14,657,train,train
knife_14_0_5974.jpg,knife,unsafe,,knife_14,657,train,train
knife_15_0_3099.jpg,knife,unsafe,,knife_15,658,train,test
knife_15_0_361.jpg,knife,unsafe,,knife_15,658,train,train
knife_15_0_4053.jpg,knife,unsafe,,knife_15,658,train,train
knife_15_0_4131.jpg,knife,unsafe,,knife_15,658,train,train
knife_15_0_5335.jpg,knife,unsafe,,knife_15,658,train,train
knife_16_0_3446.jpg,knife,unsafe,,knife_16,659,train,train
knife_16_0_3585.jpg,knife,unsafe,,knife_16,659,train,train
knife_16_0_5420.jpg,knife,unsafe,,knife_16,659,train,train
knife_16_0_8830.jpg,knife,unsafe,,knife_16,659,train,train
knife_16_0_9031.jpg,knife,unsafe,,knife_16,659,train,train
knife_17_0_3413.jpg,knife,unsafe,,knife_17,660,train,train
knife_17_0_4918.jpg,knife,unsafe,,knife_17,660,train,train
knife_17_0_6592.jpg,knife,unsafe,,knife_17,660,train,train
knife_17_0_77.jpg,knife,unsafe,,knife_17,660,train,train
knife_17_0_8851.jpg,knife,unsafe,,knife_17,660,train,train
knife_18_0_1040.jpg,knife,unsafe,,knife_18,661,val,val
knife_18_0_6865.jpg,knife,unsafe,,knife_18,661,val,train
knife_18_0_7026.jpg,knife,unsafe,,knife_18,661,val,val
knife_18_0_746.jpg,knife,unsafe,,knife_18,661,val,train
knife_18_0_9487.jpg,knife,unsafe,,knife_18,661,val,val
knife_19_0_1923.jpg,knife,unsafe,,knife_19,662,train,train
knife_19_0_3277.jpg,knife,unsafe,,knife_19,662,train,train
knife_19_0_6670.jpg,knife,unsafe,,knife_19,662,train,train
knife_19_0_7391.jpg,knife,unsafe,,knife_19,662,train,train
knife_19_0_8750.jpg,knife,unsafe,,knife_19,662,train,train
knife_1_0_2645.jpg,knife,unsafe,,knife_1,663,train,train
knife_1_0_4855.jpg,knife,unsafe,,knife_1,663,train,val
knife_1_0_4936.jpg,knife,unsafe,,knife_1,663,train,test
knife_1_0_7915.jpg,knife,unsafe,,knife_1,663,train,train
knife_1_0_8328.jpg,knife,unsafe,,knife_1,663,train,train
knife_20_0_4701.jpg,knife,unsafe,,knife_20,664,train,train
knife_20_0_6093.jpg,knife,unsafe,,knife_20,664,train,train
knife_20_0_6845.jpg,knife,unsafe,,knife_20,664,train,train
knife_20_0_7292.jpg,knife,unsafe,,knife_20,664,train,train
knife_20_0_8681.jpg,knife,unsafe,,knife_20,664,train,train
knife_21_0_1004.jpg,knife,unsafe,,knife_21,665,train,train
knife_21_0_3484.jpg,knife,unsafe,,knife_21,665,train,train
knife_21_0_3646.jpg,knife,unsafe,,knife_21,665,train,test
knife_21_0_5143.jpg,knife,unsafe,,knife_21,665,train,train
knife_21_0_9677.jpg,knife,unsafe,,knife_21,665,train,test
knife_22_0_6030.jpg,knife,unsafe,,knife_22,666,train,val
knife_22_0_8441.jpg,knife,unsafe,,knife_22,666,train,train
knife_22_0_8796.jpg,knife,unsafe,,knife_22,666,train,train
knife_22_0_9089.jpg,knife,unsafe,,knife_22,666,train,test
knife_22_0_9096.jpg,knife,unsafe,,knife_22,666,train,train
knife_23_0_1261.jpg,knife,unsafe,,knife_23,667,val,train
knife_23_0_2590.jpg,knife,unsafe,,knife_23,667,val,train
knife_23_0_725.jpg,knife,unsafe,,knife_23,667,val,test
knife_23_0_8430.jpg,knife,unsafe,,knife_23,667,val,train
knife_23_0_8718.jpg,knife,unsafe,,knife_23,667,val,train
knife_24_0_102.jpg,knife,unsafe,,knife_24,668,train,test
knife_24_0_5242.jpg,knife,unsafe,,knife_24,668,train,test
knife_24_0_5349.jpg,knife,unsafe,,knife_24,668,train,val
knife_24_0_6527.jpg,knife,unsafe,,knife_24,668,train,train
knife_24_0_838.jpg,knife,unsafe,,knife_24,668,train,train
knife_25_0_1262.jpg,knife,unsafe,,knife_25,669,test,val
knife_25_0_1511.jpg,knife,unsafe,,knife_25,669,test,test
knife_25_0_6331.jpg,knife,unsafe,,knife_25,669,test,train
knife_25_0_8150.jpg,knife,unsafe,,knife_25,669,test,train
knife_25_0_8654.jpg,knife,unsafe,,knife_25,669,test,val
knife_26_0_1828.jpg,knife,unsafe,,knife_26,670,val,test
knife_26_0_4604.jpg,knife,unsafe,,knife_26,670,val,train
knife_26_0_5118.jpg,knife,unsafe,,knife_26,670,val,train
knife_26_0_6323.jpg,knife,unsafe,,knife_26,670,val,test
knife_26_0_8464.jpg,knife,unsafe,,knife_26,670,val,test
knife_27_0_2340.jpg,knife,unsafe,,knife_27,671,train,train
knife_27_0_5363.jpg,knife,unsafe,,knife_27,671,train,train
knife_27_0_5757.jpg,knife,unsafe,,knife_27,671,train,train
knife_27_0_8758.jpg,knife,unsafe,,knife_27,671,train,train
knife_27_0_93.jpg,knife,unsafe,,knife_27,671,train,train
knife_28_0_2036.jpg,knife,unsafe,,knife_28,672,val,train
knife_28_0_4330.jpg,knife,unsafe,,knife_28,672,val,train
knife_28_0_7885.jpg,knife,unsafe,,knife_28,672,val,test
knife_28_0_7948.jpg,knife,unsafe,,knife_28,672,val,train
knife_28_0_8547.jpg,knife,unsafe,,knife_28,672,val,train
knife_29_0_336.jpg,knife,unsafe,,knife_29,673,train,train
knife_29_0_5863.jpg,knife,unsafe,,knife_29,673,train,test
knife_29_0_8110.jpg,knife,unsafe,,knife_29,673,train,val
knife_29_0_8909.jpg,knife,unsafe,,knife_29,673,train,train
knife_29_0_9043.jpg,knife,unsafe,,knife_29,673,train,train
knife_2_0_5353.jpg,knife,unsafe,,knife_2,674,train,train
knife_2_0_596.jpg,knife,unsafe,,knife_2,674,train,train
knife_2_0_6363.jpg,knife,unsafe,,knife_2,674,train,train
knife_2_0_7927.jpg,knife,unsafe,,knife_2,674,train,train
knife_2_0_978.jpg,knife,unsafe,,knife_2,674,train,train
knife_30_0_2387.jpg,knife,unsafe,,knife_30,675,train,val
knife_30_0_2749.jpg,knife,unsafe,,knife_30,675,train,val
knife_30_0_8250.jpg,knife,unsafe,,knife_30,675,train,train
knife_30_0_9244.jpg,knife,unsafe,,knife_30,675,train,test
knife_30_0_9258.jpg,knife,unsafe,,knife_30,675,train,train
knife_31_0_209.jpg,knife,unsafe,,knife_31,676,train,val
knife_31_0_2580.jpg,knife,unsafe,,knife_31,676,train,train
knife_31_0_266.jpg,knife,unsafe,,knife_31,676,train,train
knife_31_0_7206.jpg,knife,unsafe,,knife_31,676,train,test
knife_31_0_7882.jpg,knife,unsafe,,knife_31,676,train,val
knife_32_0_312.jpg,knife,unsafe,,knife_32,677,train,train
knife_32_0_3521.jpg,knife,unsafe,,knife_32,677,train,train
knife_32_0_4193.jpg,knife,unsafe,,knife_32,677,train,train
knife_32_0_726.jpg,knife,unsafe,,knife_32,677,train,train
knife_32_0_9907.jpg,knife,unsafe,,knife_32,677,train,train
knife_33_0_197.jpg,knife,unsafe,,knife_33,678,test,train
knife_33_0_2629.jpg,knife,unsafe,,knife_33,678,test,train
knife_33_0_6512.jpg,knife,unsafe,,knife_33,678,test,train
knife_33_0_9456.jpg,knife,unsafe,,knife_33,678,test,test
knife_33_0_9646.jpg,knife,unsafe,,knife_33,678,test,train
knife_34_0_1828.jpg,knife,unsafe,,knife_34,679,train,train
knife_34_0_4070.jpg,knife,unsafe,,knife_34,679,train,test
knife_34_0_6808.jpg,knife,unsafe,,knife_34,679,train,train
knife_34_0_764.jpg,knife,unsafe,,knife_34,679,train,train
knife_34_0_7700.jpg,knife,unsafe,,knife_34,679,train,train
knife_35_0_1498.jpg,knife,unsafe,,knife_35,680,train,test
knife_35_0_1945.jpg,knife,unsafe,,knife_35,680,train,train
knife_35_0_2257.jpg,knife,unsafe,,knife_35,680,train,train
knife_35_0_4367.jpg,knife,unsafe,,knife_35,680,train,val
knife_35_0_7288.jpg,knife,unsafe,,knife_35,680,train,train
knife_36_0_1785.jpg,knife,unsafe,,knife_36,681,train,train
knife_36_0_3169.jpg,knife,unsafe,,knife_36,681,train,test
knife_36_0_373.jpg,knife,unsafe,,knife_36,681,train,test
knife_36_0_4125.jpg,knife,unsafe,,knife_36,681,train,train
knife_36_0_9051.jpg,knife,unsafe,,knife_36,681,train,test
knife_37_0_1957.jpg,knife,unsafe,,knife_37,682,train,train
knife_37_0_3264.jpg,knife,unsafe,,knife_37,682,train,test
knife_37_0_5201.jpg,knife,unsafe,,knife_37,682,train,train
knife_37_0_6784.jpg,knife,unsafe,,knife_37,682,train,train
knife_37_0_9396.jpg,knife,unsafe,,knife_37,682,train,train
knife_38_0_3787.jpg,knife,unsafe,,knife_38,683,train,train
knife_38_0_5590.jpg,knife,unsafe,,knife_38,683,train,train
knife_38_0_6994.jpg,knife,unsafe,,knife_38,683,train,val
knife_38_0_7190.jpg,knife,unsafe,,knife_38,683,train,train
knife_38_0_941.jpg,knife,unsafe,,knife_38,683,train,train
knife_39_0_1236.jpg,knife,unsafe,,knife_39,684,train,train
knife_39_0_2031.jpg,knife,unsafe,,knife_39,684,train,val
knife_39_0_2805.jpg,knife,unsafe,,knife_39,684,train,val
knife_39_0_5110.jpg,knife,unsafe,,knife_39,684,train,test
knife_39_0_7878.jpg,knife,unsafe,,knife_39,684,train,train
knife_3_0_3344.jpg,knife,unsafe,,knife_3,685,train,test
knife_3_0_609.jpg,knife,unsafe,,knife_3,685,train,train
knife_3_0_625.jpg,knife,unsafe,,knife_3,685,train,train
knife_3_0_8772.jpg,knife,unsafe,,knife_3,685,train,train
knife_3_0_9913.jpg,knife,unsafe,,knife_3,685,train,train
knife_40_0_5566.jpg,knife,unsafe,,knife_40,686,train,train
knife_40_0_6717.jpg,knife,unsafe,,knife_40,686,train,train
knife_40_0_7953.jpg,knife,unsafe,,knife_40,686,train,train
knife_40_0_9093.jpg,knife,unsafe,,knife_40,686,train,test
knife_40_0_9098.jpg,knife,unsafe,,knife_40,686,train,train
knife_41_0_1442.jpg,knife,unsafe,,knife_41,687,test,train
knife_41_0_4487.jpg,knife,unsafe,,knife_41,687,test,train
knife_41_0_6152.jpg,knife,unsafe,,knife_41,687,test,train
knife_41_0_9081.jpg,knife,unsafe,,knife_41,687,test,train
knife_41_0_9541.jpg,knife,unsafe,,knife_41,687,test,test
knife_42_0_3632.jpg,knife,unsafe,,knife_42,688,train,test
knife_42_0_4622.jpg,knife,unsafe,,knife_42,688,train,val
knife_42_0_72.jpg,knife,unsafe,,knife_42,688,train,train
knife_42_0_749.jpg,knife,unsafe,,knife_42,688,train,train
knife_42_0_8000.jpg,knife,unsafe,,knife_42,688,train,train
knife_43_0_216.jpg,knife,unsafe,,knife_43,689,val,train
knife_43_0_2811.jpg,knife,unsafe,,knife_43,689,val,train
knife_43_0_4293.jpg,knife,unsafe,,knife_43,689,val,train
knife_43_0_8013.jpg,knife,unsafe,,knife_43,689,val,train
knife_43_0_9500.jpg,knife,unsafe,,knife_43,689,val,train
knife_44_0_1532.jpg,knife,unsafe,,knife_44,690,train,train
knife_44_0_2595.jpg,knife,unsafe,,knife_44,690,train,train
knife_44_0_5250.jpg,knife,unsafe,,knife_44,690,train,train
knife_44_0_6271.jpg,knife,unsafe,,knife_44,690,train,val
knife_44_0_6439.jpg,knife,unsafe,,knife_44,690,train,train
knife_45_0_3440.jpg,knife,unsafe,,knife_45,691,train,train
knife_45_0_430.jpg,knife,unsafe,,knife_45,691,train,test
knife_45_0_4752.jpg,knife,unsafe,,knife_45,691,train,test
knife_45_0_6312.jpg,knife,unsafe,,knife_45,691,train,train
knife_45_0_8036.jpg,knife,unsafe,,knife_45,691,train,val
knife_46_0_2300.jpg,knife,unsafe,,knife_46,692,train,train
knife_46_0_4907.jpg,knife,unsafe,,knife_46,692,train,val
knife_46_0_528.jpg,knife,unsafe,,knife_46,692,train,train
knife_46_0_5922.jpg,knife,unsafe,,knife_46,692,train,test
knife_46_0_6658.jpg,knife,unsafe,,knife_46,692,train,train
knife_47_0_1760.jpg,knife,unsafe,,knife_47,693,test,val
knife_47_0_183.jpg,knife,unsafe,,knife_47,693,test,train
knife_47_0_3079.jpg,knife,unsafe,,knife_47,693,test,train
knife_47_0_7554.jpg,knife,unsafe,,knife_47,693,test,train
knife_47_0_7606.jpg,knife,unsafe,,knife_47,693,test,train
knife_48_0_6187.jpg,knife,unsafe,,knife_48,694,train,val
knife_48_0_8296.jpg,knife,unsafe,,knife_48,694,train,train
knife_48_0_9088.jpg,knife,unsafe,,knife_48,694,train,train
knife_48_0_9618.jpg,knife,unsafe,,knife_48,694,train,train
knife_48_0_9632.jpg,knife,unsafe,,knife_48,694,train,train
knife_49_0_2912.jpg,knife,unsafe,,knife_49,695,train,train
knife_49_0_6803.jpg,knife,unsafe,,knife_49,695,train,train
knife_49_0_7516.jpg,knife,unsafe,,knife_49,695,train,test
knife_49_0_7560.jpg,knife,unsafe,,knife_49,695,train,train
knife_49_0_8669.jpg,knife,unsafe,,knife_49,695,train,train
knife_4_0_3319.jpg,knife,unsafe,,knife_4,696,val,train
knife_4_0_3712.jpg,knife,unsafe,,knife_4,696,val,train
knife_4_0_3868.jpg,knife,unsafe,,knife_4,696,val,test
knife_4_0_4570.jpg,knife,unsafe,,knife_4,696,val,train
knife_4_0_9451.jpg,knife,unsafe,,knife_4,696,val,test
knife_50_0_1451.jpg,knife,unsafe,,knife_50,697,train,train
knife_50_0_3978.jpg,knife,unsafe,,knife_50,697,train,train
knife_50_0_6828.jpg,knife,unsafe,,knife_50,697,train,train
knife_50_0_6920.jpg,knife,unsafe,,knife_50,697,train,val
knife_50_0_8168.jpg,knife,unsafe,,knife_50,697,train,train
knife_51_0_3107.jpg,knife,unsafe,,knife_51,698,train,train
knife_51_0_3636.jpg,knife,unsafe,,knife_51,698,train,train
knife_51_0_6930.jpg,knife,unsafe,,knife_51,698,train,train
knife_51_0_8009.jpg,knife,unsafe,,knife_51,698,train,train
knife_51_0_9720.jpg,knife,unsafe,,knife_51,698,train,train
knife_52_0_2794.jpg,knife,unsafe,,knife_52,699,train,train
knife_52_0_2821.jpg,knife,unsafe,,knife_52,699,train,test
knife_52_0_3735.jpg,knife,unsafe,,knife_52,699,train,train
knife_52_0_4317.jpg,knife,unsafe,,knife_52,699,train,val
knife_52_0_5503.jpg,knife,unsafe,,knife_52,699,train,train
knife_53_0_2191.jpg,knife,unsafe,,knife_53,700,val,train
knife_53_0_2414.jpg,knife,unsafe,,knife_53,700,val,val
knife_53_0_9033.jpg,knife,unsafe,,knife_53,700,val,train
knife_53_0_9354.jpg,knife,unsafe,,knife_53,700,val,train
knife_53_0_9547.jpg,knife,unsafe,,knife_53,700,val,val
knife_54_0_1479.jpg,knife,unsafe,,knife_54,701,train,train
knife_54_0_258.jpg,knife,unsafe,,knife_54,701,train,train
knife_54_0_259.jpg,knife,unsafe,,knife_54,701,train,train
knife_54_0_8558.jpg,knife,unsafe,,knife_54,701,train,train
knife_54_0_8745.jpg,knife,unsafe,,knife_54,701,train,test
knife_55_0_2676.jpg,knife,unsafe,,knife_55,702,train,train
knife_55_0_431.jpg,knife,unsafe,,knife_55,702,train,train
knife_55_0_477.jpg,knife,unsafe,,knife_55,702,train,val
knife_55_0_5273.jpg,knife,unsafe,,knife_55,702,train,val
knife_55_0_5984.jpg,knife,unsafe,,knife_55,702,train,train
knife_56_0_135.jpg,knife,unsafe,,knife_56,703,train,train
knife_56_0_1850.jpg,knife,unsafe,,knife_56,703,train,train
knife_56_0_2369.jpg,knife,unsafe,,knife_56,703,train,train
knife_56_0_3706.jpg,knife,unsafe,,knife_56,703,train,train
knife_56_0_3981.jpg,knife,unsafe,,knife_56,703,train,val
knife_57_0_3880.jpg,knife,unsafe,,knife_57,704,train,train
knife_57_0_6397.jpg,knife,unsafe,,knife_57,704,train,train
knife_57_0_6715.jpg,knife,unsafe,,knife_57,704,train,train
knife_57_0_8188.jpg,knife,unsafe,,knife_57,704,train,train
knife_57_0_8907.jpg,knife,unsafe,,knife_57,704,train,train
knife_58_0_1651.jpg,knife,unsafe,,knife_58,705,train,val
knife_58_0_4187.jpg,knife,unsafe,,knife_58,705,train,train
knife_58_0_5694.jpg,knife,unsafe,,knife_58,705,train,train
knife_58_0_8648.jpg,knife,unsafe,,knife_58,705,train,train
knife_58_0_8947.jpg,knife,unsafe,,knife_58,705,train,train
knife_59_0_2514.jpg,knife,unsafe,,knife_59,706,val,val
knife_59_0_4396.jpg,knife,unsafe,,knife_59,706,val,train
knife_59_0_4649.jpg,knife,unsafe,,knife_59,706,val,train
knife_59_0_6447.jpg,knife,unsafe,,knife_59,706,val,test
knife_59_0_7158.jpg,knife,unsafe,,knife_59,706,val,train
knife_60_0_1345.jpg,knife,unsafe,,knife_60,707,train,train
knife_60_0_5081.jpg,knife,unsafe,,knife_60,707,train,val
knife_60_0_5272.jpg,knife,unsafe,,knife_60,707,train,val
knife_60_0_6099.jpg,knife,unsafe,,knife_60,707,train,val
knife_60_0_6843.jpg,knife,unsafe,,knife_60,707,train,train
knife_61_0_1157.jpg,knife,unsafe,,knife_61,708,test,train
knife_61_0_1762.jpg,knife,unsafe,,knife_61,708,test,train
knife_61_0_4954.jpg,knife,unsafe,,knife_61,708,test,train
knife_61_0_6798.jpg,knife,unsafe,,knife_61,708,test,test
knife_61_0_699.jpg,knife,unsafe,,knife_61,708,test,train
knife_62_0_1920.jpg,knife,unsafe,,knife_62,709,val,train
knife_62_0_3211.jpg,knife,unsafe,,knife_62,709,val,val
knife_62_0_5708.jpg,knife,unsafe,,knife_62,709,val,train
knife_62_0_7394.jpg,knife,unsafe,,knife_62,709,val,train
knife_62_0_7718.jpg,knife,unsafe,,knife_62,709,val,train
knife_63_0_1693.jpg,knife,unsafe,,knife_63,710,train,train
knife_63_0_1864.jpg,knife,unsafe,,knife_63,710,train,test
knife_63_0_2331.jpg,knife,unsafe,,knife_63,710,train,val
knife_63_0_2833.jpg,knife,unsafe,,knife_63,710,train,test
knife_63_0_5336.jpg,knife,unsafe,,knife_63,710,train,test
knife_64_0_4744.jpg,knife,unsafe,,knife_64,711,train,train
knife_64_0_5527.jpg,knife,unsafe,,knife_64,711,train,train
knife_64_0_7764.jpg,knife,unsafe,,knife_64,711,train,train
knife_64_0_8702.jpg,knife,unsafe,,knife_64,711,train,val
knife_64_0_9453.jpg,knife,unsafe,,knife_64,711,train,val
knife_65_0_1122.jpg,knife,unsafe,,knife_65,712,test,train
knife_65_0_257.jpg,knife,unsafe,,knife_65,712,test,train
knife_65_0_3976.jpg,knife,unsafe,,knife_65,712,test,val
knife_65_0_8008.jpg,knife,unsafe,,knife_65,712,test,train
knife_65_0_8704.jpg,knife,unsafe,,knife_65,712,test,train
knife_66_0_1238.jpg,knife,unsafe,,knife_66,713,train,train
knife_66_0_3997.jpg,knife,unsafe,,knife_66,713,train,test
knife_66_0_4797.jpg,knife,unsafe,,knife_66,713,train,test
knife_66_0_660.jpg,knife,unsafe,,knife_66,713,train,test
knife_66_0_720.jpg,knife,unsafe,,knife_66,713,train,train
knife_67_0_1426.jpg,knife,unsafe,,knife_67,714,train,train
knife_67_0_461.jpg,knife,unsafe,,knife_67,714,train,train
knife_67_0_4682.jpg,knife,unsafe,,knife_67,714,train,train
knife_67_0_7064.jpg,knife,unsafe,,knife_67,714,train,val
knife_67_0_8808.jpg,knife,unsafe,,knife_67,714,train,test
knife_68_0_2161.jpg,knife,unsafe,,knife_68,715,val,test
knife_68_0_7176.jpg,knife,unsafe,,knife_68,715,val,train
knife_68_0_741.jpg,knife,unsafe,,knife_68,715,val,train
knife_68_0_7472.jpg,knife,unsafe,,knife_68,715,val,train
knife_68_0_8681.jpg,knife,unsafe,,knife_68,715,val,train
knife_69_0_3298.jpg,knife,unsafe,,knife_69,716,train,train
knife_69_0_3792.jpg,knife,unsafe,,knife_69,716,train,val
knife_69_0_5663.jpg,knife,unsafe,,knife_69,716,train,val
knife_69_0_6887.jpg,knife,unsafe,,knife_69,716,train,train
knife_69_0_901.jpg,knife,unsafe,,knife_69,716,train,train
knife_6_0_2926.jpg,knife,unsafe,,knife_6,717,train,train
knife_6_0_3319.jpg,knife,unsafe,,knife_6,717,train,train
knife_6_0_7858.jpg,knife,unsafe,,knife_6,717,train,train
knife_6_0_8705.jpg,knife,unsafe,,knife_6,717,train,test
knife_6_0_8908.jpg,knife,unsafe,,knife_6,717,train,train
knife_70_0_3357.jpg,knife,unsafe,,knife_70,718,test,train
knife_70_0_3735.jpg,knife,unsafe,,knife_70,718,test,val
knife_70_0_514.jpg,knife,unsafe,,knife_70,718,test,test
knife_70_0_5338.jpg,knife,unsafe,,knife_70,718,test,val
knife_70_0_9268.jpg,knife,unsafe,,knife_70,718,test,train
knife_71_0_1625.jpg,knife,unsafe,,knife_71,719,train,test
knife_71_0_3270.jpg,knife,unsafe,,knife_71,719,train,train
knife_71_0_3621.jpg,knife,unsafe,,knife_71,719,train,train
knife_71_0_478.jpg,knife,unsafe,,knife_71,719,train,train
knife_71_0_9443.jpg,knife,unsafe,,knife_71,719,train,train
knife_72_0_2909.jpg,knife,unsafe,,knife_72,720,val,train
knife_72_0_3469.jpg,knife,unsafe,,knife_72,720,val,train
knife_72_0_5248.jpg,knife,unsafe,,knife_72,720,val,train
knife_72_0_5998.jpg,knife,unsafe,,knife_72,720,val,test
knife_72_0_7409.jpg,knife,unsafe,,knife_72,720,val,train
knife_73_0_2535.jpg,knife,unsafe,,knife_73,721,test,train
knife_73_0_277.jpg,knife,unsafe,,knife_73,721,test,val
knife_73_0_5942.jpg,knife,unsafe,,knife_73,721,test,train
knife_73_0_6734.jpg,knife,unsafe,,knife_73,721,test,test
knife_73_0_8482.jpg,knife,unsafe,,knife_73,721,test,test
knife_74_0_4046.jpg,knife,unsafe,,knife_74,722,test,train
knife_74_0_4090.jpg,knife,unsafe,,knife_74,722,test,val
knife_74_0_4200.jpg,knife,unsafe,,knife_74,722,test,train
knife_74_0_7854.jpg,knife,unsafe,,knife_74,722,test,train
knife_74_0_9753.jpg,knife,unsafe,,knife_74,722,test,train
knife_75_0_4152.jpg,knife,unsafe,,knife_75,723,train,train
knife_75_0_938.jpg,knife,unsafe,,knife_75,723,train,train
knife_75_0_9389.jpg,knife,unsafe,,knife_75,723,train,train
knife_75_0_943.jpg,knife,unsafe,,knife_75,723,train,val
knife_75_0_9867.jpg,knife,unsafe,,knife_75,723,train,train
knife_76_0_1374.jpg,knife,unsafe,,knife_76,724,train,train
knife_76_0_248.jpg,knife,unsafe,,knife_76,724,train,train
knife_76_0_5796.jpg,knife,unsafe,,knife_76,724,train,test
knife_76_0_7589.jpg,knife,unsafe,,knife_76,724,train,val
knife_76_0_9926.jpg,knife,unsafe,,knife_76,724,train,train
knife_77_0_1860.jpg,knife,unsafe,,knife_77,725,val,test
knife_77_0_3306.jpg,knife,unsafe,,knife_77,725,val,train
knife_77_0_5048.jpg,knife,unsafe,,knife_77,725,val,train
knife_77_0_5479.jpg,knife,unsafe,,knife_77,725,val,train
knife_77_0_5745.jpg,knife,unsafe,,knife_77,725,val,val
knife_78_0_1491.jpg,knife,unsafe,,knife_78,726,test,train
knife_78_0_571.jpg,knife,unsafe,,knife_78,726,test,val
knife_78_0_7334.jpg,knife,unsafe,,knife_78,726,test,train
knife_78_0_7561.jpg,knife,unsafe,,knife_78,726,test,test
knife_78_0_8793.jpg,knife,unsafe,,knife_78,726,test,test
knife_79_0_3182.jpg,knife,unsafe,,knife_79,727,test,train
knife_79_0_6321.jpg,knife,unsafe,,knife_79,727,test,train
knife_79_0_6773.jpg,knife,unsafe,,knife_79,727,test,train
knife_79_0_8716.jpg,knife,unsafe,,knife_79,727,test,val
knife_79_0_981.jpg,knife,unsafe,,knife_79,727,test,train
knife_7_0_2146.jpg,knife,unsafe,,knife_7,728,train,train
knife_7_0_2171.jpg,knife,unsafe,,knife_7,728,train,train
knife_7_0_3042.jpg,knife,unsafe,,knife_7,728,train,val
knife_7_0_4359.jpg,knife,unsafe,,knife_7,728,train,test
knife_7_0_9465.jpg,knife,unsafe,,knife_7,728,train,train
knife_80_0_3726.jpg,knife,unsafe,,knife_80,729,train,train
knife_80_0_3922.jpg,knife,unsafe,,knife_80,729,train,test
knife_80_0_5037.jpg,knife,unsafe,,knife_80,729,train,train
knife_80_0_8201.jpg,knife,unsafe,,knife_80,729,train,train
knife_80_0_9842.jpg,knife,unsafe,,knife_80,729,train,train
knife_81_0_1323.jpg,knife,unsafe,,knife_81,730,train,train
knife_81_0_6970.jpg,knife,unsafe,,knife_81,730,train,train
knife_81_0_7818.jpg,knife,unsafe,,knife_81,730,train,train
knife_81_0_8108.jpg,knife,unsafe,,knife_81,730,train,train
knife_81_0_8889.jpg,knife,unsafe,,knife_81,730,train,train
knife_82_0_3331.jpg,knife,unsafe,,knife_82,731,train,train
knife_82_0_3920.jpg,knife,unsafe,,knife_82,731,train,val
knife_82_0_4737.jpg,knife,unsafe,,knife_82,731,train,train
knife_82_0_6964.jpg,knife,unsafe,,knife_82,731,train,train
knife_82_0_7993.jpg,knife,unsafe,,knife_82,731,train,train
knife_83_0_1762.jpg,knife,unsafe,,knife_83,732,val,train
knife_83_0_3362.jpg,knife,unsafe,,knife_83,732,val,val
knife_83_0_4724.jpg,knife,unsafe,,knife_83,732,val,train
knife_83_0_7569.jpg,knife,unsafe,,knife_83,732,val,test
knife_83_0_7662.jpg,knife,unsafe,,knife_83,732,val,train
knife_84_0_2957.jpg,knife,unsafe,,knife_84,733,test,train
knife_84_0_3013.jpg,knife,unsafe,,knife_84,733,test,train
knife_84_0_4460.jpg,knife,unsafe,,knife_84,733,test,train
knife_84_0_570.jpg,knife,unsafe,,knife_84,733,test,train
knife_84_0_8769.jpg,knife,unsafe,,knife_84,733,test,train
knife_8_0_3680.jpg,knife,unsafe,,knife_8,734,train,train
knife_8_0_3819.jpg,knife,unsafe,,knife_8,734,train,train
knife_8_0_766.jpg,knife,unsafe,,knife_8,734,train,train
knife_8_0_8483.jpg,knife,unsafe,,knife_8,734,train,train
knife_8_0_9886.jpg,knife,unsafe,,knife_8,734,train,train
knife_9_0_3065.jpg,knife,unsafe,,knife_9,735,train,train
knife_9_0_3853.jpg,knife,unsafe,,knife_9,735,train,train
knife_9_0_5411.jpg,knife,unsafe,,knife_9,735,train,train
knife_9_0_9000.jpg,knife,unsafe,,knife_9,735,train,train
knife_9_0_9518.jpg,knife,unsafe,,knife_9,735,train,val
seissor_10_0_2010.jpg,scissors,unsafe,,seissor_10,736,train,train
seissor_10_0_4451.jpg,scissors,unsafe,,seissor_10,736,train,val
seissor_10_0_4955.jpg,scissors,unsafe,,seissor_10,736,train,train
seissor_10_0_6225.jpg,scissors,unsafe,,seissor_10,736,train,train
seissor_10_0_7561.jpg,scissors,unsafe,,seissor_10,736,train,train
seissor_11_0_3267.jpg,scissors,unsafe,,seissor_11,737,train,train
seissor_11_0_5072.jpg,scissors,unsafe,,seissor_11,737,train,test
seissor_11_0_5117.jpg,scissors,unsafe,,seissor_11,737,train,train
seissor_11_0_5233.jpg,scissors,unsafe,,seissor_11,737,train,train
seissor_11_0_9223.jpg,scissors,unsafe,,seissor_11,737,train,train
seissor_13_0_1813.jpg,scissors,unsafe,,seissor_13,738,train,train
seissor_13_0_5185.jpg,scissors,unsafe,,seissor_13,738,train,train
seissor_13_0_5501.jpg,scissors,unsafe,,seissor_13,738,train,test
seissor_13_0_8988.jpg,scissors,unsafe,,seissor_13,738,train,test
seissor_13_0_9340.jpg,scissors,unsafe,,seissor_13,738,train,train
seissor_14_0_2426.jpg,scissors,unsafe,,seissor_14,739,test,train
seissor_14_0_5119.jpg,scissors,unsafe,,seissor_14,739,test,train
seissor_14_0_536.jpg,scissors,unsafe,,seissor_14,739,test,train
seissor_14_0_6430.jpg,scissors,unsafe,,seissor_14,739,test,train
seissor_14_0_7746.jpg,scissors,unsafe,,seissor_14,739,test,train
seissor_15_0_6717.jpg,scissors,unsafe,,seissor_15,740,val,train
seissor_15_0_6782.jpg,scissors,unsafe,,seissor_15,740,val,train
seissor_15_0_7713.jpg,scissors,unsafe,,seissor_15,740,val,train
seissor_15_0_9319.jpg,scissors,unsafe,,seissor_15,740,val,train
seissor_15_0_9995.jpg,scissors,unsafe,,seissor_15,740,val,val
seissor_16_0_1058.jpg,scissors,unsafe,,seissor_16,741,train,val
seissor_16_0_5324.jpg,scissors,unsafe,,seissor_16,741,train,train
seissor_16_0_8753.jpg,scissors,unsafe,,seissor_16,741,train,train
seissor_16_0_9016.jpg,scissors,unsafe,,seissor_16,741,train,train
seissor_16_0_9475.jpg,scissors,unsafe,,seissor_16,741,train,train
seissor_17_0_1403.jpg,scissors,unsafe,,seissor_17,742,val,train
seissor_17_0_5914.jpg,scissors,unsafe,,seissor_17,742,val,train
seissor_17_0_7894.jpg,scissors,unsafe,,seissor_17,742,val,train
seissor_17_0_872.jpg,scissors,unsafe,,seissor_17,742,val,train
seissor_18_0_1383.jpg,scissors,unsafe,,seissor_18,743,train,train
seissor_18_0_1872.jpg,scissors,unsafe,,seissor_18,743,train,train
seissor_18_0_4913.jpg,scissors,unsafe,,seissor_18,743,train,train
seissor_18_0_6467.jpg,scissors,unsafe,,seissor_18,743,train,train
seissor_18_0_9818.jpg,scissors,unsafe,,seissor_18,743,train,train
seissor_19_0_1010.jpg,scissors,unsafe,,seissor_19,744,train,train
seissor_19_0_589.jpg,scissors,unsafe,,seissor_19,744,train,train
seissor_19_0_7499.jpg,scissors,unsafe,,seissor_19,744,train,val
seissor_19_0_8714.jpg,scissors,unsafe,,seissor_19,744,train,val
seissor_19_0_9222.jpg,scissors,unsafe,,seissor_19,744,train,train
seissor_21_0_497.jpg,scissors,unsafe,,seissor_21,745,test,train
seissor_21_0_6175.jpg,scissors,unsafe,,seissor_21,745,test,train
seissor_21_0_7582.jpg,scissors,unsafe,,seissor_21,745,test,train
seissor_21_0_8007.jpg,scissors,unsafe,,seissor_21,745,test,val
seissor_21_0_9543.jpg,scissors,unsafe,,seissor_21,745,test,train
seissor_22_0_215.jpg,scissors,unsafe,,seissor_22,746,test,train
seissor_22_0_225.jpg,scissors,unsafe,,seissor_22,746,test,train
seissor_22_0_2564.jpg,scissors,unsafe,,seissor_22,746,test,train
seissor_22_0_5322.jpg,scissors,unsafe,,seissor_22,746,test,val
seissor_22_0_8502.jpg,scissors,unsafe,,seissor_22,746,test,val
seissor_26_0_1297.jpg,scissors,unsafe,,seissor_26,747,train,train
seissor_26_0_2997.jpg,scissors,unsafe,,seissor_26,747,train,train
seissor_26_0_3113.jpg,scissors,unsafe,,seissor_26,747,train,train
seissor_26_0_5780.jpg,scissors,unsafe,,seissor_26,747,train,test
seissor_26_0_7594.jpg,scissors,unsafe,,seissor_26,747,train,train
seissor_27_0_2673.jpg,scissors,unsafe,,seissor_27,748,train,train
seissor_27_0_3600.jpg,scissors,unsafe,,seissor_27,748,train,train
seissor_27_0_7101.jpg,scissors,unsafe,,seissor_27,748,train,test
seissor_27_0_7142.jpg,scissors,unsafe,,seissor_27,748,train,test
seissor_27_0_876.jpg,scissors,unsafe,,seissor_27,748,train,train
seissor_28_0_2127.jpg,scissors,unsafe,,seissor_28,749,train,test
seissor_28_0_7379.jpg,scissors,unsafe,,seissor_28,749,train,val
seissor_28_0_7574.jpg,scissors,unsafe,,seissor_28,749,train,train
seissor_28_0_8349.jpg,scissors,unsafe,,seissor_28,749,train,train
seissor_28_0_8889.jpg,scissors,unsafe,,seissor_28,749,train,val
seissor_29_0_1554.jpg,scissors,unsafe,,seissor_29,750,train,train
seissor_29_0_3260.jpg,scissors,unsafe,,seissor_29,750,train,train
seissor_29_0_4029.jpg,scissors,unsafe,,seissor_29,750,train,train
seissor_29_0_5792.jpg,scissors,unsafe,,seissor_29,750,train,train
seissor_29_0_9388.jpg,scissors,unsafe,,seissor_29,750,train,val
seissor_2_0_1124.jpg,scissors,unsafe,,seissor_2,751,train,train
seissor_2_0_1347.jpg,scissors,unsafe,,seissor_2,751,train,val
seissor_2_0_2932.jpg,scissors,unsafe,,seissor_2,751,train,train
seissor_2_0_3986.jpg,scissors,unsafe,,seissor_2,751,train,train
seissor_2_0_7713.jpg,scissors,unsafe,,seissor_2,751,train,test
seissor_31_0_139.jpg,scissors,unsafe,,seissor_31,752,test,val
seissor_31_0_2647.jpg,scissors,unsafe,,seissor_31,752,test,val
seissor_31_0_3170.jpg,scissors,unsafe,,seissor_31,752,test,test
seissor_31_0_3566.jpg,scissors,unsafe,,seissor_31,752,test,train
seissor_31_0_5113.jpg,scissors,unsafe,,seissor_31,752,test,train
seissor_32_0_3414.jpg,scissors,unsafe,,seissor_32,753,train,train
seissor_32_0_5927.jpg,scissors,unsafe,,seissor_32,753,train,test
seissor_32_0_6525.jpg,scissors,unsafe,,seissor_32,753,train,train
seissor_32_0_7148.jpg,scissors,unsafe,,seissor_32,753,train,train
seissor_32_0_7277.jpg,scissors,unsafe,,seissor_32,753,train,train
seissor_33_0_1635.jpg,scissors,unsafe,,seissor_33,754,train,val
seissor_33_0_365.jpg,scissors,unsafe,,seissor_33,754,train,train
seissor_33_0_4438.jpg,scissors,unsafe,,seissor_33,754,train,test
seissor_33_0_7323.jpg,scissors,unsafe,,seissor_33,754,train,train
seissor_33_0_91.jpg,scissors,unsafe,,seissor_33,754,train,train
seissor_34_0_3355.jpg,scissors,unsafe,,seissor_34,755,val,val
seissor_34_0_5147.jpg,scissors,unsafe,,seissor_34,755,val,train
seissor_34_0_6368.jpg,scissors,unsafe,,seissor_34,755,val,train
seissor_34_0_8385.jpg,scissors,unsafe,,seissor_34,755,val,train
seissor_34_0_8867.jpg,scissors,unsafe,,seissor_34,755,val,train
seissor_35_0_1568.jpg,scissors,unsafe,,seissor_35,756,train,train
seissor_35_0_1772.jpg,scissors,unsafe,,seissor_35,756,train,train
seissor_35_0_3077.jpg,scissors,unsafe,,seissor_35,756,train,train
seissor_35_0_5169.jpg,scissors,unsafe,,seissor_35,756,train,train
seissor_35_0_9835.jpg,scissors,unsafe,,seissor_35,756,train,train
seissor_38_0_3252.jpg,scissors,unsafe,,seissor_38,757,train,train
seissor_38_0_3714.jpg,scissors,unsafe,,seissor_38,757,train,train
seissor_38_0_5296.jpg,scissors,unsafe,,seissor_38,757,train,train
seissor_38_0_7346.jpg,scissors,unsafe,,seissor_38,757,train,train
seissor_38_0_8216.jpg,scissors,unsafe,,seissor_38,757,train,val
seissor_39_0_4189.jpg,scissors,unsafe,,seissor_39,758,val,train
seissor_39_0_526.jpg,scissors,unsafe,,seissor_39,758,val,train
seissor_39_0_8699.jpg,scissors,unsafe,,seissor_39,758,val,train
seissor_39_0_8732.jpg,scissors,unsafe,,seissor_39,758,val,train
seissor_39_0_9910.jpg,scissors,unsafe,,seissor_39,758,val,train
seissor_3_0_1487.jpg,scissors,unsafe,,seissor_3,759,train,train
seissor_3_0_3093.jpg,scissors,unsafe,,seissor_3,759,train,test
seissor_3_0_5384.jpg,scissors,unsafe,,seissor_3,759,train,train
seissor_3_0_7574.jpg,scissors,unsafe,,seissor_3,759,train,train
seissor_3_0_8483.jpg,scissors,unsafe,,seissor_3,759,train,train
seissor_40_0_1291.jpg,scissors,unsafe,,seissor_40,760,test,test
seissor_40_0_1754.jpg,scissors,unsafe,,seissor_40,760,test,train
seissor_40_0_3176.jpg,scissors,unsafe,,seissor_40,760,test,val
seissor_40_0_61.jpg,scissors,unsafe,,seissor_40,760,test,train
seissor_40_0_7027.jpg,scissors,unsafe,,seissor_40,760,test,train
seissor_41_0_1127.jpg,scissors,unsafe,,seissor_41,761,train,train
seissor_41_0_1617.jpg,scissors,unsafe,,seissor_41,761,train,train
seissor_41_0_5931.jpg,scissors,unsafe,,seissor_41,761,train,val
seissor_41_0_6306.jpg,scissors,unsafe,,seissor_41,761,train,train
seissor_41_0_8326.jpg,scissors,unsafe,,seissor_41,761,train,val
seissor_42_0_2533.jpg,scissors,unsafe,,seissor_42,762,train,test
seissor_42_0_2987.jpg,scissors,unsafe,,seissor_42,762,train,train
seissor_42_0_4221.jpg,scissors,unsafe,,seissor_42,762,train,train
seissor_42_0_5614.jpg,scissors,unsafe,,seissor_42,762,train,test
seissor_42_0_9914.jpg,scissors,unsafe,,seissor_42,762,train,train
seissor_43_0_1637.jpg,scissors,unsafe,,seissor_43,763,train,val
seissor_43_0_2247.jpg,scissors,unsafe,,seissor_43,763,train,train
seissor_43_0_6510.jpg,scissors,unsafe,,seissor_43,763,train,train
seissor_43_0_8415.jpg,scissors,unsafe,,seissor_43,763,train,test
seissor_43_0_9528.jpg,scissors,unsafe,,seissor_43,763,train,train
seissor_44_0_1739.jpg,scissors,unsafe,,seissor_44,764,train,train
seissor_44_0_6719.jpg,scissors,unsafe,,seissor_44,764,train,train
seissor_44_0_6740.jpg,scissors,unsafe,,seissor_44,764,train,train
seissor_44_0_8814.jpg,scissors,unsafe,,seissor_44,764,train,train
seissor_44_0_9971.jpg,scissors,unsafe,,seissor_44,764,train,train
seissor_45_0_1523.jpg,scissors,unsafe,,seissor_45,765,test,train
seissor_45_0_3968.jpg,scissors,unsafe,,seissor_45,765,test,train
seissor_45_0_4043.jpg,scissors,unsafe,,seissor_45,765,test,train
seissor_45_0_842.jpg,scissors,unsafe,,seissor_45,765,test,val
seissor_45_0_8731.jpg,scissors,unsafe,,seissor_45,765,test,test
seissor_46_0_1901.jpg,scissors,unsafe,,seissor_46,766,train,train
seissor_46_0_2205.jpg,scissors,unsafe,,seissor_46,766,train,train
seissor_46_0_4259.jpg,scissors,unsafe,,seissor_46,766,train,train
seissor_46_0_7391.jpg,scissors,unsafe,,seissor_46,766,train,val
seissor_46_0_8913.jpg,scissors,unsafe,,seissor_46,766,train,val
seissor_47_0_2490.jpg,scissors,unsafe,,seissor_47,767,train,test
seissor_47_0_627.jpg,scissors,unsafe,,seissor_47,767,train,train
seissor_47_0_8083.jpg,scissors,unsafe,,seissor_47,767,train,train
seissor_47_0_8295.jpg,scissors,unsafe,,seissor_47,767,train,train
seissor_47_0_8840.jpg,scissors,unsafe,,seissor_47,767,train,val
seissor_48_0_1378.jpg,scissors,unsafe,,seissor_48,768,train,test
seissor_48_0_2265.jpg,scissors,unsafe,,seissor_48,768,train,train
seissor_48_0_2583.jpg,scissors,unsafe,,seissor_48,768,train,train
seissor_48_0_3149.jpg,scissors,unsafe,,seissor_48,768,train,val
seissor_48_0_9026.jpg,scissors,unsafe,,seissor_48,768,train,test
seissor_4_0_1629.jpg,scissors,unsafe,,seissor_4,769,val,val
seissor_4_0_4198.jpg,scissors,unsafe,,seissor_4,769,val,train
seissor_4_0_4880.jpg,scissors,unsafe,,seissor_4,769,val,test
seissor_4_0_7038.jpg,scissors,unsafe,,seissor_4,769,val,train
seissor_4_0_9157.jpg,scissors,unsafe,,seissor_4,769,val,train
seissor_52_0_4100.jpg,scissors,unsafe,,seissor_52,770,val,val
seissor_52_0_4804.jpg,scissors,unsafe,,seissor_52,770,val,train
seissor_52_0_4932.jpg,scissors,unsafe,,seissor_52,770,val,val
seissor_52_0_5001.jpg,scissors,unsafe,,seissor_52,770,val,test
seissor_52_0_7433.jpg,scissors,unsafe,,seissor_52,770,val,train
seissor_53_0_2993.jpg,scissors,unsafe,,seissor_53,771,train,train
seissor_53_0_4161.jpg,scissors,unsafe,,seissor_53,771,train,train
seissor_53_0_6680.jpg,scissors,unsafe,,seissor_53,771,train,train
seissor_53_0_6715.jpg,scissors,unsafe,,seissor_53,771,train,test
seissor_53_0_8801.jpg,scissors,unsafe,,seissor_53,771,train,train
seissor_54_0_3006.jpg,scissors,unsafe,,seissor_54,772,train,train
seissor_54_0_3950.jpg,scissors,unsafe,,seissor_54,772,train,test
seissor_54_0_8173.jpg,scissors,unsafe,,seissor_54,772,train,train
seissor_54_0_8343.jpg,scissors,unsafe,,seissor_54,772,train,test
seissor_54_0_8828.jpg,scissors,unsafe,,seissor_54,772,train,val
seissor_55_0_1644.jpg,scissors,unsafe,,seissor_55,773,train,train
seissor_55_0_2036.jpg,scissors,unsafe,,seissor_55,773,train,train
seissor_55_0_2394.jpg,scissors,unsafe,,seissor_55,773,train,train
seissor_55_0_6269.jpg,scissors,unsafe,,seissor_55,773,train,train
seissor_55_0_7427.jpg,scissors,unsafe,,seissor_55,773,train,val
seissor_56_0_2726.jpg,scissors,unsafe,,seissor_56,774,train,val
seissor_56_0_4869.jpg,scissors,unsafe,,seissor_56,774,train,test
seissor_56_0_5224.jpg,scissors,unsafe,,seissor_56,774,train,train
seissor_56_0_569.jpg,scissors,unsafe,,seissor_56,774,train,train
seissor_56_0_7317.jpg,scissors,unsafe,,seissor_56,774,train,val
seissor_57_0_2071.jpg,scissors,unsafe,,seissor_57,775,train,train
seissor_57_0_7620.jpg,scissors,unsafe,,seissor_57,775,train,train
seissor_57_0_8134.jpg,scissors,unsafe,,seissor_57,775,train,train
seissor_57_0_9174.jpg,scissors,unsafe,,seissor_57,775,train,val
seissor_57_0_9785.jpg,scissors,unsafe,,seissor_57,775,train,test
seissor_58_0_1337.jpg,scissors,unsafe,,seissor_58,776,train,train
seissor_58_0_172.jpg,scissors,unsafe,,seissor_58,776,train,train
seissor_58_0_2157.jpg,scissors,unsafe,,seissor_58,776,train,train
seissor_58_0_6280.jpg,scissors,unsafe,,seissor_58,776,train,val
seissor_58_0_7071.jpg,scissors,unsafe,,seissor_58,776,train,train
seissor_59_0_3628.jpg,scissors,unsafe,,seissor_59,777,train,train
seissor_59_0_5208.jpg,scissors,unsafe,,seissor_59,777,train,train
seissor_59_0_5779.jpg,scissors,unsafe,,seissor_59,777,train,train
seissor_59_0_5930.jpg,scissors,unsafe,,seissor_59,777,train,train
seissor_59_0_7999.jpg,scissors,unsafe,,seissor_59,777,train,train
seissor_5_0_1100.jpg,scissors,unsafe,,seissor_5,778,test,train
seissor_5_0_2739.jpg,scissors,unsafe,,seissor_5,778,test,train
seissor_5_0_5915.jpg,scissors,unsafe,,seissor_5,778,test,train
seissor_5_0_5961.jpg,scissors,unsafe,,seissor_5,778,test,test
seissor_5_0_7445.jpg,scissors,unsafe,,seissor_5,778,test,test
seissor_61_0_1178.jpg,scissors,unsafe,,seissor_61,779,val,test
seissor_61_0_1800.jpg,scissors,unsafe,,seissor_61,779,val,train
seissor_61_0_4146.jpg,scissors,unsafe,,seissor_61,779,val,val
seissor_61_0_4389.jpg,scissors,unsafe,,seissor_61,779,val,train
seissor_61_0_5401.jpg,scissors,unsafe,,seissor_61,779,val,test
seissor_62_0_2260.jpg,scissors,unsafe,,seissor_62,780,val,train
seissor_62_0_5318.jpg,scissors,unsafe,,seissor_62,780,val,train
seissor_62_0_6851.jpg,scissors,unsafe,,seissor_62,780,val,train
seissor_62_0_756.jpg,scissors,unsafe,,seissor_62,780,val,train
seissor_62_0_9676.jpg,scissors,unsafe,,seissor_62,780,val,test
seissor_63_0_1567.jpg,scissors,unsafe,,seissor_63,781,train,train
seissor_63_0_1675.jpg,scissors,unsafe,,seissor_63,781,train,train
seissor_63_0_2253.jpg,scissors,unsafe,,seissor_63,781,train,test
seissor_63_0_5296.jpg,scissors,unsafe,,seissor_63,781,train,val
seissor_63_0_5440.jpg,scissors,unsafe,,seissor_63,781,train,test
seissor_6_0_472.jpg,scissors,unsafe,,seissor_6,782,train,train
seissor_6_0_5582.jpg,scissors,unsafe,,seissor_6,782,train,test
seissor_6_0_6557.jpg,scissors,unsafe,,seissor_6,782,train,train
seissor_6_0_8008.jpg,scissors,unsafe,,seissor_6,782,train,train
seissor_6_0_9041.jpg,scissors,unsafe,,seissor_6,782,train,train
seissor_7_0_1889.jpg,scissors,unsafe,,seissor_7,783,train,train
seissor_7_0_2375.jpg,scissors,unsafe,,seissor_7,783,train,train
seissor_7_0_3492.jpg,scissors,unsafe,,seissor_7,783,train,train
seissor_7_0_3747.jpg,scissors,unsafe,,seissor_7,783,train,train
seissor_7_0_7892.jpg,scissors,unsafe,,seissor_7,783,train,test
seissor_9_0_129.jpg,scissors,unsafe,,seissor_9,784,train,train
seissor_9_0_252.jpg,scissors,unsafe,,seissor_9,784,train,train
seissor_9_0_2803.jpg,scissors,unsafe,,seissor_9,784,train,test
seissor_9_0_4225.jpg,scissors,unsafe,,seissor_9,784,train,train
seissor_9_0_6172.jpg,scissors,unsafe,,seissor_9,784,train,train
socket_10_0_1579.jpg,socket,unsafe,,socket_10,785,test,train
socket_10_0_2753.jpg,socket,unsafe,,socket_10,785,test,train
socket_10_0_3486.jpg,socket,unsafe,,socket_10,785,test,train
socket_10_0_6155.jpg,socket,unsafe,,socket_10,785,test,train
socket_10_0_949.jpg,socket,unsafe,,socket_10,785,test,train
socket_11_0_1172.jpg,socket,unsafe,,socket_11,786,train,train
socket_11_0_167.jpg,socket,unsafe,,socket_11,786,train,train
socket_11_0_5003.jpg,socket,unsafe,,socket_11,786,train,train
socket_11_0_6563.jpg,socket,unsafe,,socket_11,786,train,train
socket_11_0_7798.jpg,socket,unsafe,,socket_11,786,train,train
socket_12_0_5041.jpg,socket,unsafe,,socket_12,787,test,train
socket_12_0_6449.jpg,socket,unsafe,,socket_12,787,test,train
socket_12_0_6494.jpg,socket,unsafe,,socket_12,787,test,val
socket_12_0_6857.jpg,socket,unsafe,,socket_12,787,test,train
socket_12_0_9361.jpg,socket,unsafe,,socket_12,787,test,train
socket_13_0_4520.jpg,socket,unsafe,,socket_13,788,val,train
socket_13_0_5575.jpg,socket,unsafe,,socket_13,788,val,train
socket_13_0_5852.jpg,socket,unsafe,,socket_13,788,val,train
socket_13_0_8327.jpg,socket,unsafe,,socket_13,788,val,test
socket_13_0_9222.jpg,socket,unsafe,,socket_13,788,val,train
socket_14_0_1241.jpg,socket,unsafe,,socket_14,789,train,train
socket_14_0_2095.jpg,socket,unsafe,,socket_14,789,train,train
socket_14_0_2516.jpg,socket,unsafe,,socket_14,789,train,train
socket_14_0_2808.jpg,socket,unsafe,,socket_14,789,train,val
socket_14_0_8579.jpg,socket,unsafe,,socket_14,789,train,train
socket_15_0_2789.jpg,socket,unsafe,,socket_15,790,train,train
socket_15_0_3274.jpg,socket,unsafe,,socket_15,790,train,train
socket_15_0_4084.jpg,socket,unsafe,,socket_15,790,train,train
socket_15_0_5178.jpg,socket,unsafe,,socket_15,790,train,train
socket_15_0_9734.jpg,socket,unsafe,,socket_15,790,train,val
socket_16_0_2547.jpg,socket,unsafe,,socket_16,791,train,train
socket_16_0_4883.jpg,socket,unsafe,,socket_16,791,train,train
socket_16_0_6607.jpg,socket,unsafe,,socket_16,791,train,val
socket_16_0_6800.jpg,socket,unsafe,,socket_16,791,train,train
socket_16_0_8100.jpg,socket,unsafe,,socket_16,791,train,train
socket_17_0_6241.jpg,socket,unsafe,,socket_17,792,train,train
socket_17_0_7074.jpg,socket,unsafe,,socket_17,792,train,train
socket_17_0_8401.jpg,socket,unsafe,,socket_17,792,train,test
socket_17_0_8995.jpg,socket,unsafe,,socket_17,792,train,train
socket_17_0_9207.jpg,socket,unsafe,,socket_17,792,train,train
socket_18_0_200.jpg,socket,unsafe,,socket_18,793,train,val
socket_18_0_6432.jpg,socket,unsafe,,socket_18,793,train,train
socket_18_0_7372.jpg,socket,unsafe,,socket_18,793,train,train
socket_18_0_7662.jpg,socket,unsafe,,socket_18,793,train,train
socket_18_0_9903.jpg,socket,unsafe,,socket_18,793,train,train
socket_19_0_4884.jpg,socket,unsafe,,socket_19,794,train,train
socket_19_0_5354.jpg,socket,unsafe,,socket_19,794,train,test
socket_19_0_6883.jpg,socket,unsafe,,socket_19,794,train,train
socket_19_0_7984.jpg,socket,unsafe,,socket_19,794,train,test
socket_19_0_8732.jpg,socket,unsafe,,socket_19,794,train,test
socket_1_0_1844.jpg,socket,unsafe,,socket_1,795,test,train
socket_1_0_3797.jpg,socket,unsafe,,socket_1,795,test,train
socket_1_0_4495.jpg,socket,unsafe,,socket_1,795,test,train
socket_1_0_7001.jpg,socket,unsafe,,socket_1,795,test,train
socket_1_0_7612.jpg,socket,unsafe,,socket_1,795,test,train
socket_20_0_1522.jpg,socket,unsafe,,socket_20,796,train,train
socket_20_0_2768.jpg,socket,unsafe,,socket_20,796,train,train
socket_20_0_6279.jpg,socket,unsafe,,socket_20,796,train,train
socket_20_0_6528.jpg,socket,unsafe,,socket_20,796,train,train
socket_20_0_8192.jpg,socket,unsafe,,socket_20,796,train,test
socket_21_0_2327.jpg,socket,unsafe,,socket_21,797,train,train
socket_21_0_3167.jpg,socket,unsafe,,socket_21,797,train,train
socket_21_0_3818.jpg,socket,unsafe,,socket_21,797,train,val
socket_21_0_7673.jpg,socket,unsafe,,socket_21,797,train,train
socket_21_0_820.jpg,socket,unsafe,,socket_21,797,train,train
socket_22_0_4420.jpg,socket,unsafe,,socket_22,798,test,train
socket_22_0_4854.jpg,socket,unsafe,,socket_22,798,test,train
socket_22_0_6305.jpg,socket,unsafe,,socket_22,798,test,train
socket_22_0_6851.jpg,socket,unsafe,,socket_22,798,test,train
socket_22_0_9822.jpg,socket,unsafe,,socket_22,798,test,train
socket_23_0_2985.jpg,socket,unsafe,,socket_23,799,train,train
socket_23_0_3547.jpg,socket,unsafe,,socket_23,799,train,train
socket_23_0_3806.jpg,socket,unsafe,,socket_23,799,train,train
socket_23_0_4001.jpg,socket,unsafe,,socket_23,799,train,test
socket_23_0_7738.jpg,socket,unsafe,,socket_23,799,train,test
socket_24_0_2288.jpg,socket,unsafe,,socket_24,800,train,test
socket_24_0_2592.jpg,socket,unsafe,,socket_24,800,train,val
socket_24_0_6001.jpg,socket,unsafe,,socket_24,800,train,test
socket_24_0_7444.jpg,socket,unsafe,,socket_24,800,train,val
socket_24_0_8827.jpg,socket,unsafe,,socket_24,800,train,train
socket_25_0_100.jpg,socket,unsafe,,socket_25,801,val,train
socket_25_0_4594.jpg,socket,unsafe,,socket_25,801,val,train
socket_25_0_4850.jpg,socket,unsafe,,socket_25,801,val,train
socket_25_0_6247.jpg,socket,unsafe,,socket_25,801,val,train
socket_25_0_9723.jpg,socket,unsafe,,socket_25,801,val,train
socket_26_0_2820.jpg,socket,unsafe,,socket_26,802,train,train
socket_26_0_5238.jpg,socket,unsafe,,socket_26,802,train,train
socket_26_0_5312.jpg,socket,unsafe,,socket_26,802,train,train
socket_26_0_5914.jpg,socket,unsafe,,socket_26,802,train,train
socket_26_0_6440.jpg,socket,unsafe,,socket_26,802,train,train
socket_27_0_4502.jpg,socket,unsafe,,socket_27,803,val,train
socket_27_0_4683.jpg,socket,unsafe,,socket_27,803,val,train
socket_27_0_4885.jpg,socket,unsafe,,socket_27,803,val,val
socket_27_0_5195.jpg,socket,unsafe,,socket_27,803,val,train
socket_27_0_9583.jpg,socket,unsafe,,socket_27,803,val,train
socket_28_0_1718.jpg,socket,unsafe,,socket_28,804,train,train
socket_28_0_3829.jpg,socket,unsafe,,socket_28,804,train,train
socket_28_0_537.jpg,socket,unsafe,,socket_28,804,train,train
socket_28_0_6547.jpg,socket,unsafe,,socket_28,804,train,val
socket_28_0_897.jpg,socket,unsafe,,socket_28,804,train,train
socket_29_0_3099.jpg,socket,unsafe,,socket_29,805,test,train
socket_29_0_5465.jpg,socket,unsafe,,socket_29,805,test,train
socket_29_0_7420.jpg,socket,unsafe,,socket_29,805,test,train
socket_29_0_8337.jpg,socket,unsafe,,socket_29,805,test,train
socket_29_0_8807.jpg,socket,unsafe,,socket_29,805,test,train
socket_2_0_1852.jpg,socket,unsafe,,socket_2,806,test,train
socket_2_0_380.jpg,socket,unsafe,,socket_2,806,test,train
socket_2_0_7242.jpg,socket,unsafe,,socket_2,806,test,train
socket_2_0_7308.jpg,socket,unsafe,,socket_2,806,test,test
socket_2_0_9363.jpg,socket,unsafe,,socket_2,806,test,train
socket_30_0_1921.jpg,socket,unsafe,,socket_30,807,train,train
socket_30_0_2860.jpg,socket,unsafe,,socket_30,807,train,val
socket_30_0_5305.jpg,socket,unsafe,,socket_30,807,train,test
socket_30_0_5629.jpg,socket,unsafe,,socket_30,807,train,train
socket_30_0_914.jpg,socket,unsafe,,socket_30,807,train,train
socket_31_0_2963.jpg,socket,unsafe,,socket_31,808,train,train
socket_31_0_3302.jpg,socket,unsafe,,socket_31,808,train,test
socket_31_0_6638.jpg,socket,unsafe,,socket_31,808,train,test
socket_31_0_6773.jpg,socket,unsafe,,socket_31,808,train,train
socket_31_0_9656.jpg,socket,unsafe,,socket_31,808,train,val
socket_32_0_230.jpg,socket,unsafe,,socket_32,809,test,train
socket_32_0_2877.jpg,socket,unsafe,,socket_32,809,test,train
socket_32_0_3333.jpg,socket,unsafe,,socket_32,809,test,val
socket_32_0_4140.jpg,socket,unsafe,,socket_32,809,test,test
socket_32_0_9409.jpg,socket,unsafe,,socket_32,809,test,val
socket_33_0_4906.jpg,socket,unsafe,,socket_33,810,train,val
socket_33_0_5248.jpg,socket,unsafe,,socket_33,810,train,train
socket_33_0_5346.jpg,socket,unsafe,,socket_33,810,train,train
socket_33_0_5710.jpg,socket,unsafe,,socket_33,810,train,train
socket_33_0_6025.jpg,socket,unsafe,,socket_33,810,train,train
socket_34_0_6607.jpg,socket,unsafe,,socket_34,811,train,train
socket_34_0_6868.jpg,socket,unsafe,,socket_34,811,train,test
socket_34_0_7005.jpg,socket,unsafe,,socket_34,811,train,train
socket_34_0_7520.jpg,socket,unsafe,,socket_34,811,train,train
socket_34_0_8080.jpg,socket,unsafe,,socket_34,811,train,train
socket_35_0_214.jpg,socket,unsafe,,socket_35,812,val,train
socket_35_0_3452.jpg,socket,unsafe,,socket_35,812,val,test
socket_35_0_6223.jpg,socket,unsafe,,socket_35,812,val,test
socket_35_0_6227.jpg,socket,unsafe,,socket_35,812,val,train
socket_35_0_6505.jpg,socket,unsafe,,socket_35,812,val,train
socket_36_0_3855.jpg,socket,unsafe,,socket_36,813,train,train
socket_36_0_4295.jpg,socket,unsafe,,socket_36,813,train,train
socket_36_0_4303.jpg,socket,unsafe,,socket_36,813,train,train
socket_36_0_8221.jpg,socket,unsafe,,socket_36,813,train,train
socket_36_0_9871.jpg,socket,unsafe,,socket_36,813,train,train
socket_37_0_2349.jpg,socket,unsafe,,socket_37,814,train,test
socket_37_0_2866.jpg,socket,unsafe,,socket_37,814,train,train
socket_37_0_3163.jpg,socket,unsafe,,socket_37,814,train,val
socket_37_0_3534.jpg,socket,unsafe,,socket_37,814,train,train
socket_37_0_7547.jpg,socket,unsafe,,socket_37,814,train,train
socket_38_0_1289.jpg,socket,unsafe,,socket_38,815,val,val
socket_38_0_4916.jpg,socket,unsafe,,socket_38,815,val,val
socket_38_0_543.jpg,socket,unsafe,,socket_38,815,val,train
socket_38_0_682.jpg,socket,unsafe,,socket_38,815,val,train
socket_38_0_9831.jpg,socket,unsafe,,socket_38,815,val,train
socket_39_0_1547.jpg,socket,unsafe,,socket_39,816,train,train
socket_39_0_3530.jpg,socket,unsafe,,socket_39,816,train,train
socket_39_0_6339.jpg,socket,unsafe,,socket_39,816,train,train
socket_39_0_7549.jpg,socket,unsafe,,socket_39,816,train,test
socket_39_0_9760.jpg,socket,unsafe,,socket_39,816,train,train
socket_3_0_2899.jpg,socket,unsafe,,socket_3,817,train,test
socket_3_0_3171.jpg,socket,unsafe,,socket_3,817,train,test
socket_3_0_5485.jpg,socket,unsafe,,socket_3,817,train,test
socket_3_0_58.jpg,socket,unsafe,,socket_3,817,train,train
socket_3_0_7035.jpg,socket,unsafe,,socket_3,817,train,train
socket_40_0_4833.jpg,socket,unsafe,,socket_40,818,train,train
socket_40_0_5086.jpg,socket,unsafe,,socket_40,818,train,train
socket_40_0_5407.jpg,socket,unsafe,,socket_40,818,train,val
socket_40_0_6658.jpg,socket,unsafe,,socket_40,818,train,train
socket_40_0_8154.jpg,socket,unsafe,,socket_40,818,train,train
socket_41_0_1144.jpg,socket,unsafe,,socket_41,819,train,train
socket_41_0_3506.jpg,socket,unsafe,,socket_41,819,train,train
socket_41_0_5025.jpg,socket,unsafe,,socket_41,819,train,train
socket_41_0_5311.jpg,socket,unsafe,,socket_41,819,train,train
socket_41_0_5696.jpg,socket,unsafe,,socket_41,819,train,train
socket_42_0_2586.jpg,socket,unsafe,,socket_42,820,train,val
socket_42_0_5050.jpg,socket,unsafe,,socket_42,820,train,train
socket_42_0_5958.jpg,socket,unsafe,,socket_42,820,train,train
socket_42_0_6300.jpg,socket,unsafe,,socket_42,820,train,train
socket_42_0_7585.jpg,socket,unsafe,,socket_42,820,train,train
socket_43_0_2088.jpg,socket,unsafe,,socket_43,821,train,train
socket_43_0_2275.jpg,socket,unsafe,,socket_43,821,train,test
socket_43_0_3299.jpg,socket,unsafe,,socket_43,821,train,test
socket_43_0_4626.jpg,socket,unsafe,,socket_43,821,train,val
socket_43_0_6114.jpg,socket,unsafe,,socket_43,821,train,train
socket_44_0_1698.jpg,socket,unsafe,,socket_44,822,train,val
socket_44_0_1819.jpg,socket,unsafe,,socket_44,822,train,train
socket_44_0_5335.jpg,socket,unsafe,,socket_44,822,train,test
socket_44_0_7383.jpg,socket,unsafe,,socket_44,822,train,train
socket_44_0_739.jpg,socket,unsafe,,socket_44,822,train,train
socket_45_0_1497.jpg,socket,unsafe,,socket_45,823,train,train
socket_45_0_4499.jpg,socket,unsafe,,socket_45,823,train,train
socket_45_0_6219.jpg,socket,unsafe,,socket_45,823,train,train
socket_45_0_6601.jpg,socket,unsafe,,socket_45,823,train,val
socket_45_0_8615.jpg,socket,unsafe,,socket_45,823,train,val
socket_46_0_2132.jpg,socket,unsafe,,socket_46,824,train,train
socket_46_0_2718.jpg,socket,unsafe,,socket_46,824,train,train
socket_46_0_4840.jpg,socket,unsafe,,socket_46,824,train,test
socket_46_0_609.jpg,socket,unsafe,,socket_46,824,train,train
socket_46_0_9903.jpg,socket,unsafe,,socket_46,824,train,train
socket_47_0_4621.jpg,socket,unsafe,,socket_47,825,val,train
socket_47_0_5593.jpg,socket,unsafe,,socket_47,825,val,test
socket_47_0_7328.jpg,socket,unsafe,,socket_47,825,val,val
socket_47_0_7623.jpg,socket,unsafe,,socket_47,825,val,val
socket_47_0_9458.jpg,socket,unsafe,,socket_47,825,val,test
socket_48_0_1084.jpg,socket,unsafe,,socket_48,826,train,train
socket_48_0_1919.jpg,socket,unsafe,,socket_48,826,train,train
socket_48_0_3844.jpg,socket,unsafe,,socket_48,826,train,train
socket_48_0_7569.jpg,socket,unsafe,,socket_48,826,train,train
socket_48_0_7927.jpg,socket,unsafe,,socket_48,826,train,train
socket_49_0_3092.jpg,socket,unsafe,,socket_49,827,train,test
socket_49_0_3785.jpg,socket,unsafe,,socket_49,827,train,val
socket_49_0_7980.jpg,socket,unsafe,,socket_49,827,train,train
socket_49_0_8586.jpg,socket,unsafe,,socket_49,827,train,train
socket_49_0_8855.jpg,socket,unsafe,,socket_49,827,train,train
socket_4_0_3901.jpg,socket,unsafe,,socket_4,828,test,train
socket_4_0_5999.jpg,socket,unsafe,,socket_4,828,test,val
socket_4_0_7066.jpg,socket,unsafe,,socket_4,828,test,val
socket_4_0_7480.jpg,socket,unsafe,,socket_4,828,test,train
socket_4_0_7728.jpg,socket,unsafe,,socket_4,828,test,test
socket_50_0_5017.jpg,socket,unsafe,,socket_50,829,train,train
socket_50_0_5388.jpg,socket,unsafe,,socket_50,829,train,train
socket_50_0_5653.jpg,socket,unsafe,,socket_50,829,train,train
socket_50_0_8864.jpg,socket,unsafe,,socket_50,829,train,train
socket_50_0_8967.jpg,socket,unsafe,,socket_50,829,train,train
socket_51_0_3454.jpg,socket,unsafe,,socket_51,830,test,train
socket_51_0_7070.jpg,socket,unsafe,,socket_51,830,test,test
socket_51_0_8416.jpg,socket,unsafe,,socket_51,830,test,train
socket_51_0_9262.jpg,socket,unsafe,,socket_51,830,test,val
socket_51_0_9828.jpg,socket,unsafe,,socket_51,830,test,train
socket_52_0_1354.jpg,socket,unsafe,,socket_52,831,train,train
socket_52_0_1380.jpg,socket,unsafe,,socket_52,831,train,train
socket_52_0_3535.jpg,socket,unsafe,,socket_52,831,train,train
socket_52_0_6279.jpg,socket,unsafe,,socket_52,831,train,train
socket_52_0_6890.jpg,socket,unsafe,,socket_52,831,train,train
socket_53_0_1947.jpg,socket,unsafe,,socket_53,832,train,train
socket_53_0_2268.jpg,socket,unsafe,,socket_53,832,train,train
socket_53_0_5449.jpg,socket,unsafe,,socket_53,832,train,train
socket_53_0_7614.jpg,socket,unsafe,,socket_53,832,train,train
socket_53_0_903.jpg,socket,unsafe,,socket_53,832,train,train
socket_54_0_2267.jpg,socket,unsafe,,socket_54,833,train,val
socket_54_0_2829.jpg,socket,unsafe,,socket_54,833,train,train
socket_54_0_2923.jpg,socket,unsafe,,socket_54,833,train,train
socket_54_0_4746.jpg,socket,unsafe,,socket_54,833,train,train
socket_54_0_5450.jpg,socket,unsafe,,socket_54,833,train,val
socket_55_0_194.jpg,socket,unsafe,,socket_55,834,val,train
socket_55_0_2955.jpg,socket,unsafe,,socket_55,834,val,val
socket_55_0_3646.jpg,socket,unsafe,,socket_55,834,val,val
socket_55_0_4894.jpg,socket,unsafe,,socket_55,834,val,train
socket_55_0_7706.jpg,socket,unsafe,,socket_55,834,val,train
socket_56_0_2353.jpg,socket,unsafe,,socket_56,835,train,val
socket_56_0_3675.jpg,socket,unsafe,,socket_56,835,train,train
socket_56_0_6454.jpg,socket,unsafe,,socket_56,835,train,train
socket_56_0_7509.jpg,socket,unsafe,,socket_56,835,train,test
socket_56_0_8167.jpg,socket,unsafe,,socket_56,835,train,train
socket_57_0_2588.jpg,socket,unsafe,,socket_57,836,val,train
socket_57_0_5573.jpg,socket,unsafe,,socket_57,836,val,train
socket_57_0_6197.jpg,socket,unsafe,,socket_57,836,val,train
socket_57_0_7428.jpg,socket,unsafe,,socket_57,836,val,val
socket_57_0_7535.jpg,socket,unsafe,,socket_57,836,val,test
socket_58_0_1597.jpg,socket,unsafe,,socket_58,837,train,test
socket_58_0_7392.jpg,socket,unsafe,,socket_58,837,train,train
socket_58_0_9243.jpg,socket,unsafe,,socket_58,837,train,val
socket_58_0_9535.jpg,socket,unsafe,,socket_58,837,train,train
socket_58_0_9585.jpg,socket,unsafe,,socket_58,837,train,train
socket_59_0_140.jpg,socket,unsafe,,socket_59,838,train,train
socket_59_0_2179.jpg,socket,unsafe,,socket_59,838,train,train
socket_59_0_4232.jpg,socket,unsafe,,socket_59,838,train,train
socket_59_0_4642.jpg,socket,unsafe,,socket_59,838,train,train
socket_59_0_6036.jpg,socket,unsafe,,socket_59,838,train,test
socket_5_0_2857.jpg,socket,unsafe,,socket_5,839,train,train
socket_5_0_514.jpg,socket,unsafe,,socket_5,839,train,train
socket_5_0_7237.jpg,socket,unsafe,,socket_5,839,train,test
socket_5_0_7664.jpg,socket,unsafe,,socket_5,839,train,train
socket_5_0_8923.jpg,socket,unsafe,,socket_5,839,train,val
socket_60_0_3459.jpg,socket,unsafe,,socket_60,840,test,val
socket_60_0_3820.jpg,socket,unsafe,,socket_60,840,test,train
socket_60_0_4802.jpg,socket,unsafe,,socket_60,840,test,train
socket_60_0_525.jpg,socket,unsafe,,socket_60,840,test,train
socket_60_0_777.jpg,socket,unsafe,,socket_60,840,test,train
socket_61_0_1701.jpg,socket,unsafe,,socket_61,841,test,train
socket_61_0_3122.jpg,socket,unsafe,,socket_61,841,test,train
socket_61_0_4198.jpg,socket,unsafe,,socket_61,841,test,train
socket_61_0_656.jpg,socket,unsafe,,socket_61,841,test,train
socket_61_0_9780.jpg,socket,unsafe,,socket_61,841,test,train
socket_62_0_1521.jpg,socket,unsafe,,socket_62,842,train,train
socket_62_0_4252.jpg,socket,unsafe,,socket_62,842,train,train
socket_62_0_4778.jpg,socket,unsafe,,socket_62,842,train,train
socket_62_0_4931.jpg,socket,unsafe,,socket_62,842,train,train
socket_62_0_9931.jpg,socket,unsafe,,socket_62,842,train,train
socket_63_0_3828.jpg,socket,unsafe,,socket_63,843,train,train
socket_63_0_4647.jpg,socket,unsafe,,socket_63,843,train,train
socket_63_0_4804.jpg,socket,unsafe,,socket_63,843,train,train
socket_63_0_7507.jpg,socket,unsafe,,socket_63,843,train,train
socket_63_0_8818.jpg,socket,unsafe,,socket_63,843,train,test
socket_64_0_4400.jpg,socket,unsafe,,socket_64,844,test,val
socket_64_0_586.jpg,socket,unsafe,,socket_64,844,test,train
socket_64_0_6487.jpg,socket,unsafe,,socket_64,844,test,train
socket_64_0_6518.jpg,socket,unsafe,,socket_64,844,test,val
socket_64_0_7319.jpg,socket,unsafe,,socket_64,844,test,train
socket_65_0_1766.jpg,socket,unsafe,,socket_65,845,train,train
socket_65_0_6423.jpg,socket,unsafe,,socket_65,845,train,test
socket_65_0_8270.jpg,socket,unsafe,,socket_65,845,train,test
socket_65_0_9711.jpg,socket,unsafe,,socket_65,845,train,train
socket_65_0_9858.jpg,socket,unsafe,,socket_65,845,train,train
socket_66_0_1407.jpg,socket,unsafe,,socket_66,846,val,train
socket_66_0_2803.jpg,socket,unsafe,,socket_66,846,val,train
socket_66_0_3724.jpg,socket,unsafe,,socket_66,846,val,val
socket_66_0_6336.jpg,socket,unsafe,,socket_66,846,val,train
socket_66_0_9895.jpg,socket,unsafe,,socket_66,846,val,val
socket_67_0_2070.jpg,socket,unsafe,,socket_67,847,train,val
socket_67_0_3426.jpg,socket,unsafe,,socket_67,847,train,train
socket_67_0_8525.jpg,socket,unsafe,,socket_67,847,train,test
socket_67_0_8600.jpg,socket,unsafe,,socket_67,847,train,test
socket_67_0_9481.jpg,socket,unsafe,,socket_67,847,train,train
socket_68_0_17.jpg,socket,unsafe,,socket_68,848,train,val
socket_68_0_2263.jpg,socket,unsafe,,socket_68,848,train,train
socket_68_0_4675.jpg,socket,unsafe,,socket_68,848,train,train
socket_68_0_4828.jpg,socket,unsafe,,socket_68,848,train,test
socket_68_0_6986.jpg,socket,unsafe,,socket_68,848,train,train
socket_69_0_1161.jpg,socket,unsafe,,socket_69,849,val,train
socket_69_0_3454.jpg,socket,unsafe,,socket_69,849,val,train
socket_69_0_43.jpg,socket,unsafe,,socket_69,849,val,train
socket_69_0_601.jpg,socket,unsafe,,socket_69,849,val,val
socket_69_0_7182.jpg,socket,unsafe,,socket_69,849,val,train
socket_6_0_4474.jpg,socket,unsafe,,socket_6,850,train,train
socket_6_0_5493.jpg,socket,unsafe,,socket_6,850,train,train
socket_6_0_5915.jpg,socket,unsafe,,socket_6,850,train,train
socket_6_0_6413.jpg,socket,unsafe,,socket_6,850,train,train
socket_6_0_7894.jpg,socket,unsafe,,socket_6,850,train,train
socket_70_0_2397.jpg,socket,unsafe,,socket_70,851,val,train
socket_70_0_3580.jpg,socket,unsafe,,socket_70,851,val,train
socket_70_0_4182.jpg,socket,unsafe,,socket_70,851,val,train
socket_70_0_7134.jpg,socket,unsafe,,socket_70,851,val,train
socket_70_0_7180.jpg,socket,unsafe,,socket_70,851,val,train
socket_71_0_1006.jpg,socket,unsafe,,socket_71,852,train,test
socket_71_0_4799.jpg,socket,unsafe,,socket_71,852,train,train
socket_71_0_8243.jpg,socket,unsafe,,socket_71,852,train,train
socket_71_0_9467.jpg,socket,unsafe,,socket_71,852,train,train
socket_71_0_9659.jpg,socket,unsafe,,socket_71,852,train,train
socket_72_0_6690.jpg,socket,unsafe,,socket_72,853,train,train
socket_72_0_7420.jpg,socket,unsafe,,socket_72,853,train,test
socket_72_0_8063.jpg,socket,unsafe,,socket_72,853,train,train
socket_72_0_8083.jpg,socket,unsafe,,socket_72,853,train,train
socket_72_0_9814.jpg,socket,unsafe,,socket_72,853,train,val
socket_73_0_4449.jpg,socket,unsafe,,socket_73,854,train,train
socket_73_0_690.jpg,socket,unsafe,,socket_73,854,train,train
socket_73_0_8156.jpg,socket,unsafe,,socket_73,854,train,val
socket_73_0_8445.jpg,socket,unsafe,,socket_73,854,train,test
socket_73_0_9174.jpg,socket,unsafe,,socket_73,854,train,test
socket_74_0_2245.jpg,socket,unsafe,,socket_74,855,val,train
socket_74_0_4597.jpg,socket,unsafe,,socket_74,855,val,train
socket_74_0_557.jpg,socket,unsafe,,socket_74,855,val,train
socket_74_0_5680.jpg,socket,unsafe,,socket_74,855,val,val
socket_74_0_6225.jpg,socket,unsafe,,socket_74,855,val,train
socket_75_0_1117.jpg,socket,unsafe,,socket_75,856,train,test
socket_75_0_1989.jpg,socket,unsafe,,socket_75,856,train,val
socket_75_0_2472.jpg,socket,unsafe,,socket_75,856,train,train
socket_75_0_6395.jpg,socket,unsafe,,socket_75,856,train,train
socket_75_0_9960.jpg,socket,unsafe,,socket_75,856,train,train
socket_76_0_1540.jpg,socket,unsafe,,socket_76,857,train,train
socket_76_0_352.jpg,socket,unsafe,,socket_76,857,train,train
socket_76_0_3980.jpg,socket,unsafe,,socket_76,857,train,train
socket_76_0_5701.jpg,socket,unsafe,,socket_76,857,train,test
socket_76_0_6677.jpg,socket,unsafe,,socket_76,857,train,train
socket_77_0_2387.jpg,socket,unsafe,,socket_77,858,val,val
socket_77_0_3002.jpg,socket,unsafe,,socket_77,858,val,train
socket_77_0_3104.jpg,socket,unsafe,,socket_77,858,val,train
socket_77_0_6480.jpg,socket,unsafe,,socket_77,858,val,train
socket_77_0_8189.jpg,socket,unsafe,,socket_77,858,val,val
socket_78_0_2375.jpg,socket,unsafe,,socket_78,859,train,train
socket_78_0_6196.jpg,socket,unsafe,,socket_78,859,train,val
socket_78_0_9773.jpg,socket,unsafe,,socket_78,859,train,train
socket_78_0_983.jpg,socket,unsafe,,socket_78,859,train,train
socket_78_0_9909.jpg,socket,unsafe,,socket_78,859,train,val
socket_79_0_5067.jpg,socket,unsafe,,socket_79,860,train,train
socket_79_0_5522.jpg,socket,unsafe,,socket_79,860,train,train
socket_79_0_598.jpg,socket,unsafe,,socket_79,860,train,train
socket_79_0_8410.jpg,socket,unsafe,,socket_79,860,train,train
socket_79_0_9765.jpg,socket,unsafe,,socket_79,860,train,train
socket_7_0_148.jpg,socket,unsafe,,socket_7,861,test,val
socket_7_0_5439.jpg,socket,unsafe,,socket_7,861,test,val
socket_7_0_6516.jpg,socket,unsafe,,socket_7,861,test,train
socket_7_0_7646.jpg,socket,unsafe,,socket_7,861,test,train
socket_7_0_8395.jpg,socket,unsafe,,socket_7,861,test,train
socket_80_0_2109.jpg,socket,unsafe,,socket_80,862,val,train
socket_80_0_2450.jpg,socket,unsafe,,socket_80,862,val,train
socket_80_0_4453.jpg,socket,unsafe,,socket_80,862,val,train
socket_80_0_8827.jpg,socket,unsafe,,socket_80,862,val,train
socket_80_0_9007.jpg,socket,unsafe,,socket_80,862,val,train
socket_81_0_1931.jpg,socket,unsafe,,socket_81,863,train,val
socket_81_0_2165.jpg,socket,unsafe,,socket_81,863,train,train
socket_81_0_2428.jpg,socket,unsafe,,socket_81,863,train,val
socket_81_0_3307.jpg,socket,unsafe,,socket_81,863,train,train
socket_81_0_9409.jpg,socket,unsafe,,socket_81,863,train,train
socket_82_0_4804.jpg,socket,unsafe,,socket_82,864,train,train
socket_82_0_5089.jpg,socket,unsafe,,socket_82,864,train,train
socket_82_0_5429.jpg,socket,unsafe,,socket_82,864,train,train
socket_82_0_7543.jpg,socket,unsafe,,socket_82,864,train,test
socket_82_0_895.jpg,socket,unsafe,,socket_82,864,train,test
socket_83_0_2086.jpg,socket,unsafe,,socket_83,865,train,test
socket_83_0_2585.jpg,socket,unsafe,,socket_83,865,train,train
socket_83_0_3317.jpg,socket,unsafe,,socket_83,865,train,val
socket_83_0_3927.jpg,socket,unsafe,,socket_83,865,train,train
socket_83_0_9702.jpg,socket,unsafe,,socket_83,865,train,test
socket_84_0_3054.jpg,socket,unsafe,,socket_84,866,train,test
socket_84_0_3805.jpg,socket,unsafe,,socket_84,866,train,train
socket_84_0_3987.jpg,socket,unsafe,,socket_84,866,train,test
socket_84_0_4108.jpg,socket,unsafe,,socket_84,866,train,train
socket_84_0_5062.jpg,socket,unsafe,,socket_84,866,train,train
socket_85_0_1246.jpg,socket,unsafe,,socket_85,867,val,val
socket_85_0_4995.jpg,socket,unsafe,,socket_85,867,val,train
socket_85_0_5845.jpg,socket,unsafe,,socket_85,867,val,train
socket_85_0_7636.jpg,socket,unsafe,,socket_85,867,val,test
socket_85_0_7712.jpg,socket,unsafe,,socket_85,867,val,train
socket_86_0_5087.jpg,socket,unsafe,,socket_86,868,train,train
socket_86_0_5700.jpg,socket,unsafe,,socket_86,868,train,train
socket_86_0_7966.jpg,socket,unsafe,,socket_86,868,train,train
socket_86_0_85.jpg,socket,unsafe,,socket_86,868,train,train
socket_86_0_9219.jpg,socket,unsafe,,socket_86,868,train,train
socket_8_0_3830.jpg,socket,unsafe,,socket_8,869,train,train
socket_8_0_3839.jpg,socket,unsafe,,socket_8,869,train,train
socket_8_0_396.jpg,socket,unsafe,,socket_8,869,train,train
socket_8_0_4694.jpg,socket,unsafe,,socket_8,869,train,val
socket_8_0_5012.jpg,socket,unsafe,,socket_8,869,train,train
socket_9_0_1924.jpg,socket,unsafe,,socket_9,870,train,test
socket_9_0_2615.jpg,socket,unsafe,,socket_9,870,train,train
socket_9_0_2627.jpg,socket,unsafe,,socket_9,870,train,train
socket_9_0_4899.jpg,socket,unsafe,,socket_9,870,train,train
socket_9_0_7517.jpg,socket,unsafe,,socket_9,870,train,train
sockets_101_0_2546.jpg,socket,unsafe,,sockets_101,871,train,train
sockets_101_0_3135.jpg,socket,unsafe,,sockets_101,871,train,test
sockets_101_0_4258.jpg,socket,unsafe,,sockets_101,871,train,val
sockets_101_0_6819.jpg,socket,unsafe,,sockets_101,871,train,train
sockets_101_0_9919.jpg,socket,unsafe,,sockets_101,871,train,val
sockets_102_0_1612.jpg,socket,unsafe,,sockets_102,872,train,train
sockets_102_0_1740.jpg,socket,unsafe,,sockets_102,872,train,train
sockets_102_0_2310.jpg,socket,unsafe,,sockets_102,872,train,train
sockets_102_0_4045.jpg,socket,unsafe,,sockets_102,872,train,train
sockets_102_0_9014.jpg,socket,unsafe,,sockets_102,872,train,val
sockets_103_0_1268.jpg,socket,unsafe,,sockets_103,873,val,val
sockets_103_0_1777.jpg,socket,unsafe,,sockets_103,873,val,train
sockets_103_0_3486.jpg,socket,unsafe,,sockets_103,873,val,train
sockets_103_0_5233.jpg,socket,unsafe,,sockets_103,873,val,train
sockets_103_0_9931.jpg,socket,unsafe,,sockets_103,873,val,train
sockets_104_0_3464.jpg,socket,unsafe,,sockets_104,874,train,val
sockets_104_0_4863.jpg,socket,unsafe,,sockets_104,874,train,train
sockets_104_0_5182.jpg,socket,unsafe,,sockets_104,874,train,train
sockets_104_0_6256.jpg,socket,unsafe,,sockets_104,874,train,train
sockets_104_0_7066.jpg,socket,unsafe,,sockets_104,874,train,train
sockets_105_0_1259.jpg,socket,unsafe,,sockets_105,875,train,train
sockets_105_0_1599.jpg,socket,unsafe,,sockets_105,875,train,val
sockets_105_0_2722.jpg,socket,unsafe,,sockets_105,875,train,train
sockets_105_0_4253.jpg,socket,unsafe,,sockets_105,875,train,train
sockets_105_0_7190.jpg,socket,unsafe,,sockets_105,875,train,train
sockets_106_0_1839.jpg,socket,unsafe,,sockets_106,876,train,train
sockets_106_0_2004.jpg,socket,unsafe,,sockets_106,876,train,val
sockets_106_0_3364.jpg,socket,unsafe,,sockets_106,876,train,train
sockets_106_0_4034.jpg,socket,unsafe,,sockets_106,876,train,val
sockets_106_0_9145.jpg,socket,unsafe,,sockets_106,876,train,train
sockets_107_0_1138.jpg,socket,unsafe,,sockets_107,877,val,val
sockets_107_0_4939.jpg,socket,unsafe,,sockets_107,877,val,train
sockets_107_0_514.jpg,socket,unsafe,,sockets_107,877,val,train
sockets_107_0_915.jpg,socket,unsafe,,sockets_107,877,val,train
sockets_107_0_9556.jpg,socket,unsafe,,sockets_107,877,val,train
sockets_108_0_2172.jpg,socket,unsafe,,sockets_108,878,train,val
sockets_108_0_3477.jpg,socket,unsafe,,sockets_108,878,train,train
sockets_108_0_3590.jpg,socket,unsafe,,sockets_108,878,train,train
sockets_108_0_604.jpg,socket,unsafe,,sockets_108,878,train,train
sockets_108_0_713.jpg,socket,unsafe,,sockets_108,878,train,train
sockets_10_0_1528.jpg,socket,unsafe,,sockets_10,879,train,train
sockets_10_0_3518.jpg,socket,unsafe,,sockets_10,879,train,val
sockets_10_0_4786.jpg,socket,unsafe,,sockets_10,879,train,train
sockets_10_0_6089.jpg,socket,unsafe,,sockets_10,879,train,train
sockets_10_0_8731.jpg,socket,unsafe,,sockets_10,879,train,train
sockets_110_0_4443.jpg,socket,unsafe,,sockets_110,880,train,test
sockets_110_0_5183.jpg,socket,unsafe,,sockets_110,880,train,val
sockets_110_0_6125.jpg,socket,unsafe,,sockets_110,880,train,train
sockets_110_0_734.jpg,socket,unsafe,,sockets_110,880,train,train
sockets_110_0_830.jpg,socket,unsafe,,sockets_110,880,train,val
sockets_111_0_2092.jpg,socket,unsafe,,sockets_111,881,val,test
sockets_111_0_7936.jpg,socket,unsafe,,sockets_111,881,val,train
sockets_111_0_8267.jpg,socket,unsafe,,sockets_111,881,val,train
sockets_111_0_8491.jpg,socket,unsafe,,sockets_111,881,val,train
sockets_111_0_8966.jpg,socket,unsafe,,sockets_111,881,val,val
sockets_112_0_2315.jpg,socket,unsafe,,sockets_112,882,val,train
sockets_112_0_459.jpg,socket,unsafe,,sockets_112,882,val,train
sockets_112_0_4664.jpg,socket,unsafe,,sockets_112,882,val,val
sockets_112_0_7559.jpg,socket,unsafe,,sockets_112,882,val,train
sockets_112_0_8216.jpg,socket,unsafe,,sockets_112,882,val,train
sockets_113_0_182.jpg,socket,unsafe,,sockets_113,883,train,val
sockets_113_0_3242.jpg,socket,unsafe,,sockets_113,883,train,train
sockets_113_0_7345.jpg,socket,unsafe,,sockets_113,883,train,train
sockets_113_0_7658.jpg,socket,unsafe,,sockets_113,883,train,train
sockets_113_0_774.jpg,socket,unsafe,,sockets_113,883,train,train
sockets_114_0_140.jpg,socket,unsafe,,sockets_114,884,train,val
sockets_114_0_2017.jpg,socket,unsafe,,sockets_114,884,train,train
sockets_114_0_2277.jpg,socket,unsafe,,sockets_114,884,train,train
sockets_114_0_5106.jpg,socket,unsafe,,sockets_114,884,train,train
sockets_114_0_6935.jpg,socket,unsafe,,sockets_114,884,train,test
sockets_115_0_1449.jpg,socket,unsafe,,sockets_115,885,test,train
sockets_115_0_2447.jpg,socket,unsafe,,sockets_115,885,test,train
sockets_115_0_7361.jpg,socket,unsafe,,sockets_115,885,test,train
sockets_115_0_7617.jpg,socket,unsafe,,sockets_115,885,test,train
sockets_115_0_9933.jpg,socket,unsafe,,sockets_115,885,test,test
sockets_116_0_2642.jpg,socket,unsafe,,sockets_116,886,train,val
sockets_116_0_4693.jpg,socket,unsafe,,sockets_116,886,train,train
sockets_116_0_5324.jpg,socket,unsafe,,sockets_116,886,train,train
sockets_116_0_7379.jpg,socket,unsafe,,sockets_116,886,train,train
sockets_116_0_7586.jpg,socket,unsafe,,sockets_116,886,train,val
sockets_117_0_1207.jpg,socket,unsafe,,sockets_117,887,test,train
sockets_117_0_1465.jpg,socket,unsafe,,sockets_117,887,test,train
sockets_117_0_4658.jpg,socket,unsafe,,sockets_117,887,test,val
sockets_117_0_7392.jpg,socket,unsafe,,sockets_117,887,test,train
sockets_117_0_7495.jpg,socket,unsafe,,sockets_117,887,test,train
sockets_118_0_200.jpg,socket,unsafe,,sockets_118,888,test,test
sockets_118_0_3077.jpg,socket,unsafe,,sockets_118,888,test,train
sockets_118_0_4748.jpg,socket,unsafe,,sockets_118,888,test,train
sockets_118_0_9355.jpg,socket,unsafe,,sockets_118,888,test,val
sockets_118_0_9775.jpg,socket,unsafe,,sockets_118,888,test,val
sockets_119_0_2141.jpg,socket,unsafe,,sockets_119,889,train,val
sockets_119_0_4045.jpg,socket,unsafe,,sockets_119,889,train,train
sockets_119_0_4874.jpg,socket,unsafe,,sockets_119,889,train,train
sockets_119_0_6537.jpg,socket,unsafe,,sockets_119,889,train,train
sockets_119_0_8534.jpg,socket,unsafe,,sockets_119,889,train,train
sockets_11_0_2932.jpg,socket,unsafe,,sockets_11,890,train,test
sockets_11_0_2997.jpg,socket,unsafe,,sockets_11,890,train,train
sockets_11_0_3916.jpg,socket,unsafe,,sockets_11,890,train,train
sockets_11_0_4721.jpg,socket,unsafe,,sockets_11,890,train,train
sockets_11_0_9039.jpg,socket,unsafe,,sockets_11,890,train,train
sockets_121_0_2670.jpg,socket,unsafe,,sockets_121,891,train,train
sockets_121_0_6110.jpg,socket,unsafe,,sockets_121,891,train,train
sockets_121_0_7510.jpg,socket,unsafe,,sockets_121,891,train,train
sockets_121_0_8524.jpg,socket,unsafe,,sockets_121,891,train,train
sockets_121_0_9025.jpg,socket,unsafe,,sockets_121,891,train,train
sockets_122_0_2234.jpg,socket,unsafe,,sockets_122,892,train,test
sockets_122_0_2785.jpg,socket,unsafe,,sockets_122,892,train,train
sockets_122_0_3205.jpg,socket,unsafe,,sockets_122,892,train,train
sockets_122_0_5674.jpg,socket,unsafe,,sockets_122,892,train,train
sockets_122_0_5897.jpg,socket,unsafe,,sockets_122,892,train,train
sockets_123_0_6192.jpg,socket,unsafe,,sockets_123,893,test,train
sockets_123_0_7462.jpg,socket,unsafe,,sockets_123,893,test,test
sockets_123_0_7467.jpg,socket,unsafe,,sockets_123,893,test,test
sockets_123_0_7944.jpg,socket,unsafe,,sockets_123,893,test,val
sockets_123_0_7986.jpg,socket,unsafe,,sockets_123,893,test,val
sockets_124_0_2977.jpg,socket,unsafe,,sockets_124,894,val,train
sockets_124_0_3986.jpg,socket,unsafe,,sockets_124,894,val,test
sockets_124_0_4696.jpg,socket,unsafe,,sockets_124,894,val,test
sockets_124_0_8431.jpg,socket,unsafe,,sockets_124,894,val,val
sockets_124_0_9043.jpg,socket,unsafe,,sockets_124,894,val,train
sockets_125_0_2343.jpg,socket,unsafe,,sockets_125,895,train,train
sockets_125_0_3662.jpg,socket,unsafe,,sockets_125,895,train,train
sockets_125_0_4374.jpg,socket,unsafe,,sockets_125,895,train,val
sockets_125_0_8469.jpg,socket,unsafe,,sockets_125,895,train,train
sockets_125_0_9758.jpg,socket,unsafe,,sockets_125,895,train,train
sockets_126_0_2228.jpg,socket,unsafe,,sockets_126,896,test,train
sockets_126_0_2267.jpg,socket,unsafe,,sockets_126,896,test,train
sockets_126_0_8676.jpg,socket,unsafe,,sockets_126,896,test,train
sockets_126_0_9106.jpg,socket,unsafe,,sockets_126,896,test,test
sockets_126_0_9727.jpg,socket,unsafe,,sockets_126,896,test,train
sockets_128_0_2354.jpg,socket,unsafe,,sockets_128,897,test,train
sockets_128_0_3852.jpg,socket,unsafe,,sockets_128,897,test,train
sockets_128_0_5922.jpg,socket,unsafe,,sockets_128,897,test,test
sockets_128_0_6210.jpg,socket,unsafe,,sockets_128,897,test,train
sockets_128_0_9336.jpg,socket,unsafe,,sockets_128,897,test,train
sockets_129_0_3771.jpg,socket,unsafe,,sockets_129,898,train,train
sockets_129_0_5527.jpg,socket,unsafe,,sockets_129,898,train,val
sockets_129_0_632.jpg,socket,unsafe,,sockets_129,898,train,train
sockets_129_0_8116.jpg,socket,unsafe,,sockets_129,898,train,test
sockets_129_0_9252.jpg,socket,unsafe,,sockets_129,898,train,val
sockets_12_0_1719.jpg,socket,unsafe,,sockets_12,899,train,train
sockets_12_0_2310.jpg,socket,unsafe,,sockets_12,899,train,train
sockets_12_0_4417.jpg,socket,unsafe,,sockets_12,899,train,test
sockets_12_0_8537.jpg,socket,unsafe,,sockets_12,899,train,train
sockets_12_0_9754.jpg,socket,unsafe,,sockets_12,899,train,train
sockets_130_0_3458.jpg,socket,unsafe,,sockets_130,900,train,train
sockets_130_0_4463.jpg,socket,unsafe,,sockets_130,900,train,train
sockets_130_0_8965.jpg,socket,unsafe,,sockets_130,900,train,train
sockets_130_0_9094.jpg,socket,unsafe,,sockets_130,900,train,train
sockets_130_0_9849.jpg,socket,unsafe,,sockets_130,900,train,train
sockets_131_0_2149.jpg,socket,unsafe,,sockets_131,901,train,test
sockets_131_0_3499.jpg,socket,unsafe,,sockets_131,901,train,test
sockets_131_0_6107.jpg,socket,unsafe,,sockets_131,901,train,test
sockets_131_0_7672.jpg,socket,unsafe,,sockets_131,901,train,train
sockets_131_0_8956.jpg,socket,unsafe,,sockets_131,901,train,train
sockets_133_0_1462.jpg,socket,unsafe,,sockets_133,902,train,train
sockets_133_0_407.jpg,socket,unsafe,,sockets_133,902,train,train
sockets_133_0_5984.jpg,socket,unsafe,,sockets_133,902,train,train
sockets_133_0_721.jpg,socket,unsafe,,sockets_133,902,train,train
sockets_133_0_7947.jpg,socket,unsafe,,sockets_133,902,train,train
sockets_134_0_198.jpg,socket,unsafe,,sockets_134,903,train,train
sockets_134_0_3548.jpg,socket,unsafe,,sockets_134,903,train,train
sockets_134_0_7923.jpg,socket,unsafe,,sockets_134,903,train,train
sockets_134_0_8997.jpg,socket,unsafe,,sockets_134,903,train,test
sockets_134_0_9250.jpg,socket,unsafe,,sockets_134,903,train,val
sockets_135_0_1646.jpg,socket,unsafe,,sockets_135,904,train,test
sockets_135_0_2314.jpg,socket,unsafe,,sockets_135,904,train,train
sockets_135_0_2976.jpg,socket,unsafe,,sockets_135,904,train,val
sockets_135_0_5378.jpg,socket,unsafe,,sockets_135,904,train,train
sockets_135_0_6202.jpg,socket,unsafe,,sockets_135,904,train,train
sockets_136_0_2475.jpg,socket,unsafe,,sockets_136,905,train,train
sockets_136_0_3586.jpg,socket,unsafe,,sockets_136,905,train,test
sockets_136_0_4539.jpg,socket,unsafe,,sockets_136,905,train,train
sockets_136_0_4599.jpg,socket,unsafe,,sockets_136,905,train,train
sockets_136_0_5996.jpg,socket,unsafe,,sockets_136,905,train,train
sockets_137_0_1575.jpg,socket,unsafe,,sockets_137,906,train,train
sockets_137_0_3538.jpg,socket,unsafe,,sockets_137,906,train,val
sockets_137_0_3993.jpg,socket,unsafe,,sockets_137,906,train,train
sockets_137_0_8588.jpg,socket,unsafe,,sockets_137,906,train,train
sockets_137_0_9011.jpg,socket,unsafe,,sockets_137,906,train,train
sockets_138_0_315.jpg,socket,unsafe,,sockets_138,907,train,train
sockets_138_0_8342.jpg,socket,unsafe,,sockets_138,907,train,test
sockets_138_0_8533.jpg,socket,unsafe,,sockets_138,907,train,train
sockets_138_0_9081.jpg,socket,unsafe,,sockets_138,907,train,train
sockets_138_0_9902.jpg,socket,unsafe,,sockets_138,907,train,val
sockets_139_0_318.jpg,socket,unsafe,,sockets_139,908,train,train
sockets_139_0_5649.jpg,socket,unsafe,,sockets_139,908,train,val
sockets_139_0_7861.jpg,socket,unsafe,,sockets_139,908,train,test
sockets_139_0_9292.jpg,socket,unsafe,,sockets_139,908,train,test
sockets_139_0_9918.jpg,socket,unsafe,,sockets_139,908,train,val
sockets_13_0_1989.jpg,socket,unsafe,,sockets_13,909,train,train
sockets_13_0_4330.jpg,socket,unsafe,,sockets_13,909,train,train
sockets_13_0_4710.jpg,socket,unsafe,,sockets_13,909,train,val
sockets_13_0_4854.jpg,socket,unsafe,,sockets_13,909,train,val
sockets_13_0_681.jpg,socket,unsafe,,sockets_13,909,train,test
sockets_140_0_5506.jpg,socket,unsafe,,sockets_140,910,train,val
sockets_140_0_7612.jpg,socket,unsafe,,sockets_140,910,train,train
sockets_140_0_8059.jpg,socket,unsafe,,sockets_140,910,train,test
sockets_140_0_8733.jpg,socket,unsafe,,sockets_140,910,train,val
sockets_140_0_9743.jpg,socket,unsafe,,sockets_140,910,train,test
sockets_141_0_2794.jpg,socket,unsafe,,sockets_141,911,train,train
sockets_141_0_310.jpg,socket,unsafe,,sockets_141,911,train,test
sockets_141_0_3425.jpg,socket,unsafe,,sockets_141,911,train,train
sockets_141_0_4318.jpg,socket,unsafe,,sockets_141,911,train,train
sockets_141_0_831.jpg,socket,unsafe,,sockets_141,911,train,train
sockets_142_0_3199.jpg,socket,unsafe,,sockets_142,912,train,train
sockets_142_0_445.jpg,socket,unsafe,,sockets_142,912,train,train
sockets_142_0_4804.jpg,socket,unsafe,,sockets_142,912,train,test
sockets_142_0_5306.jpg,socket,unsafe,,sockets_142,912,train,train
sockets_142_0_7681.jpg,socket,unsafe,,sockets_142,912,train,train
sockets_143_0_3451.jpg,socket,unsafe,,sockets_143,913,train,test
sockets_143_0_4461.jpg,socket,unsafe,,sockets_143,913,train,train
sockets_143_0_535.jpg,socket,unsafe,,sockets_143,913,train,val
sockets_143_0_5646.jpg,socket,unsafe,,sockets_143,913,train,train
sockets_143_0_8109.jpg,socket,unsafe,,sockets_143,913,train,train
sockets_144_0_2137.jpg,socket,unsafe,,sockets_144,914,train,val
sockets_144_0_2317.jpg,socket,unsafe,,sockets_144,914,train,train
sockets_144_0_3984.jpg,socket,unsafe,,sockets_144,914,train,train
sockets_144_0_4086.jpg,socket,unsafe,,sockets_144,914,train,train
sockets_144_0_6412.jpg,socket,unsafe,,sockets_144,914,train,train
sockets_15_0_1017.jpg,socket,unsafe,,sockets_15,915,val,val
sockets_15_0_1205.jpg,socket,unsafe,,sockets_15,915,val,train
sockets_15_0_4152.jpg,socket,unsafe,,sockets_15,915,val,train
sockets_15_0_6630.jpg,socket,unsafe,,sockets_15,915,val,train
sockets_15_0_8280.jpg,socket,unsafe,,sockets_15,915,val,val
sockets_16_0_3984.jpg,socket,unsafe,,sockets_16,916,train,val
sockets_16_0_5778.jpg,socket,unsafe,,sockets_16,916,train,train
sockets_16_0_6689.jpg,socket,unsafe,,sockets_16,916,train,train
sockets_16_0_7991.jpg,socket,unsafe,,sockets_16,916,train,train
sockets_16_0_9368.jpg,socket,unsafe,,sockets_16,916,train,test
sockets_18_0_13.jpg,socket,unsafe,,sockets_18,917,train,train
sockets_18_0_5379.jpg,socket,unsafe,,sockets_18,917,train,train
sockets_18_0_7492.jpg,socket,unsafe,,sockets_18,917,train,train
sockets_18_0_7855.jpg,socket,unsafe,,sockets_18,917,train,test
sockets_18_0_9740.jpg,socket,unsafe,,sockets_18,917,train,val
sockets_19_0_3987.jpg,socket,unsafe,,sockets_19,918,train,train
sockets_19_0_4430.jpg,socket,unsafe,,sockets_19,918,train,train
sockets_19_0_5598.jpg,socket,unsafe,,sockets_19,918,train,train
sockets_19_0_8160.jpg,socket,unsafe,,sockets_19,918,train,train
sockets_19_0_9496.jpg,socket,unsafe,,sockets_19,918,train,test
sockets_21_0_3443.jpg,socket,unsafe,,sockets_21,919,test,train
sockets_21_0_3921.jpg,socket,unsafe,,sockets_21,919,test,train
sockets_21_0_4313.jpg,socket,unsafe,,sockets_21,919,test,train
sockets_21_0_7766.jpg,socket,unsafe,,sockets_21,919,test,train
sockets_21_0_7902.jpg,socket,unsafe,,sockets_21,919,test,train
sockets_23_0_3121.jpg,socket,unsafe,,sockets_23,920,train,train
sockets_23_0_4310.jpg,socket,unsafe,,sockets_23,920,train,train
sockets_23_0_552.jpg,socket,unsafe,,sockets_23,920,train,train
sockets_23_0_6585.jpg,socket,unsafe,,sockets_23,920,train,train
sockets_23_0_7559.jpg,socket,unsafe,,sockets_23,920,train,train
sockets_27_0_1397.jpg,socket,unsafe,,sockets_27,921,val,train
sockets_27_0_2747.jpg,socket,unsafe,,sockets_27,921,val,train
sockets_27_0_6342.jpg,socket,unsafe,,sockets_27,921,val,val
sockets_27_0_7563.jpg,socket,unsafe,,sockets_27,921,val,val
sockets_27_0_9419.jpg,socket,unsafe,,sockets_27,921,val,test
sockets_28_0_1177.jpg,socket,unsafe,,sockets_28,922,val,test
sockets_28_0_2412.jpg,socket,unsafe,,sockets_28,922,val,train
sockets_28_0_3275.jpg,socket,unsafe,,sockets_28,922,val,train
sockets_28_0_6246.jpg,socket,unsafe,,sockets_28,922,val,val
sockets_28_0_8067.jpg,socket,unsafe,,sockets_28,922,val,train
sockets_29_0_2862.jpg,socket,unsafe,,sockets_29,923,train,train
sockets_29_0_3641.jpg,socket,unsafe,,sockets_29,923,train,train
sockets_29_0_5258.jpg,socket,unsafe,,sockets_29,923,train,train
sockets_29_0_5667.jpg,socket,unsafe,,sockets_29,923,train,train
sockets_29_0_910.jpg,socket,unsafe,,sockets_29,923,train,train
sockets_30_0_2192.jpg,socket,unsafe,,sockets_30,924,train,train
sockets_30_0_3506.jpg,socket,unsafe,,sockets_30,924,train,train
sockets_30_0_4426.jpg,socket,unsafe,,sockets_30,924,train,test
sockets_30_0_6117.jpg,socket,unsafe,,sockets_30,924,train,train
sockets_30_0_7918.jpg,socket,unsafe,,sockets_30,924,train,train
sockets_31_0_1049.jpg,socket,unsafe,,sockets_31,925,train,train
sockets_31_0_3553.jpg,socket,unsafe,,sockets_31,925,train,val
sockets_31_0_6404.jpg,socket,unsafe,,sockets_31,925,train,val
sockets_31_0_8759.jpg,socket,unsafe,,sockets_31,925,train,test
sockets_31_0_9158.jpg,socket,unsafe,,sockets_31,925,train,train
sockets_32_0_1301.jpg,socket,unsafe,,sockets_32,926,test,train
sockets_32_0_3712.jpg,socket,unsafe,,sockets_32,926,test,train
sockets_32_0_3862.jpg,socket,unsafe,,sockets_32,926,test,val
sockets_32_0_5237.jpg,socket,unsafe,,sockets_32,926,test,val
sockets_32_0_9942.jpg,socket,unsafe,,sockets_32,926,test,test
sockets_33_0_2684.jpg,socket,unsafe,,sockets_33,927,train,train
sockets_33_0_353.jpg,socket,unsafe,,sockets_33,927,train,train
sockets_33_0_6347.jpg,socket,unsafe,,sockets_33,927,train,train
sockets_33_0_7034.jpg,socket,unsafe,,sockets_33,927,train,train
sockets_33_0_7594.jpg,socket,unsafe,,sockets_33,927,train,train
sockets_34_0_2910.jpg,socket,unsafe,,sockets_34,928,train,train
sockets_34_0_5285.jpg,socket,unsafe,,sockets_34,928,train,train
sockets_34_0_6699.jpg,socket,unsafe,,sockets_34,928,train,train
sockets_34_0_8635.jpg,socket,unsafe,,sockets_34,928,train,train
sockets_34_0_926.jpg,socket,unsafe,,sockets_34,928,train,train
sockets_37_0_2707.jpg,socket,unsafe,,sockets_37,929,test,train
sockets_37_0_5462.jpg,socket,unsafe,,sockets_37,929,test,train
sockets_37_0_5750.jpg,socket,unsafe,,sockets_37,929,test,train
sockets_37_0_6556.jpg,socket,unsafe,,sockets_37,929,test,train
sockets_37_0_6782.jpg,socket,unsafe,,sockets_37,929,test,test
sockets_38_0_1398.jpg,socket,unsafe,,sockets_38,930,train,train
sockets_38_0_1500.jpg,socket,unsafe,,sockets_38,930,train,train
sockets_38_0_3864.jpg,socket,unsafe,,sockets_38,930,train,train
sockets_38_0_3932.jpg,socket,unsafe,,sockets_38,930,train,test
sockets_38_0_5927.jpg,socket,unsafe,,sockets_38,930,train,train
sockets_39_0_1251.jpg,socket,unsafe,,sockets_39,931,train,train
sockets_39_0_2337.jpg,socket,unsafe,,sockets_39,931,train,train
sockets_39_0_3460.jpg,socket,unsafe,,sockets_39,931,train,val
sockets_39_0_3484.jpg,socket,unsafe,,sockets_39,931,train,train
sockets_39_0_5031.jpg,socket,unsafe,,sockets_39,931,train,val
sockets_40_0_1929.jpg,socket,unsafe,,sockets_40,932,val,train
sockets_40_0_3095.jpg,socket,unsafe,,sockets_40,932,val,train
sockets_40_0_5748.jpg,socket,unsafe,,sockets_40,932,val,train
sockets_40_0_7813.jpg,socket,unsafe,,sockets_40,932,val,train
sockets_40_0_7983.jpg,socket,unsafe,,sockets_40,932,val,test
sockets_41_0_4480.jpg,socket,unsafe,,sockets_41,933,train,train
sockets_41_0_7096.jpg,socket,unsafe,,sockets_41,933,train,val
sockets_41_0_7917.jpg,socket,unsafe,,sockets_41,933,train,test
sockets_41_0_89.jpg,socket,unsafe,,sockets_41,933,train,train
sockets_41_0_972.jpg,socket,unsafe,,sockets_41,933,train,train
sockets_42_0_1401.jpg,socket,unsafe,,sockets_42,934,train,train
sockets_42_0_3230.jpg,socket,unsafe,,sockets_42,934,train,val
sockets_42_0_3412.jpg,socket,unsafe,,sockets_42,934,train,train
sockets_42_0_4086.jpg,socket,unsafe,,sockets_42,934,train,train
sockets_42_0_4182.jpg,socket,unsafe,,sockets_42,934,train,val
sockets_43_0_6696.jpg,socket,unsafe,,sockets_43,935,train,train
sockets_43_0_7286.jpg,socket,unsafe,,sockets_43,935,train,test
sockets_43_0_9519.jpg,socket,unsafe,,sockets_43,935,train,train
sockets_43_0_9730.jpg,socket,unsafe,,sockets_43,935,train,val
sockets_43_0_9977.jpg,socket,unsafe,,sockets_43,935,train,val
sockets_44_0_4024.jpg,socket,unsafe,,sockets_44,936,train,train
sockets_44_0_6858.jpg,socket,unsafe,,sockets_44,936,train,test
sockets_44_0_7948.jpg,socket,unsafe,,sockets_44,936,train,train
sockets_44_0_7996.jpg,socket,unsafe,,sockets_44,936,train,train
sockets_44_0_8957.jpg,socket,unsafe,,sockets_44,936,train,train
sockets_45_0_2789.jpg,socket,unsafe,,sockets_45,937,train,train
sockets_45_0_4080.jpg,socket,unsafe,,sockets_45,937,train,train
sockets_45_0_4983.jpg,socket,unsafe,,sockets_45,937,train,test
sockets_45_0_6851.jpg,socket,unsafe,,sockets_45,937,train,train
sockets_45_0_8018.jpg,socket,unsafe,,sockets_45,937,train,train
sockets_46_0_1528.jpg,socket,unsafe,,sockets_46,938,test,test
sockets_46_0_2832.jpg,socket,unsafe,,sockets_46,938,test,train
sockets_46_0_8249.jpg,socket,unsafe,,sockets_46,938,test,train
sockets_46_0_8650.jpg,socket,unsafe,,sockets_46,938,test,train
sockets_46_0_9477.jpg,socket,unsafe,,sockets_46,938,test,train
sockets_47_0_3101.jpg,socket,unsafe,,sockets_47,939,train,test
sockets_47_0_6970.jpg,socket,unsafe,,sockets_47,939,train,train
sockets_47_0_722.jpg,socket,unsafe,,sockets_47,939,train,train
sockets_47_0_8178.jpg,socket,unsafe,,sockets_47,939,train,val
sockets_47_0_9492.jpg,socket,unsafe,,sockets_47,939,train,train
sockets_48_0_2089.jpg,socket,unsafe,,sockets_48,940,train,train
sockets_48_0_5054.jpg,socket,unsafe,,sockets_48,940,train,train
sockets_48_0_5519.jpg,socket,unsafe,,sockets_48,940,train,train
sockets_48_0_658.jpg,socket,unsafe,,sockets_48,940,train,val
sockets_48_0_9930.jpg,socket,unsafe,,sockets_48,940,train,train
sockets_49_0_2277.jpg,socket,unsafe,,sockets_49,941,train,train
sockets_49_0_2906.jpg,socket,unsafe,,sockets_49,941,train,val
sockets_49_0_3821.jpg,socket,unsafe,,sockets_49,941,train,val
sockets_49_0_5756.jpg,socket,unsafe,,sockets_49,941,train,train
sockets_49_0_8115.jpg,socket,unsafe,,sockets_49,941,train,train
sockets_4_0_2233.jpg,socket,unsafe,,sockets_4,942,train,train
sockets_4_0_4431.jpg,socket,unsafe,,sockets_4,942,train,train
sockets_4_0_474.jpg,socket,unsafe,,sockets_4,942,train,train
sockets_4_0_5865.jpg,socket,unsafe,,sockets_4,942,train,train
sockets_4_0_7243.jpg,socket,unsafe,,sockets_4,942,train,train
sockets_50_0_2576.jpg,socket,unsafe,,sockets_50,943,train,val
sockets_50_0_4349.jpg,socket,unsafe,,sockets_50,943,train,test
sockets_50_0_503.jpg,socket,unsafe,,sockets_50,943,train,train
sockets_50_0_6360.jpg,socket,unsafe,,sockets_50,943,train,train
sockets_50_0_6532.jpg,socket,unsafe,,sockets_50,943,train,train
sockets_51_0_1873.jpg,socket,unsafe,,sockets_51,944,train,train
sockets_51_0_6567.jpg,socket,unsafe,,sockets_51,944,train,val
sockets_51_0_7693.jpg,socket,unsafe,,sockets_51,944,train,train
sockets_51_0_8743.jpg,socket,unsafe,,sockets_51,944,train,train
sockets_51_0_949.jpg,socket,unsafe,,sockets_51,944,train,train
sockets_52_0_275.jpg,socket,unsafe,,sockets_52,945,train,train
sockets_52_0_5644.jpg,socket,unsafe,,sockets_52,945,train,train
sockets_52_0_7093.jpg,socket,unsafe,,sockets_52,945,train,train
sockets_52_0_7740.jpg,socket,unsafe,,sockets_52,945,train,train
sockets_52_0_8152.jpg,socket,unsafe,,sockets_52,945,train,train
sockets_53_0_2026.jpg,socket,unsafe,,sockets_53,946,train,val
sockets_53_0_3083.jpg,socket,unsafe,,sockets_53,946,train,train
sockets_53_0_6793.jpg,socket,unsafe,,sockets_53,946,train,train
sockets_53_0_7466.jpg,socket,unsafe,,sockets_53,946,train,test
sockets_53_0_7872.jpg,socket,unsafe,,sockets_53,946,train,train
sockets_54_0_1522.jpg,socket,unsafe,,sockets_54,947,train,train
sockets_54_0_1557.jpg,socket,unsafe,,sockets_54,947,train,train
sockets_54_0_2828.jpg,socket,unsafe,,sockets_54,947,train,test
sockets_54_0_3910.jpg,socket,unsafe,,sockets_54,947,train,train
sockets_54_0_5381.jpg,socket,unsafe,,sockets_54,947,train,train
sockets_55_0_1192.jpg,socket,unsafe,,sockets_55,948,test,train
sockets_55_0_2054.jpg,socket,unsafe,,sockets_55,948,test,test
sockets_55_0_461.jpg,socket,unsafe,,sockets_55,948,test,val
sockets_55_0_6439.jpg,socket,unsafe,,sockets_55,948,test,train
sockets_55_0_8030.jpg,socket,unsafe,,sockets_55,948,test,train
sockets_56_0_1437.jpg,socket,unsafe,,sockets_56,949,test,train
sockets_56_0_4047.jpg,socket,unsafe,,sockets_56,949,test,test
sockets_56_0_74.jpg,socket,unsafe,,sockets_56,949,test,val
sockets_56_0_8921.jpg,socket,unsafe,,sockets_56,949,test,val
sockets_56_0_9904.jpg,socket,unsafe,,sockets_56,949,test,test
sockets_57_0_1328.jpg,socket,unsafe,,sockets_57,950,train,train
sockets_57_0_4723.jpg,socket,unsafe,,sockets_57,950,train,train
sockets_57_0_9274.jpg,socket,unsafe,,sockets_57,950,train,test
sockets_57_0_9295.jpg,socket,unsafe,,sockets_57,950,train,train
sockets_57_0_968.jpg,socket,unsafe,,sockets_57,950,train,val
sockets_58_0_1135.jpg,socket,unsafe,,sockets_58,951,val,test
sockets_58_0_2041.jpg,socket,unsafe,,sockets_58,951,val,test
sockets_58_0_5689.jpg,socket,unsafe,,sockets_58,951,val,train
sockets_58_0_811.jpg,socket,unsafe,,sockets_58,951,val,train
sockets_58_0_8341.jpg,socket,unsafe,,sockets_58,951,val,train
sockets_59_0_5393.jpg,socket,unsafe,,sockets_59,952,train,val
sockets_59_0_794.jpg,socket,unsafe,,sockets_59,952,train,test
sockets_59_0_8352.jpg,socket,unsafe,,sockets_59,952,train,test
sockets_59_0_8578.jpg,socket,unsafe,,sockets_59,952,train,train
sockets_59_0_9931.jpg,socket,unsafe,,sockets_59,952,train,train
sockets_5_0_260.jpg,socket,unsafe,,sockets_5,953,train,train
sockets_5_0_681.jpg,socket,unsafe,,sockets_5,953,train,val
sockets_5_0_6978.jpg,socket,unsafe,,sockets_5,953,train,train
sockets_5_0_7339.jpg,socket,unsafe,,sockets_5,953,train,train
sockets_5_0_9446.jpg,socket,unsafe,,sockets_5,953,train,train
sockets_60_0_4562.jpg,socket,unsafe,,sockets_60,954,train,train
sockets_60_0_6263.jpg,socket,unsafe,,sockets_60,954,train,train
sockets_60_0_709.jpg,socket,unsafe,,sockets_60,954,train,test
sockets_60_0_9697.jpg,socket,unsafe,,sockets_60,954,train,test
sockets_60_0_9741.jpg,socket,unsafe,,sockets_60,954,train,train
sockets_61_0_3415.jpg,socket,unsafe,,sockets_61,955,train,train
sockets_61_0_4327.jpg,socket,unsafe,,sockets_61,955,train,train
sockets_61_0_5640.jpg,socket,unsafe,,sockets_61,955,train,train
sockets_61_0_6797.jpg,socket,unsafe,,sockets_61,955,train,train
sockets_61_0_9417.jpg,socket,unsafe,,sockets_61,955,train,train
sockets_62_0_3267.jpg,socket,unsafe,,sockets_62,956,train,train
sockets_62_0_3937.jpg,socket,unsafe,,sockets_62,956,train,train
sockets_62_0_4223.jpg,socket,unsafe,,sockets_62,956,train,train
sockets_62_0_9362.jpg,socket,unsafe,,sockets_62,956,train,train
sockets_62_0_9913.jpg,socket,unsafe,,sockets_62,956,train,val
sockets_63_0_1425.jpg,socket,unsafe,,sockets_63,957,train,val
sockets_63_0_2672.jpg,socket,unsafe,,sockets_63,957,train,test
sockets_63_0_3089.jpg,socket,unsafe,,sockets_63,957,train,train
sockets_63_0_3388.jpg,socket,unsafe,,sockets_63,957,train,train
sockets_63_0_4610.jpg,socket,unsafe,,sockets_63,957,train,train
sockets_64_0_2045.jpg,socket,unsafe,,sockets_64,958,train,train
sockets_64_0_3343.jpg,socket,unsafe,,sockets_64,958,train,train
sockets_64_0_7284.jpg,socket,unsafe,,sockets_64,958,train,test
sockets_64_0_7343.jpg,socket,unsafe,,sockets_64,958,train,train
sockets_64_0_7420.jpg,socket,unsafe,,sockets_64,958,train,val
sockets_65_0_1513.jpg,socket,unsafe,,sockets_65,959,train,test
sockets_65_0_2612.jpg,socket,unsafe,,sockets_65,959,train,train
sockets_65_0_5341.jpg,socket,unsafe,,sockets_65,959,train,train
sockets_65_0_5958.jpg,socket,unsafe,,sockets_65,959,train,train
sockets_65_0_9364.jpg,socket,unsafe,,sockets_65,959,train,train
sockets_66_0_1948.jpg,socket,unsafe,,sockets_66,960,val,test
sockets_66_0_4413.jpg,socket,unsafe,,sockets_66,960,val,test
sockets_66_0_7117.jpg,socket,unsafe,,sockets_66,960,val,test
sockets_66_0_9017.jpg,socket,unsafe,,sockets_66,960,val,val
sockets_66_0_9760.jpg,socket,unsafe,,sockets_66,960,val,val
sockets_67_0_1099.jpg,socket,unsafe,,sockets_67,961,val,val
sockets_67_0_3479.jpg,socket,unsafe,,sockets_67,961,val,train
sockets_67_0_6458.jpg,socket,unsafe,,sockets_67,961,val,train
sockets_67_0_6496.jpg,socket,unsafe,,sockets_67,961,val,train
sockets_67_0_6775.jpg,socket,unsafe,,sockets_67,961,val,train
sockets_68_0_2275.jpg,socket,unsafe,,sockets_68,962,test,train
sockets_68_0_2762.jpg,socket,unsafe,,sockets_68,962,test,train
sockets_68_0_4024.jpg,socket,unsafe,,sockets_68,962,test,train
sockets_68_0_4913.jpg,socket,unsafe,,sockets_68,962,test,train
sockets_68_0_9993.jpg,socket,unsafe,,sockets_68,962,test,train
sockets_69_0_1204.jpg,socket,unsafe,,sockets_69,963,train,train
sockets_69_0_3243.jpg,socket,unsafe,,sockets_69,963,train,train
sockets_69_0_6271.jpg,socket,unsafe,,sockets_69,963,train,train
sockets_69_0_6768.jpg,socket,unsafe,,sockets_69,963,train,train
sockets_69_0_988.jpg,socket,unsafe,,sockets_69,963,train,val
sockets_6_0_1743.jpg,socket,unsafe,,sockets_6,964,train,train
sockets_6_0_4752.jpg,socket,unsafe,,sockets_6,964,train,train
sockets_6_0_5227.jpg,socket,unsafe,,sockets_6,964,train,train
sockets_6_0_7525.jpg,socket,unsafe,,sockets_6,964,train,train
sockets_6_0_7825.jpg,socket,unsafe,,sockets_6,964,train,test
sockets_70_0_1668.jpg,socket,unsafe,,sockets_70,965,train,train
sockets_70_0_2151.jpg,socket,unsafe,,sockets_70,965,train,train
sockets_70_0_3235.jpg,socket,unsafe,,sockets_70,965,train,test
sockets_70_0_4544.jpg,socket,unsafe,,sockets_70,965,train,test
sockets_70_0_8031.jpg,socket,unsafe,,sockets_70,965,train,val
sockets_71_0_1710.jpg,socket,unsafe,,sockets_71,966,train,train
sockets_71_0_2622.jpg,socket,unsafe,,sockets_71,966,train,test
sockets_71_0_5711.jpg,socket,unsafe,,sockets_71,966,train,train
sockets_71_0_8371.jpg,socket,unsafe,,sockets_71,966,train,train
sockets_71_0_9384.jpg,socket,unsafe,,sockets_71,966,train,train
sockets_72_0_1158.jpg,socket,unsafe,,sockets_72,967,train,train
sockets_72_0_2148.jpg,socket,unsafe,,sockets_72,967,train,train
sockets_72_0_2457.jpg,socket,unsafe,,sockets_72,967,train,train
sockets_72_0_441.jpg,socket,unsafe,,sockets_72,967,train,train
sockets_72_0_929.jpg,socket,unsafe,,sockets_72,967,train,test
sockets_73_0_2532.jpg,socket,unsafe,,sockets_73,968,train,train
sockets_73_0_5148.jpg,socket,unsafe,,sockets_73,968,train,test
sockets_73_0_5718.jpg,socket,unsafe,,sockets_73,968,train,train
sockets_73_0_8255.jpg,socket,unsafe,,sockets_73,968,train,train
sockets_73_0_9213.jpg,socket,unsafe,,sockets_73,968,train,test
sockets_74_0_1195.jpg,socket,unsafe,,sockets_74,969,val,train
sockets_74_0_3692.jpg,socket,unsafe,,sockets_74,969,val,test
sockets_74_0_8209.jpg,socket,unsafe,,sockets_74,969,val,train
sockets_74_0_94.jpg,socket,unsafe,,sockets_74,969,val,val
sockets_74_0_9740.jpg,socket,unsafe,,sockets_74,969,val,train
sockets_75_0_1624.jpg,socket,unsafe,,sockets_75,970,test,train
sockets_75_0_7292.jpg,socket,unsafe,,sockets_75,970,test,test
sockets_75_0_8191.jpg,socket,unsafe,,sockets_75,970,test,train
sockets_75_0_8577.jpg,socket,unsafe,,sockets_75,970,test,val
sockets_75_0_8875.jpg,socket,unsafe,,sockets_75,970,test,train
sockets_76_0_1357.jpg,socket,unsafe,,sockets_76,971,test,train
sockets_76_0_1397.jpg,socket,unsafe,,sockets_76,971,test,train
sockets_76_0_3061.jpg,socket,unsafe,,sockets_76,971,test,test
sockets_76_0_3406.jpg,socket,unsafe,,sockets_76,971,test,train
sockets_76_0_7.jpg,socket,unsafe,,sockets_76,971,test,val
sockets_77_0_1224.jpg,socket,unsafe,,sockets_77,972,train,train
sockets_77_0_2775.jpg,socket,unsafe,,sockets_77,972,train,train
sockets_77_0_3964.jpg,socket,unsafe,,sockets_77,972,train,train
sockets_77_0_6017.jpg,socket,unsafe,,sockets_77,972,train,train
sockets_77_0_8746.jpg,socket,unsafe,,sockets_77,972,train,train
sockets_78_0_1282.jpg,socket,unsafe,,sockets_78,973,val,test
sockets_78_0_1714.jpg,socket,unsafe,,sockets_78,973,val,test
sockets_78_0_4460.jpg,socket,unsafe,,sockets_78,973,val,train
sockets_78_0_4536.jpg,socket,unsafe,,sockets_78,973,val,train
sockets_78_0_8545.jpg,socket,unsafe,,sockets_78,973,val,train
sockets_79_0_3156.jpg,socket,unsafe,,sockets_79,974,train,train
sockets_79_0_5418.jpg,socket,unsafe,,sockets_79,974,train,train
sockets_79_0_5800.jpg,socket,unsafe,,sockets_79,974,train,train
sockets_79_0_6971.jpg,socket,unsafe,,sockets_79,974,train,train
sockets_79_0_7491.jpg,socket,unsafe,,sockets_79,974,train,train
sockets_7_0_4358.jpg,socket,unsafe,,sockets_7,975,train,val
sockets_7_0_6078.jpg,socket,unsafe,,sockets_7,975,train,train
sockets_7_0_6807.jpg,socket,unsafe,,sockets_7,975,train,train
sockets_7_0_7372.jpg,socket,unsafe,,sockets_7,975,train,train
sockets_7_0_7562.jpg,socket,unsafe,,sockets_7,975,train,train
sockets_80_0_2440.jpg,socket,unsafe,,sockets_80,976,val,train
sockets_80_0_3565.jpg,socket,unsafe,,sockets_80,976,val,train
sockets_80_0_5038.jpg,socket,unsafe,,sockets_80,976,val,train
sockets_80_0_725.jpg,socket,unsafe,,sockets_80,976,val,train
sockets_80_0_8639.jpg,socket,unsafe,,sockets_80,976,val,train
sockets_81_0_2194.jpg,socket,unsafe,,sockets_81,977,train,train
sockets_81_0_380.jpg,socket,unsafe,,sockets_81,977,train,train
sockets_81_0_6253.jpg,socket,unsafe,,sockets_81,977,train,train
sockets_81_0_7530.jpg,socket,unsafe,,sockets_81,977,train,val
sockets_81_0_7993.jpg,socket,unsafe,,sockets_81,977,train,test
sockets_82_0_6383.jpg,socket,unsafe,,sockets_82,978,train,test
sockets_82_0_6696.jpg,socket,unsafe,,sockets_82,978,train,train
sockets_82_0_712.jpg,socket,unsafe,,sockets_82,978,train,train
sockets_82_0_8285.jpg,socket,unsafe,,sockets_82,978,train,train
sockets_82_0_9491.jpg,socket,unsafe,,sockets_82,978,train,train
sockets_83_0_2312.jpg,socket,unsafe,,sockets_83,979,val,test
sockets_83_0_6916.jpg,socket,unsafe,,sockets_83,979,val,train
sockets_83_0_7015.jpg,socket,unsafe,,sockets_83,979,val,test
sockets_83_0_7867.jpg,socket,unsafe,,sockets_83,979,val,train
sockets_83_0_832.jpg,socket,unsafe,,sockets_83,979,val,train
sockets_84_0_5197.jpg,socket,unsafe,,sockets_84,980,test,train
sockets_84_0_572.jpg,socket,unsafe,,sockets_84,980,test,train
sockets_84_0_654.jpg,socket,unsafe,,sockets_84,980,test,test
sockets_84_0_9685.jpg,socket,unsafe,,sockets_84,980,test,train
sockets_84_0_9935.jpg,socket,unsafe,,sockets_84,980,test,val
sockets_85_0_1097.jpg,socket,unsafe,,sockets_85,981,train,train
sockets_85_0_2605.jpg,socket,unsafe,,sockets_85,981,train,train
sockets_85_0_2858.jpg,socket,unsafe,,sockets_85,981,train,train
sockets_85_0_3486.jpg,socket,unsafe,,sockets_85,981,train,train
sockets_85_0_5102.jpg,socket,unsafe,,sockets_85,981,train,train
sockets_86_0_2545.jpg,socket,unsafe,,sockets_86,982,val,train
sockets_86_0_3320.jpg,socket,unsafe,,sockets_86,982,val,train
sockets_86_0_5220.jpg,socket,unsafe,,sockets_86,982,val,train
sockets_86_0_6325.jpg,socket,unsafe,,sockets_86,982,val,test
sockets_86_0_8931.jpg,socket,unsafe,,sockets_86,982,val,train
sockets_87_0_1390.jpg,socket,unsafe,,sockets_87,983,train,train
sockets_87_0_30.jpg,socket,unsafe,,sockets_87,983,train,train
sockets_87_0_4313.jpg,socket,unsafe,,sockets_87,983,train,train
sockets_87_0_6038.jpg,socket,unsafe,,sockets_87,983,train,train
sockets_87_0_6379.jpg,socket,unsafe,,sockets_87,983,train,train
sockets_88_0_111.jpg,socket,unsafe,,sockets_88,984,train,test
sockets_88_0_8044.jpg,socket,unsafe,,sockets_88,984,train,test
sockets_88_0_8502.jpg,socket,unsafe,,sockets_88,984,train,test
sockets_88_0_8798.jpg,socket,unsafe,,sockets_88,984,train,val
sockets_88_0_9881.jpg,socket,unsafe,,sockets_88,984,train,val
sockets_89_0_2147.jpg,socket,unsafe,,sockets_89,985,train,train
sockets_89_0_2643.jpg,socket,unsafe,,sockets_89,985,train,train
sockets_89_0_7299.jpg,socket,unsafe,,sockets_89,985,train,test
sockets_89_0_8268.jpg,socket,unsafe,,sockets_89,985,train,val
sockets_89_0_9743.jpg,socket,unsafe,,sockets_89,985,train,train
sockets_8_0_4182.jpg,socket,unsafe,,sockets_8,986,train,train
sockets_8_0_5755.jpg,socket,unsafe,,sockets_8,986,train,train
sockets_8_0_5964.jpg,socket,unsafe,,sockets_8,986,train,val
sockets_8_0_6823.jpg,socket,unsafe,,sockets_8,986,train,train
sockets_8_0_9930.jpg,socket,unsafe,,sockets_8,986,train,val
sockets_90_0_4138.jpg,socket,unsafe,,sockets_90,987,train,train
sockets_90_0_4383.jpg,socket,unsafe,,sockets_90,987,train,test
sockets_90_0_5759.jpg,socket,unsafe,,sockets_90,987,train,train
sockets_90_0_5792.jpg,socket,unsafe,,sockets_90,987,train,test
sockets_90_0_7275.jpg,socket,unsafe,,sockets_90,987,train,test
sockets_92_0_1361.jpg,socket,unsafe,,sockets_92,988,train,train
sockets_92_0_5236.jpg,socket,unsafe,,sockets_92,988,train,val
sockets_92_0_7983.jpg,socket,unsafe,,sockets_92,988,train,test
sockets_92_0_9203.jpg,socket,unsafe,,sockets_92,988,train,train
sockets_92_0_9466.jpg,socket,unsafe,,sockets_92,988,train,test
sockets_93_0_1043.jpg,socket,unsafe,,sockets_93,989,train,train
sockets_93_0_3289.jpg,socket,unsafe,,sockets_93,989,train,test
sockets_93_0_7715.jpg,socket,unsafe,,sockets_93,989,train,train
sockets_93_0_9277.jpg,socket,unsafe,,sockets_93,989,train,train
sockets_93_0_9791.jpg,socket,unsafe,,sockets_93,989,train,train
sockets_94_0_5578.jpg,socket,unsafe,,sockets_94,990,test,train
sockets_94_0_6258.jpg,socket,unsafe,,sockets_94,990,test,val
sockets_94_0_6688.jpg,socket,unsafe,,sockets_94,990,test,train
sockets_94_0_7043.jpg,socket,unsafe,,sockets_94,990,test,test
sockets_94_0_8091.jpg,socket,unsafe,,sockets_94,990,test,train
sockets_95_0_1269.jpg,socket,unsafe,,sockets_95,991,train,val
sockets_95_0_2393.jpg,socket,unsafe,,sockets_95,991,train,val
sockets_95_0_7566.jpg,socket,unsafe,,sockets_95,991,train,train
sockets_95_0_8260.jpg,socket,unsafe,,sockets_95,991,train,train
sockets_95_0_9777.jpg,socket,unsafe,,sockets_95,991,train,train
sockets_98_0_1126.jpg,socket,unsafe,,sockets_98,992,train,train
sockets_98_0_2348.jpg,socket,unsafe,,sockets_98,992,train,test
sockets_98_0_2871.jpg,socket,unsafe,,sockets_98,992,train,train
sockets_98_0_5157.jpg,socket,unsafe,,sockets_98,992,train,train
sockets_98_0_6835.jpg,socket,unsafe,,sockets_98,992,train,train
sockets_99_0_3339.jpg,socket,unsafe,,sockets_99,993,train,train
sockets_99_0_4285.jpg,socket,unsafe,,sockets_99,993,train,test
sockets_99_0_5125.jpg,socket,unsafe,,sockets_99,993,train,train
sockets_99_0_5665.jpg,socket,unsafe,,sockets_99,993,train,test
sockets_99_0_9044.jpg,socket,unsafe,,sockets_99,993,train,train
sockets_9_0_2917.jpg,socket,unsafe,,sockets_9,994,test,test
sockets_9_0_4212.jpg,socket,unsafe,,sockets_9,994,test,val
sockets_9_0_7346.jpg,socket,unsafe,,sockets_9,994,test,train
sockets_9_0_8710.jpg,socket,unsafe,,sockets_9,994,test,test
sockets_9_0_9517.jpg,socket,unsafe,,sockets_9,994,test,train
spoon_16_0_6200.jpg,spoon,safe,,spoon_16,995,val,train
spoon_16_0_8221.jpg,spoon,safe,,spoon_16,995,val,val
spoon_16_0_8299.jpg,spoon,safe,,spoon_16,995,val,val
spoon_16_0_8835.jpg,spoon,safe,,spoon_16,995,val,train
spoon_16_0_9382.jpg,spoon,safe,,spoon_16,995,val,train
spoon_19_0_1268.jpg,spoon,safe,,spoon_19,996,train,train
spoon_19_0_2276.jpg,spoon,safe,,spoon_19,996,train,train
spoon_19_0_3668.jpg,spoon,safe,,spoon_19,996,train,train
spoon_19_0_5266.jpg,spoon,safe,,spoon_19,996,train,train
spoon_19_0_8536.jpg,spoon,safe,,spoon_19,996,train,train
spoon_23_0_2003.jpg,spoon,safe,,spoon_23,997,val,test
spoon_23_0_507.jpg,spoon,safe,,spoon_23,997,val,train
spoon_23_0_6190.jpg,spoon,safe,,spoon_23,997,val,train
spoon_23_0_6361.jpg,spoon,safe,,spoon_23,997,val,train
spoon_23_0_9110.jpg,spoon,safe,,spoon_23,997,val,val
spoon_35_0_2070.jpg,spoon,safe,,spoon_35,998,train,val
spoon_35_0_5027.jpg,spoon,safe,,spoon_35,998,train,train
spoon_35_0_8253.jpg,spoon,safe,,spoon_35,998,train,train
spoon_35_0_9614.jpg,spoon,safe,,spoon_35,998,train,train
spoon_35_0_9897.jpg,spoon,safe,,spoon_35,998,train,val
spoon_42_0_4209.jpg,spoon,safe,,spoon_42,996,train,val
spoon_42_0_7301.jpg,spoon,safe,,spoon_42,996,train,test
spoon_42_0_9590.jpg,spoon,safe,,spoon_42,996,train,test
spoon_42_0_9975.jpg,spoon,safe,,spoon_42,996,train,train
spoon_42_0_9996.jpg,spoon,safe,,spoon_42,996,train,train
spoon_50_0_1923.jpg,spoon,safe,,spoon_50,999,train,train
spoon_50_0_4009.jpg,spoon,safe,,spoon_50,999,train,train
spoon_50_0_5128.jpg,spoon,safe,,spoon_50,999,train,val
spoon_50_0_6239.jpg,spoon,safe,,spoon_50,999,train,train
spoon_50_0_669.jpg,spoon,safe,,spoon_50,999,train,train
spoon_65_0_3701.jpg,spoon,safe,,spoon_65,1000,train,train
spoon_65_0_4879.jpg,spoon,safe,,spoon_65,1000,train,train
spoon_65_0_5031.jpg,spoon,safe,,spoon_65,1000,train,test
spoon_65_0_6054.jpg,spoon,safe,,spoon_65,1000,train,train
spoon_65_0_6722.jpg,spoon,safe,,spoon_65,1000,train,train
spoon_68_0_1239.jpg,spoon,safe,,spoon_68,1001,train,test
spoon_68_0_3424.jpg,spoon,safe,,spoon_68,1001,train,train
spoon_68_0_5227.jpg,spoon,safe,,spoon_68,1001,train,test
spoon_68_0_7752.jpg,spoon,safe,,spoon_68,1001,train,train
spoon_68_0_9949.jpg,spoon,safe,,spoon_68,1001,train,train
spoon_73_0_2155.jpg,spoon,safe,,spoon_73,1002,test,train
spoon_73_0_260.jpg,spoon,safe,,spoon_73,1002,test,train
spoon_73_0_3617.jpg,spoon,safe,,spoon_73,1002,test,train
spoon_73_0_446.jpg,spoon,safe,,spoon_73,1002,test,train
spoon_73_0_8318.jpg,spoon,safe,,spoon_73,1002,test,test
spoon_83_0_3541.jpg,spoon,safe,,spoon_83,1003,train,train
spoon_83_0_4544.jpg,spoon,safe,,spoon_83,1003,train,train
spoon_83_0_5856.jpg,spoon,safe,,spoon_83,1003,train,val
spoon_83_0_6500.jpg,spoon,safe,,spoon_83,1003,train,train
spoon_83_0_9703.jpg,spoon,safe,,spoon_83,1003,train,test
spoon_88_0_5525.jpg,spoon,safe,,spoon_88,1004,train,train
spoon_88_0_6415.jpg,spoon,safe,,spoon_88,1004,train,train
spoon_88_0_7794.jpg,spoon,safe,,spoon_88,1004,train,train
spoon_88_0_9083.jpg,spoon,safe,,spoon_88,1004,train,train
spoon_88_0_9803.jpg,spoon,safe,,spoon_88,1004,train,train
toy_100_0_1464.jpg,toy,safe,,toy_100,1005,train,train
toy_100_0_2044.jpg,toy,safe,,toy_100,1005,train,train
toy_100_0_5644.jpg,toy,safe,,toy_100,1005,train,test
toy_100_0_806.jpg,toy,safe,,toy_100,1005,train,train
toy_100_0_9176.jpg,toy,safe,,toy_100,1005,train,test
toy_101_0_1281.jpg,toy,safe,,toy_101,1006,test,train
toy_101_0_219.jpg,toy,safe,,toy_101,1006,test,train
toy_101_0_7294.jpg,toy,safe,,toy_101,1006,test,test
toy_101_0_8382.jpg,toy,safe,,toy_101,1006,test,val
toy_101_0_9595.jpg,toy,safe,,toy_101,1006,test,test
toy_102_0_3447.jpg,toy,safe,,toy_102,1007,test,train
toy_102_0_3455.jpg,toy,safe,,toy_102,1007,test,test
toy_102_0_3756.jpg,toy,safe,,toy_102,1007,test,val
toy_102_0_4944.jpg,toy,safe,,toy_102,1007,test,train
toy_102_0_5078.jpg,toy,safe,,toy_102,1007,test,test
toy_103_0_2668.jpg,toy,safe,,toy_103,1008,train,train
toy_103_0_3439.jpg,toy,safe,,toy_103,1008,train,train
toy_103_0_451.jpg,toy,safe,,toy_103,1008,train,train
toy_103_0_5653.jpg,toy,safe,,toy_103,1008,train,train
toy_103_0_8352.jpg,toy,safe,,toy_103,1008,train,train
toy_104_0_1326.jpg,toy,safe,,toy_104,1009,test,train
toy_104_0_1767.jpg,toy,safe,,toy_104,1009,test,val
toy_104_0_1996.jpg,toy,safe,,toy_104,1009,test,train
toy_104_0_5005.jpg,toy,safe,,toy_104,1009,test,train
toy_104_0_5501.jpg,toy,safe,,toy_104,1009,test,train
toy_105_0_1991.jpg,toy,safe,,toy_105,1010,train,test
toy_105_0_2077.jpg,toy,safe,,toy_105,1010,train,train
toy_105_0_2496.jpg,toy,safe,,toy_105,1010,train,val
toy_105_0_5442.jpg,toy,safe,,toy_105,1010,train,train
toy_105_0_9782.jpg,toy,safe,,toy_105,1010,train,train
toy_106_0_1623.jpg,toy,safe,,toy_106,1011,train,val
toy_106_0_5658.jpg,toy,safe,,toy_106,1011,train,val
toy_106_0_6738.jpg,toy,safe,,toy_106,1011,train,test
toy_106_0_8166.jpg,toy,safe,,toy_106,1011,train,train
toy_106_0_9862.jpg,toy,safe,,toy_106,1011,train,val
toy_107_0_2883.jpg,toy,safe,,toy_107,1012,train,val
toy_107_0_3948.jpg,toy,safe,,toy_107,1012,train,train
toy_107_0_6755.jpg,toy,safe,,toy_107,1012,train,train
toy_107_0_8343.jpg,toy,safe,,toy_107,1012,train,train
toy_107_0_8993.jpg,toy,safe,,toy_107,1012,train,train
toy_108_0_7751.jpg,toy,safe,,toy_108,1013,train,train
toy_108_0_8829.jpg,toy,safe,,toy_108,1013,train,train
toy_108_0_9028.jpg,toy,safe,,toy_108,1013,train,train
toy_108_0_9206.jpg,toy,safe,,toy_108,1013,train,val
toy_108_0_940.jpg,toy,safe,,toy_108,1013,train,train
toy_109_0_3441.jpg,toy,safe,,toy_109,1014,train,train
toy_109_0_5164.jpg,toy,safe,,toy_109,1014,train,train
toy_109_0_5881.jpg,toy,safe,,toy_109,1014,train,train
toy_109_0_8066.jpg,toy,safe,,toy_109,1014,train,train
toy_109_0_8111.jpg,toy,safe,,toy_109,1014,train,test
toy_10_0_2916.jpg,toy,safe,,toy_10,1015,test,train
toy_10_0_6615.jpg,toy,safe,,toy_10,1015,test,val
toy_10_0_7946.jpg,toy,safe,,toy_10,1015,test,train
toy_10_0_8684.jpg,toy,safe,,toy_10,1015,test,train
toy_10_0_9712.jpg,toy,safe,,toy_10,1015,test,test
toy_110_0_323.jpg,toy,safe,,toy_110,1016,test,train
toy_110_0_6223.jpg,toy,safe,,toy_110,1016,test,train
toy_110_0_8170.jpg,toy,safe,,toy_110,1016,test,train
toy_110_0_8909.jpg,toy,safe,,toy_110,1016,test,train
toy_110_0_9840.jpg,toy,safe,,toy_110,1016,test,train
toy_111_0_4852.jpg,toy,safe,,toy_111,1017,train,train
toy_111_0_5255.jpg,toy,safe,,toy_111,1017,train,val
toy_111_0_6308.jpg,toy,safe,,toy_111,1017,train,train
toy_111_0_7879.jpg,toy,safe,,toy_111,1017,train,train
toy_111_0_908.jpg,toy,safe,,toy_111,1017,train,train
toy_112_0_3097.jpg,toy,safe,,toy_112,1018,val,train
toy_112_0_3513.jpg,toy,safe,,toy_112,1018,val,train
toy_112_0_4142.jpg,toy,safe,,toy_112,1018,val,train
toy_112_0_422.jpg,toy,safe,,toy_112,1018,val,train
toy_112_0_56.jpg,toy,safe,,toy_112,1018,val,train
toy_113_0_1279.jpg,toy,safe,,toy_113,1019,train,train
toy_113_0_2480.jpg,toy,safe,,toy_113,1019,train,train
toy_113_0_8014.jpg,toy,safe,,toy_113,1019,train,train
toy_113_0_8280.jpg,toy,safe,,toy_113,1019,train,test
toy_113_0_8906.jpg,toy,safe,,toy_113,1019,train,train
toy_114_0_228.jpg,toy,safe,,toy_114,1020,train,val
toy_114_0_3336.jpg,toy,safe,,toy_114,1020,train,train
toy_114_0_3577.jpg,toy,safe,,toy_114,1020,train,val
toy_114_0_4758.jpg,toy,safe,,toy_114,1020,train,train
toy_114_0_9077.jpg,toy,safe,,toy_114,1020,train,train
toy_115_0_307.jpg,toy,safe,,toy_115,1021,train,train
toy_115_0_5357.jpg,toy,safe,,toy_115,1021,train,train
toy_115_0_6686.jpg,toy,safe,,toy_115,1021,train,train
toy_115_0_9431.jpg,toy,safe,,toy_115,1021,train,train
toy_115_0_9519.jpg,toy,safe,,toy_115,1021,train,train
toy_116_0_1294.jpg,toy,safe,,toy_116,1022,val,test
toy_116_0_3458.jpg,toy,safe,,toy_116,1022,val,val
toy_116_0_4810.jpg,toy,safe,,toy_116,1022,val,train
toy_116_0_7916.jpg,toy,safe,,toy_116,1022,val,train
toy_116_0_8888.jpg,toy,safe,,toy_116,1022,val,train
toy_117_0_5266.jpg,toy,safe,,toy_117,1023,train,train
toy_117_0_5606.jpg,toy,safe,,toy_117,1023,train,train
toy_117_0_7492.jpg,toy,safe,,toy_117,1023,train,test
toy_117_0_7597.jpg,toy,safe,,toy_117,1023,train,train
toy_117_0_8730.jpg,toy,safe,,toy_117,1023,train,train
toy_118_0_4290.jpg,toy,safe,,toy_118,1024,train,train
toy_118_0_526.jpg,toy,safe,,toy_118,1024,train,train
toy_118_0_6407.jpg,toy,safe,,toy_118,1024,train,train
toy_118_0_6559.jpg,toy,safe,,toy_118,1024,train,train
toy_118_0_956.jpg,toy,safe,,toy_118,1024,train,val
toy_119_0_3320.jpg,toy,safe,,toy_119,1025,train,train
toy_119_0_3890.jpg,toy,safe,,toy_119,1025,train,train
toy_119_0_4822.jpg,toy,safe,,toy_119,1025,train,train
toy_119_0_4830.jpg,toy,safe,,toy_119,1025,train,test
toy_119_0_5991.jpg,toy,safe,,toy_119,1025,train,test
toy_11_0_3615.jpg,toy,safe,,toy_11,1026,train,val
toy_11_0_5320.jpg,toy,safe,,toy_11,1026,train,train
toy_11_0_7551.jpg,toy,safe,,toy_11,1026,train,train
toy_11_0_8014.jpg,toy,safe,,toy_11,1026,train,train
toy_11_0_9240.jpg,toy,safe,,toy_11,1026,train,train
toy_120_0_3961.jpg,toy,safe,,toy_120,1027,val,train
toy_120_0_4656.jpg,toy,safe,,toy_120,1027,val,val
toy_120_0_5821.jpg,toy,safe,,toy_120,1027,val,train
toy_120_0_9166.jpg,toy,safe,,toy_120,1027,val,test
toy_120_0_9566.jpg,toy,safe,,toy_120,1027,val,train
toy_121_0_3134.jpg,toy,safe,,toy_121,1028,test,test
toy_121_0_333.jpg,toy,safe,,toy_121,1028,test,train
toy_121_0_4153.jpg,toy,safe,,toy_121,1028,test,val
toy_121_0_4344.jpg,toy,safe,,toy_121,1028,test,test
toy_121_0_8789.jpg,toy,safe,,toy_121,1028,test,train
toy_122_0_227.jpg,toy,safe,,toy_122,1029,train,train
toy_122_0_2750.jpg,toy,safe,,toy_122,1029,train,train
toy_122_0_7254.jpg,toy,safe,,toy_122,1029,train,train
toy_122_0_9101.jpg,toy,safe,,toy_122,1029,train,train
toy_122_0_9110.jpg,toy,safe,,toy_122,1029,train,test
toy_123_0_2107.jpg,toy,safe,,toy_123,1030,train,train
toy_123_0_2915.jpg,toy,safe,,toy_123,1030,train,train
toy_123_0_4384.jpg,toy,safe,,toy_123,1030,train,train
toy_123_0_771.jpg,toy,safe,,toy_123,1030,train,test
toy_123_0_879.jpg,toy,safe,,toy_123,1030,train,train
toy_124_0_3563.jpg,toy,safe,,toy_124,1031,train,val
toy_124_0_396.jpg,toy,safe,,toy_124,1031,train,train
toy_124_0_4733.jpg,toy,safe,,toy_124,1031,train,train
toy_124_0_6361.jpg,toy,safe,,toy_124,1031,train,train
toy_124_0_8970.jpg,toy,safe,,toy_124,1031,train,train
toy_125_0_1583.jpg,toy,safe,,toy_125,1032,train,train
toy_125_0_3686.jpg,toy,safe,,toy_125,1032,train,test
toy_125_0_6107.jpg,toy,safe,,toy_125,1032,train,train
toy_125_0_9618.jpg,toy,safe,,toy_125,1032,train,train
toy_125_0_9845.jpg,toy,safe,,toy_125,1032,train,train
toy_126_0_1682.jpg,toy,safe,,toy_126,1033,train,train
toy_126_0_3226.jpg,toy,safe,,toy_126,1033,train,train
toy_126_0_4027.jpg,toy,safe,,toy_126,1033,train,val
toy_126_0_4220.jpg,toy,safe,,toy_126,1033,train,train
toy_126_0_6275.jpg,toy,safe,,toy_126,1033,train,train
toy_127_0_2546.jpg,toy,safe,,toy_127,1034,test,val
toy_127_0_4132.jpg,toy,safe,,toy_127,1034,test,train
toy_127_0_6055.jpg,toy,safe,,toy_127,1034,test,train
toy_127_0_6751.jpg,toy,safe,,toy_127,1034,test,train
toy_127_0_8060.jpg,toy,safe,,toy_127,1034,test,train
toy_128_0_4727.jpg,toy,safe,,toy_128,1035,train,val
toy_128_0_4742.jpg,toy,safe,,toy_128,1035,train,train
toy_128_0_5613.jpg,toy,safe,,toy_128,1035,train,train
toy_128_0_6278.jpg,toy,safe,,toy_128,1035,train,test
toy_128_0_9823.jpg,toy,safe,,toy_128,1035,train,train
toy_129_0_1181.jpg,toy,safe,,toy_129,1036,train,train
toy_129_0_2098.jpg,toy,safe,,toy_129,1036,train,test
toy_129_0_5006.jpg,toy,safe,,toy_129,1036,train,train
toy_129_0_5517.jpg,toy,safe,,toy_129,1036,train,test
toy_129_0_5542.jpg,toy,safe,,toy_129,1036,train,test
toy_12_0_3957.jpg,toy,safe,,toy_12,1037,train,train
toy_12_0_5238.jpg,toy,safe,,toy_12,1037,train,train
toy_12_0_618.jpg,toy,safe,,toy_12,1037,train,train
toy_12_0_8260.jpg,toy,safe,,toy_12,1037,train,train
toy_12_0_9129.jpg,toy,safe,,toy_12,1037,train,train
toy_130_0_1167.jpg,toy,safe,,toy_130,1038,train,train
toy_130_0_144.jpg,toy,safe,,toy_130,1038,train,train
toy_130_0_4873.jpg,toy,safe,,toy_130,1038,train,train
toy_130_0_639.jpg,toy,safe,,toy_130,1038,train,train
toy_130_0_6973.jpg,toy,safe,,toy_130,1038,train,test
toy_131_0_1140.jpg,toy,safe,,toy_131,1039,val,test
toy_131_0_4801.jpg,toy,safe,,toy_131,1039,val,train
toy_131_0_5053.jpg,toy,safe,,toy_131,1039,val,train
toy_131_0_8747.jpg,toy,safe,,toy_131,1039,val,val
toy_131_0_9094.jpg,toy,safe,,toy_131,1039,val,test
toy_132_0_6731.jpg,toy,safe,,toy_132,1040,train,train
toy_132_0_7557.jpg,toy,safe,,toy_132,1040,train,val
toy_132_0_8349.jpg,toy,safe,,toy_132,1040,train,train
toy_132_0_9006.jpg,toy,safe,,toy_132,1040,train,train
toy_132_0_9186.jpg,toy,safe,,toy_132,1040,train,train
toy_133_0_1276.jpg,toy,safe,,toy_133,1041,test,val
toy_133_0_667.jpg,toy,safe,,toy_133,1041,test,train
toy_133_0_8728.jpg,toy,safe,,toy_133,1041,test,train
toy_133_0_9698.jpg,toy,safe,,toy_133,1041,test,val
toy_133_0_9897.jpg,toy,safe,,toy_133,1041,test,train
toy_134_0_2239.jpg,toy,safe,,toy_134,1042,test,train
toy_134_0_2816.jpg,toy,safe,,toy_134,1042,test,test
toy_134_0_3861.jpg,toy,safe,,toy_134,1042,test,train
toy_134_0_5368.jpg,toy,safe,,toy_134,1042,test,train
toy_134_0_5948.jpg,toy,safe,,toy_134,1042,test,val
toy_135_0_1644.jpg,toy,safe,,toy_135,1043,train,train
toy_135_0_4356.jpg,toy,safe,,toy_135,1043,train,train
toy_135_0_633.jpg,toy,safe,,toy_135,1043,train,train
toy_135_0_6856.jpg,toy,safe,,toy_135,1043,train,train
toy_135_0_7776.jpg,toy,safe,,toy_135,1043,train,test
toy_136_0_2995.jpg,toy,safe,,toy_136,1044,train,val
toy_136_0_4509.jpg,toy,safe,,toy_136,1044,train,train
toy_136_0_5851.jpg,toy,safe,,toy_136,1044,train,val
toy_136_0_6772.jpg,toy,safe,,toy_136,1044,train,train
toy_136_0_7384.jpg,toy,safe,,toy_136,1044,train,train
toy_137_0_2438.jpg,toy,safe,,toy_137,1045,train,train
toy_137_0_4073.jpg,toy,safe,,toy_137,1045,train,train
toy_137_0_4799.jpg,toy,safe,,toy_137,1045,train,val
toy_137_0_6098.jpg,toy,safe,,toy_137,1045,train,val
toy_137_0_6252.jpg,toy,safe,,toy_137,1045,train,train
toy_138_0_2811.jpg,toy,safe,,toy_138,1046,val,train
toy_138_0_2939.jpg,toy,safe,,toy_138,1046,val,val
toy_138_0_4405.jpg,toy,safe,,toy_138,1046,val,train
toy_138_0_8869.jpg,toy,safe,,toy_138,1046,val,train
toy_138_0_985.jpg,toy,safe,,toy_138,1046,val,test
toy_139_0_2187.jpg,toy,safe,,toy_139,1047,val,train
toy_139_0_2354.jpg,toy,safe,,toy_139,1047,val,train
toy_139_0_536.jpg,toy,safe,,toy_139,1047,val,train
toy_139_0_5863.jpg,toy,safe,,toy_139,1047,val,train
toy_139_0_7537.jpg,toy,safe,,toy_139,1047,val,train
toy_13_0_2297.jpg,toy,safe,,toy_13,1048,train,test
toy_13_0_2986.jpg,toy,safe,,toy_13,1048,train,train
toy_13_0_3897.jpg,toy,safe,,toy_13,1048,train,val
toy_13_0_7514.jpg,toy,safe,,toy_13,1048,train,val
toy_13_0_9202.jpg,toy,safe,,toy_13,1048,train,val
toy_140_0_525.jpg,toy,safe,,toy_140,1049,train,train
toy_140_0_527.jpg,toy,safe,,toy_140,1049,train,train
toy_140_0_6034.jpg,toy,safe,,toy_140,1049,train,train
toy_140_0_7130.jpg,toy,safe,,toy_140,1049,train,train
toy_140_0_996.jpg,toy,safe,,toy_140,1049,train,test
toy_141_0_3007.jpg,toy,safe,,toy_141,1050,train,train
toy_141_0_5308.jpg,toy,safe,,toy_141,1050,train,train
toy_141_0_7387.jpg,toy,safe,,toy_141,1050,train,train
toy_141_0_9332.jpg,toy,safe,,toy_141,1050,train,val
toy_141_0_9406.jpg,toy,safe,,toy_141,1050,train,test
toy_142_0_2312.jpg,toy,safe,,toy_142,1051,train,train
toy_142_0_2324.jpg,toy,safe,,toy_142,1051,train,train
toy_142_0_3828.jpg,toy,safe,,toy_142,1051,train,test
toy_142_0_4878.jpg,toy,safe,,toy_142,1051,train,train
toy_142_0_6213.jpg,toy,safe,,toy_142,1051,train,train
toy_143_0_5199.jpg,toy,safe,,toy_143,1052,train,train
toy_143_0_5386.jpg,toy,safe,,toy_143,1052,train,train
toy_143_0_6544.jpg,toy,safe,,toy_143,1052,train,val
toy_143_0_6776.jpg,toy,safe,,toy_143,1052,train,train
toy_143_0_9856.jpg,toy,safe,,toy_143,1052,train,val
toy_144_0_2176.jpg,toy,safe,,toy_144,1053,train,train
toy_144_0_6307.jpg,toy,safe,,toy_144,1053,train,train
toy_144_0_8598.jpg,toy,safe,,toy_144,1053,train,train
toy_144_0_920.jpg,toy,safe,,toy_144,1053,train,train
toy_144_0_9690.jpg,toy,safe,,toy_144,1053,train,test
toy_145_0_3589.jpg,toy,safe,,toy_145,1054,val,train
toy_145_0_4601.jpg,toy,safe,,toy_145,1054,val,train
toy_145_0_5491.jpg,toy,safe,,toy_145,1054,val,test
toy_145_0_7302.jpg,toy,safe,,toy_145,1054,val,train
toy_145_0_786.jpg,toy,safe,,toy_145,1054,val,train
toy_146_0_1408.jpg,toy,safe,,toy_146,1055,test,train
toy_146_0_3919.jpg,toy,safe,,toy_146,1055,test,train
toy_146_0_4001.jpg,toy,safe,,toy_146,1055,test,train
toy_146_0_9187.jpg,toy,safe,,toy_146,1055,test,train
toy_146_0_9734.jpg,toy,safe,,toy_146,1055,test,val
toy_147_0_3909.jpg,toy,safe,,toy_147,1056,val,train
toy_147_0_4014.jpg,toy,safe,,toy_147,1056,val,train
toy_147_0_4793.jpg,toy,safe,,toy_147,1056,val,val
toy_147_0_5978.jpg,toy,safe,,toy_147,1056,val,train
toy_147_0_6111.jpg,toy,safe,,toy_147,1056,val,val
toy_148_0_1812.jpg,toy,safe,,toy_148,1057,train,val
toy_148_0_3854.jpg,toy,safe,,toy_148,1057,train,train
toy_148_0_400.jpg,toy,safe,,toy_148,1057,train,val
toy_148_0_6530.jpg,toy,safe,,toy_148,1057,train,train
toy_148_0_7172.jpg,toy,safe,,toy_148,1057,train,val
toy_149_0_1277.jpg,toy,safe,,toy_149,1058,test,train
toy_149_0_2113.jpg,toy,safe,,toy_149,1058,test,test
toy_149_0_4951.jpg,toy,safe,,toy_149,1058,test,train
toy_149_0_6869.jpg,toy,safe,,toy_149,1058,test,train
toy_149_0_7204.jpg,toy,safe,,toy_149,1058,test,train
toy_14_0_376.jpg,toy,safe,,toy_14,1059,val,train
toy_14_0_5736.jpg,toy,safe,,toy_14,1059,val,test
toy_14_0_6277.jpg,toy,safe,,toy_14,1059,val,train
toy_14_0_8524.jpg,toy,safe,,toy_14,1059,val,test
toy_14_0_8822.jpg,toy,safe,,toy_14,1059,val,train
toy_150_0_1102.jpg,toy,safe,,toy_150,1060,train,train
toy_150_0_2776.jpg,toy,safe,,toy_150,1060,train,test
toy_150_0_3695.jpg,toy,safe,,toy_150,1060,train,test
toy_150_0_8477.jpg,toy,safe,,toy_150,1060,train,train
toy_150_0_9226.jpg,toy,safe,,toy_150,1060,train,train
toy_151_0_1730.jpg,toy,safe,,toy_151,1061,val,val
toy_151_0_204.jpg,toy,safe,,toy_151,1061,val,train
toy_151_0_3068.jpg,toy,safe,,toy_151,1061,val,train
toy_151_0_3229.jpg,toy,safe,,toy_151,1061,val,train
toy_151_0_4887.jpg,toy,safe,,toy_151,1061,val,train
toy_152_0_4037.jpg,toy,safe,,toy_152,1062,train,train
toy_152_0_5896.jpg,toy,safe,,toy_152,1062,train,train
toy_152_0_7643.jpg,toy,safe,,toy_152,1062,train,val
toy_152_0_8875.jpg,toy,safe,,toy_152,1062,train,train
toy_152_0_9586.jpg,toy,safe,,toy_152,1062,train,train
toy_153_0_1870.jpg,toy,safe,,toy_153,1063,train,train
toy_153_0_3242.jpg,toy,safe,,toy_153,1063,train,train
toy_153_0_5443.jpg,toy,safe,,toy_153,1063,train,val
toy_153_0_8404.jpg,toy,safe,,toy_153,1063,train,train
toy_153_0_849.jpg,toy,safe,,toy_153,1063,train,train
toy_154_0_307.jpg,toy,safe,,toy_154,1064,test,val
toy_154_0_4101.jpg,toy,safe,,toy_154,1064,test,val
toy_154_0_4599.jpg,toy,safe,,toy_154,1064,test,val
toy_154_0_8534.jpg,toy,safe,,toy_154,1064,test,train
toy_154_0_9722.jpg,toy,safe,,toy_154,1064,test,train
toy_155_0_1889.jpg,toy,safe,,toy_155,1065,test,val
toy_155_0_6808.jpg,toy,safe,,toy_155,1065,test,test
toy_155_0_8258.jpg,toy,safe,,toy_155,1065,test,train
toy_155_0_9742.jpg,toy,safe,,toy_155,1065,test,train
toy_155_0_9826.jpg,toy,safe,,toy_155,1065,test,train
toy_156_0_1966.jpg,toy,safe,,toy_156,1066,train,train
toy_156_0_2005.jpg,toy,safe,,toy_156,1066,train,test
toy_156_0_416.jpg,toy,safe,,toy_156,1066,train,train
toy_156_0_5317.jpg,toy,safe,,toy_156,1066,train,train
toy_156_0_8126.jpg,toy,safe,,toy_156,1066,train,train
toy_157_0_1241.jpg,toy,safe,,toy_157,1067,train,test
toy_157_0_2752.jpg,toy,safe,,toy_157,1067,train,train
toy_157_0_3784.jpg,toy,safe,,toy_157,1067,train,train
toy_157_0_5427.jpg,toy,safe,,toy_157,1067,train,train
toy_157_0_6045.jpg,toy,safe,,toy_157,1067,train,test
toy_158_0_1457.jpg,toy,safe,,toy_158,1068,train,train
toy_158_0_6513.jpg,toy,safe,,toy_158,1068,train,val
toy_158_0_7071.jpg,toy,safe,,toy_158,1068,train,train
toy_158_0_7189.jpg,toy,safe,,toy_158,1068,train,train
toy_158_0_7657.jpg,toy,safe,,toy_158,1068,train,train
toy_159_0_2099.jpg,toy,safe,,toy_159,1069,train,train
toy_159_0_2496.jpg,toy,safe,,toy_159,1069,train,test
toy_159_0_4684.jpg,toy,safe,,toy_159,1069,train,train
toy_159_0_5114.jpg,toy,safe,,toy_159,1069,train,test
toy_159_0_9161.jpg,toy,safe,,toy_159,1069,train,train
toy_15_0_1663.jpg,toy,safe,,toy_15,1070,train,train
toy_15_0_1809.jpg,toy,safe,,toy_15,1070,train,test
toy_15_0_2455.jpg,toy,safe,,toy_15,1070,train,train
toy_15_0_4920.jpg,toy,safe,,toy_15,1070,train,val
toy_15_0_6992.jpg,toy,safe,,toy_15,1070,train,train
toy_160_0_2294.jpg,toy,safe,,toy_160,1071,test,test
toy_160_0_427.jpg,toy,safe,,toy_160,1071,test,train
toy_160_0_4720.jpg,toy,safe,,toy_160,1071,test,test
toy_160_0_7166.jpg,toy,safe,,toy_160,1071,test,train
toy_160_0_8498.jpg,toy,safe,,toy_160,1071,test,train
toy_161_0_3201.jpg,toy,safe,,toy_161,1072,train,val
toy_161_0_3686.jpg,toy,safe,,toy_161,1072,train,train
toy_161_0_3929.jpg,toy,safe,,toy_161,1072,train,train
toy_161_0_8377.jpg,toy,safe,,toy_161,1072,train,train
toy_161_0_8602.jpg,toy,safe,,toy_161,1072,train,val
toy_162_0_1684.jpg,toy,safe,,toy_162,1073,train,train
toy_162_0_181.jpg,toy,safe,,toy_162,1073,train,train
toy_162_0_5046.jpg,toy,safe,,toy_162,1073,train,train
toy_162_0_711.jpg,toy,safe,,toy_162,1073,train,train
toy_162_0_9343.jpg,toy,safe,,toy_162,1073,train,train
toy_163_0_5292.jpg,toy,safe,,toy_163,1074,train,train
toy_163_0_8410.jpg,toy,safe,,toy_163,1074,train,train
toy_163_0_8855.jpg,toy,safe,,toy_163,1074,train,train
toy_163_0_9437.jpg,toy,safe,,toy_163,1074,train,test
toy_163_0_963.jpg,toy,safe,,toy_163,1074,train,train
toy_164_0_7340.jpg,toy,safe,,toy_164,1075,train,val
toy_164_0_829.jpg,toy,safe,,toy_164,1075,train,train
toy_164_0_8365.jpg,toy,safe,,toy_164,1075,train,val
toy_164_0_8918.jpg,toy,safe,,toy_164,1075,train,test
toy_164_0_9331.jpg,toy,safe,,toy_164,1075,train,train
toy_165_0_3161.jpg,toy,safe,,toy_165,1076,val,train
toy_165_0_6987.jpg,toy,safe,,toy_165,1076,val,test
toy_165_0_7042.jpg,toy,safe,,toy_165,1076,val,train
toy_165_0_7182.jpg,toy,safe,,toy_165,1076,val,train
toy_165_0_8526.jpg,toy,safe,,toy_165,1076,val,train
toy_166_0_1611.jpg,toy,safe,,toy_166,1077,train,test
toy_166_0_1886.jpg,toy,safe,,toy_166,1077,train,test
toy_166_0_2854.jpg,toy,safe,,toy_166,1077,train,train
toy_166_0_3552.jpg,toy,safe,,toy_166,1077,train,train
toy_166_0_8959.jpg,toy,safe,,toy_166,1077,train,train
toy_167_0_1916.jpg,toy,safe,,toy_167,1078,train,val
toy_167_0_2952.jpg,toy,safe,,toy_167,1078,train,val
toy_167_0_3164.jpg,toy,safe,,toy_167,1078,train,train
toy_167_0_6917.jpg,toy,safe,,toy_167,1078,train,val
toy_167_0_6929.jpg,toy,safe,,toy_167,1078,train,train
toy_168_0_2760.jpg,toy,safe,,toy_168,1079,train,train
toy_168_0_6369.jpg,toy,safe,,toy_168,1079,train,train
toy_168_0_7800.jpg,toy,safe,,toy_168,1079,train,test
toy_168_0_8803.jpg,toy,safe,,toy_168,1079,train,val
toy_168_0_9669.jpg,toy,safe,,toy_168,1079,train,train
toy_169_0_1960.jpg,toy,safe,,toy_169,1080,test,train
toy_169_0_1995.jpg,toy,safe,,toy_169,1080,test,train
toy_169_0_4604.jpg,toy,safe,,toy_169,1080,test,train
toy_169_0_5363.jpg,toy,safe,,toy_169,1080,test,train
toy_169_0_7308.jpg,toy,safe,,toy_169,1080,test,train
toy_16_0_1238.jpg,toy,safe,,toy_16,1081,train,train
toy_16_0_2346.jpg,toy,safe,,toy_16,1081,train,train
toy_16_0_4399.jpg,toy,safe,,toy_16,1081,train,train
toy_16_0_6274.jpg,toy,safe,,toy_16,1081,train,val
toy_16_0_7162.jpg,toy,safe,,toy_16,1081,train,train
toy_170_0_1163.jpg,toy,safe,,toy_170,1082,train,train
toy_170_0_1313.jpg,toy,safe,,toy_170,1082,train,val
toy_170_0_3595.jpg,toy,safe,,toy_170,1082,train,train
toy_170_0_4857.jpg,toy,safe,,toy_170,1082,train,train
toy_170_0_5494.jpg,toy,safe,,toy_170,1082,train,train
toy_171_0_1364.jpg,toy,safe,,toy_171,1083,val,train
toy_171_0_1765.jpg,toy,safe,,toy_171,1083,val,train
toy_171_0_4535.jpg,toy,safe,,toy_171,1083,val,test
toy_171_0_775.jpg,toy,safe,,toy_171,1083,val,train
toy_171_0_7823.jpg,toy,safe,,toy_171,1083,val,train
toy_172_0_1498.jpg,toy,safe,,toy_172,1084,train,train
toy_172_0_2408.jpg,toy,safe,,toy_172,1084,train,train
toy_172_0_2460.jpg,toy,safe,,toy_172,1084,train,train
toy_172_0_5907.jpg,toy,safe,,toy_172,1084,train,train
toy_172_0_6196.jpg,toy,safe,,toy_172,1084,train,train
toy_173_0_1205.jpg,toy,safe,,toy_173,1085,train,train
toy_173_0_14.jpg,toy,safe,,toy_173,1085,train,train
toy_173_0_1705.jpg,toy,safe,,toy_173,1085,train,train
toy_173_0_4073.jpg,toy,safe,,toy_173,1085,train,test
toy_173_0_5007.jpg,toy,safe,,toy_173,1085,train,train
toy_174_0_4315.jpg,toy,safe,,toy_174,1086,train,train
toy_174_0_5748.jpg,toy,safe,,toy_174,1086,train,train
toy_174_0_7473.jpg,toy,safe,,toy_174,1086,train,val
toy_174_0_8645.jpg,toy,safe,,toy_174,1086,train,test
toy_174_0_958.jpg,toy,safe,,toy_174,1086,train,train
toy_175_0_4034.jpg,toy,safe,,toy_175,1087,test,train
toy_175_0_5231.jpg,toy,safe,,toy_175,1087,test,test
toy_175_0_5732.jpg,toy,safe,,toy_175,1087,test,train
toy_175_0_6304.jpg,toy,safe,,toy_175,1087,test,val
toy_175_0_8726.jpg,toy,safe,,toy_175,1087,test,train
toy_176_0_1708.jpg,toy,safe,,toy_176,1088,train,train
toy_176_0_1991.jpg,toy,safe,,toy_176,1088,train,val
toy_176_0_4248.jpg,toy,safe,,toy_176,1088,train,train
toy_176_0_521.jpg,toy,safe,,toy_176,1088,train,train
toy_176_0_6129.jpg,toy,safe,,toy_176,1088,train,train
toy_177_0_1020.jpg,toy,safe,,toy_177,1089,train,test
toy_177_0_60.jpg,toy,safe,,toy_177,1089,train,test
toy_177_0_721.jpg,toy,safe,,toy_177,1089,train,train
toy_177_0_8020.jpg,toy,safe,,toy_177,1089,train,val
toy_177_0_8558.jpg,toy,safe,,toy_177,1089,train,train
toy_178_0_412.jpg,toy,safe,,toy_178,1090,train,train
toy_178_0_4513.jpg,toy,safe,,toy_178,1090,train,train
toy_178_0_4661.jpg,toy,safe,,toy_178,1090,train,train
toy_178_0_5041.jpg,toy,safe,,toy_178,1090,train,val
toy_178_0_8775.jpg,toy,safe,,toy_178,1090,train,train
toy_179_0_4750.jpg,toy,safe,,toy_179,1091,train,val
toy_179_0_4992.jpg,toy,safe,,toy_179,1091,train,train
toy_179_0_6589.jpg,toy,safe,,toy_179,1091,train,train
toy_179_0_6643.jpg,toy,safe,,toy_179,1091,train,test
toy_179_0_8651.jpg,toy,safe,,toy_179,1091,train,train
toy_17_0_2082.jpg,toy,safe,,toy_17,1092,train,train
toy_17_0_2254.jpg,toy,safe,,toy_17,1092,train,train
toy_17_0_2533.jpg,toy,safe,,toy_17,1092,train,train
toy_17_0_5356.jpg,toy,safe,,toy_17,1092,train,train
toy_17_0_7950.jpg,toy,safe,,toy_17,1092,train,train
toy_180_0_1398.jpg,toy,safe,,toy_180,1093,val,train
toy_180_0_2674.jpg,toy,safe,,toy_180,1093,val,train
toy_180_0_5018.jpg,toy,safe,,toy_180,1093,val,train
toy_180_0_6355.jpg,toy,safe,,toy_180,1093,val,train
toy_180_0_9946.jpg,toy,safe,,toy_180,1093,val,train
toy_181_0_3547.jpg,toy,safe,,toy_181,1094,train,test
toy_181_0_391.jpg,toy,safe,,toy_181,1094,train,train
toy_181_0_404.jpg,toy,safe,,toy_181,1094,train,train
toy_181_0_4959.jpg,toy,safe,,toy_181,1094,train,val
toy_181_0_832.jpg,toy,safe,,toy_181,1094,train,test
toy_182_0_160.jpg,toy,safe,,toy_182,1095,train,train
toy_182_0_2285.jpg,toy,safe,,toy_182,1095,train,train
toy_182_0_5552.jpg,toy,safe,,toy_182,1095,train,test
toy_182_0_7343.jpg,toy,safe,,toy_182,1095,train,train
toy_182_0_8186.jpg,toy,safe,,toy_182,1095,train,train
toy_183_0_1712.jpg,toy,safe,,toy_183,1096,train,test
toy_183_0_1738.jpg,toy,safe,,toy_183,1096,train,test
toy_183_0_2375.jpg,toy,safe,,toy_183,1096,train,train
toy_183_0_3303.jpg,toy,safe,,toy_183,1096,train,test
toy_183_0_3437.jpg,toy,safe,,toy_183,1096,train,val
toy_184_0_2132.jpg,toy,safe,,toy_184,1097,train,test
toy_184_0_2815.jpg,toy,safe,,toy_184,1097,train,train
toy_184_0_4141.jpg,toy,safe,,toy_184,1097,train,train
toy_184_0_573.jpg,toy,safe,,toy_184,1097,train,train
toy_184_0_8468.jpg,toy,safe,,toy_184,1097,train,test
toy_185_0_3756.jpg,toy,safe,,toy_185,1098,train,train
toy_185_0_5119.jpg,toy,safe,,toy_185,1098,train,train
toy_185_0_7582.jpg,toy,safe,,toy_185,1098,train,train
toy_185_0_9785.jpg,toy,safe,,toy_185,1098,train,val
toy_185_0_987.jpg,toy,safe,,toy_185,1098,train,train
toy_186_0_2512.jpg,toy,safe,,toy_186,1099,val,test
toy_186_0_5299.jpg,toy,safe,,toy_186,1099,val,train
toy_186_0_6309.jpg,toy,safe,,toy_186,1099,val,train
toy_186_0_9138.jpg,toy,safe,,toy_186,1099,val,val
toy_186_0_9365.jpg,toy,safe,,toy_186,1099,val,train
toy_187_0_338.jpg,toy,safe,,toy_187,1100,val,train
toy_187_0_3955.jpg,toy,safe,,toy_187,1100,val,train
toy_187_0_7128.jpg,toy,safe,,toy_187,1100,val,test
toy_187_0_726.jpg,toy,safe,,toy_187,1100,val,val
toy_187_0_8282.jpg,toy,safe,,toy_187,1100,val,train
toy_188_0_1939.jpg,toy,safe,,toy_188,1101,test,val
toy_188_0_2666.jpg,toy,safe,,toy_188,1101,test,val
toy_188_0_3196.jpg,toy,safe,,toy_188,1101,test,train
toy_188_0_6882.jpg,toy,safe,,toy_188,1101,test,test
toy_188_0_766.jpg,toy,safe,,toy_188,1101,test,train
toy_189_0_2466.jpg,toy,safe,,toy_189,1102,val,train
toy_189_0_3291.jpg,toy,safe,,toy_189,1102,val,val
toy_189_0_4221.jpg,toy,safe,,toy_189,1102,val,train
toy_189_0_9556.jpg,toy,safe,,toy_189,1102,val,test
toy_189_0_9575.jpg,toy,safe,,toy_189,1102,val,train
toy_18_0_3858.jpg,toy,safe,,toy_18,1103,train,test
toy_18_0_5966.jpg,toy,safe,,toy_18,1103,train,train
toy_18_0_7383.jpg,toy,safe,,toy_18,1103,train,test
toy_18_0_7681.jpg,toy,safe,,toy_18,1103,train,train
toy_18_0_9684.jpg,toy,safe,,toy_18,1103,train,test
toy_190_0_1346.jpg,toy,safe,,toy_190,1104,test,train
toy_190_0_2765.jpg,toy,safe,,toy_190,1104,test,train
toy_190_0_4430.jpg,toy,safe,,toy_190,1104,test,val
toy_190_0_7732.jpg,toy,safe,,toy_190,1104,test,val
toy_190_0_9673.jpg,toy,safe,,toy_190,1104,test,val
toy_191_0_1148.jpg,toy,safe,,toy_191,1105,train,train
toy_191_0_2055.jpg,toy,safe,,toy_191,1105,train,train
toy_191_0_3979.jpg,toy,safe,,toy_191,1105,train,train
toy_191_0_5867.jpg,toy,safe,,toy_191,1105,train,val
toy_191_0_741.jpg,toy,safe,,toy_191,1105,train,train
toy_192_0_2150.jpg,toy,safe,,toy_192,1106,train,train
toy_192_0_5747.jpg,toy,safe,,toy_192,1106,train,test
toy_192_0_5897.jpg,toy,safe,,toy_192,1106,train,val
toy_192_0_8816.jpg,toy,safe,,toy_192,1106,train,train
toy_192_0_9075.jpg,toy,safe,,toy_192,1106,train,train
toy_193_0_154.jpg,toy,safe,,toy_193,1107,train,train
toy_193_0_5446.jpg,toy,safe,,toy_193,1107,train,train
toy_193_0_5600.jpg,toy,safe,,toy_193,1107,train,train
toy_193_0_5782.jpg,toy,safe,,toy_193,1107,train,train
toy_193_0_8644.jpg,toy,safe,,toy_193,1107,train,train
toy_194_0_1359.jpg,toy,safe,,toy_194,1108,train,test
toy_194_0_1500.jpg,toy,safe,,toy_194,1108,train,train
toy_194_0_6274.jpg,toy,safe,,toy_194,1108,train,train
toy_194_0_6277.jpg,toy,safe,,toy_194,1108,train,test
toy_194_0_6808.jpg,toy,safe,,toy_194,1108,train,test
toy_195_0_1943.jpg,toy,safe,,toy_195,1109,train,train
toy_195_0_6068.jpg,toy,safe,,toy_195,1109,train,train
toy_195_0_7243.jpg,toy,safe,,toy_195,1109,train,train
toy_195_0_7566.jpg,toy,safe,,toy_195,1109,train,train
toy_195_0_9644.jpg,toy,safe,,toy_195,1109,train,train
toy_196_0_2688.jpg,toy,safe,,toy_196,1110,train,train
toy_196_0_2725.jpg,toy,safe,,toy_196,1110,train,train
toy_196_0_3295.jpg,toy,safe,,toy_196,1110,train,train
toy_196_0_5097.jpg,toy,safe,,toy_196,1110,train,train
toy_196_0_6116.jpg,toy,safe,,toy_196,1110,train,train
toy_197_0_1641.jpg,toy,safe,,toy_197,1111,val,train
toy_197_0_1890.jpg,toy,safe,,toy_197,1111,val,train
toy_197_0_3477.jpg,toy,safe,,toy_197,1111,val,val
toy_197_0_8395.jpg,toy,safe,,toy_197,1111,val,train
toy_197_0_9888.jpg,toy,safe,,toy_197,1111,val,val
toy_198_0_1591.jpg,toy,safe,,toy_198,1112,test,train
toy_198_0_2538.jpg,toy,safe,,toy_198,1112,test,train
toy_198_0_4769.jpg,toy,safe,,toy_198,1112,test,train
toy_198_0_5433.jpg,toy,safe,,toy_198,1112,test,train
toy_198_0_5880.jpg,toy,safe,,toy_198,1112,test,test
toy_199_0_1106.jpg,toy,safe,,toy_199,1113,train,test
toy_199_0_3004.jpg,toy,safe,,toy_199,1113,train,train
toy_199_0_4599.jpg,toy,safe,,toy_199,1113,train,val
toy_199_0_6412.jpg,toy,safe,,toy_199,1113,train,train
toy_199_0_9518.jpg,toy,safe,,toy_199,1113,train,train
toy_19_0_1902.jpg,toy,safe,,toy_19,1114,train,test
toy_19_0_2134.jpg,toy,safe,,toy_19,1114,train,train
toy_19_0_772.jpg,toy,safe,,toy_19,1114,train,train
toy_19_0_8746.jpg,toy,safe,,toy_19,1114,train,train
toy_19_0_938.jpg,toy,safe,,toy_19,1114,train,val
toy_1_0_1374.jpg,toy,safe,,toy_1,1115,train,val
toy_1_0_4246.jpg,toy,safe,,toy_1,1115,train,train
toy_1_0_4808.jpg,toy,safe,,toy_1,1115,train,train
toy_1_0_4908.jpg,toy,safe,,toy_1,1115,train,train
toy_1_0_8119.jpg,toy,safe,,toy_1,1115,train,test
toy_200_0_1516.jpg,toy,safe,,toy_200,1116,train,val
toy_200_0_2420.jpg,toy,safe,,toy_200,1116,train,train
toy_200_0_3505.jpg,toy,safe,,toy_200,1116,train,val
toy_200_0_5158.jpg,toy,safe,,toy_200,1116,train,test
toy_200_0_8923.jpg,toy,safe,,toy_200,1116,train,test
toy_201_0_2324.jpg,toy,safe,,toy_201,1117,test,train
toy_201_0_5122.jpg,toy,safe,,toy_201,1117,test,val
toy_201_0_8328.jpg,toy,safe,,toy_201,1117,test,train
toy_201_0_8407.jpg,toy,safe,,toy_201,1117,test,test
toy_201_0_9657.jpg,toy,safe,,toy_201,1117,test,val
toy_202_0_1449.jpg,toy,safe,,toy_202,1118,train,train
toy_202_0_4056.jpg,toy,safe,,toy_202,1118,train,train
toy_202_0_5787.jpg,toy,safe,,toy_202,1118,train,train
toy_202_0_8310.jpg,toy,safe,,toy_202,1118,train,train
toy_202_0_9164.jpg,toy,safe,,toy_202,1118,train,train
toy_203_0_1478.jpg,toy,safe,,toy_203,1119,train,train
toy_203_0_4281.jpg,toy,safe,,toy_203,1119,train,train
toy_203_0_6192.jpg,toy,safe,,toy_203,1119,train,train
toy_203_0_6762.jpg,toy,safe,,toy_203,1119,train,train
toy_203_0_6911.jpg,toy,safe,,toy_203,1119,train,train
toy_204_0_256.jpg,toy,safe,,toy_204,1120,train,train
toy_204_0_3650.jpg,toy,safe,,toy_204,1120,train,val
toy_204_0_5721.jpg,toy,safe,,toy_204,1120,train,val
toy_204_0_6011.jpg,toy,safe,,toy_204,1120,train,train
toy_204_0_991.jpg,toy,safe,,toy_204,1120,train,test
toy_205_0_1532.jpg,toy,safe,,toy_205,1121,val,val
toy_205_0_3822.jpg,toy,safe,,toy_205,1121,val,train
toy_205_0_6006.jpg,toy,safe,,toy_205,1121,val,train
toy_205_0_6551.jpg,toy,safe,,toy_205,1121,val,val
toy_205_0_9840.jpg,toy,safe,,toy_205,1121,val,train
toy_206_0_2667.jpg,toy,safe,,toy_206,1122,train,train
toy_206_0_5307.jpg,toy,safe,,toy_206,1122,train,train
toy_206_0_5478.jpg,toy,safe,,toy_206,1122,train,train
toy_206_0_6627.jpg,toy,safe,,toy_206,1122,train,train
toy_206_0_7459.jpg,toy,safe,,toy_206,1122,train,train
toy_207_0_1728.jpg,toy,safe,,toy_207,1123,train,train
toy_207_0_5156.jpg,toy,safe,,toy_207,1123,train,train
toy_207_0_7429.jpg,toy,safe,,toy_207,1123,train,test
toy_207_0_82.jpg,toy,safe,,toy_207,1123,train,val
toy_207_0_8816.jpg,toy,safe,,toy_207,1123,train,train
toy_208_0_2578.jpg,toy,safe,,toy_208,1124,train,train
toy_208_0_4409.jpg,toy,safe,,toy_208,1124,train,val
toy_208_0_45.jpg,toy,safe,,toy_208,1124,train,test
toy_208_0_4971.jpg,toy,safe,,toy_208,1124,train,train
toy_208_0_892.jpg,toy,safe,,toy_208,1124,train,train
toy_209_0_3070.jpg,toy,safe,,toy_209,1125,train,test
toy_209_0_4518.jpg,toy,safe,,toy_209,1125,train,train
toy_209_0_7431.jpg,toy,safe,,toy_209,1125,train,val
toy_209_0_8027.jpg,toy,safe,,toy_209,1125,train,train
toy_209_0_9350.jpg,toy,safe,,toy_209,1125,train,test
toy_20_0_3949.jpg,toy,safe,,toy_20,1126,train,train
toy_20_0_4983.jpg,toy,safe,,toy_20,1126,train,val
toy_20_0_7788.jpg,toy,safe,,toy_20,1126,train,test
toy_20_0_8464.jpg,toy,safe,,toy_20,1126,train,train
toy_20_0_8933.jpg,toy,safe,,toy_20,1126,train,test
toy_210_0_1915.jpg,toy,safe,,toy_210,1127,train,train
toy_210_0_2740.jpg,toy,safe,,toy_210,1127,train,train
toy_210_0_2892.jpg,toy,safe,,toy_210,1127,train,test
toy_210_0_4376.jpg,toy,safe,,toy_210,1127,train,train
toy_210_0_5620.jpg,toy,safe,,toy_210,1127,train,train
toy_211_0_1938.jpg,toy,safe,,toy_211,1128,train,test
toy_211_0_2379.jpg,toy,safe,,toy_211,1128,train,train
toy_211_0_5631.jpg,toy,safe,,toy_211,1128,train,train
toy_211_0_7252.jpg,toy,safe,,toy_211,1128,train,test
toy_211_0_9692.jpg,toy,safe,,toy_211,1128,train,test
toy_212_0_1640.jpg,toy,safe,,toy_212,1129,train,train
toy_212_0_4936.jpg,toy,safe,,toy_212,1129,train,train
toy_212_0_5490.jpg,toy,safe,,toy_212,1129,train,test
toy_212_0_5678.jpg,toy,safe,,toy_212,1129,train,train
toy_212_0_7422.jpg,toy,safe,,toy_212,1129,train,train
toy_213_0_2315.jpg,toy,safe,,toy_213,1130,train,val
toy_213_0_2466.jpg,toy,safe,,toy_213,1130,train,val
toy_213_0_445.jpg,toy,safe,,toy_213,1130,train,train
toy_213_0_8877.jpg,toy,safe,,toy_213,1130,train,test
toy_213_0_9635.jpg,toy,safe,,toy_213,1130,train,train
toy_214_0_1851.jpg,toy,safe,,toy_214,1131,train,test
toy_214_0_1909.jpg,toy,safe,,toy_214,1131,train,train
toy_214_0_2791.jpg,toy,safe,,toy_214,1131,train,train
toy_214_0_4480.jpg,toy,safe,,toy_214,1131,train,train
toy_214_0_7074.jpg,toy,safe,,toy_214,1131,train,val
toy_215_0_3828.jpg,toy,safe,,toy_215,1132,val,train
toy_215_0_4242.jpg,toy,safe,,toy_215,1132,val,train
toy_215_0_703.jpg,toy,safe,,toy_215,1132,val,train
toy_215_0_8236.jpg,toy,safe,,toy_215,1132,val,train
toy_215_0_8934.jpg,toy,safe,,toy_215,1132,val,val
toy_216_0_2120.jpg,toy,safe,,toy_216,1133,test,val
toy_216_0_2525.jpg,toy,safe,,toy_216,1133,test,val
toy_216_0_5576.jpg,toy,safe,,toy_216,1133,test,train
toy_216_0_7994.jpg,toy,safe,,toy_216,1133,test,test
toy_216_0_8191.jpg,toy,safe,,toy_216,1133,test,val
toy_217_0_1567.jpg,toy,safe,,toy_217,1134,test,train
toy_217_0_1880.jpg,toy,safe,,toy_217,1134,test,train
toy_217_0_2512.jpg,toy,safe,,toy_217,1134,test,train
toy_217_0_3827.jpg,toy,safe,,toy_217,1134,test,train
toy_217_0_6080.jpg,toy,safe,,toy_217,1134,test,train
toy_218_0_1647.jpg,toy,safe,,toy_218,1135,train,test
toy_218_0_1801.jpg,toy,safe,,toy_218,1135,train,train
toy_218_0_6733.jpg,toy,safe,,toy_218,1135,train,train
toy_218_0_8675.jpg,toy,safe,,toy_218,1135,train,val
toy_218_0_9951.jpg,toy,safe,,toy_218,1135,train,train
toy_219_0_1037.jpg,toy,safe,,toy_219,1136,val,train
toy_219_0_3078.jpg,toy,safe,,toy_219,1136,val,val
toy_219_0_7459.jpg,toy,safe,,toy_219,1136,val,train
toy_219_0_9253.jpg,toy,safe,,toy_219,1136,val,val
toy_219_0_9261.jpg,toy,safe,,toy_219,1136,val,train
toy_21_0_1803.jpg,toy,safe,,toy_21,1137,test,val
toy_21_0_2506.jpg,toy,safe,,toy_21,1137,test,train
toy_21_0_4007.jpg,toy,safe,,toy_21,1137,test,train
toy_21_0_8656.jpg,toy,safe,,toy_21,1137,test,train
toy_21_0_9539.jpg,toy,safe,,toy_21,1137,test,test
toy_220_0_2871.jpg,toy,safe,,toy_220,1138,test,train
toy_220_0_4050.jpg,toy,safe,,toy_220,1138,test,test
toy_220_0_830.jpg,toy,safe,,toy_220,1138,test,train
toy_220_0_8314.jpg,toy,safe,,toy_220,1138,test,train
toy_220_0_909.jpg,toy,safe,,toy_220,1138,test,train
toy_221_0_3159.jpg,toy,safe,,toy_221,1139,val,train
toy_221_0_340.jpg,toy,safe,,toy_221,1139,val,train
toy_221_0_3607.jpg,toy,safe,,toy_221,1139,val,train
toy_221_0_4871.jpg,toy,safe,,toy_221,1139,val,val
toy_221_0_8241.jpg,toy,safe,,toy_221,1139,val,train
toy_222_0_170.jpg,toy,safe,,toy_222,1140,train,val
toy_222_0_3172.jpg,toy,safe,,toy_222,1140,train,train
toy_222_0_4588.jpg,toy,safe,,toy_222,1140,train,val
toy_222_0_6596.jpg,toy,safe,,toy_222,1140,train,train
toy_222_0_777.jpg,toy,safe,,toy_222,1140,train,train
toy_223_0_1159.jpg,toy,safe,,toy_223,1141,test,val
toy_223_0_2851.jpg,toy,safe,,toy_223,1141,test,train
toy_223_0_2853.jpg,toy,safe,,toy_223,1141,test,val
toy_223_0_515.jpg,toy,safe,,toy_223,1141,test,train
toy_223_0_8295.jpg,toy,safe,,toy_223,1141,test,test
toy_224_0_3498.jpg,toy,safe,,toy_224,1142,train,test
toy_224_0_6551.jpg,toy,safe,,toy_224,1142,train,train
toy_224_0_6877.jpg,toy,safe,,toy_224,1142,train,train
toy_224_0_760.jpg,toy,safe,,toy_224,1142,train,val
toy_224_0_9700.jpg,toy,safe,,toy_224,1142,train,train
toy_225_0_1943.jpg,toy,safe,,toy_225,1143,train,val
toy_225_0_362.jpg,toy,safe,,toy_225,1143,train,train
toy_225_0_516.jpg,toy,safe,,toy_225,1143,train,test
toy_225_0_5373.jpg,toy,safe,,toy_225,1143,train,train
toy_225_0_9448.jpg,toy,safe,,toy_225,1143,train,train
toy_226_0_4880.jpg,toy,safe,,toy_226,1144,val,test
toy_226_0_501.jpg,toy,safe,,toy_226,1144,val,train
toy_226_0_6700.jpg,toy,safe,,toy_226,1144,val,test
toy_226_0_7088.jpg,toy,safe,,toy_226,1144,val,train
toy_226_0_9660.jpg,toy,safe,,toy_226,1144,val,train
toy_227_0_1782.jpg,toy,safe,,toy_227,1145,train,val
toy_227_0_1963.jpg,toy,safe,,toy_227,1145,train,val
toy_227_0_2433.jpg,toy,safe,,toy_227,1145,train,train
toy_227_0_2910.jpg,toy,safe,,toy_227,1145,train,val
toy_227_0_8847.jpg,toy,safe,,toy_227,1145,train,val
toy_228_0_4477.jpg,toy,safe,,toy_228,1146,test,val
toy_228_0_4712.jpg,toy,safe,,toy_228,1146,test,train
toy_228_0_5062.jpg,toy,safe,,toy_228,1146,test,train
toy_228_0_5968.jpg,toy,safe,,toy_228,1146,test,test
toy_228_0_7802.jpg,toy,safe,,toy_228,1146,test,val
toy_229_0_3875.jpg,toy,safe,,toy_229,1147,val,train
toy_229_0_5760.jpg,toy,safe,,toy_229,1147,val,train
toy_229_0_8148.jpg,toy,safe,,toy_229,1147,val,train
toy_229_0_8167.jpg,toy,safe,,toy_229,1147,val,test
toy_229_0_9541.jpg,toy,safe,,toy_229,1147,val,train
toy_22_0_1002.jpg,toy,safe,,toy_22,1148,train,val
toy_22_0_1048.jpg,toy,safe,,toy_22,1148,train,train
toy_22_0_5282.jpg,toy,safe,,toy_22,1148,train,test
toy_22_0_6744.jpg,toy,safe,,toy_22,1148,train,train
toy_22_0_9384.jpg,toy,safe,,toy_22,1148,train,test
toy_230_0_2279.jpg,toy,safe,,toy_230,1149,train,train
toy_230_0_3169.jpg,toy,safe,,toy_230,1149,train,train
toy_230_0_4079.jpg,toy,safe,,toy_230,1149,train,train
toy_230_0_6303.jpg,toy,safe,,toy_230,1149,train,train
toy_230_0_7218.jpg,toy,safe,,toy_230,1149,train,train
toy_231_0_1100.jpg,toy,safe,,toy_231,1150,train,train
toy_231_0_2545.jpg,toy,safe,,toy_231,1150,train,train
toy_231_0_4354.jpg,toy,safe,,toy_231,1150,train,train
toy_231_0_4410.jpg,toy,safe,,toy_231,1150,train,test
toy_231_0_9603.jpg,toy,safe,,toy_231,1150,train,train
toy_232_0_3858.jpg,toy,safe,,toy_232,1151,train,train
toy_232_0_4698.jpg,toy,safe,,toy_232,1151,train,train
toy_232_0_5422.jpg,toy,safe,,toy_232,1151,train,test
toy_232_0_7422.jpg,toy,safe,,toy_232,1151,train,train
toy_232_0_8645.jpg,toy,safe,,toy_232,1151,train,train
toy_233_0_2303.jpg,toy,safe,,toy_233,1152,train,test
toy_233_0_2526.jpg,toy,safe,,toy_233,1152,train,val
toy_233_0_2582.jpg,toy,safe,,toy_233,1152,train,train
toy_233_0_34.jpg,toy,safe,,toy_233,1152,train,val
toy_233_0_7262.jpg,toy,safe,,toy_233,1152,train,test
toy_234_0_2180.jpg,toy,safe,,toy_234,1153,train,train
toy_234_0_6371.jpg,toy,safe,,toy_234,1153,train,train
toy_234_0_7309.jpg,toy,safe,,toy_234,1153,train,train
toy_234_0_7989.jpg,toy,safe,,toy_234,1153,train,train
toy_234_0_9662.jpg,toy,safe,,toy_234,1153,train,test
toy_235_0_3113.jpg,toy,safe,,toy_235,1154,test,val
toy_235_0_547.jpg,toy,safe,,toy_235,1154,test,train
toy_235_0_7274.jpg,toy,safe,,toy_235,1154,test,train
toy_235_0_9109.jpg,toy,safe,,toy_235,1154,test,train
toy_235_0_9640.jpg,toy,safe,,toy_235,1154,test,train
toy_236_0_1020.jpg,toy,safe,,toy_236,1155,train,train
toy_236_0_5829.jpg,toy,safe,,toy_236,1155,train,train
toy_236_0_6449.jpg,toy,safe,,toy_236,1155,train,val
toy_236_0_8155.jpg,toy,safe,,toy_236,1155,train,test
toy_236_0_9099.jpg,toy,safe,,toy_236,1155,train,train
toy_237_0_4049.jpg,toy,safe,,toy_237,1156,train,test
toy_237_0_4082.jpg,toy,safe,,toy_237,1156,train,train
toy_237_0_4360.jpg,toy,safe,,toy_237,1156,train,train
toy_237_0_4550.jpg,toy,safe,,toy_237,1156,train,train
toy_237_0_5230.jpg,toy,safe,,toy_237,1156,train,train
toy_238_0_2001.jpg,toy,safe,,toy_238,1157,train,train
toy_238_0_2195.jpg,toy,safe,,toy_238,1157,train,train
toy_238_0_5407.jpg,toy,safe,,toy_238,1157,train,train
toy_238_0_6939.jpg,toy,safe,,toy_238,1157,train,train
toy_238_0_8626.jpg,toy,safe,,toy_238,1157,train,train
toy_239_0_3059.jpg,toy,safe,,toy_239,1158,train,train
toy_239_0_3289.jpg,toy,safe,,toy_239,1158,train,train
toy_239_0_3560.jpg,toy,safe,,toy_239,1158,train,train
toy_239_0_6970.jpg,toy,safe,,toy_239,1158,train,train
toy_239_0_844.jpg,toy,safe,,toy_239,1158,train,train
toy_23_0_2255.jpg,toy,safe,,toy_23,1159,train,train
toy_23_0_2344.jpg,toy,safe,,toy_23,1159,train,val
toy_23_0_3224.jpg,toy,safe,,toy_23,1159,train,train
toy_23_0_6115.jpg,toy,safe,,toy_23,1159,train,train
toy_23_0_8728.jpg,toy,safe,,toy_23,1159,train,train
toy_240_0_117.jpg,toy,safe,,toy_240,1160,train,test
toy_240_0_1804.jpg,toy,safe,,toy_240,1160,train,train
toy_240_0_3964.jpg,toy,safe,,toy_240,1160,train,train
toy_240_0_519.jpg,toy,safe,,toy_240,1160,train,train
toy_240_0_6137.jpg,toy,safe,,toy_240,1160,train,train
toy_241_0_1123.jpg,toy,safe,,toy_241,1161,test,train
toy_241_0_2117.jpg,toy,safe,,toy_241,1161,test,val
toy_241_0_3191.jpg,toy,safe,,toy_241,1161,test,train
toy_241_0_4067.jpg,toy,safe,,toy_241,1161,test,test
toy_241_0_667.jpg,toy,safe,,toy_241,1161,test,train
toy_242_0_285.jpg,toy,safe,,toy_242,1162,val,train
toy_242_0_5544.jpg,toy,safe,,toy_242,1162,val,train
toy_242_0_7116.jpg,toy,safe,,toy_242,1162,val,val
toy_242_0_8464.jpg,toy,safe,,toy_242,1162,val,val
toy_242_0_9530.jpg,toy,safe,,toy_242,1162,val,test
toy_243_0_1018.jpg,toy,safe,,toy_243,1163,val,train
toy_243_0_5115.jpg,toy,safe,,toy_243,1163,val,train
toy_243_0_5617.jpg,toy,safe,,toy_243,1163,val,val
toy_243_0_6092.jpg,toy,safe,,toy_243,1163,val,test
toy_243_0_8308.jpg,toy,safe,,toy_243,1163,val,train
toy_244_0_1479.jpg,toy,safe,,toy_244,1164,train,train
toy_244_0_2500.jpg,toy,safe,,toy_244,1164,train,test
toy_244_0_3317.jpg,toy,safe,,toy_244,1164,train,train
toy_244_0_8554.jpg,toy,safe,,toy_244,1164,train,train
toy_244_0_9002.jpg,toy,safe,,toy_244,1164,train,val
toy_245_0_3791.jpg,toy,safe,,toy_245,1165,train,train
toy_245_0_6741.jpg,toy,safe,,toy_245,1165,train,train
toy_245_0_9193.jpg,toy,safe,,toy_245,1165,train,train
toy_245_0_9527.jpg,toy,safe,,toy_245,1165,train,train
toy_245_0_9679.jpg,toy,safe,,toy_245,1165,train,val
toy_246_0_1281.jpg,toy,safe,,toy_246,1166,val,train
toy_246_0_1584.jpg,toy,safe,,toy_246,1166,val,train
toy_246_0_5520.jpg,toy,safe,,toy_246,1166,val,val
toy_246_0_563.jpg,toy,safe,,toy_246,1166,val,train
toy_246_0_6457.jpg,toy,safe,,toy_246,1166,val,train
toy_247_0_3579.jpg,toy,safe,,toy_247,1167,train,train
toy_247_0_5712.jpg,toy,safe,,toy_247,1167,train,train
toy_247_0_6825.jpg,toy,safe,,toy_247,1167,train,train
toy_247_0_7267.jpg,toy,safe,,toy_247,1167,train,train
toy_247_0_9747.jpg,toy,safe,,toy_247,1167,train,train
toy_248_0_3383.jpg,toy,safe,,toy_248,1168,train,train
toy_248_0_5981.jpg,toy,safe,,toy_248,1168,train,val
toy_248_0_7751.jpg,toy,safe,,toy_248,1168,train,train
toy_248_0_8775.jpg,toy,safe,,toy_248,1168,train,train
toy_248_0_9063.jpg,toy,safe,,toy_248,1168,train,train
toy_249_0_3533.jpg,toy,safe,,toy_249,1169,val,train
toy_249_0_4791.jpg,toy,safe,,toy_249,1169,val,train
toy_249_0_5420.jpg,toy,safe,,toy_249,1169,val,train
toy_249_0_5855.jpg,toy,safe,,toy_249,1169,val,test
toy_249_0_9022.jpg,toy,safe,,toy_249,1169,val,train
toy_24_0_2145.jpg,toy,safe,,toy_24,1170,val,train
toy_24_0_2964.jpg,toy,safe,,toy_24,1170,val,train
toy_24_0_5417.jpg,toy,safe,,toy_24,1170,val,train
toy_24_0_7947.jpg,toy,safe,,toy_24,1170,val,val
toy_24_0_8559.jpg,toy,safe,,toy_24,1170,val,train
toy_250_0_2009.jpg,toy,safe,,toy_250,1171,train,val
toy_250_0_3639.jpg,toy,safe,,toy_250,1171,train,val
toy_250_0_4147.jpg,toy,safe,,toy_250,1171,train,test
toy_250_0_9206.jpg,toy,safe,,toy_250,1171,train,val
toy_250_0_9449.jpg,toy,safe,,toy_250,1171,train,train
toy_251_0_1112.jpg,toy,safe,,toy_251,1172,train,train
toy_251_0_4490.jpg,toy,safe,,toy_251,1172,train,train
toy_251_0_5039.jpg,toy,safe,,toy_251,1172,train,train
toy_251_0_7982.jpg,toy,safe,,toy_251,1172,train,train
toy_251_0_8484.jpg,toy,safe,,toy_251,1172,train,val
toy_252_0_7989.jpg,toy,safe,,toy_252,1173,train,train
toy_252_0_8321.jpg,toy,safe,,toy_252,1173,train,train
toy_252_0_8763.jpg,toy,safe,,toy_252,1173,train,train
toy_252_0_9325.jpg,toy,safe,,toy_252,1173,train,val
toy_252_0_9847.jpg,toy,safe,,toy_252,1173,train,train
toy_253_0_3712.jpg,toy,safe,,toy_253,1174,train,train
toy_253_0_5641.jpg,toy,safe,,toy_253,1174,train,train
toy_253_0_5709.jpg,toy,safe,,toy_253,1174,train,test
toy_253_0_8573.jpg,toy,safe,,toy_253,1174,train,train
toy_253_0_9921.jpg,toy,safe,,toy_253,1174,train,train
toy_254_0_1285.jpg,toy,safe,,toy_254,1175,train,train
toy_254_0_1816.jpg,toy,safe,,toy_254,1175,train,train
toy_254_0_2619.jpg,toy,safe,,toy_254,1175,train,train
toy_254_0_3479.jpg,toy,safe,,toy_254,1175,train,train
toy_254_0_463.jpg,toy,safe,,toy_254,1175,train,train
toy_255_0_3338.jpg,toy,safe,,toy_255,1176,train,train
toy_255_0_350.jpg,toy,safe,,toy_255,1176,train,train
toy_255_0_4437.jpg,toy,safe,,toy_255,1176,train,test
toy_255_0_6142.jpg,toy,safe,,toy_255,1176,train,train
toy_255_0_7055.jpg,toy,safe,,toy_255,1176,train,val
toy_256_0_2321.jpg,toy,safe,,toy_256,1177,train,test
toy_256_0_3155.jpg,toy,safe,,toy_256,1177,train,train
toy_256_0_5726.jpg,toy,safe,,toy_256,1177,train,test
toy_256_0_9432.jpg,toy,safe,,toy_256,1177,train,train
toy_256_0_9924.jpg,toy,safe,,toy_256,1177,train,val
toy_257_0_1544.jpg,toy,safe,,toy_257,1178,train,train
toy_257_0_2400.jpg,toy,safe,,toy_257,1178,train,train
toy_257_0_4185.jpg,toy,safe,,toy_257,1178,train,train
toy_257_0_710.jpg,toy,safe,,toy_257,1178,train,train
toy_257_0_7160.jpg,toy,safe,,toy_257,1178,train,test
toy_258_0_2886.jpg,toy,safe,,toy_258,1179,train,train
toy_258_0_4848.jpg,toy,safe,,toy_258,1179,train,test
toy_258_0_8946.jpg,toy,safe,,toy_258,1179,train,val
toy_258_0_9061.jpg,toy,safe,,toy_258,1179,train,train
toy_258_0_9422.jpg,toy,safe,,toy_258,1179,train,val
toy_259_0_0.jpg,toy,safe,,toy_259,1180,train,val
toy_259_0_1804.jpg,toy,safe,,toy_259,1180,train,train
toy_259_0_6776.jpg,toy,safe,,toy_259,1180,train,train
toy_259_0_7307.jpg,toy,safe,,toy_259,1180,train,train
toy_259_0_8628.jpg,toy,safe,,toy_259,1180,train,train
toy_25_0_1611.jpg,toy,safe,,toy_25,1181,train,train
toy_25_0_3438.jpg,toy,safe,,toy_25,1181,train,val
toy_25_0_6815.jpg,toy,safe,,toy_25,1181,train,test
toy_25_0_7225.jpg,toy,safe,,toy_25,1181,train,train
toy_25_0_8448.jpg,toy,safe,,toy_25,1181,train,train
toy_260_0_1843.jpg,toy,safe,,toy_260,1182,val,test
toy_260_0_3875.jpg,toy,safe,,toy_260,1182,val,val
toy_260_0_5378.jpg,toy,safe,,toy_260,1182,val,train
toy_260_0_8669.jpg,toy,safe,,toy_260,1182,val,test
toy_260_0_9122.jpg,toy,safe,,toy_260,1182,val,val
toy_261_0_1997.jpg,toy,safe,,toy_261,1183,train,train
toy_261_0_3911.jpg,toy,safe,,toy_261,1183,train,train
toy_261_0_6215.jpg,toy,safe,,toy_261,1183,train,train
toy_261_0_713.jpg,toy,safe,,toy_261,1183,train,val
toy_261_0_9196.jpg,toy,safe,,toy_261,1183,train,train
toy_265_0_1430.jpg,toy,safe,,toy_265,1184,train,train
toy_265_0_1637.jpg,toy,safe,,toy_265,1184,train,train
toy_265_0_5230.jpg,toy,safe,,toy_265,1184,train,val
toy_265_0_7233.jpg,toy,safe,,toy_265,1184,train,train
toy_265_0_8888.jpg,toy,safe,,toy_265,1184,train,train
toy_266_0_2287.jpg,toy,safe,,toy_266,1185,train,train
toy_266_0_2299.jpg,toy,safe,,toy_266,1185,train,train
toy_266_0_2390.jpg,toy,safe,,toy_266,1185,train,train
toy_266_0_6753.jpg,toy,safe,,toy_266,1185,train,train
toy_266_0_9222.jpg,toy,safe,,toy_266,1185,train,train
toy_267_0_1691.jpg,toy,safe,,toy_267,1186,train,val
toy_267_0_2066.jpg,toy,safe,,toy_267,1186,train,train
toy_267_0_4045.jpg,toy,safe,,toy_267,1186,train,train
toy_267_0_7037.jpg,toy,safe,,toy_267,1186,train,train
toy_267_0_9164.jpg,toy,safe,,toy_267,1186,train,train
toy_268_0_2402.jpg,toy,safe,,toy_268,1187,train,train
toy_268_0_3139.jpg,toy,safe,,toy_268,1187,train,val
toy_268_0_3625.jpg,toy,safe,,toy_268,1187,train,train
toy_268_0_599.jpg,toy,safe,,toy_268,1187,train,train
toy_268_0_8033.jpg,toy,safe,,toy_268,1187,train,train
toy_269_0_1312.jpg,toy,safe,,toy_269,1188,train,train
toy_269_0_1879.jpg,toy,safe,,toy_269,1188,train,train
toy_269_0_4016.jpg,toy,safe,,toy_269,1188,train,train
toy_269_0_5255.jpg,toy,safe,,toy_269,1188,train,train
toy_269_0_5705.jpg,toy,safe,,toy_269,1188,train,train
toy_26_0_5637.jpg,toy,safe,,toy_26,1189,train,train
toy_26_0_6185.jpg,toy,safe,,toy_26,1189,train,train
toy_26_0_777.jpg,toy,safe,,toy_26,1189,train,train
toy_26_0_7826.jpg,toy,safe,,toy_26,1189,train,val
toy_26_0_9990.jpg,toy,safe,,toy_26,1189,train,train
toy_270_0_1037.jpg,toy,safe,,toy_270,1190,train,train
toy_270_0_1102.jpg,toy,safe,,toy_270,1190,train,train
toy_270_0_2411.jpg,toy,safe,,toy_270,1190,train,train
toy_270_0_3105.jpg,toy,safe,,toy_270,1190,train,val
toy_270_0_8012.jpg,toy,safe,,toy_270,1190,train,test
toy_271_0_1527.jpg,toy,safe,,toy_271,1191,train,train
toy_271_0_3868.jpg,toy,safe,,toy_271,1191,train,train
toy_271_0_476.jpg,toy,safe,,toy_271,1191,train,train
toy_271_0_5936.jpg,toy,safe,,toy_271,1191,train,train
toy_271_0_6128.jpg,toy,safe,,toy_271,1191,train,val
toy_272_0_2950.jpg,toy,safe,,toy_272,1192,val,train
toy_272_0_4880.jpg,toy,safe,,toy_272,1192,val,train
toy_272_0_5469.jpg,toy,safe,,toy_272,1192,val,train
toy_272_0_8702.jpg,toy,safe,,toy_272,1192,val,val
toy_272_0_9657.jpg,toy,safe,,toy_272,1192,val,train
toy_273_0_1842.jpg,toy,safe,,toy_273,1193,train,train
toy_273_0_239.jpg,toy,safe,,toy_273,1193,train,train
toy_273_0_2490.jpg,toy,safe,,toy_273,1193,train,train
toy_273_0_4298.jpg,toy,safe,,toy_273,1193,train,test
toy_273_0_9550.jpg,toy,safe,,toy_273,1193,train,train
toy_274_0_2536.jpg,toy,safe,,toy_274,1194,train,test
toy_274_0_3397.jpg,toy,safe,,toy_274,1194,train,train
toy_274_0_5045.jpg,toy,safe,,toy_274,1194,train,train
toy_274_0_5956.jpg,toy,safe,,toy_274,1194,train,train
toy_274_0_8747.jpg,toy,safe,,toy_274,1194,train,train
toy_275_0_1637.jpg,toy,safe,,toy_275,1195,test,train
toy_275_0_2362.jpg,toy,safe,,toy_275,1195,test,val
toy_275_0_589.jpg,toy,safe,,toy_275,1195,test,train
toy_275_0_6109.jpg,toy,safe,,toy_275,1195,test,train
toy_275_0_8005.jpg,toy,safe,,toy_275,1195,test,val
toy_276_0_435.jpg,toy,safe,,toy_276,1196,train,train
toy_276_0_5352.jpg,toy,safe,,toy_276,1196,train,val
toy_276_0_5757.jpg,toy,safe,,toy_276,1196,train,train
toy_276_0_5956.jpg,toy,safe,,toy_276,1196,train,train
toy_276_0_8234.jpg,toy,safe,,toy_276,1196,train,train
toy_277_0_1087.jpg,toy,safe,,toy_277,1197,val,test
toy_277_0_1145.jpg,toy,safe,,toy_277,1197,val,train
toy_277_0_7697.jpg,toy,safe,,toy_277,1197,val,train
toy_277_0_7923.jpg,toy,safe,,toy_277,1197,val,train
toy_277_0_8976.jpg,toy,safe,,toy_277,1197,val,val
toy_278_0_1094.jpg,toy,safe,,toy_278,1198,train,train
toy_278_0_5357.jpg,toy,safe,,toy_278,1198,train,train
toy_278_0_5536.jpg,toy,safe,,toy_278,1198,train,train
toy_278_0_5888.jpg,toy,safe,,toy_278,1198,train,train
toy_278_0_9236.jpg,toy,safe,,toy_278,1198,train,train
toy_279_0_4110.jpg,toy,safe,,toy_279,1199,test,train
toy_279_0_6865.jpg,toy,safe,,toy_279,1199,test,train
toy_279_0_7938.jpg,toy,safe,,toy_279,1199,test,test
toy_279_0_798.jpg,toy,safe,,toy_279,1199,test,train
toy_279_0_8801.jpg,toy,safe,,toy_279,1199,test,train
toy_27_0_1138.jpg,toy,safe,,toy_27,1200,train,train
toy_27_0_2101.jpg,toy,safe,,toy_27,1200,train,val
toy_27_0_327.jpg,toy,safe,,toy_27,1200,train,train
toy_27_0_5497.jpg,toy,safe,,toy_27,1200,train,train
toy_27_0_9189.jpg,toy,safe,,toy_27,1200,train,test
toy_280_0_1043.jpg,toy,safe,,toy_280,1201,train,train
toy_280_0_3760.jpg,toy,safe,,toy_280,1201,train,train
toy_280_0_5387.jpg,toy,safe,,toy_280,1201,train,test
toy_280_0_550.jpg,toy,safe,,toy_280,1201,train,train
toy_280_0_802.jpg,toy,safe,,toy_280,1201,train,train
toy_281_0_3375.jpg,toy,safe,,toy_281,1202,train,train
toy_281_0_4195.jpg,toy,safe,,toy_281,1202,train,train
toy_281_0_5346.jpg,toy,safe,,toy_281,1202,train,train
toy_281_0_8574.jpg,toy,safe,,toy_281,1202,train,train
toy_281_0_9952.jpg,toy,safe,,toy_281,1202,train,test
toy_282_0_3377.jpg,toy,safe,,toy_282,1203,test,train
toy_282_0_3587.jpg,toy,safe,,toy_282,1203,test,train
toy_282_0_3913.jpg,toy,safe,,toy_282,1203,test,train
toy_282_0_9154.jpg,toy,safe,,toy_282,1203,test,train
toy_282_0_9253.jpg,toy,safe,,toy_282,1203,test,train
toy_283_0_1496.jpg,toy,safe,,toy_283,1204,train,train
toy_283_0_4767.jpg,toy,safe,,toy_283,1204,train,train
toy_283_0_7481.jpg,toy,safe,,toy_283,1204,train,train
toy_283_0_8477.jpg,toy,safe,,toy_283,1204,train,train
toy_283_0_8779.jpg,toy,safe,,toy_283,1204,train,train
toy_284_0_2193.jpg,toy,safe,,toy_284,1205,train,train
toy_284_0_306.jpg,toy,safe,,toy_284,1205,train,train
toy_284_0_3503.jpg,toy,safe,,toy_284,1205,train,val
toy_284_0_5019.jpg,toy,safe,,toy_284,1205,train,train
toy_284_0_6189.jpg,toy,safe,,toy_284,1205,train,train
toy_285_0_1797.jpg,toy,safe,,toy_285,1206,test,train
toy_285_0_1949.jpg,toy,safe,,toy_285,1206,test,test
toy_285_0_242.jpg,toy,safe,,toy_285,1206,test,train
toy_285_0_7427.jpg,toy,safe,,toy_285,1206,test,train
toy_285_0_8895.jpg,toy,safe,,toy_285,1206,test,train
toy_286_0_1634.jpg,toy,safe,,toy_286,1207,train,train
toy_286_0_2724.jpg,toy,safe,,toy_286,1207,train,train
toy_286_0_4500.jpg,toy,safe,,toy_286,1207,train,train
toy_286_0_7816.jpg,toy,safe,,toy_286,1207,train,val
toy_286_0_8945.jpg,toy,safe,,toy_286,1207,train,train
toy_287_0_3602.jpg,toy,safe,,toy_287,1208,train,test
toy_287_0_3722.jpg,toy,safe,,toy_287,1208,train,test
toy_287_0_4203.jpg,toy,safe,,toy_287,1208,train,train
toy_287_0_5689.jpg,toy,safe,,toy_287,1208,train,train
toy_287_0_9815.jpg,toy,safe,,toy_287,1208,train,train
toy_288_0_2049.jpg,toy,safe,,toy_288,1209,train,val
toy_288_0_2679.jpg,toy,safe,,toy_288,1209,train,train
toy_288_0_3428.jpg,toy,safe,,toy_288,1209,train,val
toy_288_0_5201.jpg,toy,safe,,toy_288,1209,train,train
toy_288_0_9078.jpg,toy,safe,,toy_288,1209,train,train
toy_289_0_2142.jpg,toy,safe,,toy_289,1210,train,train
toy_289_0_3878.jpg,toy,safe,,toy_289,1210,train,val
toy_289_0_3889.jpg,toy,safe,,toy_289,1210,train,train
toy_289_0_6369.jpg,toy,safe,,toy_289,1210,train,train
toy_289_0_7485.jpg,toy,safe,,toy_289,1210,train,val
toy_28_0_1480.jpg,toy,safe,,toy_28,1211,test,val
toy_28_0_3067.jpg,toy,safe,,toy_28,1211,test,train
toy_28_0_6148.jpg,toy,safe,,toy_28,1211,test,train
toy_28_0_686.jpg,toy,safe,,toy_28,1211,test,train
toy_28_0_704.jpg,toy,safe,,toy_28,1211,test,train
toy_290_0_3289.jpg,toy,safe,,toy_290,1212,train,train
toy_290_0_3979.jpg,toy,safe,,toy_290,1212,train,train
toy_290_0_4245.jpg,toy,safe,,toy_290,1212,train,train
toy_290_0_5895.jpg,toy,safe,,toy_290,1212,train,train
toy_290_0_6913.jpg,toy,safe,,toy_290,1212,train,train
toy_291_0_2235.jpg,toy,safe,,toy_291,1213,train,test
toy_291_0_4963.jpg,toy,safe,,toy_291,1213,train,train
toy_291_0_5963.jpg,toy,safe,,toy_291,1213,train,test
toy_291_0_7937.jpg,toy,safe,,toy_291,1213,train,train
toy_291_0_8136.jpg,toy,safe,,toy_291,1213,train,train
toy_292_0_1419.jpg,toy,safe,,toy_292,1214,train,train
toy_292_0_1628.jpg,toy,safe,,toy_292,1214,train,train
toy_292_0_27.jpg,toy,safe,,toy_292,1214,train,train
toy_292_0_4903.jpg,toy,safe,,toy_292,1214,train,test
toy_292_0_936.jpg,toy,safe,,toy_292,1214,train,train
toy_293_0_1590.jpg,toy,safe,,toy_293,1215,val,train
toy_293_0_226.jpg,toy,safe,,toy_293,1215,val,train
toy_293_0_3112.jpg,toy,safe,,toy_293,1215,val,train
toy_293_0_3868.jpg,toy,safe,,toy_293,1215,val,train
toy_293_0_7548.jpg,toy,safe,,toy_293,1215,val,train
toy_294_0_4373.jpg,toy,safe,,toy_294,1216,train,train
toy_294_0_5003.jpg,toy,safe,,toy_294,1216,train,train
toy_294_0_7897.jpg,toy,safe,,toy_294,1216,train,train
toy_294_0_8217.jpg,toy,safe,,toy_294,1216,train,train
toy_294_0_9737.jpg,toy,safe,,toy_294,1216,train,train
toy_295_0_2730.jpg,toy,safe,,toy_295,1217,train,train
toy_295_0_4719.jpg,toy,safe,,toy_295,1217,train,train
toy_295_0_5001.jpg,toy,safe,,toy_295,1217,train,test
toy_295_0_5142.jpg,toy,safe,,toy_295,1217,train,train
toy_295_0_9644.jpg,toy,safe,,toy_295,1217,train,test
toy_296_0_1091.jpg,toy,safe,,toy_296,1218,train,train
toy_296_0_2426.jpg,toy,safe,,toy_296,1218,train,train
toy_296_0_3938.jpg,toy,safe,,toy_296,1218,train,test
toy_296_0_7388.jpg,toy,safe,,toy_296,1218,train,test
toy_296_0_9371.jpg,toy,safe,,toy_296,1218,train,train
toy_297_0_2042.jpg,toy,safe,,toy_297,1219,val,train
toy_297_0_4940.jpg,toy,safe,,toy_297,1219,val,train
toy_297_0_5435.jpg,toy,safe,,toy_297,1219,val,test
toy_297_0_7703.jpg,toy,safe,,toy_297,1219,val,train
toy_297_0_9802.jpg,toy,safe,,toy_297,1219,val,val
toy_298_0_2566.jpg,toy,safe,,toy_298,1220,train,val
toy_298_0_8482.jpg,toy,safe,,toy_298,1220,train,train
toy_298_0_8630.jpg,toy,safe,,toy_298,1220,train,train
toy_298_0_957.jpg,toy,safe,,toy_298,1220,train,train
toy_298_0_9986.jpg,toy,safe,,toy_298,1220,train,train
toy_29_0_3023.jpg,toy,safe,,toy_29,1221,train,train
toy_29_0_5030.jpg,toy,safe,,toy_29,1221,train,test
toy_29_0_5678.jpg,toy,safe,,toy_29,1221,train,test
toy_29_0_7061.jpg,toy,safe,,toy_29,1221,train,train
toy_29_0_9990.jpg,toy,safe,,toy_29,1221,train,train
toy_2_0_1535.jpg,toy,safe,,toy_2,1222,train,train
toy_2_0_1585.jpg,toy,safe,,toy_2,1222,train,train
toy_2_0_4614.jpg,toy,safe,,toy_2,1222,train,train
toy_2_0_8584.jpg,toy,safe,,toy_2,1222,train,train
toy_2_0_8763.jpg,toy,safe,,toy_2,1222,train,train
toy_302_0_1022.jpg,toy,safe,,toy_302,1223,test,train
toy_302_0_3384.jpg,toy,safe,,toy_302,1223,test,test
toy_302_0_4316.jpg,toy,safe,,toy_302,1223,test,train
toy_302_0_5411.jpg,toy,safe,,toy_302,1223,test,train
toy_302_0_6255.jpg,toy,safe,,toy_302,1223,test,train
toy_303_0_2594.jpg,toy,safe,,toy_303,1224,train,train
toy_303_0_2902.jpg,toy,safe,,toy_303,1224,train,train
toy_303_0_673.jpg,toy,safe,,toy_303,1224,train,train
toy_303_0_7445.jpg,toy,safe,,toy_303,1224,train,train
toy_303_0_8432.jpg,toy,safe,,toy_303,1224,train,val
toy_304_0_2146.jpg,toy,safe,,toy_304,1225,test,train
toy_304_0_3804.jpg,toy,safe,,toy_304,1225,test,train
toy_304_0_553.jpg,toy,safe,,toy_304,1225,test,test
toy_304_0_5588.jpg,toy,safe,,toy_304,1225,test,train
toy_304_0_5975.jpg,toy,safe,,toy_304,1225,test,train
toy_305_0_1391.jpg,toy,safe,,toy_305,1226,train,test
toy_305_0_2766.jpg,toy,safe,,toy_305,1226,train,train
toy_305_0_2882.jpg,toy,safe,,toy_305,1226,train,val
toy_305_0_5190.jpg,toy,safe,,toy_305,1226,train,train
toy_305_0_6321.jpg,toy,safe,,toy_305,1226,train,val
toy_306_0_1733.jpg,toy,safe,,toy_306,1227,train,train
toy_306_0_4696.jpg,toy,safe,,toy_306,1227,train,val
toy_306_0_7204.jpg,toy,safe,,toy_306,1227,train,train
toy_306_0_7638.jpg,toy,safe,,toy_306,1227,train,test
toy_306_0_8759.jpg,toy,safe,,toy_306,1227,train,test
toy_307_0_3827.jpg,toy,safe,,toy_307,1228,val,train
toy_307_0_4504.jpg,toy,safe,,toy_307,1228,val,test
toy_307_0_5952.jpg,toy,safe,,toy_307,1228,val,train
toy_307_0_7032.jpg,toy,safe,,toy_307,1228,val,train
toy_307_0_8088.jpg,toy,safe,,toy_307,1228,val,train
toy_308_0_2794.jpg,toy,safe,,toy_308,1229,train,train
toy_308_0_2849.jpg,toy,safe,,toy_308,1229,train,train
toy_308_0_5729.jpg,toy,safe,,toy_308,1229,train,train
toy_308_0_5815.jpg,toy,safe,,toy_308,1229,train,train
toy_308_0_6636.jpg,toy,safe,,toy_308,1229,train,train
toy_309_0_425.jpg,toy,safe,,toy_309,1230,val,test
toy_309_0_4313.jpg,toy,safe,,toy_309,1230,val,val
toy_309_0_5183.jpg,toy,safe,,toy_309,1230,val,test
toy_309_0_8613.jpg,toy,safe,,toy_309,1230,val,train
toy_309_0_9231.jpg,toy,safe,,toy_309,1230,val,train
toy_30_0_3612.jpg,toy,safe,,toy_30,1231,train,val
toy_30_0_393.jpg,toy,safe,,toy_30,1231,train,val
toy_30_0_454.jpg,toy,safe,,toy_30,1231,train,train
toy_30_0_651.jpg,toy,safe,,toy_30,1231,train,train
toy_30_0_9457.jpg,toy,safe,,toy_30,1231,train,train
toy_312_0_1101.jpg,toy,safe,,toy_312,1232,val,train
toy_312_0_2474.jpg,toy,safe,,toy_312,1232,val,train
toy_312_0_5852.jpg,toy,safe,,toy_312,1232,val,train
toy_312_0_5887.jpg,toy,safe,,toy_312,1232,val,train
toy_312_0_7315.jpg,toy,safe,,toy_312,1232,val,train
toy_313_0_3868.jpg,toy,safe,,toy_313,1233,val,train
toy_313_0_5566.jpg,toy,safe,,toy_313,1233,val,train
toy_313_0_7194.jpg,toy,safe,,toy_313,1233,val,train
toy_313_0_8818.jpg,toy,safe,,toy_313,1233,val,val
toy_313_0_9732.jpg,toy,safe,,toy_313,1233,val,train
toy_315_0_4933.jpg,toy,safe,,toy_315,1234,train,test
toy_315_0_5432.jpg,toy,safe,,toy_315,1234,train,train
toy_315_0_6211.jpg,toy,safe,,toy_315,1234,train,test
toy_315_0_6398.jpg,toy,safe,,toy_315,1234,train,val
toy_315_0_8634.jpg,toy,safe,,toy_315,1234,train,train
toy_317_0_1933.jpg,toy,safe,,toy_317,1235,train,val
toy_317_0_2296.jpg,toy,safe,,toy_317,1235,train,train
toy_317_0_2756.jpg,toy,safe,,toy_317,1235,train,val
toy_317_0_2914.jpg,toy,safe,,toy_317,1235,train,train
toy_317_0_5697.jpg,toy,safe,,toy_317,1235,train,val
toy_318_0_2291.jpg,toy,safe,,toy_318,1236,train,train
toy_318_0_4203.jpg,toy,safe,,toy_318,1236,train,train
toy_318_0_9217.jpg,toy,safe,,toy_318,1236,train,test
toy_318_0_9498.jpg,toy,safe,,toy_318,1236,train,train
toy_318_0_9538.jpg,toy,safe,,toy_318,1236,train,val
toy_319_0_1996.jpg,toy,safe,,toy_319,1237,train,train
toy_319_0_40.jpg,toy,safe,,toy_319,1237,train,val
toy_319_0_5517.jpg,toy,safe,,toy_319,1237,train,test
toy_319_0_564.jpg,toy,safe,,toy_319,1237,train,train
toy_319_0_6203.jpg,toy,safe,,toy_319,1237,train,test
toy_31_0_3161.jpg,toy,safe,,toy_31,1238,train,train
toy_31_0_5217.jpg,toy,safe,,toy_31,1238,train,test
toy_31_0_5945.jpg,toy,safe,,toy_31,1238,train,train
toy_31_0_6184.jpg,toy,safe,,toy_31,1238,train,train
toy_31_0_7535.jpg,toy,safe,,toy_31,1238,train,train
toy_324_0_8343.jpg,toy,safe,,toy_324,1239,train,train
toy_324_0_9074.jpg,toy,safe,,toy_324,1239,train,train
toy_324_0_9158.jpg,toy,safe,,toy_324,1239,train,train
toy_324_0_9382.jpg,toy,safe,,toy_324,1239,train,train
toy_324_0_9444.jpg,toy,safe,,toy_324,1239,train,train
toy_325_0_3290.jpg,toy,safe,,toy_325,1240,train,train
toy_325_0_3336.jpg,toy,safe,,toy_325,1240,train,train
toy_325_0_3855.jpg,toy,safe,,toy_325,1240,train,train
toy_325_0_4700.jpg,toy,safe,,toy_325,1240,train,test
toy_325_0_7189.jpg,toy,safe,,toy_325,1240,train,test
toy_326_0_1760.jpg,toy,safe,,toy_326,1241,val,val
toy_326_0_2630.jpg,toy,safe,,toy_326,1241,val,val
toy_326_0_5089.jpg,toy,safe,,toy_326,1241,val,train
toy_326_0_5310.jpg,toy,safe,,toy_326,1241,val,train
toy_326_0_7273.jpg,toy,safe,,toy_326,1241,val,test
toy_327_0_2163.jpg,toy,safe,,toy_327,1242,train,train
toy_327_0_2763.jpg,toy,safe,,toy_327,1242,train,train
toy_327_0_4003.jpg,toy,safe,,toy_327,1242,train,train
toy_327_0_5266.jpg,toy,safe,,toy_327,1242,train,train
toy_327_0_6010.jpg,toy,safe,,toy_327,1242,train,train
toy_328_0_1423.jpg,toy,safe,,toy_328,1243,train,train
toy_328_0_3938.jpg,toy,safe,,toy_328,1243,train,val
toy_328_0_4248.jpg,toy,safe,,toy_328,1243,train,train
toy_328_0_5307.jpg,toy,safe,,toy_328,1243,train,train
toy_328_0_7227.jpg,toy,safe,,toy_328,1243,train,train
toy_329_0_2424.jpg,toy,safe,,toy_329,1244,test,train
toy_329_0_3105.jpg,toy,safe,,toy_329,1244,test,test
toy_329_0_6998.jpg,toy,safe,,toy_329,1244,test,train
toy_329_0_7855.jpg,toy,safe,,toy_329,1244,test,train
toy_329_0_7859.jpg,toy,safe,,toy_329,1244,test,train
toy_32_0_2239.jpg,toy,safe,,toy_32,1245,train,train
toy_32_0_2861.jpg,toy,safe,,toy_32,1245,train,train
toy_32_0_4046.jpg,toy,safe,,toy_32,1245,train,train
toy_32_0_6023.jpg,toy,safe,,toy_32,1245,train,train
toy_32_0_8271.jpg,toy,safe,,toy_32,1245,train,train
toy_330_0_1973.jpg,toy,safe,,toy_330,1246,train,train
toy_330_0_5981.jpg,toy,safe,,toy_330,1246,train,train
toy_330_0_6962.jpg,toy,safe,,toy_330,1246,train,train
toy_330_0_7923.jpg,toy,safe,,toy_330,1246,train,train
toy_330_0_927.jpg,toy,safe,,toy_330,1246,train,val
toy_331_0_1236.jpg,toy,safe,,toy_331,1247,train,train
toy_331_0_4856.jpg,toy,safe,,toy_331,1247,train,test
toy_331_0_4871.jpg,toy,safe,,toy_331,1247,train,train
toy_331_0_7848.jpg,toy,safe,,toy_331,1247,train,train
toy_331_0_9073.jpg,toy,safe,,toy_331,1247,train,train
toy_332_0_1427.jpg,toy,safe,,toy_332,1248,train,train
toy_332_0_1763.jpg,toy,safe,,toy_332,1248,train,test
toy_332_0_2156.jpg,toy,safe,,toy_332,1248,train,train
toy_332_0_5245.jpg,toy,safe,,toy_332,1248,train,val
toy_332_0_5573.jpg,toy,safe,,toy_332,1248,train,train
toy_333_0_3586.jpg,toy,safe,,toy_333,1249,train,train
toy_333_0_4437.jpg,toy,safe,,toy_333,1249,train,train
toy_333_0_5025.jpg,toy,safe,,toy_333,1249,train,test
toy_333_0_5968.jpg,toy,safe,,toy_333,1249,train,train
toy_333_0_8486.jpg,toy,safe,,toy_333,1249,train,train
toy_334_0_2110.jpg,toy,safe,,toy_334,1250,train,val
toy_334_0_4601.jpg,toy,safe,,toy_334,1250,train,val
toy_334_0_5058.jpg,toy,safe,,toy_334,1250,train,train
toy_334_0_6286.jpg,toy,safe,,toy_334,1250,train,test
toy_334_0_8160.jpg,toy,safe,,toy_334,1250,train,train
toy_335_0_2269.jpg,toy,safe,,toy_335,1251,test,train
toy_335_0_4130.jpg,toy,safe,,toy_335,1251,test,train
toy_335_0_67.jpg,toy,safe,,toy_335,1251,test,train
toy_335_0_6812.jpg,toy,safe,,toy_335,1251,test,train
toy_335_0_9718.jpg,toy,safe,,toy_335,1251,test,train
toy_336_0_404.jpg,toy,safe,,toy_336,1252,train,val
toy_336_0_4934.jpg,toy,safe,,toy_336,1252,train,train
toy_336_0_8733.jpg,toy,safe,,toy_336,1252,train,test
toy_336_0_8930.jpg,toy,safe,,toy_336,1252,train,train
toy_336_0_9557.jpg,toy,safe,,toy_336,1252,train,val
toy_337_0_2225.jpg,toy,safe,,toy_337,1253,train,train
toy_337_0_2641.jpg,toy,safe,,toy_337,1253,train,test
toy_337_0_4006.jpg,toy,safe,,toy_337,1253,train,train
toy_337_0_7262.jpg,toy,safe,,toy_337,1253,train,val
toy_337_0_9472.jpg,toy,safe,,toy_337,1253,train,train
toy_338_0_4221.jpg,toy,safe,,toy_338,1254,train,train
toy_338_0_5245.jpg,toy,safe,,toy_338,1254,train,test
toy_338_0_7185.jpg,toy,safe,,toy_338,1254,train,train
toy_338_0_7461.jpg,toy,safe,,toy_338,1254,train,train
toy_338_0_7760.jpg,toy,safe,,toy_338,1254,train,test
toy_339_0_3285.jpg,toy,safe,,toy_339,1255,train,val
toy_339_0_5908.jpg,toy,safe,,toy_339,1255,train,test
toy_339_0_8315.jpg,toy,safe,,toy_339,1255,train,train
toy_339_0_8705.jpg,toy,safe,,toy_339,1255,train,train
toy_339_0_9034.jpg,toy,safe,,toy_339,1255,train,val
toy_33_0_673.jpg,toy,safe,,toy_33,1256,train,train
toy_33_0_6733.jpg,toy,safe,,toy_33,1256,train,train
toy_33_0_7163.jpg,toy,safe,,toy_33,1256,train,train
toy_33_0_7870.jpg,toy,safe,,toy_33,1256,train,test
toy_33_0_9497.jpg,toy,safe,,toy_33,1256,train,train
toy_340_0_1534.jpg,toy,safe,,toy_340,1257,test,train
toy_340_0_778.jpg,toy,safe,,toy_340,1257,test,train
toy_340_0_8597.jpg,toy,safe,,toy_340,1257,test,train
toy_340_0_8841.jpg,toy,safe,,toy_340,1257,test,train
toy_340_0_9170.jpg,toy,safe,,toy_340,1257,test,train
toy_341_0_5058.jpg,toy,safe,,toy_341,1258,train,test
toy_341_0_5853.jpg,toy,safe,,toy_341,1258,train,train
toy_341_0_6335.jpg,toy,safe,,toy_341,1258,train,train
toy_341_0_9264.jpg,toy,safe,,toy_341,1258,train,train
toy_341_0_9574.jpg,toy,safe,,toy_341,1258,train,train
toy_342_0_338.jpg,toy,safe,,toy_342,1259,train,test
toy_342_0_3570.jpg,toy,safe,,toy_342,1259,train,val
toy_342_0_3733.jpg,toy,safe,,toy_342,1259,train,train
toy_342_0_4984.jpg,toy,safe,,toy_342,1259,train,train
toy_342_0_9398.jpg,toy,safe,,toy_342,1259,train,train
toy_343_0_2804.jpg,toy,safe,,toy_343,1260,train,test
toy_343_0_5467.jpg,toy,safe,,toy_343,1260,train,train
toy_343_0_7592.jpg,toy,safe,,toy_343,1260,train,train
toy_343_0_7956.jpg,toy,safe,,toy_343,1260,train,test
toy_343_0_898.jpg,toy,safe,,toy_343,1260,train,test
toy_344_0_1126.jpg,toy,safe,,toy_344,1261,test,train
toy_344_0_1269.jpg,toy,safe,,toy_344,1261,test,train
toy_344_0_1875.jpg,toy,safe,,toy_344,1261,test,val
toy_344_0_2563.jpg,toy,safe,,toy_344,1261,test,test
toy_344_0_3499.jpg,toy,safe,,toy_344,1261,test,train
toy_345_0_2796.jpg,toy,safe,,toy_345,1262,train,train
toy_345_0_4803.jpg,toy,safe,,toy_345,1262,train,train
toy_345_0_5930.jpg,toy,safe,,toy_345,1262,train,train
toy_345_0_8509.jpg,toy,safe,,toy_345,1262,train,test
toy_345_0_9096.jpg,toy,safe,,toy_345,1262,train,test
toy_346_0_1324.jpg,toy,safe,,toy_346,1263,test,train
toy_346_0_4024.jpg,toy,safe,,toy_346,1263,test,val
toy_346_0_6926.jpg,toy,safe,,toy_346,1263,test,train
toy_346_0_9248.jpg,toy,safe,,toy_346,1263,test,train
toy_346_0_9583.jpg,toy,safe,,toy_346,1263,test,train
toy_347_0_1430.jpg,toy,safe,,toy_347,1264,test,val
toy_347_0_2031.jpg,toy,safe,,toy_347,1264,test,test
toy_347_0_2112.jpg,toy,safe,,toy_347,1264,test,train
toy_347_0_2274.jpg,toy,safe,,toy_347,1264,test,train
toy_347_0_9591.jpg,toy,safe,,toy_347,1264,test,val
toy_348_0_5105.jpg,toy,safe,,toy_348,1265,test,train
toy_348_0_6471.jpg,toy,safe,,toy_348,1265,test,test
toy_348_0_7643.jpg,toy,safe,,toy_348,1265,test,train
toy_348_0_8554.jpg,toy,safe,,toy_348,1265,test,train
toy_348_0_9379.jpg,toy,safe,,toy_348,1265,test,train
toy_349_0_1339.jpg,toy,safe,,toy_349,1266,train,train
toy_349_0_1402.jpg,toy,safe,,toy_349,1266,train,train
toy_349_0_2199.jpg,toy,safe,,toy_349,1266,train,train
toy_349_0_2737.jpg,toy,safe,,toy_349,1266,train,train
toy_349_0_643.jpg,toy,safe,,toy_349,1266,train,train
toy_34_0_1253.jpg,toy,safe,,toy_34,1267,val,train
toy_34_0_2096.jpg,toy,safe,,toy_34,1267,val,train
toy_34_0_4957.jpg,toy,safe,,toy_34,1267,val,train
toy_34_0_5364.jpg,toy,safe,,toy_34,1267,val,val
toy_34_0_6873.jpg,toy,safe,,toy_34,1267,val,val
toy_350_0_207.jpg,toy,safe,,toy_350,1268,val,train
toy_350_0_2300.jpg,toy,safe,,toy_350,1268,val,val
toy_350_0_3954.jpg,toy,safe,,toy_350,1268,val,test
toy_350_0_6313.jpg,toy,safe,,toy_350,1268,val,train
toy_350_0_8090.jpg,toy,safe,,toy_350,1268,val,train
toy_351_0_174.jpg,toy,safe,,toy_351,1269,val,train
toy_351_0_4254.jpg,toy,safe,,toy_351,1269,val,train
toy_351_0_4777.jpg,toy,safe,,toy_351,1269,val,train
toy_351_0_6523.jpg,toy,safe,,toy_351,1269,val,train
toy_351_0_7897.jpg,toy,safe,,toy_351,1269,val,test
toy_352_0_134.jpg,toy,safe,,toy_352,1270,test,train
toy_352_0_1629.jpg,toy,safe,,toy_352,1270,test,train
toy_352_0_5968.jpg,toy,safe,,toy_352,1270,test,train
toy_352_0_776.jpg,toy,safe,,toy_352,1270,test,train
toy_352_0_9723.jpg,toy,safe,,toy_352,1270,test,train
toy_353_0_1905.jpg,toy,safe,,toy_353,1271,val,train
toy_353_0_3344.jpg,toy,safe,,toy_353,1271,val,val
toy_353_0_3379.jpg,toy,safe,,toy_353,1271,val,train
toy_353_0_8017.jpg,toy,safe,,toy_353,1271,val,val
toy_353_0_9413.jpg,toy,safe,,toy_353,1271,val,train
toy_354_0_2539.jpg,toy,safe,,toy_354,1272,train,train
toy_354_0_3943.jpg,toy,safe,,toy_354,1272,train,train
toy_354_0_438.jpg,toy,safe,,toy_354,1272,train,train
toy_354_0_7536.jpg,toy,safe,,toy_354,1272,train,train
toy_354_0_8653.jpg,toy,safe,,toy_354,1272,train,train
toy_356_0_386.jpg,toy,safe,,toy_356,1273,train,train
toy_356_0_6125.jpg,toy,safe,,toy_356,1273,train,train
toy_356_0_6324.jpg,toy,safe,,toy_356,1273,train,train
toy_356_0_8451.jpg,toy,safe,,toy_356,1273,train,train
toy_356_0_9731.jpg,toy,safe,,toy_356,1273,train,train
toy_357_0_2320.jpg,toy,safe,,toy_357,1274,train,val
toy_357_0_4751.jpg,toy,safe,,toy_357,1274,train,train
toy_357_0_7057.jpg,toy,safe,,toy_357,1274,train,train
toy_357_0_8474.jpg,toy,safe,,toy_357,1274,train,train
toy_357_0_9044.jpg,toy,safe,,toy_357,1274,train,val
toy_358_0_4209.jpg,toy,safe,,toy_358,1275,train,train
toy_358_0_5791.jpg,toy,safe,,toy_358,1275,train,train
toy_358_0_7038.jpg,toy,safe,,toy_358,1275,train,train
toy_358_0_7619.jpg,toy,safe,,toy_358,1275,train,val
toy_358_0_77.jpg,toy,safe,,toy_358,1275,train,test
toy_359_0_5129.jpg,toy,safe,,toy_359,1276,train,test
toy_359_0_550.jpg,toy,safe,,toy_359,1276,train,train
toy_359_0_5653.jpg,toy,safe,,toy_359,1276,train,train
toy_359_0_5776.jpg,toy,safe,,toy_359,1276,train,train
toy_359_0_7337.jpg,toy,safe,,toy_359,1276,train,train
toy_35_0_2058.jpg,toy,safe,,toy_35,1277,train,val
toy_35_0_2672.jpg,toy,safe,,toy_35,1277,train,train
toy_35_0_2926.jpg,toy,safe,,toy_35,1277,train,test
toy_35_0_647.jpg,toy,safe,,toy_35,1277,train,val
toy_35_0_7056.jpg,toy,safe,,toy_35,1277,train,train
toy_360_0_1808.jpg,toy,safe,,toy_360,1278,train,train
toy_360_0_192.jpg,toy,safe,,toy_360,1278,train,train
toy_360_0_2138.jpg,toy,safe,,toy_360,1278,train,train
toy_360_0_4641.jpg,toy,safe,,toy_360,1278,train,train
toy_360_0_959.jpg,toy,safe,,toy_360,1278,train,train
toy_361_0_2436.jpg,toy,safe,,toy_361,1279,train,train
toy_361_0_2846.jpg,toy,safe,,toy_361,1279,train,train
toy_361_0_4379.jpg,toy,safe,,toy_361,1279,train,train
toy_361_0_5263.jpg,toy,safe,,toy_361,1279,train,train
toy_361_0_5662.jpg,toy,safe,,toy_361,1279,train,train
toy_362_0_2673.jpg,toy,safe,,toy_362,1280,train,test
toy_362_0_5214.jpg,toy,safe,,toy_362,1280,train,train
toy_362_0_7497.jpg,toy,safe,,toy_362,1280,train,test
toy_362_0_8354.jpg,toy,safe,,toy_362,1280,train,train
toy_362_0_9336.jpg,toy,safe,,toy_362,1280,train,train
toy_363_0_3928.jpg,toy,safe,,toy_363,1281,train,train
toy_363_0_4585.jpg,toy,safe,,toy_363,1281,train,train
toy_363_0_5600.jpg,toy,safe,,toy_363,1281,train,train
toy_363_0_5698.jpg,toy,safe,,toy_363,1281,train,test
toy_363_0_7273.jpg,toy,safe,,toy_363,1281,train,train
toy_364_0_2432.jpg,toy,safe,,toy_364,1282,train,train
toy_364_0_4972.jpg,toy,safe,,toy_364,1282,train,train
toy_364_0_5100.jpg,toy,safe,,toy_364,1282,train,train
toy_364_0_7072.jpg,toy,safe,,toy_364,1282,train,train
toy_364_0_9230.jpg,toy,safe,,toy_364,1282,train,val
toy_365_0_1091.jpg,toy,safe,,toy_365,1283,train,val
toy_365_0_1890.jpg,toy,safe,,toy_365,1283,train,train
toy_365_0_5729.jpg,toy,safe,,toy_365,1283,train,train
toy_365_0_7511.jpg,toy,safe,,toy_365,1283,train,train
toy_365_0_9529.jpg,toy,safe,,toy_365,1283,train,train
toy_366_0_3457.jpg,toy,safe,,toy_366,1284,val,val
toy_366_0_5123.jpg,toy,safe,,toy_366,1284,val,train
toy_366_0_6847.jpg,toy,safe,,toy_366,1284,val,train
toy_366_0_9255.jpg,toy,safe,,toy_366,1284,val,val
toy_366_0_9743.jpg,toy,safe,,toy_366,1284,val,train
toy_367_0_239.jpg,toy,safe,,toy_367,1285,train,train
toy_367_0_5012.jpg,toy,safe,,toy_367,1285,train,test
toy_367_0_5221.jpg,toy,safe,,toy_367,1285,train,train
toy_367_0_7580.jpg,toy,safe,,toy_367,1285,train,train
toy_367_0_9087.jpg,toy,safe,,toy_367,1285,train,val
toy_368_0_1492.jpg,toy,safe,,toy_368,1286,train,train
toy_368_0_7823.jpg,toy,safe,,toy_368,1286,train,train
toy_368_0_7896.jpg,toy,safe,,toy_368,1286,train,train
toy_368_0_8976.jpg,toy,safe,,toy_368,1286,train,val
toy_368_0_9004.jpg,toy,safe,,toy_368,1286,train,train
toy_369_0_1780.jpg,toy,safe,,toy_369,1287,train,train
toy_369_0_6754.jpg,toy,safe,,toy_369,1287,train,train
toy_369_0_7317.jpg,toy,safe,,toy_369,1287,train,train
toy_369_0_822.jpg,toy,safe,,toy_369,1287,train,test
toy_369_0_9050.jpg,toy,safe,,toy_369,1287,train,test
toy_36_0_2113.jpg,toy,safe,,toy_36,1288,train,train
toy_36_0_3081.jpg,toy,safe,,toy_36,1288,train,train
toy_36_0_5345.jpg,toy,safe,,toy_36,1288,train,test
toy_36_0_6302.jpg,toy,safe,,toy_36,1288,train,train
toy_36_0_8296.jpg,toy,safe,,toy_36,1288,train,train
toy_370_0_2753.jpg,toy,safe,,toy_370,1289,train,train
toy_370_0_3171.jpg,toy,safe,,toy_370,1289,train,train
toy_370_0_3417.jpg,toy,safe,,toy_370,1289,train,train
toy_370_0_6479.jpg,toy,safe,,toy_370,1289,train,test
toy_370_0_9232.jpg,toy,safe,,toy_370,1289,train,train
toy_371_0_1266.jpg,toy,safe,,toy_371,1290,train,train
toy_371_0_5506.jpg,toy,safe,,toy_371,1290,train,train
toy_371_0_7404.jpg,toy,safe,,toy_371,1290,train,train
toy_371_0_7875.jpg,toy,safe,,toy_371,1290,train,train
toy_371_0_9228.jpg,toy,safe,,toy_371,1290,train,train
toy_372_0_2557.jpg,toy,safe,,toy_372,1291,train,val
toy_372_0_3052.jpg,toy,safe,,toy_372,1291,train,train
toy_372_0_5673.jpg,toy,safe,,toy_372,1291,train,train
toy_372_0_5791.jpg,toy,safe,,toy_372,1291,train,val
toy_372_0_6249.jpg,toy,safe,,toy_372,1291,train,train
toy_373_0_1250.jpg,toy,safe,,toy_373,1292,train,test
toy_373_0_2021.jpg,toy,safe,,toy_373,1292,train,train
toy_373_0_3773.jpg,toy,safe,,toy_373,1292,train,test
toy_373_0_4088.jpg,toy,safe,,toy_373,1292,train,train
toy_373_0_9208.jpg,toy,safe,,toy_373,1292,train,val
toy_374_0_2402.jpg,toy,safe,,toy_374,1293,val,train
toy_374_0_3009.jpg,toy,safe,,toy_374,1293,val,train
toy_374_0_578.jpg,toy,safe,,toy_374,1293,val,test
toy_374_0_8130.jpg,toy,safe,,toy_374,1293,val,train
toy_374_0_874.jpg,toy,safe,,toy_374,1293,val,test
toy_375_0_2489.jpg,toy,safe,,toy_375,1294,train,val
toy_375_0_2566.jpg,toy,safe,,toy_375,1294,train,train
toy_375_0_3763.jpg,toy,safe,,toy_375,1294,train,train
toy_375_0_4791.jpg,toy,safe,,toy_375,1294,train,val
toy_375_0_6699.jpg,toy,safe,,toy_375,1294,train,train
toy_376_0_5863.jpg,toy,safe,,toy_376,1295,test,train
toy_376_0_6049.jpg,toy,safe,,toy_376,1295,test,test
toy_376_0_661.jpg,toy,safe,,toy_376,1295,test,train
toy_376_0_7507.jpg,toy,safe,,toy_376,1295,test,train
toy_376_0_8257.jpg,toy,safe,,toy_376,1295,test,train
toy_377_0_1741.jpg,toy,safe,,toy_377,1296,train,train
toy_377_0_1806.jpg,toy,safe,,toy_377,1296,train,train
toy_377_0_5790.jpg,toy,safe,,toy_377,1296,train,train
toy_377_0_6126.jpg,toy,safe,,toy_377,1296,train,train
toy_377_0_9138.jpg,toy,safe,,toy_377,1296,train,test
toy_378_0_1068.jpg,toy,safe,,toy_378,1297,train,train
toy_378_0_178.jpg,toy,safe,,toy_378,1297,train,test
toy_378_0_2483.jpg,toy,safe,,toy_378,1297,train,train
toy_378_0_4499.jpg,toy,safe,,toy_378,1297,train,test
toy_378_0_5833.jpg,toy,safe,,toy_378,1297,train,test
toy_379_0_1193.jpg,toy,safe,,toy_379,1298,train,train
toy_379_0_3919.jpg,toy,safe,,toy_379,1298,train,train
toy_379_0_4129.jpg,toy,safe,,toy_379,1298,train,train
toy_379_0_8717.jpg,toy,safe,,toy_379,1298,train,train
toy_379_0_9406.jpg,toy,safe,,toy_379,1298,train,train
toy_37_0_2164.jpg,toy,safe,,toy_37,1299,train,val
toy_37_0_2635.jpg,toy,safe,,toy_37,1299,train,test
toy_37_0_4501.jpg,toy,safe,,toy_37,1299,train,train
toy_37_0_5933.jpg,toy,safe,,toy_37,1299,train,train
toy_37_0_7579.jpg,toy,safe,,toy_37,1299,train,test
toy_380_0_3376.jpg,toy,safe,,toy_380,1300,train,train
toy_380_0_5506.jpg,toy,safe,,toy_380,1300,train,train
toy_380_0_6835.jpg,toy,safe,,toy_380,1300,train,train
toy_380_0_7214.jpg,toy,safe,,toy_380,1300,train,train
toy_380_0_93.jpg,toy,safe,,toy_380,1300,train,test
toy_381_0_1464.jpg,toy,safe,,toy_381,1301,val,train
toy_381_0_1820.jpg,toy,safe,,toy_381,1301,val,val
toy_381_0_478.jpg,toy,safe,,toy_381,1301,val,val
toy_381_0_5201.jpg,toy,safe,,toy_381,1301,val,val
toy_381_0_7467.jpg,toy,safe,,toy_381,1301,val,train
toy_382_0_2106.jpg,toy,safe,,toy_382,1302,train,train
toy_382_0_2954.jpg,toy,safe,,toy_382,1302,train,train
toy_382_0_6754.jpg,toy,safe,,toy_382,1302,train,train
toy_382_0_912.jpg,toy,safe,,toy_382,1302,train,val
toy_382_0_9894.jpg,toy,safe,,toy_382,1302,train,train
toy_383_0_1178.jpg,toy,safe,,toy_383,1303,train,train
toy_383_0_4734.jpg,toy,safe,,toy_383,1303,train,train
toy_383_0_572.jpg,toy,safe,,toy_383,1303,train,train
toy_383_0_6577.jpg,toy,safe,,toy_383,1303,train,train
toy_383_0_8656.jpg,toy,safe,,toy_383,1303,train,train
toy_384_0_4157.jpg,toy,safe,,toy_384,1304,train,train
toy_384_0_450.jpg,toy,safe,,toy_384,1304,train,train
toy_384_0_5376.jpg,toy,safe,,toy_384,1304,train,val
toy_384_0_6393.jpg,toy,safe,,toy_384,1304,train,train
toy_384_0_7187.jpg,toy,safe,,toy_384,1304,train,test
toy_385_0_1130.jpg,toy,safe,,toy_385,1305,train,val
toy_385_0_2240.jpg,toy,safe,,toy_385,1305,train,val
toy_385_0_2350.jpg,toy,safe,,toy_385,1305,train,train
toy_385_0_7968.jpg,toy,safe,,toy_385,1305,train,test
toy_385_0_8071.jpg,toy,safe,,toy_385,1305,train,train
toy_386_0_1069.jpg,toy,safe,,toy_386,1306,val,train
toy_386_0_2946.jpg,toy,safe,,toy_386,1306,val,train
toy_386_0_330.jpg,toy,safe,,toy_386,1306,val,train
toy_386_0_5635.jpg,toy,safe,,toy_386,1306,val,train
toy_386_0_9869.jpg,toy,safe,,toy_386,1306,val,train
toy_387_0_2008.jpg,toy,safe,,toy_387,1307,train,train
toy_387_0_2718.jpg,toy,safe,,toy_387,1307,train,train
toy_387_0_3046.jpg,toy,safe,,toy_387,1307,train,train
toy_387_0_6236.jpg,toy,safe,,toy_387,1307,train,test
toy_387_0_8725.jpg,toy,safe,,toy_387,1307,train,val
toy_388_0_1512.jpg,toy,safe,,toy_388,1308,train,test
toy_388_0_3488.jpg,toy,safe,,toy_388,1308,train,train
toy_388_0_3999.jpg,toy,safe,,toy_388,1308,train,train
toy_388_0_7708.jpg,toy,safe,,toy_388,1308,train,val
toy_388_0_7835.jpg,toy,safe,,toy_388,1308,train,train
toy_389_0_2638.jpg,toy,safe,,toy_389,1309,train,train
toy_389_0_5837.jpg,toy,safe,,toy_389,1309,train,train
toy_389_0_6775.jpg,toy,safe,,toy_389,1309,train,test
toy_389_0_9130.jpg,toy,safe,,toy_389,1309,train,train
toy_389_0_9618.jpg,toy,safe,,toy_389,1309,train,test
toy_38_0_2833.jpg,toy,safe,,toy_38,1310,train,test
toy_38_0_3670.jpg,toy,safe,,toy_38,1310,train,train
toy_38_0_3971.jpg,toy,safe,,toy_38,1310,train,test
toy_38_0_6345.jpg,toy,safe,,toy_38,1310,train,train
toy_38_0_7865.jpg,toy,safe,,toy_38,1310,train,train
toy_390_0_1025.jpg,toy,safe,,toy_390,1311,test,train
toy_390_0_2206.jpg,toy,safe,,toy_390,1311,test,train
toy_390_0_3434.jpg,toy,safe,,toy_390,1311,test,train
toy_390_0_6816.jpg,toy,safe,,toy_390,1311,test,train
toy_390_0_9507.jpg,toy,safe,,toy_390,1311,test,train
toy_391_0_3784.jpg,toy,safe,,toy_391,1312,val,val
toy_391_0_3792.jpg,toy,safe,,toy_391,1312,val,train
toy_391_0_3997.jpg,toy,safe,,toy_391,1312,val,test
toy_391_0_4856.jpg,toy,safe,,toy_391,1312,val,train
toy_391_0_5974.jpg,toy,safe,,toy_391,1312,val,train
toy_392_0_2383.jpg,toy,safe,,toy_392,1313,train,train
toy_392_0_2678.jpg,toy,safe,,toy_392,1313,train,train
toy_392_0_5178.jpg,toy,safe,,toy_392,1313,train,train
toy_392_0_8120.jpg,toy,safe,,toy_392,1313,train,train
toy_392_0_8170.jpg,toy,safe,,toy_392,1313,train,train
toy_393_0_2262.jpg,toy,safe,,toy_393,1314,val,train
toy_393_0_5416.jpg,toy,safe,,toy_393,1314,val,train
toy_393_0_5510.jpg,toy,safe,,toy_393,1314,val,train
toy_393_0_5576.jpg,toy,safe,,toy_393,1314,val,val
toy_393_0_7543.jpg,toy,safe,,toy_393,1314,val,train
toy_394_0_170.jpg,toy,safe,,toy_394,1315,train,test
toy_394_0_3441.jpg,toy,safe,,toy_394,1315,train,test
toy_394_0_794.jpg,toy,safe,,toy_394,1315,train,train
toy_394_0_9273.jpg,toy,safe,,toy_394,1315,train,train
toy_394_0_9453.jpg,toy,safe,,toy_394,1315,train,train
toy_395_0_1844.jpg,toy,safe,,toy_395,1316,train,train
toy_395_0_4533.jpg,toy,safe,,toy_395,1316,train,train
toy_395_0_5140.jpg,toy,safe,,toy_395,1316,train,train
toy_395_0_5308.jpg,toy,safe,,toy_395,1316,train,train
toy_395_0_7973.jpg,toy,safe,,toy_395,1316,train,test
toy_39_0_1597.jpg,toy,safe,,toy_39,1317,train,train
toy_39_0_1770.jpg,toy,safe,,toy_39,1317,train,train
toy_39_0_53.jpg,toy,safe,,toy_39,1317,train,val
toy_39_0_7395.jpg,toy,safe,,toy_39,1317,train,train
toy_39_0_7969.jpg,toy,safe,,toy_39,1317,train,train
toy_3_0_1971.jpg,toy,safe,,toy_3,1318,train,train
toy_3_0_3475.jpg,toy,safe,,toy_3,1318,train,train
toy_3_0_7000.jpg,toy,safe,,toy_3,1318,train,train
toy_3_0_7296.jpg,toy,safe,,toy_3,1318,train,train
toy_3_0_8093.jpg,toy,safe,,toy_3,1318,train,train
toy_40_0_1102.jpg,toy,safe,,toy_40,1319,test,train
toy_40_0_3902.jpg,toy,safe,,toy_40,1319,test,train
toy_40_0_4124.jpg,toy,safe,,toy_40,1319,test,train
toy_40_0_5182.jpg,toy,safe,,toy_40,1319,test,train
toy_40_0_6332.jpg,toy,safe,,toy_40,1319,test,train
toy_41_0_1553.jpg,toy,safe,,toy_41,1320,train,train
toy_41_0_2977.jpg,toy,safe,,toy_41,1320,train,train
toy_41_0_6207.jpg,toy,safe,,toy_41,1320,train,train
toy_41_0_7104.jpg,toy,safe,,toy_41,1320,train,train
toy_41_0_8825.jpg,toy,safe,,toy_41,1320,train,test
toy_42_0_1330.jpg,toy,safe,,toy_42,1321,test,train
toy_42_0_5561.jpg,toy,safe,,toy_42,1321,test,train
toy_42_0_6437.jpg,toy,safe,,toy_42,1321,test,train
toy_42_0_7065.jpg,toy,safe,,toy_42,1321,test,train
toy_42_0_8777.jpg,toy,safe,,toy_42,1321,test,train
toy_4_0_33.jpg,toy,safe,,toy_4,1322,test,test
toy_4_0_3783.jpg,toy,safe,,toy_4,1322,test,train
toy_4_0_8445.jpg,toy,safe,,toy_4,1322,test,test
toy_4_0_890.jpg,toy,safe,,toy_4,1322,test,train
toy_4_0_9177.jpg,toy,safe,,toy_4,1322,test,train


In [ ]:
%%writefile src/childsafe/data/__init__.py
# package


In [ ]:
%%writefile src/childsafe/data/catalog.py
"""Build an image catalog (path, category, binary label) from a folder tree.

The ChildSUn folder layout was not verified before writing this, so the category is
inferred from folder names: the deepest folder whose normalized name is in
``label_map`` wins, and a specific category (``knife``) beats a generic one (``unsafe``).
Folders called train/test/val are kept as ``source_split`` for information only;
they are NOT used for our split because they may contain leaked augmentations.
"""
from __future__ import annotations

import re
from pathlib import Path

import pandas as pd

from childsafe.config import normalize_name

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SPLIT_NAMES = {"train", "training", "test", "testing", "val", "valid", "validation"}
GENERIC = {"safe", "unsafe"}


def infer_labels(rel_parts: tuple[str, ...], label_map: dict) -> tuple[str | None, str | None, str | None]:
    """Return (category, binary_label, source_split) from the folder parts of a relative path."""
    category = binary = generic = source_split = None
    for part in rel_parts:
        norm = normalize_name(part)
        if norm in SPLIT_NAMES:
            source_split = norm
        if norm in label_map:
            if norm in GENERIC:
                generic = norm
            else:
                category, binary = norm, label_map[norm]
    if category is None and generic is not None:
        category, binary = generic, generic
    return category, binary, source_split


AUG_NAME = re.compile(r"^(?P<prefix>.+?)_(?P<src>\d+)_\d+_\d+$")   # Keras save_to_dir: <prefix>_<index>_<n>_<rand>
PLAIN_NAME = re.compile(r"^(?P<prefix>.+?)_\d+$")                   # e.g. fruits_12


def parse_filename(stem: str) -> tuple[str | None, str | None]:
    """Return (class prefix, source photo index or None)."""
    m = AUG_NAME.match(stem)
    if m:
        return m["prefix"], m["src"]
    m = PLAIN_NAME.match(stem)
    return (m["prefix"], None) if m else (None, None)


def source_roots(cfg: dict) -> dict[str, Path]:
    """Image root per source: ChildSUn plus any `extra_sources` (name -> folder relative to the repo)."""
    from childsafe.config import ROOT
    roots = {"childsun": Path(cfg["paths"]["raw_data"])}
    roots.update({name: (ROOT / folder).resolve() for name, folder in (cfg.get("extra_sources") or {}).items()})
    return roots


def build_full_catalog(cfg: dict) -> pd.DataFrame:
    """ChildSUn catalog plus extra sources (e.g. Open Images crops), with a `source` column.
    Extra-source crops are grouped by their original photo id (file name `<imageid>_<class>.jpg`)."""
    frames = []
    for name, root in source_roots(cfg).items():
        df = build_catalog(root, cfg["label_map"], cfg.get("category_aliases"))
        df["source"] = name
        if name != "childsun":
            df["source_id"] = name + "_" + df["path"].map(lambda p: Path(p).stem.split("_")[0])
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


def build_catalog(root: str | Path, label_map: dict, aliases: dict | None = None) -> pd.DataFrame:
    root = Path(root)
    aliases = aliases or {}
    rows, unmatched = [], {}
    # sort by the posix string: Path sorting is case-insensitive on Windows, case-sensitive on Linux,
    # which gave different image orders (and different group splits) locally and on Kaggle
    for p in sorted(root.rglob("*"), key=lambda q: q.relative_to(root).as_posix()):
        if p.suffix.lower() not in IMG_EXT or not p.is_file():
            continue
        rel = p.relative_to(root)
        category, binary, source_split = infer_labels(rel.parts[:-1], label_map)
        prefix, source = parse_filename(p.stem)
        if category is None or category in GENERIC:
            # flat layout (as ChildSUn ships): class is the filename prefix
            norm = normalize_name(prefix) if prefix else ""
            if norm in label_map and norm not in GENERIC:
                category, binary = norm, label_map[norm]
        if category is None:
            key = f"{rel.parent} ({prefix or p.stem})"
            unmatched[key] = unmatched.get(key, 0) + 1
            continue
        category = aliases.get(category, category)
        rows.append({"path": rel.as_posix(), "category": category, "label": binary,
                     "source_split": source_split or "",
                     # augmented copies of one original photo share this id
                     "source_id": f"{normalize_name(prefix)}_{source}" if source is not None else rel.as_posix()})
    if unmatched:
        print(f"[catalog] {sum(unmatched.values())} images not matched to label_map (skipped):")
        for k, v in sorted(unmatched.items()):
            print(f"    {k}: {v}")
    if not rows:
        raise SystemExit(f"No labelled images found under {root}. Check paths.raw_data and label_map.")
    return pd.DataFrame(rows)


In [ ]:
%%writefile src/childsafe/data/corruptions.py
"""A5: image corruptions with a 1-5 severity scale.

Used two ways: (1) as random training augmentation (``RobustAug``), (2) at fixed
severity on the test set to draw degradation curves (accuracy vs severity).
Severity 0 means no corruption. The parameter tables are our own choice, loosely
following the ImageNet-C idea; they are not taken from that benchmark.
"""
from __future__ import annotations

import random

import cv2
import numpy as np
from PIL import Image

SEVERITY = {
    "gaussian_noise": [0.04, 0.08, 0.12, 0.18, 0.26],   # std, image in [0,1]
    "iso_noise":      [0.04, 0.08, 0.12, 0.18, 0.26],   # shot + colour noise
    "motion_blur":    [5, 9, 13, 19, 27],               # kernel length (px at 224)
    "low_light":      [1.5, 2.0, 2.6, 3.3, 4.2],        # gamma > 1 darkens
    "occlusion":      [0.08, 0.15, 0.22, 0.30, 0.40],   # erased area fraction
    "jpeg":           [60, 40, 25, 15, 8],              # quality
}
KINDS = list(SEVERITY)


def _rng(seed):
    return np.random.default_rng(seed)


def corrupt(img: np.ndarray, kind: str, severity: int, seed: int | None = None) -> np.ndarray:
    """img: HxWx3 uint8 RGB. Returns uint8 RGB."""
    if severity <= 0:
        return img
    p = SEVERITY[kind][severity - 1]
    rng = _rng(seed)
    x = img.astype(np.float32) / 255.0
    if kind == "gaussian_noise":
        x = x + rng.normal(0, p, x.shape)
    elif kind == "iso_noise":
        s = 1.0 / p ** 2                      # shot-noise std is p at full brightness
        x = rng.poisson(x * s) / s + rng.normal(0, p / 2, x.shape)
    elif kind == "motion_blur":
        k = int(p)
        kernel = np.zeros((k, k), np.float32)
        kernel[k // 2, :] = 1.0 / k
        angle = rng.uniform(0, 180)
        m = cv2.getRotationMatrix2D((k / 2 - 0.5, k / 2 - 0.5), angle, 1.0)
        kernel = cv2.warpAffine(kernel, m, (k, k))
        kernel /= max(kernel.sum(), 1e-6)
        x = cv2.filter2D(x, -1, kernel)
    elif kind == "low_light":
        x = np.power(x, p)
        x = x + rng.normal(0, 0.01 * p, x.shape)
    elif kind == "occlusion":
        h, w = x.shape[:2]
        area = p * h * w
        ar = rng.uniform(0.5, 2.0)
        eh, ew = int(min(h, np.sqrt(area * ar))), int(min(w, np.sqrt(area / ar)))
        y0, x0 = rng.integers(0, h - eh + 1), rng.integers(0, w - ew + 1)
        x[y0:y0 + eh, x0:x0 + ew] = rng.uniform(0, 1, 3)
    elif kind == "jpeg":
        ok, buf = cv2.imencode(".jpg", (np.clip(x, 0, 1) * 255).astype(np.uint8)[..., ::-1],
                               [cv2.IMWRITE_JPEG_QUALITY, int(p)])
        return cv2.imdecode(buf, cv2.IMREAD_COLOR)[..., ::-1].copy()
    else:
        raise ValueError(kind)
    return (np.clip(x, 0, 1) * 255).astype(np.uint8)


class FixedCorruption:
    """PIL -> PIL transform with a fixed kind/severity. The seed advances per call, so each image gets
    different noise / occlusion placement, and a fresh instance reproduces the same sequence."""

    def __init__(self, kind: str, severity: int, seed: int = 0):
        self.kind, self.severity, self.seed = kind, severity, seed

    def __call__(self, img: Image.Image) -> Image.Image:
        self.seed += 1
        return Image.fromarray(corrupt(np.asarray(img.convert("RGB")), self.kind, self.severity, self.seed))


class RobustAug:
    """Training-time A5 augmentation: with prob p apply one random corruption at severity 1-3."""

    KINDS = ("iso_noise", "motion_blur", "low_light", "gaussian_noise", "jpeg")

    def __init__(self, p: float = 0.5, max_severity: int = 3):
        self.p, self.max_severity = p, max_severity

    def __call__(self, img: Image.Image) -> Image.Image:
        if random.random() > self.p:
            return img
        kind = random.choice(self.KINDS)
        sev = random.randint(1, self.max_severity)
        return Image.fromarray(corrupt(np.asarray(img.convert("RGB")), kind, sev, random.randrange(1 << 30)))


In [ ]:
%%writefile src/childsafe/data/dataset.py
"""Torch dataset and transforms for Module A."""
from __future__ import annotations

from pathlib import Path

import pandas as pd
import torch
from PIL import Image, ImageOps
from torch.utils.data import Dataset
from torchvision import transforms as T

from childsafe.data.corruptions import FixedCorruption, RobustAug

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)


class Letterbox:
    """Keep the full thin object visible rather than center-cropping its endpoints."""
    def __init__(self, size):
        self.size = size

    def __call__(self, image):
        return ImageOps.pad(image, (self.size, self.size), method=Image.Resampling.BILINEAR,
                            color=(124, 116, 104))


def train_transform(size: int, robust: bool = False, preprocessing: str = "center_crop"):
    if preprocessing not in ('center_crop','letterbox'):
        raise ValueError('unsupported checkpoint preprocessing')
    tf = [Letterbox(size) if preprocessing == "letterbox" else T.RandomResizedCrop(size, scale=(0.6, 1.0)),
          T.RandomHorizontalFlip(),
          T.ColorJitter(0.3, 0.3, 0.2, 0.02)]
    if robust:
        tf.append(RobustAug(p=0.5))
    tf += [T.ToTensor(), T.Normalize(MEAN, STD)]
    if robust:
        tf.append(T.RandomErasing(p=0.4, scale=(0.02, 0.25), value="random"))
    return T.Compose(tf)


def eval_transform(size: int, corruption: tuple[str, int] | None = None, preprocessing: str = "center_crop"):
    if preprocessing not in ('center_crop','letterbox'):
        raise ValueError('unsupported checkpoint preprocessing')
    tf = [Letterbox(size)] if preprocessing == "letterbox" else [T.Resize(int(size * 1.14)), T.CenterCrop(size)]
    if corruption and corruption[1] > 0:
        tf.append(FixedCorruption(*corruption))
    tf += [T.ToTensor(), T.Normalize(MEAN, STD)]
    return T.Compose(tf)


def image_paths(df: pd.DataFrame, root: str | Path | dict) -> list[Path]:
    """root: one folder, or {source: folder} when df has a `source` column (ChildSUn + extra sources)."""
    if isinstance(root, dict):
        src = df["source"] if "source" in df else ["childsun"] * len(df)
        # Expanded reviewed manifests already contain absolute paths and may have
        # no source value for supplemental rows. Resolve those before looking up roots.
        return [Path(p) if Path(p).is_absolute() else Path(root['childsun' if pd.isna(s) else s]) / p
                for s, p in zip(src, df['path'])]
    return [Path(root) / p for p in df["path"]]


class CatalogDataset(Dataset):
    def __init__(self, df: pd.DataFrame, root: str | Path | dict, classes: list[str], transform):
        self.paths = image_paths(df, root)
        idx = {c: i for i, c in enumerate(classes)}
        self.targets = torch.tensor([idx[c] for c in df["category"]])
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.transform(img), self.targets[i]


In [ ]:
%%writefile src/childsafe/data/dedup.py
"""A1: perceptual-hash near-duplicate grouping.

Each image gets a 64-bit pHash and dHash. With ``dihedral=True`` we also hash the 7
flipped/rotated variants, and the distance between images i and j is the minimum over
j's variants. That catches copies produced by flip / 90-degree rotation augmentation,
which a plain pHash misses. Small-angle rotations, crops and colour jitter mostly keep
pHash close anyway. Two images are near-duplicates when BOTH hash distances are under
threshold; near-duplicate pairs are merged into groups with union-find.

Hash thresholds are a judgement call: inspect the montage from scripts/find_duplicates.py
at the chosen threshold and report the threshold in the write-up.
"""
from __future__ import annotations

from pathlib import Path

import imagehash
import numpy as np
from PIL import Image
from tqdm import tqdm

N_VARIANTS = 8


def _dihedral(img: Image.Image) -> list[Image.Image]:
    out = []
    for flip in (False, True):
        base = img.transpose(Image.Transpose.FLIP_LEFT_RIGHT) if flip else img
        for k in range(4):
            out.append(base.rotate(90 * k, expand=True) if k else base)
    return out


def _to_int(h: imagehash.ImageHash) -> np.uint64:
    bits = np.packbits(h.hash.flatten().astype(np.uint8))
    return np.frombuffer(bits.tobytes(), dtype=">u8")[0].astype(np.uint64)


def compute_hashes(paths: list[str | Path], hash_size: int = 8, dihedral: bool = True):
    """Return (phash, dhash) arrays of shape (N, 8) uint64. Unreadable images get all-ones hashes."""
    if hash_size != 8:
        raise ValueError("only 64-bit hashes (hash_size=8) are supported")
    n = len(paths)
    ph = np.full((n, N_VARIANTS), np.iinfo(np.uint64).max, dtype=np.uint64)
    dh = ph.copy()
    bad = []
    for i, p in enumerate(tqdm(paths, desc="hashing")):
        try:
            img = Image.open(p).convert("L")
        except Exception:  # corrupt file: keep sentinel hash so it matches nothing
            bad.append(str(p))
            continue
        variants = _dihedral(img) if dihedral else [img] * N_VARIANTS
        for v, im in enumerate(variants):
            ph[i, v] = _to_int(imagehash.phash(im, hash_size))
            dh[i, v] = _to_int(imagehash.dhash(im, hash_size))
    if bad:
        print(f"[dedup] {len(bad)} unreadable images, e.g. {bad[:3]}")
    return ph, dh


def _popcount(x: np.ndarray) -> np.ndarray:
    if hasattr(np, "bitwise_count"):
        return np.bitwise_count(x)
    b = x.view(np.uint8).reshape(*x.shape, 8)
    return np.unpackbits(b, axis=-1).sum(-1)


def min_distance_matrix_block(h: np.ndarray, rows: slice, cols: np.ndarray | None = None) -> np.ndarray:
    """Min Hamming distance between canonical hashes h[rows, 0] and all variants of h[cols]."""
    a = h[rows, 0][:, None, None]
    b = h[cols] if cols is not None else h
    return _popcount(a ^ b[None, :, :]).min(-1)


def find_duplicate_pairs(ph, dh, phash_thr: int, dhash_thr: int, chunk: int = 256):
    """Return array of (i, j, d_phash, d_dhash) for i < j that are near-duplicates."""
    n = len(ph)
    out = []
    for s in tqdm(range(0, n, chunk), desc="pairwise"):
        rows = slice(s, min(s + chunk, n))
        dp = min_distance_matrix_block(ph, rows)
        dd = min_distance_matrix_block(dh, rows)
        ii, jj = np.nonzero((dp <= phash_thr) & (dd <= dhash_thr))
        ii = ii + s
        keep = ii != jj
        # the variant-min distance is not symmetric, so keep a match found in either direction
        for i, j in zip(ii[keep], jj[keep]):
            out.append((min(i, j), max(i, j), dp[i - s, j], dd[i - s, j]))
    if not out:
        return np.zeros((0, 4), dtype=np.int64)
    arr = np.array(out, dtype=np.int64)
    arr = arr[np.lexsort((arr[:, 2] + arr[:, 3], arr[:, 1], arr[:, 0]))]
    _, first = np.unique(arr[:, :2], axis=0, return_index=True)
    return arr[np.sort(first)]


def union_find_groups(n: int, pairs: np.ndarray) -> np.ndarray:
    parent = np.arange(n)

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for i, j in pairs[:, :2]:
        ri, rj = find(i), find(j)
        if ri != rj:
            parent[max(ri, rj)] = min(ri, rj)
    roots = np.array([find(i) for i in range(n)])
    _, group = np.unique(roots, return_inverse=True)
    return group


def cross_split_leakage(ph, dh, split: np.ndarray, phash_thr: int, dhash_thr: int,
                        query: str = "test", ref: str = "train") -> float:
    """Fraction of `query` images that have a near-duplicate in `ref`."""
    q = np.nonzero(split == query)[0]
    r = np.nonzero(split == ref)[0]
    if len(q) == 0 or len(r) == 0:
        return float("nan")
    hits = 0
    for s in range(0, len(q), 256):
        idx = q[s:s + 256]
        dp = _popcount(ph[idx, 0][:, None, None] ^ ph[r][None]).min(-1)
        dd = _popcount(dh[idx, 0][:, None, None] ^ dh[r][None]).min(-1)
        hits += int(((dp <= phash_thr) & (dd <= dhash_thr)).any(1).sum())
    return hits / len(q)


In [ ]:
%%writefile src/childsafe/data/split.py
"""A1: group-based split vs plain random split.

Both splits use the same train/val/test fractions and stratify by category, so the
only difference is whether near-duplicate groups are kept together. The gap in test
accuracy between the two is our estimate of how much leakage inflates the score.

The group split uses its own greedy assignment with numpy's PCG64 generator instead of
sklearn's StratifiedGroupKFold: the sklearn result changed between library versions
(local vs Kaggle gave different splits from identical groups), which breaks reproducibility.
"""
from __future__ import annotations

import numpy as np
from sklearn.model_selection import train_test_split

PARTS = ("train", "val", "test")


def _strata(y: np.ndarray, min_count: int = 3) -> np.ndarray:
    """Stratification labels; classes too small to stratify are pooled (sklearn needs >= 2 per class)."""
    y = np.asarray(y, dtype=object)
    vals, counts = np.unique(y, return_counts=True)
    rare = set(vals[counts < min_count])
    if not rare:
        return y
    s = np.array(["__rare__" if v in rare else v for v in y], dtype=object)
    if (s == "__rare__").sum() < min_count:
        s[s == "__rare__"] = vals[counts.argmax()]
    return s


def random_split(y: np.ndarray, test_frac: float, val_frac: float, seed: int) -> np.ndarray:
    idx = np.arange(len(y))
    trval, test = train_test_split(idx, test_size=test_frac, stratify=_strata(y), random_state=seed)
    train, val = train_test_split(trval, test_size=val_frac / (1 - test_frac),
                                  stratify=_strata(np.asarray(y)[trval]), random_state=seed)
    out = np.empty(len(y), dtype=object)
    out[train], out[val], out[test] = "train", "val", "test"
    return out


def group_split(y: np.ndarray, groups: np.ndarray, test_frac: float, val_frac: float,
                seed: int) -> np.ndarray:
    """Greedy stratified group split: groups are visited largest first (random order within a
    size), and each goes to the part that still needs the most images of the group's classes."""
    y = np.asarray(y)
    groups = np.asarray(groups)
    classes, y_idx = np.unique(y, return_inverse=True)
    gids, g_idx = np.unique(groups, return_inverse=True)
    counts = np.zeros((len(gids), len(classes)), dtype=np.int64)
    np.add.at(counts, (g_idx, y_idx), 1)

    frac = np.array([1 - test_frac - val_frac, val_frac, test_frac])
    target = frac[:, None] * counts.sum(0)[None, :]          # (3, n_classes)
    have = np.zeros_like(target)
    rng = np.random.Generator(np.random.PCG64(seed))
    order = rng.permutation(len(gids))
    order = order[np.argsort(-counts[order].sum(1), kind="stable")]

    assign = np.empty(len(gids), dtype=np.int64)
    for g in order:
        need = ((target - have) * counts[g][None, :]).sum(1) / counts[g].sum()
        part = int(np.argmax(need))
        assign[g] = part
        have[part] += counts[g]

    out = np.array(PARTS, dtype=object)[assign[g_idx]]
    for a in ("val", "test"):
        assert not (set(groups[out == "train"]) & set(groups[out == a])), f"group leaked into {a}"
    return out


In [ ]:
%%writefile src/childsafe/classifier/__init__.py
# package


In [ ]:
%%writefile src/childsafe/classifier/calibration.py
"""A3: temperature scaling, ECE, reliability diagrams (Guo et al., ICML 2017 method)."""
from __future__ import annotations

from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F


def fit_temperature(logits: np.ndarray, targets: np.ndarray, max_iter: int = 200) -> float:
    """Fit a single temperature T > 0 minimising NLL on held-out (val) logits."""
    lg = torch.as_tensor(logits, dtype=torch.float64)
    y = torch.as_tensor(targets, dtype=torch.long)
    log_t = torch.zeros(1, dtype=torch.float64, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=max_iter, line_search_fn="strong_wolfe")

    def closure():
        opt.zero_grad()
        loss = F.cross_entropy(lg / log_t.exp(), y)
        loss.backward()
        return loss

    opt.step(closure)
    return float(log_t.exp().item())


def softmax(logits: np.ndarray, temperature: float = 1.0) -> np.ndarray:
    z = logits / temperature
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def ece(confidence: np.ndarray, correct: np.ndarray, n_bins: int = 15) -> float:
    """Expected calibration error with equal-width bins."""
    edges = np.linspace(0, 1, n_bins + 1)
    total, n = 0.0, len(confidence)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (confidence > lo) & (confidence <= hi)
        if m.any():
            total += m.sum() / n * abs(correct[m].mean() - confidence[m].mean())
    return float(total)


def binary_ece(p_pos: np.ndarray, y_pos: np.ndarray, n_bins: int = 15) -> float:
    """ECE of the binary (unsafe) prediction, using confidence of the predicted side."""
    pred = p_pos >= 0.5
    conf = np.where(pred, p_pos, 1 - p_pos)
    return ece(conf, (pred == y_pos).astype(float), n_bins)


def reliability_diagram(confidence, correct, path: str | Path, n_bins: int = 15, title: str = ""):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    edges = np.linspace(0, 1, n_bins + 1)
    centers = (edges[:-1] + edges[1:]) / 2
    acc, cnt = np.zeros(n_bins), np.zeros(n_bins)
    for b, (lo, hi) in enumerate(zip(edges[:-1], edges[1:])):
        m = (confidence > lo) & (confidence <= hi)
        cnt[b] = m.sum()
        acc[b] = correct[m].mean() if m.any() else np.nan
    fig, (ax, ax2) = plt.subplots(2, 1, figsize=(4.5, 5.5), gridspec_kw={"height_ratios": [3, 1]}, sharex=True)
    ax.plot([0, 1], [0, 1], "--", color="gray", lw=1)
    ax.bar(centers, np.nan_to_num(acc), width=1 / n_bins, edgecolor="black", alpha=0.8)
    ax.set_ylabel("accuracy")
    ax.set_title(f"{title}\nECE = {ece(confidence, correct, n_bins):.3f}")
    ax2.bar(centers, cnt, width=1 / n_bins, color="gray")
    ax2.set_xlabel("confidence")
    ax2.set_ylabel("count")
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    plt.close(fig)


In [ ]:
%%writefile src/childsafe/classifier/engine.py
"""Inference and metric summaries shared by train.py and evaluate.py."""
from __future__ import annotations

from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_recall_fscore_support)

from childsafe.classifier.calibration import binary_ece, ece, reliability_diagram, softmax


@torch.no_grad()
def run_inference(model, loader, device, amp: bool = True):
    model.eval()
    logits, targets = [], []
    for x, y in loader:
        with torch.autocast(device_type=device.type, enabled=amp and device.type == "cuda"):
            out = model(x.to(device, non_blocking=True))
        logits.append(out.float().cpu())
        targets.append(y)
    return torch.cat(logits).numpy(), torch.cat(targets).numpy()


def summarize(logits: np.ndarray, targets: np.ndarray, classes: list[str], binary_map: dict,
              temperature: float = 1.0, reject_threshold: float | None = None,
              n_bins: int = 15, plot_prefix: str | Path | None = None, title: str = "") -> dict:
    """Per-class and binary (unsafe-vs-safe) metrics, before and after temperature scaling."""
    unsafe = np.array([binary_map[c] == "unsafe" for c in classes])
    res: dict = {"n": int(len(targets)), "temperature": temperature}
    present = sorted(set(targets.tolist()))
    for tag, t in (("raw", 1.0), ("calibrated", temperature)):
        p = softmax(logits, t)
        pred = p.argmax(1)
        conf = p.max(1)
        correct = (pred == targets).astype(float)
        p_unsafe = p[:, unsafe].sum(1)
        y_unsafe = unsafe[targets]
        res[f"ece_{tag}"] = ece(conf, correct, n_bins)
        res[f"binary_ece_{tag}"] = binary_ece(p_unsafe, y_unsafe, n_bins)
        res[f"nll_{tag}"] = float(-np.log(np.clip(p[np.arange(len(targets)), targets], 1e-12, 1)).mean())
        if plot_prefix is not None:
            reliability_diagram(conf, correct, f"{plot_prefix}_reliability_{tag}.png", n_bins, f"{title} ({tag})")

    p = softmax(logits, temperature)
    pred = p.argmax(1)
    res["accuracy"] = float(accuracy_score(targets, pred))
    res["macro_f1"] = float(f1_score(targets, pred, labels=present, average="macro", zero_division=0))
    prec, rec, f1, sup = precision_recall_fscore_support(targets, pred, labels=range(len(classes)), zero_division=0)
    res["per_class"] = {c: {"precision": float(prec[i]), "recall": float(rec[i]), "f1": float(f1[i]),
                            "support": int(sup[i])} for i, c in enumerate(classes)}
    res["confusion_matrix"] = confusion_matrix(targets, pred, labels=range(len(classes))).tolist()

    # Binary view: P(unsafe) = sum of unsafe-category probabilities.
    p_unsafe = p[:, unsafe].sum(1)
    y_unsafe = unsafe[targets]
    b_pred = p_unsafe >= 0.5
    tp = int((b_pred & y_unsafe).sum()); fn = int((~b_pred & y_unsafe).sum())
    fp = int((b_pred & ~y_unsafe).sum()); tn = int((~b_pred & ~y_unsafe).sum())
    res["binary"] = {
        "accuracy": (tp + tn) / max(len(targets), 1),
        "unsafe_recall": tp / max(tp + fn, 1),
        "unsafe_precision": tp / max(tp + fp, 1),
        "unsafe_f1": 2 * tp / max(2 * tp + fp + fn, 1),
        "tp": tp, "fn": fn, "fp": fp, "tn": tn,
    }
    if reject_threshold is not None:
        # A3 reject option: low calibrated confidence -> "uncertain" (caution), not a hard label.
        b_conf = np.maximum(p_unsafe, 1 - p_unsafe)
        keep = (b_conf >= reject_threshold) & (p.max(1) >= reject_threshold)
        unknown = np.array([binary_map[c] == 'unknown' or c in ('other','unknown','empty_hand') for c in classes])
        keep &= ~unknown[pred] & (p[:,unknown].sum(1) < .5)
        res["reject"] = {
            "threshold": reject_threshold,
            "coverage": float(keep.mean()),
            "binary_accuracy_on_kept": float((b_pred[keep] == y_unsafe[keep]).mean()) if keep.any() else float("nan"),
            "unsafe_missed_as_safe_confidently": int((~b_pred & y_unsafe & keep).sum()),
            "unsafe_rejected_to_caution": int((y_unsafe & ~keep).sum()),
        }
    return res


In [ ]:
%%writefile src/childsafe/classifier/evaluate.py
"""Evaluate a trained checkpoint in domain, out of domain (A2) and under corruptions (A5).

    python -m childsafe.classifier.evaluate --run outputs/runs/efficientnet_b0_group --ood data/ood
    python -m childsafe.classifier.evaluate --run outputs/runs/efficientnet_b0_group --corruptions

The OOD folder uses the same class folder names as ChildSUn (e.g. data/ood/knife/*.jpg);
extra sub-folders such as data/ood/low_light/knife/... are fine and reported by `condition`.
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms as T

from childsafe.classifier.engine import run_inference, summarize
from childsafe.classifier.models import load_checkpoint
from childsafe.config import load_config, normalize_name
from childsafe.data.catalog import build_catalog, source_roots
from childsafe.data.corruptions import KINDS, FixedCorruption
from childsafe.data.dataset import MEAN, STD, eval_transform, image_paths


_CACHE: dict = {}


def _cached_images(df, root, size, preprocessing="center_crop"):
    """Decode + resize + center-crop once; corruption passes reuse these (no DataLoader worker re-spawn,
    which costs tens of seconds per pass on Windows)."""
    paths = image_paths(df, root)
    key = (tuple(str(p.resolve()) for p in paths), size, preprocessing)
    if key not in _CACHE:
        tf = T.Compose(eval_transform(size, preprocessing=preprocessing).transforms[:-2])
        _CACHE[key] = [tf(Image.open(p).convert('RGB')) for p in paths]
    return _CACHE[key]


def _eval_df(model, ckpt, df, root, device, cfg, corruption=None, plot_prefix=None, title=""):
    size = ckpt["img_size"]
    imgs = _cached_images(df, root, size, ckpt.get("preprocessing", "center_crop"))
    tail = T.Compose(([FixedCorruption(*corruption)] if corruption and corruption[1] > 0 else [])
                     + [T.ToTensor(), T.Normalize(MEAN, STD)])
    idx = {c: i for i, c in enumerate(ckpt["classes"])}
    x = torch.stack([tail(im) for im in imgs])
    y = torch.tensor([idx[c] for c in df["category"]])
    lg, y = run_inference(model, DataLoader(TensorDataset(x, y), batch_size=64), device)
    rc = cfg["calibration"]
    return summarize(lg, y, ckpt["classes"], ckpt["binary_map"], ckpt["temperature"],
                     rc["reject_threshold"], rc["n_bins"], plot_prefix, title)


def _unknown_report(model, ckpt, df, root, device, cfg) -> dict:
    from childsafe.classifier.calibration import softmax
    size = ckpt["img_size"]
    imgs = _cached_images(df, root, size, ckpt.get('preprocessing', 'center_crop'))
    tail = T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)])
    x = torch.stack([tail(im) for im in imgs])
    lg, _ = run_inference(model, DataLoader(TensorDataset(x, torch.zeros(len(x), dtype=torch.long)), batch_size=64), device)
    p = softmax(lg, ckpt["temperature"])
    unsafe = np.array([ckpt["binary_map"][c] == "unsafe" for c in ckpt["classes"]])
    pu = p[:, unsafe].sum(1)
    thr = cfg["calibration"]["reject_threshold"]
    status = np.where(np.maximum(pu, 1 - pu) < thr, "uncertain", np.where(pu >= 0.5, "unsafe", "safe"))
    pred = np.array(ckpt["classes"])[p.argmax(1)]
    unknown = np.array([ckpt['binary_map'][c] == 'unknown' or c in ('other','unknown','empty_hand')
                        for c in ckpt['classes']])
    rejected = (p.max(1) < thr) | np.isin(pred,['other','unknown','empty_hand']) | (p[:,unknown].sum(1) >= .5)
    status[rejected] = 'uncertain'
    out = {}
    for c in sorted(df["category"].unique()):
        m = (df["category"] == c).to_numpy()
        vc = pd.Series(pred[m]).value_counts(normalize=True).head(3).round(3).to_dict()
        out[c] = {"n": int(m.sum()), "unsafe": float((status[m] == "unsafe").mean()),
                  "uncertain": float((status[m] == "uncertain").mean()), "safe": float((status[m] == "safe").mean()),
                  "mean_max_prob": float(p[m].max(1).mean()), "top_predicted": vc}
    return out


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--run", required=True, help="run folder containing best.pt")
    ap.add_argument("--ood", default=None, help="out-of-domain image folder (A2)")
    ap.add_argument("--corruptions", action="store_true", help="A5 degradation curves on the test split")
    ap.add_argument("--config", default=None)
    ap.add_argument("--manifest", default=None, help="Expanded split manifest used to train the checkpoint")
    args = ap.parse_args(argv)

    cfg = load_config(args.config)
    run = Path(args.run)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model, ckpt = load_checkpoint(run / "best.pt", device)
    cfg["label_map"] = {**cfg["label_map"], **ckpt["binary_map"]}
    split_col = f"split_{ckpt.get('split', 'group')}"
    report: dict = {"run": run.name}

    if args.ood:
        ood = build_catalog(args.ood, cfg["label_map"], cfg.get("category_aliases"))
        unknown = set(ood["category"]) - set(ckpt["classes"])
        if unknown:
            # objects the model was never trained on (e.g. "other": pen, phone, hand): there is no correct
            # class, so report how the alert layer would treat them (unsafe = would raise danger near mouth)
            report["unknown_objects"] = _unknown_report(model, ckpt, ood[~ood["category"].isin(ckpt["classes"])],
                                                        args.ood, device, cfg)
            for c, r in report["unknown_objects"].items():
                print(f"[ood] '{c}' (not a trained class, n={r['n']}): called unsafe {r['unsafe']:.1%}, "
                      f"uncertain {r['uncertain']:.1%}, safe {r['safe']:.1%}; top predictions {r['top_predicted']}")
            ood = ood[ood["category"].isin(ckpt["classes"])]
        # condition = first folder level that is not the class name (e.g. low_light, dim, bright)
        ood["condition"] = [next((p for p in Path(x).parts[:-1] if normalize_name(p) not in cfg["label_map"]), "all")
                            for x in ood["path"]]
        report["ood"] = _eval_df(model, ckpt, ood, args.ood, device, cfg, plot_prefix=run / "ood", title="OOD")
        report["ood_by_condition"] = {c: _eval_df(model, ckpt, g, args.ood, device, cfg)
                                      for c, g in ood.groupby("condition")} if ood["condition"].nunique() > 1 else {}
        test = pd.read_csv(args.manifest or cfg["paths"]["splits"])
        test = test[test[split_col] == "test"]
        ind = _eval_df(model, ckpt, test, source_roots(cfg), device, cfg)
        report["in_domain_test"] = ind
        report["ood_drop"] = {"accuracy": ind["accuracy"] - report["ood"]["accuracy"],
                              "unsafe_recall": ind["binary"]["unsafe_recall"] - report["ood"]["binary"]["unsafe_recall"]}
        print(f"in-domain acc {ind['accuracy']:.4f} -> OOD acc {report['ood']['accuracy']:.4f}; "
              f"unsafe recall {ind['binary']['unsafe_recall']:.4f} -> {report['ood']['binary']['unsafe_recall']:.4f}; "
              f"ECE(cal) {ind['ece_calibrated']:.4f} -> {report['ood']['ece_calibrated']:.4f}")

    if args.corruptions:
        test = pd.read_csv(args.manifest or cfg["paths"]["splits"])
        test = test[test[split_col] == "test"]
        curves = {}
        for kind in KINDS:
            curves[kind] = []
            for sev in range(0, 6):
                m = _eval_df(model, ckpt, test, source_roots(cfg), device, cfg, (kind, sev))
                curves[kind].append({"severity": sev, "accuracy": m["accuracy"],
                                     "unsafe_recall": m["binary"]["unsafe_recall"], "ece": m["ece_calibrated"]})
                print(f"{kind} s{sev}: acc {m['accuracy']:.4f} unsafe_recall {m['binary']['unsafe_recall']:.4f}")
        report["corruption_curves"] = curves
        _plot_curves(curves, run / "degradation.png", run.name)

    (run / "eval_report.json").write_text(json.dumps(report, indent=2))
    print(f"wrote {run / 'eval_report.json'}")


def _plot_curves(curves, path, title):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
    for kind, pts in curves.items():
        s = [p["severity"] for p in pts]
        axes[0].plot(s, [p["accuracy"] for p in pts], marker="o", label=kind)
        axes[1].plot(s, [p["unsafe_recall"] for p in pts], marker="o", label=kind)
    axes[0].set_title("accuracy"); axes[1].set_title("unsafe recall")
    for a in axes:
        a.set_xlabel("severity"); a.set_ylim(0, 1.02); a.grid(alpha=0.3)
    axes[1].legend(fontsize=7)
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    plt.close(fig)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/childsafe/classifier/gradcam.py
"""Grad-CAM (Selvaraju et al., ICCV 2017) overlays for qualitative inspection.

    python -m childsafe.classifier.gradcam --run outputs/runs/efficientnet_b0_group --n 40
    python -m childsafe.classifier.gradcam --run ... --images data/ood/knife   # any folder
"""
from __future__ import annotations

import argparse
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms as T

from childsafe.classifier.calibration import softmax
from childsafe.classifier.models import gradcam_target_layer, load_checkpoint
from childsafe.config import load_config
from childsafe.data.catalog import IMG_EXT, source_roots
from childsafe.data.dataset import eval_transform, image_paths


class GradCAM:
    def __init__(self, model, layer):
        self.model, self.acts, self.grads = model, None, None
        layer.register_forward_hook(self._hook)

    def _hook(self, module, inp, out):
        # tensor hook instead of a module backward hook: timm uses in-place activations
        self.acts = out
        out.register_hook(lambda g: setattr(self, "grads", g))

    def __call__(self, x: torch.Tensor, target: int | None = None):
        self.model.zero_grad(set_to_none=True)
        logits = self.model(x)
        target = int(logits.argmax(1)) if target is None else target
        logits[0, target].backward()
        w = self.grads.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((w * self.acts).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam.detach().cpu().numpy(), logits.detach().cpu().numpy()[0], target


def overlay(rgb: np.ndarray, cam: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    heat = cv2.applyColorMap((cam * 255).astype(np.uint8), cv2.COLORMAP_JET)[..., ::-1]
    return (rgb * (1 - alpha) + heat * alpha).astype(np.uint8)


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--run", required=True)
    ap.add_argument("--images", default=None, help="folder of images; default: random test-split images")
    ap.add_argument("--n", type=int, default=30)
    ap.add_argument("--config", default=None)
    args = ap.parse_args(argv)

    cfg = load_config(args.config)
    run = Path(args.run)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model, ckpt = load_checkpoint(run / "best.pt", device)
    cam = GradCAM(model, gradcam_target_layer(model, ckpt["arch"]))
    size = ckpt["img_size"]
    tf = eval_transform(size, preprocessing=ckpt.get("preprocessing", "center_crop"))
    show = T.Compose(tf.transforms[:-2])  # same spatial preprocessing as model input

    if args.images:
        paths = sorted(p for p in Path(args.images).rglob("*") if p.suffix.lower() in IMG_EXT)[: args.n]
        truth = [""] * len(paths)
    else:
        df = pd.read_csv(cfg["paths"]["splits"])
        df = df[df[f"split_{ckpt.get('split', 'group')}"] == "test"].sample(frac=1, random_state=0).head(args.n)
        paths = image_paths(df, source_roots(cfg))
        truth = list(df["category"])

    out = run / "gradcam"
    out.mkdir(exist_ok=True)
    for k, (p, t) in enumerate(zip(paths, truth)):
        img = Image.open(p).convert("RGB")
        x = tf(img).unsqueeze(0).to(device).requires_grad_(True)
        heat, logits, target = cam(x)
        prob = softmax(logits[None], ckpt["temperature"])[0]
        disp = np.asarray(show(img))
        vis = overlay(disp, heat)
        label = f"pred {ckpt['classes'][target]} {prob[target]:.2f}" + (f" | true {t}" if t else "")
        vis = cv2.putText(vis.copy(), label, (4, 16), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv2.LINE_AA)
        Image.fromarray(vis).save(out / f"{k:03d}_{Path(p).stem}.jpg")
    print(f"wrote {len(paths)} overlays to {out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/childsafe/classifier/models.py
"""Model factory (timm) and checkpoint I/O."""
from __future__ import annotations

from pathlib import Path

import timm
import torch
import torch.nn as nn

ARCHS = ("efficientnet_b0", "mobilenetv3_large_100", "vgg19_bn")


def create_model(arch: str, num_classes: int, pretrained: bool = True) -> nn.Module:
    return timm.create_model(arch, pretrained=pretrained, num_classes=num_classes)


def gradcam_target_layer(model: nn.Module, arch: str) -> nn.Module:
    """Last spatial feature layer. (In timm MobileNetV3 conv_head comes after pooling, so use blocks.)"""
    if arch.startswith(("efficientnet", "mobilenetv3")):
        return model.blocks[-1]
    if arch.startswith("vgg"):
        return model.pre_logits
    raise ValueError(f"no Grad-CAM layer rule for {arch}")


def save_checkpoint(path: str | Path, model: nn.Module, arch: str, classes: list[str],
                    binary_map: dict[str, str], img_size: int, temperature: float = 1.0, **extra):
    torch.save({"arch": arch, "classes": classes, "binary_map": binary_map, "img_size": img_size,
                "temperature": temperature, "state_dict": model.state_dict(), **extra}, path)


def load_checkpoint(path: str | Path, device: str | torch.device = "cpu"):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model = create_model(ckpt["arch"], len(ckpt["classes"]), pretrained=False)
    model.load_state_dict(ckpt["state_dict"])
    return model.to(device).eval(), ckpt


def unsafe_mask(classes: list[str], binary_map: dict[str, str]) -> torch.Tensor:
    return torch.tensor([binary_map[c] == "unsafe" for c in classes])


In [ ]:
%%writefile src/childsafe/classifier/train.py
"""Train one Module A model on one split, then calibrate on val and evaluate on test.

    python -m childsafe.classifier.train --arch efficientnet_b0 --split group
    python -m childsafe.classifier.train --arch efficientnet_b0 --split random   # leakage comparison
    python -m childsafe.classifier.train --arch efficientnet_b0 --split group --robust  # A5

Outputs go to outputs/runs/<arch>_<split>[_robust]/.
"""
from __future__ import annotations

import argparse
import hashlib
import json
import random
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from childsafe.classifier.calibration import fit_temperature
from childsafe.classifier.engine import run_inference, summarize
from childsafe.classifier.models import create_model, save_checkpoint
from childsafe.config import load_config
from childsafe.data.catalog import source_roots
from childsafe.data.dataset import CatalogDataset, eval_transform, train_transform


def seed_everything(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


def class_balanced_sampler(targets, num_classes, balance):
    if balance != 'sqrt':
        return None
    counts = torch.bincount(targets,minlength=num_classes).float().clamp(min=1)
    return torch.utils.data.WeightedRandomSampler(counts.rsqrt()[targets],len(targets),replacement=True)


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--arch", required=True)
    ap.add_argument("--split", choices=["group", "random"], default="group")
    ap.add_argument("--robust", action="store_true", help="A5 robustness augmentation")
    ap.add_argument("--config", default=None)
    ap.add_argument("--epochs", type=int)
    ap.add_argument("--batch-size", type=int)
    ap.add_argument("--no-pretrained", action="store_true")
    ap.add_argument("--seed", type=int, default=None)
    ap.add_argument("--out", default=None)
    ap.add_argument("--manifest", help="Reviewed expanded manifest with train/val/cal/test in split_group")
    ap.add_argument("--balanced", action="store_true", help="Inverse-square-root training class weights")
    ap.add_argument("--preprocessing", choices=["center_crop", "letterbox"], default="center_crop")
    args = ap.parse_args(argv)
    sys.stdout.reconfigure(line_buffering=True)  # progress visible when redirected to a log

    cfg = load_config(args.config)
    tc = cfg["train"]
    epochs = args.epochs or tc["epochs"]
    bs = args.batch_size or tc["batch_size"]
    seed = cfg["split"]["seed"] if args.seed is None else args.seed
    seed_everything(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp = tc["amp"] and device.type == "cuda"

    df = pd.read_csv(args.manifest or cfg["paths"]["splits"])
    col = f"split_{args.split}"
    classes = sorted(df["category"].unique())
    binary_map = dict(df[["category", "label"]].drop_duplicates().values)
    if df.groupby("category")["label"].nunique().max() != 1:
        raise ValueError("Conflicting category risk labels")
    if args.manifest:
        if args.split != "group" or not set(df[col]).issubset({"train", "val", "cal", "test"}):
            raise ValueError("Expanded training requires group split with train/val/cal/test")
        if df.groupby("group")[col].nunique().max() != 1:
            raise ValueError("Source group leakage across partitions")
        for part in ("train", "val", "cal", "test"):
            missing = set(classes) - set(df.loc[df[col] == part, "category"])
            if missing:
                raise ValueError(f"{part} missing classes: {sorted(missing)}; collect more independent sources")
    root = source_roots(cfg)   # ChildSUn + any extra sources
    size = tc["img_size"]

    def loader(part, tf, shuffle):
        ds = CatalogDataset(df[df[col] == part], root, classes, tf)
        sampler = class_balanced_sampler(ds.targets,len(classes),tc.get('balance')) if shuffle else None
        return DataLoader(ds, batch_size=bs, shuffle=shuffle and sampler is None, sampler=sampler,
                          num_workers=tc["num_workers"], pin_memory=device.type == "cuda",
                          persistent_workers=tc["num_workers"] > 0, drop_last=shuffle and len(ds) > bs)

    eval_tf = eval_transform(size, preprocessing=args.preprocessing)
    train_dl = loader("train", train_transform(size, args.robust, args.preprocessing), True)
    val_dl = loader("val", eval_tf, False)
    cal_dl = loader("cal", eval_tf, False) if args.manifest else val_dl
    test_dl = loader("test", eval_tf, False)

    run_name = f"{args.arch}_{args.split}" + ("_robust" if args.robust else "")
    out = Path(args.out or Path(cfg["paths"]["outputs"]) / "runs" / run_name)
    out.mkdir(parents=True, exist_ok=True)
    if args.manifest:
        df.to_csv(out / "training_manifest.csv", index=False)
    provenance = {"arguments": vars(args), "seed": seed, "config": cfg,
                  "manifest_sha256": hashlib.sha256(Path(args.manifest or cfg["paths"]["splits"]).read_bytes()).hexdigest(),
                  "partition_counts": df.groupby(col).size().to_dict()}
    (out / "run_config.json").write_text(json.dumps(provenance, indent=2, default=str), encoding="utf-8")

    lr = tc.get("lr_overrides", {}).get(args.arch, tc["lr"])
    print(f"[{run_name}] lr={lr} epochs={epochs} batch={bs}", flush=True)
    model = create_model(args.arch, len(classes), pretrained=not args.no_pretrained).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=tc["weight_decay"])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(train_dl),
                                                pct_start=0.1)
    scaler = torch.amp.GradScaler(enabled=amp)
    weights = None
    if args.balanced:
        counts = df[df[col] == "train"].category.value_counts().reindex(classes).to_numpy()
        weights = torch.tensor(1 / np.sqrt(counts), dtype=torch.float32, device=device)
        weights /= weights.mean()
    crit = nn.CrossEntropyLoss(weight=weights)

    best_f1, best_epoch, history = -1.0, -1, []
    for epoch in range(epochs):
        model.train()
        t0, tot, n = time.time(), 0.0, 0
        for x, y in train_dl:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=amp):
                loss = crit(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            tot += loss.item() * len(y); n += len(y)
        vl, vy = run_inference(model, val_dl, device, amp)
        vm = summarize(vl, vy, classes, binary_map)
        history.append({"epoch": epoch, "train_loss": tot / max(n, 1), "val_acc": vm["accuracy"],
                        "val_macro_f1": vm["macro_f1"], "val_unsafe_recall": vm["binary"]["unsafe_recall"],
                        "sec": time.time() - t0})
        print(json.dumps(history[-1]), flush=True)
        if vm["macro_f1"] > best_f1:
            best_f1, best_epoch = vm["macro_f1"], epoch
            save_checkpoint(out / "best.pt", model, args.arch, classes, binary_map, size,
                            preprocessing=args.preprocessing)
        elif epoch - best_epoch >= tc["patience"]:
            print(f"early stop at epoch {epoch} (best {best_epoch})")
            break

    # Reload best, calibrate on val, evaluate on test.
    model.load_state_dict(torch.load(out / "best.pt", weights_only=False)["state_dict"])
    vl, vy = run_inference(model, val_dl, device, amp)
    cl, cy = run_inference(model, cal_dl, device, amp)
    temp = fit_temperature(cl, cy)
    save_checkpoint(out / "best.pt", model, args.arch, classes, binary_map, size, temperature=temp,
                    split=args.split, robust=args.robust, best_epoch=best_epoch,
                    preprocessing=args.preprocessing, balanced=args.balanced,
                    calibration_partition="cal" if args.manifest else "val")
    tl, ty = run_inference(model, test_dl, device, amp)
    np.savez(out / "logits.npz", val_logits=vl, val_targets=vy, test_logits=tl, test_targets=ty,
             cal_logits=cl, cal_targets=cy, classes=np.array(classes))
    rc = cfg["calibration"]
    metrics = {
        "arch": args.arch, "split": args.split, "robust": args.robust, "best_epoch": best_epoch,
        "val": summarize(vl, vy, classes, binary_map, temp, rc["reject_threshold"], rc["n_bins"]),
        "test": summarize(tl, ty, classes, binary_map, temp, rc["reject_threshold"], rc["n_bins"],
                          plot_prefix=out / "test", title=f"{run_name} test"),
        "history": history,
        "preprocessing": args.preprocessing, "balanced": args.balanced,
        "calibration_partition": "cal" if args.manifest else "val",
        "test_by_domain": {str(domain): summarize(tl[mask], ty[mask], classes, binary_map, temp,
                                  rc["reject_threshold"], rc["n_bins"])
                           for domain in df.loc[df[col] == "test", "domain"].unique()
                           for mask in [(df.loc[df[col] == "test", "domain"].to_numpy() == domain)]}
                          if "domain" in df else {},
    }
    (out / "metrics.json").write_text(json.dumps(metrics, indent=2))
    t = metrics["test"]
    print(f"[{run_name}] T={temp:.3f} test acc={t['accuracy']:.4f} macroF1={t['macro_f1']:.4f} "
          f"unsafe recall={t['binary']['unsafe_recall']:.4f} ECE raw={t['ece_raw']:.4f} cal={t['ece_calibrated']:.4f}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/childsafe/__init__.py
"""Child dangerous-object warning system."""


In [ ]:
%%writefile src/childsafe/config.py
"""Config loading. Paths in the YAML are relative to the repo root.

A config file may start with `_base_: default.yaml` and list only the keys it changes
(e.g. configs/tuned_overrides.yaml written by the grid search).
"""
from __future__ import annotations

from pathlib import Path

import yaml

ROOT = Path(__file__).resolve().parents[2]
DEFAULT_CONFIG = ROOT / "configs" / "default.yaml"


def _merge(base: dict, over: dict) -> dict:
    out = dict(base)
    for k, v in over.items():
        out[k] = _merge(out[k], v) if isinstance(v, dict) and isinstance(out.get(k), dict) else v
    return out


def _read(path: Path) -> dict:
    with open(path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f) or {}
    base = cfg.pop("_base_", None)
    return _merge(_read(path.parent / base), cfg) if base else cfg


def load_config(path: str | Path | None = None) -> dict:
    cfg = _read(Path(path) if path else DEFAULT_CONFIG)
    cfg["paths"] = {k: str((ROOT / v).resolve()) for k, v in cfg["paths"].items()}
    if cfg.get('detector',{}).get('weights'):
        cfg['detector']['weights'] = str((ROOT / cfg['detector']['weights']).resolve())
    return cfg


def normalize_name(name: str) -> str:
    return "".join(ch for ch in name.lower() if ch.isalnum())


## Explicit dataset paths — no automatic selection of the biggest image folder

In [ ]:
# Edit these paths using the Kaggle Input panel. RAW_ROOT is the folder containing
# canonical filenames such as baby_items_10_0_1720.jpg (not the outer dataset folder).
RAW_ROOT=Path('/kaggle/input/CHANGE-ME-CHILDSUN/CHANGE-ME-IMAGE-FOLDER')
SUPPLEMENT=Path('/kaggle/input/CHANGE-ME-SUPPLEMENT/manifest.csv')
assert RAW_ROOT.is_dir(), RAW_ROOT
assert SUPPLEMENT.is_file(), SUPPLEMENT
subprocess.run([sys.executable,'scripts/prepare_expanded_data.py','--raw-root',str(RAW_ROOT),
                '--supplement',str(SUPPLEMENT)],check=True)
import pandas as pd
df=pd.read_csv('data/splits/expanded.csv')
assert {'pen','coin'}.issubset(set(df.category)), 'Approve enough pen/coin sources before training'
display(df.groupby(['category','domain','split_group']).size().unstack(fill_value=0))


## Training and controlled comparisons
First compare preprocessing/augmentation using validation, then choose one variant before interpreting test.
The notebook runs one chosen improved model by default; optional baseline/MobileNet runs cost extra GPU time.
Do not choose thresholds or model variants by looking at test scores repeatedly.


In [ ]:
ARCH='efficientnet_b0'  # Alternative: mobilenetv3_large_100
EPOCHS=30
cmd=[sys.executable,'-m','childsafe.classifier.train','--arch',ARCH,'--split','group',
     '--manifest','data/splits/expanded.csv','--robust','--balanced','--preprocessing','letterbox',
     '--epochs',str(EPOCHS),'--batch-size','32','--out',f'outputs/expanded/{ARCH}']
subprocess.run(cmd,check=True)


## Inspect results, then collect deployment-domain holdouts
Read pen/coin recall and confusion with fruit/spoon, unsafe recall, ECE, rejection coverage,
confident unsafe-to-safe errors, and test metrics by source domain. Small new holdouts have wide uncertainty.
An Open Images score does not validate webcam behavior.
Add consented adult/doll hand crops with different people, rooms, devices, and physical objects;
use one `group` per session/object so neighboring frames never cross partitions.
For actual alerts, measure event recall, false alarms/hour, and detection delay on separate scenarios.


In [ ]:
import json, shutil
metrics=json.loads(Path(f'outputs/expanded/{ARCH}/metrics.json').read_text())
print(json.dumps(metrics['test'],indent=2))
print('By domain:',json.dumps(metrics['test_by_domain'],indent=2))
shutil.make_archive('/kaggle/working/childsafe_expanded_results','zip','outputs/expanded')
print('Download childsafe_expanded_results.zip. The deployable checkpoint is best.pt.')
